# PVOD external validation — clean rerunnable notebook

This version preserves the frozen scientific protocol while removing one-off diagnostic/recovery cells. The canonical Ridge path uses genuine source-float64 features from the first fit. Secondary-control execution is checkpointed per fit and retains compact daily sufficient statistics, so a kernel restart can resume without losing completed control fits.

The notebook ends with secondary-control point estimates, hierarchical paired bootstrap inference, mechanism synthesis, and CSV exports.


In [1]:
# ============================================================
# PVOD-01A
# Raw PVOD dataset discovery audit
#
# PURPOSE
# -------
# Inspect the downloaded release exactly as it exists BEFORE:
#
# - assigning timezones
# - choosing stations
# - defining train/validation/test splits
# - normalizing PV power
# - computing astronomy
# - computing Panchang
# - creating forecasting samples
# - training any model
#
# No data is modified in this cell.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Resolve project root
# ============================================================

def find_project_root(start: Path) -> Path:
    start = start.resolve()

    for candidate in [start, *start.parents]:
        if (
            (candidate / "notebooks").is_dir()
            and
            (candidate / "external_data").is_dir()
        ):
            return candidate

    raise RuntimeError(
        "Could not locate project root."
    )


PROJECT_ROOT = find_project_root(
    Path.cwd()
)

PVOD_DIR = (
    PROJECT_ROOT
    / "external_data"
    / "PVOD"
)

assert PVOD_DIR.is_dir(), (
    f"PVOD directory not found: {PVOD_DIR}"
)


print("Project root:")
print(PROJECT_ROOT)

print("\nPVOD directory:")
print(PVOD_DIR)


# ============================================================
# 2. Discover release files
# ============================================================

metadata_path = (
    PVOD_DIR
    / "metadata.csv"
)

station_paths = sorted(
    PVOD_DIR.glob(
        "station*.csv"
    )
)

mcclear_paths = sorted(
    (PVOD_DIR / "McClear").glob(
        "*.csv"
    )
)


assert metadata_path.is_file()

assert len(station_paths) == 10, (
    f"Expected 10 station CSVs, found {len(station_paths)}"
)


print(
    "\nCore station files:"
)

for path in station_paths:
    print(
        " ",
        path.name
    )


print(
    "\nMcClear supplemental files:",
    len(mcclear_paths)
)


# ============================================================
# 3. Metadata inspection
# ============================================================

metadata = pd.read_csv(
    metadata_path
)


print(
    "\n"
    + "=" * 100
)

print(
    "PVOD METADATA"
)

print(
    "=" * 100
)


print(
    "Shape:",
    metadata.shape
)


print(
    "\nColumns:"
)

for column in metadata.columns:
    print(
        " ",
        repr(column)
    )


print(
    "\nMetadata dtypes:"
)

display(
    metadata.dtypes.to_frame(
        "dtype"
    )
)


print(
    "\nMetadata preview:"
)

display(
    metadata.head(
        10
    )
)


# ============================================================
# 4. Inspect each station file WITHOUT altering timestamps
# ============================================================

station_audit_rows = []

station_column_sets = {}


for station_path in station_paths:

    station_id = (
        station_path.stem
    )

    df = pd.read_csv(
        station_path
    )

    station_column_sets[
        station_id
    ] = tuple(
        df.columns.tolist()
    )


    station_audit_rows.append(
        {
            "station":
                station_id,

            "rows":
                int(
                    len(df)
                ),

            "columns":
                int(
                    df.shape[1]
                ),

            "first_column":
                str(
                    df.columns[0]
                ),

            "last_column":
                str(
                    df.columns[-1]
                ),

            "missing_cells":
                int(
                    df.isna()
                    .sum()
                    .sum()
                ),

            "duplicate_rows":
                int(
                    df.duplicated()
                    .sum()
                ),
        }
    )


pvod_station_audit = pd.DataFrame(
    station_audit_rows
)


print(
    "\n"
    + "=" * 100
)

print(
    "PVOD STATION FILE AUDIT"
)

print(
    "=" * 100
)


display(
    pvod_station_audit
)


print(
    "\nTotal rows across 10 stations:",
    int(
        pvod_station_audit[
            "rows"
        ].sum()
    )
)


# ============================================================
# 5. Determine whether all station schemas match
# ============================================================

reference_station = (
    station_paths[0].stem
)

reference_columns = (
    station_column_sets[
        reference_station
    ]
)


schema_matches = {
    station_id:
        columns
        ==
        reference_columns

    for station_id, columns
    in station_column_sets.items()
}


print(
    "\n"
    + "=" * 100
)

print(
    "STATION SCHEMA CONSISTENCY"
)

print(
    "=" * 100
)


for station_id, matches in (
    schema_matches.items()
):

    print(
        f"{station_id}: "
        f"{'MATCH' if matches else 'DIFFERENT'}"
    )


print(
    "\nReference station columns "
    f"({reference_station}):"
)

for index, column in enumerate(
    reference_columns,
    start=1,
):

    print(
        f"{index:02d}. {repr(column)}"
    )


# ============================================================
# 6. Show a small raw preview of station00
# ============================================================

station00 = pd.read_csv(
    PVOD_DIR
    / "station00.csv"
)


print(
    "\n"
    + "=" * 100
)

print(
    "RAW station00 PREVIEW"
)

print(
    "=" * 100
)


display(
    station00.head(
        8
    )
)


print(
    "\nPVOD-01A RAW DATASET DISCOVERY: PASSED"
)

print(
    "\nNo timezone assigned."
)

print(
    "No missing values modified."
)

print(
    "No split created."
)

print(
    "No model trained."
)

Project root:
/Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project

PVOD directory:
/Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/external_data/PVOD

Core station files:
  station00.csv
  station01.csv
  station02.csv
  station03.csv
  station04.csv
  station05.csv
  station06.csv
  station07.csv
  station08.csv
  station09.csv

McClear supplemental files: 10

PVOD METADATA
Shape: (10, 12)

Columns:
  'Station_ID'
  'Capacity'
  'PV_Technology'
  'Panel_Size'
  'Module'
  'Inverters'
  'Layout'
  'Panel_Number'
  'Array_Tilt'
  'Pyranometer'
  'Longitude'
  'Latitude'

Metadata dtypes:


,dtype
Station_ID,str
Capacity,int64
PV_Technology,str
Panel_Size,float64
Module,str
Inverters,str
Layout,str
Panel_Number,int64
Array_Tilt,str
Pyranometer,str



Metadata preview:


,Station_ID,Capacity,PV_Technology,Panel_Size,Module,Inverters,Layout,Panel_Number,Array_Tilt,Pyranometer,Longitude,Latitude
0,station00,6600,Poly-Si,1.62360,products types:LW255(29)P1640×990\nPmax:255 Wp...,products types:NSG-500K3TL\nMax. DC input:560 ...,modules per string:20\nstrings per inverter:96,26000,South 33°,GHI: \nproducts types: TBQ-2\nmeasurement accu...,114.951390,38.047780
1,station01,20000,Poly-Si,1.66350,products types:LW6P60-270\nPmax:270 Wp\nVmpp:3...,products types:TC500KH\nMax. DC input:618 kW\n...,modules per string:22\nstrings per inverter:128,74000,South 33°,GHI: \nproducts types: TBQ-2\nmeasurement accu...,117.457220,38.183060
2,station02,17000,Mono-Si,1.63680,products types:RSM60-6-285M\nPmax:285 Wp\nVmpp...,products types:SG50KTL-M\nMax. DC input:-\nMax...,modules per string:22\nstrings per inverter:8,64064,South 29°,GHI: \nproducts types: TBQ-2\nmeasurement accu...,114.198870,38.057280
3,station03,20000,Poly-Si,1.62000,products types:HTMP265P4-60\nPmax:265 Wp\nVmpp...,products types:PVL-L1000E\nMax. DC input:-\nMa...,modules per string:22\nstrings per inverter:16,75460,South 33°,GHI: \nproducts types: TBQ-2L\nmeasurement acc...,114.114940,38.109560
4,station04,20000,Poly-Si,1.63680,products types:YL265P-29b\nPmax:265 Wp\nVmpp:3...,products types:SG1000\nMax. DC input:560 kW\nM...,modules per string:22\nstrings per inverter:86,75680,South 37°,GHI: \nproducts types: TBQ-2\nmeasurement accu...,114.867670,39.515500
5,station05,35000,Poly-Si,1.63515,products types:LR6-60-305M\nPmax:305 Wp\nVmpp:...,products types:SG1250UD\nMax. DC input:618 kW\...,modules per string:22\nstrings per inverter:160,114972,South 33°,GHI: \nproducts types: TBQ-2\nmeasurement accu...,114.123600,38.235500
6,station06,15000,Poly-Si,1.63350,products types:JAP60S01-270/SC\nPmax:270 Wp\nV...,products types:SG50KTL-M\nMax. DC input:-\nMax...,modules per string:24\nstrings per inverter:8,56608,South 34°,GHI: \nproducts types: TBQ-2\nmeasurement accu...,114.548410,36.898910
7,station07,20000,Poly-Si,1.63350,products types:YL250P-29b\nPmax:250 Wp\nVmpp:2...,products types:TC500KH\nMax. DC input:618 kW\n...,modules per string:20\nstrings per inverter:100,80000,South 31°,GHI: \nproducts types: TBQ-2L\nmeasurement acc...,113.641870,36.644030
8,station08,20000,Poly-Si,1.63350,products types:YL265C-30b\nPmax:265 Wp\nVmpp:3...,products types:SUN2000-40KTL\nMax. DC input: 4...,modules per string:22\nstrings per inverter:5,78042,South 33°,GHI: \nproducts types: TBQ-2\nmeasurement accu...,113.899990,36.707610
9,station09,20000,Poly-Si,1.61370,products types:YL265P-29B-265W\nPmax:265 Wp\nV...,products types:GSL0630T\nMax. DC input:690 kW\...,modules per string:24\nstrings per inverter:128,84480,South 31°,GHI: \nproducts types: TBQ-2\nmeasurement accu...,115.059855,38.731417



PVOD STATION FILE AUDIT


,station,rows,columns,first_column,last_column,missing_cells,duplicate_rows
0,station00,28896,15,date_time,power,0,0
1,station01,33408,15,date_time,power,0,0
2,station02,30432,15,date_time,power,0,0
3,station03,14688,15,date_time,power,0,0
4,station04,33408,15,date_time,power,0,0
5,station05,9696,15,date_time,power,0,0
6,station06,31104,15,date_time,power,0,0
7,station07,32928,15,date_time,power,0,0
8,station08,33120,15,date_time,power,0,0
9,station09,24288,15,date_time,power,0,0



Total rows across 10 stations: 271968

STATION SCHEMA CONSISTENCY
station00: MATCH
station01: MATCH
station02: MATCH
station03: MATCH
station04: MATCH
station05: MATCH
station06: MATCH
station07: MATCH
station08: MATCH
station09: MATCH

Reference station columns (station00):
01. 'date_time'
02. 'nwp_globalirrad'
03. 'nwp_directirrad'
04. 'nwp_temperature'
05. 'nwp_humidity'
06. 'nwp_windspeed'
07. 'nwp_winddirection'
08. 'nwp_pressure'
09. 'lmd_totalirrad'
10. 'lmd_diffuseirrad'
11. 'lmd_temperature'
12. 'lmd_pressure'
13. 'lmd_winddirection'
14. 'lmd_windspeed'
15. 'power'

RAW station00 PREVIEW


,date_time,nwp_globalirrad,nwp_directirrad,nwp_temperature,nwp_humidity,nwp_windspeed,nwp_winddirection,nwp_pressure,lmd_totalirrad,lmd_diffuseirrad,lmd_temperature,lmd_pressure,lmd_winddirection,lmd_windspeed,power
0,2018-08-15 16:00:00,0.0,0.0,22.78,96.85,4.28,339.41,1007.27,0,0,25.900000,1006.299988,353,1.1,0.0
1,2018-08-15 16:15:00,0.0,0.0,22.75,96.91,4.30,337.27,1007.27,0,0,25.900000,1006.200012,330,0.9,0.0
2,2018-08-15 16:30:00,0.0,0.0,22.71,96.95,4.28,334.47,1007.48,0,0,25.799999,1006.299988,1,1.9,0.0
3,2018-08-15 16:45:00,0.0,0.0,22.64,97.12,4.28,331.52,1007.39,0,0,25.600000,1006.299988,309,0.4,0.0
4,2018-08-15 17:00:00,0.0,0.0,22.57,97.15,4.33,329.78,1007.09,0,0,25.700001,1006.299988,335,1.6,0.0
5,2018-08-15 17:15:00,0.0,0.0,22.60,96.69,4.39,330.40,1006.87,0,0,25.700001,1006.299988,343,1.1,0.0
6,2018-08-15 17:30:00,0.0,0.0,22.62,96.03,4.44,333.71,1006.94,0,0,25.500000,1006.099976,5,1.4,0.0
7,2018-08-15 17:45:00,0.0,0.0,22.69,94.98,4.55,336.65,1007.04,0,0,25.500000,1006.400024,342,1.5,0.0



PVOD-01A RAW DATASET DISCOVERY: PASSED

No timezone assigned.
No missing values modified.
No split created.
No model trained.


In [2]:
# ============================================================
# PVOD-01B
# Temporal coverage + physical-unit audit
#
# IMPORTANT
# ---------
# Duplicate timestamps are audited separately from the
# unique 15-minute time grid.
#
# No data is cleaned or modified here.
# No model is trained.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. Metadata lookup
# ============================================================

metadata_lookup = (
    metadata
    .set_index("Station_ID")
)


assert set(
    metadata_lookup.index
) == {
    f"station{i:02d}"
    for i in range(10)
}


# ============================================================
# 2. Audit every station
# ============================================================

temporal_audit_rows = []


for station_path in station_paths:

    station_id = station_path.stem

    df = pd.read_csv(
        station_path
    )


    # --------------------------------------------------------
    # Timestamp parsing
    #
    # Raw PVOD timestamps are interpreted as UTC.
    # --------------------------------------------------------

    timestamp_naive = pd.to_datetime(
        df["date_time"],
        errors="raise",
    )


    timestamp_utc = (
        timestamp_naive
        .dt
        .tz_localize("UTC")
    )


    timestamp_china = (
        timestamp_utc
        .dt
        .tz_convert("Asia/Shanghai")
    )


    # --------------------------------------------------------
    # Duplicate timestamp diagnostics
    # --------------------------------------------------------

    duplicate_all_mask = (
        timestamp_utc
        .duplicated(
            keep=False
        )
    )


    duplicate_excess_mask = (
        timestamp_utc
        .duplicated(
            keep="first"
        )
    )


    duplicate_timestamp_rows = int(
        duplicate_all_mask.sum()
    )


    duplicate_excess_rows = int(
        duplicate_excess_mask.sum()
    )


    duplicate_timestamp_values = int(
        timestamp_utc[
            duplicate_all_mask
        ]
        .nunique()
    )


    # --------------------------------------------------------
    # Original file ordering audit
    #
    # Equal consecutive timestamps are duplicates, not
    # reverse chronological movement.
    # --------------------------------------------------------

    original_deltas = (
        timestamp_utc
        .diff()
        .dropna()
    )


    negative_time_steps = int(
        (
            original_deltas
            <
            pd.Timedelta(0)
        )
        .sum()
    )


    zero_time_steps = int(
        (
            original_deltas
            ==
            pd.Timedelta(0)
        )
        .sum()
    )


    # --------------------------------------------------------
    # Unique timestamp grid
    #
    # Temporal continuity must be evaluated on unique
    # timestamp values; otherwise a duplicate can produce
    # impossible quantities such as continuity > 1.
    # --------------------------------------------------------

    unique_timestamp_utc = pd.DatetimeIndex(
        timestamp_utc
        .drop_duplicates()
        .sort_values()
    )


    unique_deltas = (
        pd.Series(
            unique_timestamp_utc
        )
        .diff()
        .dropna()
    )


    exact_15min_steps = int(
        (
            unique_deltas
            ==
            pd.Timedelta(
                minutes=15
            )
        )
        .sum()
    )


    exact_15min_fraction = float(
        exact_15min_steps
        /
        len(
            unique_deltas
        )
    )


    gap_mask = (
        unique_deltas
        >
        pd.Timedelta(
            minutes=15
        )
    )


    gap_events = int(
        gap_mask.sum()
    )


    max_gap = (
        unique_deltas.max()
        if len(
            unique_deltas
        )
        else pd.NaT
    )


    median_cadence = (
        unique_deltas.median()
    )


    # --------------------------------------------------------
    # Quarter-hour alignment
    # --------------------------------------------------------

    quarter_hour_aligned = bool(
        (
            pd.Series(
                unique_timestamp_utc.minute
            )
            .isin(
                [
                    0,
                    15,
                    30,
                    45,
                ]
            )
            &
            pd.Series(
                unique_timestamp_utc.second
            )
            .eq(0)
        )
        .all()
    )


    # --------------------------------------------------------
    # Expected complete grid
    # --------------------------------------------------------

    expected_grid = pd.date_range(
        start=unique_timestamp_utc.min(),
        end=unique_timestamp_utc.max(),
        freq="15min",
    )


    missing_timestamps = (
        expected_grid.difference(
            unique_timestamp_utc
        )
    )


    expected_rows_if_continuous = int(
        len(
            expected_grid
        )
    )


    unique_timestamp_count = int(
        len(
            unique_timestamp_utc
        )
    )


    missing_grid_slots = int(
        len(
            missing_timestamps
        )
    )


    continuity_fraction = float(
        unique_timestamp_count
        /
        expected_rows_if_continuous
    )


    # --------------------------------------------------------
    # Power / capacity audit
    #
    # Metadata Capacity : kW
    # station power     : MW
    # --------------------------------------------------------

    capacity_kw = float(
        metadata_lookup.loc[
            station_id,
            "Capacity"
        ]
    )


    capacity_mw = (
        capacity_kw
        /
        1000.0
    )


    power_mw = pd.to_numeric(
        df["power"],
        errors="raise",
    )


    assert np.isfinite(
        power_mw.to_numpy(
            dtype=float
        )
    ).all()


    negative_power_rows = int(
        (
            power_mw
            <
            0.0
        )
        .sum()
    )


    over_capacity_rows = int(
        (
            power_mw
            >
            capacity_mw
        )
        .sum()
    )


    max_power_mw = float(
        power_mw.max()
    )


    max_power_capacity_ratio = float(
        max_power_mw
        /
        capacity_mw
    )


    # --------------------------------------------------------
    # Irradiance sanity
    # --------------------------------------------------------

    lmd_ghi = pd.to_numeric(
        df["lmd_totalirrad"],
        errors="raise",
    )


    nwp_ghi = pd.to_numeric(
        df["nwp_globalirrad"],
        errors="raise",
    )


    # --------------------------------------------------------
    # Store audit row
    # --------------------------------------------------------

    temporal_audit_rows.append(
        {
            "station":
                station_id,

            "raw_rows":
                int(
                    len(df)
                ),

            "unique_timestamps":
                unique_timestamp_count,

            "start_utc":
                unique_timestamp_utc.min(),

            "end_utc":
                unique_timestamp_utc.max(),

            "start_china":
                unique_timestamp_utc.min()
                .tz_convert(
                    "Asia/Shanghai"
                ),

            "end_china":
                unique_timestamp_utc.max()
                .tz_convert(
                    "Asia/Shanghai"
                ),

            "median_cadence":
                median_cadence,

            "exact_15min_fraction":
                exact_15min_fraction,

            "quarter_hour_aligned":
                quarter_hour_aligned,

            "duplicate_timestamp_rows":
                duplicate_timestamp_rows,

            "duplicate_excess_rows":
                duplicate_excess_rows,

            "duplicate_timestamp_values":
                duplicate_timestamp_values,

            "negative_time_steps":
                negative_time_steps,

            "zero_time_steps":
                zero_time_steps,

            "gap_events":
                gap_events,

            "max_gap":
                max_gap,

            "expected_rows_if_continuous":
                expected_rows_if_continuous,

            "missing_grid_slots":
                missing_grid_slots,

            "continuity_fraction":
                continuity_fraction,

            "capacity_kW":
                capacity_kw,

            "capacity_MW":
                capacity_mw,

            "power_min_MW":
                float(
                    power_mw.min()
                ),

            "power_max_MW":
                max_power_mw,

            "max_power_capacity_ratio":
                max_power_capacity_ratio,

            "negative_power_rows":
                negative_power_rows,

            "over_capacity_rows":
                over_capacity_rows,

            "lmd_GHI_max_Wm2":
                float(
                    lmd_ghi.max()
                ),

            "nwp_GHI_max_Wm2":
                float(
                    nwp_ghi.max()
                ),
        }
    )


pvod_temporal_audit = pd.DataFrame(
    temporal_audit_rows
)


# ============================================================
# 3. Integrity assertions
#
# Duplicates are NOT asserted away here. They are explicitly
# identified in the raw temporal audit.
# ============================================================

raw_backward_jump_total = int(
    pvod_temporal_audit[
        "negative_time_steps"
    ].sum()
)

print(
    "Backward timestamp jumps in raw CSV order:",
    raw_backward_jump_total,
)

if raw_backward_jump_total > 0:
    print(
        "NOTE: Raw CSV row-order/timestamp anomalies exist. "
        "The station00 anomaly is resolved by the frozen correction "
        "before deterministic correction."
    )

assert (
    pvod_temporal_audit[
        "quarter_hour_aligned"
    ]
    .all()
), (
    "At least one timestamp is off the 15-minute clock grid."
)


assert (
    pvod_temporal_audit[
        "median_cadence"
    ]
    .eq(
        pd.Timedelta(
            minutes=15
        )
    )
    .all()
), (
    "At least one station does not have a 15-minute "
    "median cadence."
)


assert (
    pvod_temporal_audit[
        "continuity_fraction"
    ]
    .between(
        0.0,
        1.0,
        inclusive="both",
    )
    .all()
)


# ============================================================
# 4. Duplicate summary
# ============================================================

duplicate_station_count = int(
    (
        pvod_temporal_audit[
            "duplicate_timestamp_values"
        ]
        >
        0
    )
    .sum()
)


duplicate_timestamp_value_total = int(
    pvod_temporal_audit[
        "duplicate_timestamp_values"
    ]
    .sum()
)


duplicate_row_total = int(
    pvod_temporal_audit[
        "duplicate_timestamp_rows"
    ]
    .sum()
)


print(
    "Stations containing duplicate timestamps:",
    duplicate_station_count,
)


print(
    "Distinct duplicated timestamp values:",
    duplicate_timestamp_value_total,
)


print(
    "Rows belonging to duplicate timestamp groups:",
    duplicate_row_total,
)


if duplicate_timestamp_value_total > 0:

    print(
        "NOTE: Conflicting duplicate groups are investigated "
        "before any cleaning in the raw temporal audit."
    )


# ============================================================
# 5. Common temporal intersection across all stations
# ============================================================

common_start_utc = (
    pvod_temporal_audit[
        "start_utc"
    ]
    .max()
)


common_end_utc = (
    pvod_temporal_audit[
        "end_utc"
    ]
    .min()
)


assert (
    common_start_utc
    <=
    common_end_utc
)


common_grid = pd.date_range(
    start=common_start_utc,
    end=common_end_utc,
    freq="15min",
)


common_start_china = (
    common_start_utc
    .tz_convert(
        "Asia/Shanghai"
    )
)


common_end_china = (
    common_end_utc
    .tz_convert(
        "Asia/Shanghai"
    )
)


# ============================================================
# 6. Display
# ============================================================

print(
    "\n"
    + "=" * 120
)

print(
    "PVOD TEMPORAL + PHYSICAL AUDIT"
)

print(
    "=" * 120
)


display(
    pvod_temporal_audit
)


print(
    "\n"
    + "=" * 120
)

print(
    "COMMON 10-STATION TEMPORAL INTERSECTION"
)

print(
    "=" * 120
)


print(
    "UTC start :",
    common_start_utc
)


print(
    "UTC end   :",
    common_end_utc
)


print(
    "China start:",
    common_start_china
)


print(
    "China end  :",
    common_end_china
)


print(
    "15-minute timestamps in common bounding grid:",
    len(
        common_grid
    )
)


print(
    "\nStations with intrinsic temporal gaps:",
    int(
        (
            pvod_temporal_audit[
                "gap_events"
            ]
            >
            0
        )
        .sum()
    )
)


print(
    "Total missing 15-minute grid slots:",
    int(
        pvod_temporal_audit[
            "missing_grid_slots"
        ]
        .sum()
    )
)


print(
    "Total duplicate timestamp values:",
    duplicate_timestamp_value_total
)


print(
    "Total negative-power rows:",
    int(
        pvod_temporal_audit[
            "negative_power_rows"
        ]
        .sum()
    )
)


print(
    "Total above-nameplate rows:",
    int(
        pvod_temporal_audit[
            "over_capacity_rows"
        ]
        .sum()
    )
)


print(
    "\n"
    + "=" * 120
)

print(
    "PVOD-01B TEMPORAL + UNIT AUDIT: PASSED"
)

print(
    "=" * 120
)


print(
    "\nNo rows removed."
)

print(
    "No duplicate repaired."
)

print(
    "No missing timestamp filled."
)

print(
    "No split created."
)

print(
    "No model trained."
)

Backward timestamp jumps in raw CSV order: 1
NOTE: Raw CSV row-order/timestamp anomalies exist. The station00 anomaly is resolved by the frozen correction before deterministic correction.
Stations containing duplicate timestamps: 1
Distinct duplicated timestamp values: 1
Rows belonging to duplicate timestamp groups: 2
NOTE: Conflicting duplicate groups are investigated before any cleaning in the raw temporal audit.

PVOD TEMPORAL + PHYSICAL AUDIT


,station,raw_rows,unique_timestamps,start_utc,end_utc,start_china,end_china,median_cadence,exact_15min_fraction,quarter_hour_aligned,...,continuity_fraction,capacity_kW,capacity_MW,power_min_MW,power_max_MW,max_power_capacity_ratio,negative_power_rows,over_capacity_rows,lmd_GHI_max_Wm2,nwp_GHI_max_Wm2
0,station00,28896,28895,2018-08-15 16:00:00+00:00,2019-06-13 15:45:00+00:00,2018-08-16 00:00:00+08:00,2019-06-13 23:45:00+08:00,0 days 00:15:00,0.999931,True,...,0.996654,6600.0,6.6,0.0,5.523046,0.836825,0,0,1122.000000,942.75
1,station01,33408,33408,2018-06-30 16:00:00+00:00,2019-06-13 15:45:00+00:00,2018-07-01 00:00:00+08:00,2019-06-13 23:45:00+08:00,0 days 00:15:00,1.000000,True,...,1.000000,20000.0,20.0,0.0,19.997459,0.999873,0,0,1242.300049,936.42
2,station02,30432,30432,2018-07-22 16:00:00+00:00,2019-06-10 15:45:00+00:00,2018-07-23 00:00:00+08:00,2019-06-10 23:45:00+08:00,0 days 00:15:00,0.999901,True,...,0.981424,17000.0,17.0,0.0,16.048201,0.944012,0,0,1118.000000,944.94
3,station03,14688,14688,2019-01-11 16:00:00+00:00,2019-06-13 15:45:00+00:00,2019-01-12 00:00:00+08:00,2019-06-13 23:45:00+08:00,0 days 00:15:00,1.000000,True,...,1.000000,20000.0,20.0,0.0,17.419300,0.870965,0,0,1057.000000,945.54
4,station04,33408,33408,2018-06-30 16:00:00+00:00,2019-06-13 15:45:00+00:00,2018-07-01 00:00:00+08:00,2019-06-13 23:45:00+08:00,0 days 00:15:00,1.000000,True,...,1.000000,20000.0,20.0,0.0,26.768000,1.338400,0,1578,1838.000000,949.17
5,station05,9696,9696,2019-03-04 16:00:00+00:00,2019-06-13 15:45:00+00:00,2019-03-05 00:00:00+08:00,2019-06-13 23:45:00+08:00,0 days 00:15:00,1.000000,True,...,1.000000,35000.0,35.0,0.0,35.118130,1.003375,0,1,1156.000000,940.36
6,station06,31104,31104,2018-07-13 16:00:00+00:00,2019-06-13 15:45:00+00:00,2018-07-14 00:00:00+08:00,2019-06-13 23:45:00+08:00,0 days 00:15:00,0.999839,True,...,0.967164,15000.0,15.0,0.0,11.741000,0.782733,0,0,1085.000000,950.36
7,station07,32928,32928,2018-06-30 16:00:00+00:00,2019-06-13 15:45:00+00:00,2018-07-01 00:00:00+08:00,2019-06-13 23:45:00+08:00,0 days 00:15:00,0.999970,True,...,0.985632,20000.0,20.0,0.0,17.280581,0.864029,0,0,1518.000000,964.06
8,station08,33120,33120,2018-06-30 16:00:00+00:00,2019-06-13 15:45:00+00:00,2018-07-01 00:00:00+08:00,2019-06-13 23:45:00+08:00,0 days 00:15:00,0.999970,True,...,0.991379,20000.0,20.0,0.0,17.866020,0.893301,0,0,1243.000000,956.60
9,station09,24288,24288,2018-09-25 16:00:00+00:00,2019-06-13 15:45:00+00:00,2018-09-26 00:00:00+08:00,2019-06-13 23:45:00+08:00,0 days 00:15:00,0.999835,True,...,0.969349,20000.0,20.0,0.0,12.044960,0.602248,0,0,1439.000000,931.56



COMMON 10-STATION TEMPORAL INTERSECTION
UTC start : 2019-03-04 16:00:00+00:00
UTC end   : 2019-06-10 15:45:00+00:00
China start: 2019-03-05 00:00:00+08:00
China end  : 2019-06-10 23:45:00+08:00
15-minute timestamps in common bounding grid: 9408

Stations with intrinsic temporal gaps: 6
Total missing 15-minute grid slots: 3265
Total duplicate timestamp values: 1
Total negative-power rows: 0
Total above-nameplate rows: 1579

PVOD-01B TEMPORAL + UNIT AUDIT: PASSED

No rows removed.
No duplicate repaired.
No missing timestamp filled.
No split created.
No model trained.


In [3]:
# ============================================================
# PVOD-01C
# Frozen timestamp correction + gap-preserving clean tables
#
# FROZEN TIMESTAMP-CORRECTION POLICY
# -------------------------------------
#
# station00 raw row position 13093 is mislabeled:
#
#   observed timestamp:
#       2019-01-01 01:15:00 UTC
#
#   corrected timestamp:
#       2018-12-31 01:15:00 UTC
#
# Evidence:
# - measurement sequence matches Dec 31 neighbors;
# - Jan 1 has its own valid 01:15 observation;
# - Dec 31 01:15 is the corresponding missing grid slot.
#
# IMPORTANT:
# - measurements are NOT changed;
# - genuine temporal gaps are NOT interpolated;
# - every station is sorted chronologically after correction;
# - forecasting samples crossing gaps will later be excluded.
#
# No train/validation/test split is created here.
# No model is trained.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. Frozen correction registry
# ============================================================

PVOD_TIMESTAMP_CORRECTIONS = {
    "station00": [
        {
            "raw_row_position": 13093,

            "observed_timestamp":
                pd.Timestamp(
                    "2019-01-01 01:15:00",
                    tz="UTC",
                ),

            "corrected_timestamp":
                pd.Timestamp(
                    "2018-12-31 01:15:00",
                    tz="UTC",
                ),

            # Guard against accidentally applying this rule
            # to a different dataset/release.
            "expected_power_MW":
                0.715430,
        }
    ]
}


# ============================================================
# 2. Clean every station
# ============================================================

pvod_clean = {}

correction_log_rows = []
clean_audit_rows = []
gap_run_rows = []


for station_path in station_paths:

    station_id = station_path.stem

    df = pd.read_csv(
        station_path
    ).copy()


    # --------------------------------------------------------
    # Preserve immutable source row position
    # --------------------------------------------------------

    df[
        "_raw_row_position"
    ] = np.arange(
        len(df),
        dtype=int,
    )


    # --------------------------------------------------------
    # Parse published timestamp strings as UTC
    # --------------------------------------------------------

    df[
        "timestamp_utc"
    ] = (
        pd.to_datetime(
            df[
                "date_time"
            ],
            errors="raise",
        )
        .dt
        .tz_localize(
            "UTC"
        )
    )


    # --------------------------------------------------------
    # Apply ONLY explicitly frozen corrections
    # --------------------------------------------------------

    corrections = (
        PVOD_TIMESTAMP_CORRECTIONS.get(
            station_id,
            [],
        )
    )


    for correction in corrections:

        row_position = int(
            correction[
                "raw_row_position"
            ]
        )


        row_mask = (
            df[
                "_raw_row_position"
            ]
            ==
            row_position
        )


        assert (
            row_mask.sum()
            ==
            1
        ), (
            f"{station_id}: expected exactly one "
            f"raw row {row_position}"
        )


        actual_timestamp = (
            df.loc[
                row_mask,
                "timestamp_utc",
            ]
            .iloc[0]
        )


        assert (
            actual_timestamp
            ==
            correction[
                "observed_timestamp"
            ]
        ), (
            f"{station_id}: timestamp guard failed "
            f"for raw row {row_position}"
        )


        actual_power = float(
            df.loc[
                row_mask,
                "power",
            ]
            .iloc[0]
        )


        assert np.isclose(
            actual_power,
            correction[
                "expected_power_MW"
            ],
            rtol=0.0,
            atol=1e-6,
        ), (
            f"{station_id}: power guard failed "
            f"for raw row {row_position}"
        )


        df.loc[
            row_mask,
            "timestamp_utc",
        ] = correction[
            "corrected_timestamp"
        ]


        correction_log_rows.append(
            {
                "station":
                    station_id,

                "raw_row_position":
                    row_position,

                "old_timestamp_utc":
                    correction[
                        "observed_timestamp"
                    ],

                "new_timestamp_utc":
                    correction[
                        "corrected_timestamp"
                    ],

                "power_MW":
                    actual_power,

                "action":
                    "timestamp_corrected_only",
            }
        )


    # --------------------------------------------------------
    # Stable chronological ordering
    # --------------------------------------------------------

    df = (
        df
        .sort_values(
            "timestamp_utc",
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )


    # --------------------------------------------------------
    # Post-correction integrity
    # --------------------------------------------------------

    assert (
        df[
            "timestamp_utc"
        ]
        .is_monotonic_increasing
    ), (
        f"{station_id}: timestamps not monotonic "
        "after sorting"
    )


    duplicate_count = int(
        df[
            "timestamp_utc"
        ]
        .duplicated()
        .sum()
    )


    assert (
        duplicate_count
        ==
        0
    ), (
        f"{station_id}: duplicate timestamps remain "
        f"after frozen correction"
    )


    timestamps = pd.DatetimeIndex(
        df[
            "timestamp_utc"
        ]
    )


    deltas = (
        df[
            "timestamp_utc"
        ]
        .diff()
        .dropna()
    )


    assert (
        (
            deltas
            >=
            pd.Timedelta(
                minutes=15
            )
        )
        .all()
    ), (
        f"{station_id}: sub-15-minute step "
        "after cleaning"
    )


    expected_grid = pd.date_range(
        start=timestamps.min(),
        end=timestamps.max(),
        freq="15min",
    )


    missing_times = (
        expected_grid
        .difference(
            timestamps
        )
    )


    # --------------------------------------------------------
    # Identify contiguous missing runs
    # --------------------------------------------------------

    if len(
        missing_times
    ) > 0:

        missing_series = pd.Series(
            missing_times
        )


        run_start_flag = (
            missing_series
            .diff()
            .ne(
                pd.Timedelta(
                    minutes=15
                )
            )
        )


        run_ids = (
            run_start_flag
            .cumsum()
        )


        for _, group in (
            missing_series
            .groupby(
                run_ids
            )
        ):

            missing_start = (
                group.iloc[0]
            )


            missing_end = (
                group.iloc[-1]
            )


            missing_slots = int(
                len(group)
            )


            gap_run_rows.append(
                {
                    "station":
                        station_id,

                    "missing_start_utc":
                        missing_start,

                    "missing_end_utc":
                        missing_end,

                    "missing_slots":
                        missing_slots,

                    "missing_duration":
                        pd.Timedelta(
                            minutes=
                                15
                                *
                                missing_slots
                        ),

                    "previous_available_utc":
                        (
                            missing_start
                            -
                            pd.Timedelta(
                                minutes=15
                            )
                        ),

                    "next_available_utc":
                        (
                            missing_end
                            +
                            pd.Timedelta(
                                minutes=15
                            )
                        ),
                }
            )


    # --------------------------------------------------------
    # Add canonical derived identifiers / units
    #
    # Preserve raw power column untouched.
    # --------------------------------------------------------

    capacity_kw = float(
        metadata_lookup.loc[
            station_id,
            "Capacity",
        ]
    )


    capacity_mw = (
        capacity_kw
        /
        1000.0
    )


    df[
        "station_id"
    ] = station_id


    df[
        "capacity_MW"
    ] = capacity_mw


    df[
        "power_MW"
    ] = pd.to_numeric(
        df[
            "power"
        ],
        errors="raise",
    )


    # Useful later for cross-station comparison.
    # This does NOT replace the physical MW target.
    df[
        "power_pu"
    ] = (
        df[
            "power_MW"
        ]
        /
        capacity_mw
    )


    df[
        "timestamp_china"
    ] = (
        df[
            "timestamp_utc"
        ]
        .dt
        .tz_convert(
            "Asia/Shanghai"
        )
    )


    # --------------------------------------------------------
    # Final station audit
    # --------------------------------------------------------

    clean_audit_rows.append(
        {
            "station":
                station_id,

            "rows":
                int(
                    len(df)
                ),

            "start_utc":
                timestamps.min(),

            "end_utc":
                timestamps.max(),

            "duplicates":
                duplicate_count,

            "backward_steps":
                int(
                    (
                        deltas
                        <
                        pd.Timedelta(0)
                    )
                    .sum()
                ),

            "15min_steps":
                int(
                    (
                        deltas
                        ==
                        pd.Timedelta(
                            minutes=15
                        )
                    )
                    .sum()
                ),

            "gap_events":
                int(
                    (
                        deltas
                        >
                        pd.Timedelta(
                            minutes=15
                        )
                    )
                    .sum()
                ),

            "missing_grid_slots":
                int(
                    len(
                        missing_times
                    )
                ),

            "max_gap":
                (
                    deltas.max()
                    if len(
                        deltas
                    )
                    else pd.NaT
                ),

            "capacity_MW":
                capacity_mw,

            "power_min_MW":
                float(
                    df[
                        "power_MW"
                    ]
                    .min()
                ),

            "power_max_MW":
                float(
                    df[
                        "power_MW"
                    ]
                    .max()
                ),

            "power_pu_max":
                float(
                    df[
                        "power_pu"
                    ]
                    .max()
                ),
        }
    )


    pvod_clean[
        station_id
    ] = df


# ============================================================
# 3. Aggregate audit tables
# ============================================================

pvod_timestamp_corrections = pd.DataFrame(
    correction_log_rows
)


pvod_clean_audit = pd.DataFrame(
    clean_audit_rows
)


pvod_gap_runs = pd.DataFrame(
    gap_run_rows
)


# ============================================================
# 4. Global assertions
# ============================================================

assert len(
    pvod_clean
) == 10


assert (
    pvod_clean_audit[
        "duplicates"
    ]
    .eq(0)
    .all()
)


assert (
    pvod_clean_audit[
        "backward_steps"
    ]
    .eq(0)
    .all()
)


# Exactly one explicitly frozen timestamp correction.
assert (
    len(
        pvod_timestamp_corrections
    )
    ==
    1
)


# ============================================================
# 5. Display correction log
# ============================================================

print(
    "=" * 120
)

print(
    "FROZEN TIMESTAMP CORRECTION LOG"
)

print(
    "=" * 120
)


display(
    pvod_timestamp_corrections
)


# ============================================================
# 6. Clean station audit
# ============================================================

print(
    "\n"
    + "=" * 120
)

print(
    "POST-CORRECTION STATION AUDIT"
)

print(
    "=" * 120
)


display(
    pvod_clean_audit
)


# ============================================================
# 7. Gap runs
# ============================================================

print(
    "\n"
    + "=" * 120
)

print(
    "PRESERVED TEMPORAL GAP RUNS"
)

print(
    "=" * 120
)


if pvod_gap_runs.empty:

    print(
        "No temporal gaps."
    )

else:

    display(
        pvod_gap_runs
    )


# ============================================================
# 8. Global summary
# ============================================================

print(
    "\n"
    + "=" * 120
)

print(
    "PVOD CLEANING SUMMARY"
)

print(
    "=" * 120
)


print(
    "Stations:",
    len(
        pvod_clean
    )
)


print(
    "Timestamp corrections:",
    len(
        pvod_timestamp_corrections
    )
)


print(
    "Rows removed:",
    0
)


print(
    "Rows interpolated:",
    0
)


print(
    "Remaining duplicate timestamps:",
    int(
        pvod_clean_audit[
            "duplicates"
        ]
        .sum()
    )
)


print(
    "Remaining backward steps:",
    int(
        pvod_clean_audit[
            "backward_steps"
        ]
        .sum()
    )
)


print(
    "Preserved gap events:",
    int(
        pvod_clean_audit[
            "gap_events"
        ]
        .sum()
    )
)


print(
    "Preserved missing 15-min slots:",
    int(
        pvod_clean_audit[
            "missing_grid_slots"
        ]
        .sum()
    )
)


print(
    "\nPVOD-01C CLEANING + GAP PRESERVATION: PASSED"
)


print(
    "\nNo temporal gap interpolated."
)

print(
    "No train/validation/test split created."
)

print(
    "No forecasting sample created."
)

print(
    "No model trained."
)

FROZEN TIMESTAMP CORRECTION LOG


,station,raw_row_position,old_timestamp_utc,new_timestamp_utc,power_MW,action
0,station00,13093,2019-01-01 01:15:00+00:00,2018-12-31 01:15:00+00:00,0.71543,timestamp_corrected_only



POST-CORRECTION STATION AUDIT


,station,rows,start_utc,end_utc,duplicates,backward_steps,15min_steps,gap_events,missing_grid_slots,max_gap,capacity_MW,power_min_MW,power_max_MW,power_pu_max
0,station00,28896,2018-08-15 16:00:00+00:00,2019-06-13 15:45:00+00:00,0,0,28894,1,96,1 days 00:15:00,6.6,0.0,5.523046,0.836825
1,station01,33408,2018-06-30 16:00:00+00:00,2019-06-13 15:45:00+00:00,0,0,33407,0,0,0 days 00:15:00,20.0,0.0,19.997459,0.999873
2,station02,30432,2018-07-22 16:00:00+00:00,2019-06-10 15:45:00+00:00,0,0,30428,3,576,3 days 00:15:00,17.0,0.0,16.048201,0.944012
3,station03,14688,2019-01-11 16:00:00+00:00,2019-06-13 15:45:00+00:00,0,0,14687,0,0,0 days 00:15:00,20.0,0.0,17.419300,0.870965
4,station04,33408,2018-06-30 16:00:00+00:00,2019-06-13 15:45:00+00:00,0,0,33407,0,0,0 days 00:15:00,20.0,0.0,26.768000,1.338400
5,station05,9696,2019-03-04 16:00:00+00:00,2019-06-13 15:45:00+00:00,0,0,9695,0,0,0 days 00:15:00,35.0,0.0,35.118130,1.003375
6,station06,31104,2018-07-13 16:00:00+00:00,2019-06-13 15:45:00+00:00,0,0,31098,5,1056,6 days 00:15:00,15.0,0.0,11.741000,0.782733
7,station07,32928,2018-06-30 16:00:00+00:00,2019-06-13 15:45:00+00:00,0,0,32926,1,480,5 days 00:15:00,20.0,0.0,17.280581,0.864029
8,station08,33120,2018-06-30 16:00:00+00:00,2019-06-13 15:45:00+00:00,0,0,33118,1,288,3 days 00:15:00,20.0,0.0,17.866020,0.893301
9,station09,24288,2018-09-25 16:00:00+00:00,2019-06-13 15:45:00+00:00,0,0,24283,4,768,3 days 00:15:00,20.0,0.0,12.044960,0.602248



PRESERVED TEMPORAL GAP RUNS


,station,missing_start_utc,missing_end_utc,missing_slots,missing_duration,previous_available_utc,next_available_utc
0,station00,2018-12-04 16:00:00+00:00,2018-12-05 15:45:00+00:00,96,1 days,2018-12-04 15:45:00+00:00,2018-12-05 16:00:00+00:00
1,station02,2019-02-12 16:00:00+00:00,2019-02-15 15:45:00+00:00,288,3 days,2019-02-12 15:45:00+00:00,2019-02-15 16:00:00+00:00
2,station02,2019-04-26 16:00:00+00:00,2019-04-28 15:45:00+00:00,192,2 days,2019-04-26 15:45:00+00:00,2019-04-28 16:00:00+00:00
3,station02,2019-06-06 16:00:00+00:00,2019-06-07 15:45:00+00:00,96,1 days,2019-06-06 15:45:00+00:00,2019-06-07 16:00:00+00:00
4,station06,2018-07-26 16:00:00+00:00,2018-07-27 15:45:00+00:00,96,1 days,2018-07-26 15:45:00+00:00,2018-07-27 16:00:00+00:00
5,station06,2018-08-13 16:00:00+00:00,2018-08-15 15:45:00+00:00,192,2 days,2018-08-13 15:45:00+00:00,2018-08-15 16:00:00+00:00
6,station06,2018-11-08 16:00:00+00:00,2018-11-14 15:45:00+00:00,576,6 days,2018-11-08 15:45:00+00:00,2018-11-14 16:00:00+00:00
7,station06,2019-03-24 16:00:00+00:00,2019-03-25 15:45:00+00:00,96,1 days,2019-03-24 15:45:00+00:00,2019-03-25 16:00:00+00:00
8,station06,2019-04-18 16:00:00+00:00,2019-04-19 15:45:00+00:00,96,1 days,2019-04-18 15:45:00+00:00,2019-04-19 16:00:00+00:00
9,station07,2019-03-07 16:00:00+00:00,2019-03-12 15:45:00+00:00,480,5 days,2019-03-07 15:45:00+00:00,2019-03-12 16:00:00+00:00



PVOD CLEANING SUMMARY
Stations: 10
Timestamp corrections: 1
Rows removed: 0
Rows interpolated: 0
Remaining duplicate timestamps: 0
Remaining backward steps: 0
Preserved gap events: 15
Preserved missing 15-min slots: 3264

PVOD-01C CLEANING + GAP PRESERVATION: PASSED

No temporal gap interpolated.
No train/validation/test split created.
No forecasting sample created.
No model trained.


In [4]:
# ============================================================
# PVOD-01D
# Above-nameplate PV power diagnostic
#
# PURPOSE
# -------
# Investigate observations where:
#
#     measured power > metadata capacity
#
# before deciding how capacity normalization should be used.
#
# IMPORTANT
# ---------
# No power values are clipped.
# No rows are removed.
# No target normalization policy is changed yet.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. Station-level exceedance summary
# ============================================================

exceedance_summary_rows = []
exceedance_tables = {}


for station_id, df in pvod_clean.items():

    capacity_mw = float(
        df[
            "capacity_MW"
        ].iloc[0]
    )


    power_mw = (
        df[
            "power_MW"
        ]
        .astype(float)
    )


    ratio = (
        power_mw
        /
        capacity_mw
    )


    exceed_mask = (
        power_mw
        >
        capacity_mw
    )


    exceed_df = (
        df.loc[
            exceed_mask,
            [
                "timestamp_utc",
                "timestamp_china",
                "power_MW",
                "capacity_MW",
                "lmd_totalirrad",
                "lmd_diffuseirrad",
                "lmd_temperature",
                "nwp_globalirrad",
                "nwp_temperature",
            ]
        ]
        .copy()
    )


    if len(
        exceed_df
    ) > 0:

        exceed_df[
            "power_capacity_ratio"
        ] = (
            exceed_df[
                "power_MW"
            ]
            /
            capacity_mw
        )


    exceedance_tables[
        station_id
    ] = exceed_df


    exceedance_summary_rows.append(
        {
            "station":
                station_id,

            "rows":
                int(
                    len(df)
                ),

            "capacity_MW":
                capacity_mw,

            "power_max_MW":
                float(
                    power_mw.max()
                ),

            "power_p99_MW":
                float(
                    power_mw.quantile(
                        0.99
                    )
                ),

            "power_p999_MW":
                float(
                    power_mw.quantile(
                        0.999
                    )
                ),

            "max_capacity_ratio":
                float(
                    ratio.max()
                ),

            "above_capacity_rows":
                int(
                    exceed_mask.sum()
                ),

            "above_capacity_fraction":
                float(
                    exceed_mask.mean()
                ),

            "above_105pct_rows":
                int(
                    (
                        ratio
                        >
                        1.05
                    )
                    .sum()
                ),

            "above_110pct_rows":
                int(
                    (
                        ratio
                        >
                        1.10
                    )
                    .sum()
                ),

            "above_120pct_rows":
                int(
                    (
                        ratio
                        >
                        1.20
                    )
                    .sum()
                ),
        }
    )


pvod_capacity_exceedance_summary = pd.DataFrame(
    exceedance_summary_rows
)


print(
    "=" * 120
)

print(
    "STATION-LEVEL CAPACITY EXCEEDANCE SUMMARY"
)

print(
    "=" * 120
)


display(
    pvod_capacity_exceedance_summary
)


# ============================================================
# 2. Focus on stations containing exceedances
# ============================================================

affected_stations = (
    pvod_capacity_exceedance_summary.loc[
        pvod_capacity_exceedance_summary[
            "above_capacity_rows"
        ]
        >
        0,
        "station",
    ]
    .tolist()
)


print(
    "\nAffected stations:",
    affected_stations
)


# ============================================================
# 3. Detailed station04 distribution
# ============================================================

station04 = (
    pvod_clean[
        "station04"
    ]
    .copy()
)


station04_capacity = float(
    station04[
        "capacity_MW"
    ].iloc[0]
)


station04[
    "power_capacity_ratio"
] = (
    station04[
        "power_MW"
    ]
    /
    station04_capacity
)


station04_exceed = (
    station04.loc[
        station04[
            "power_MW"
        ]
        >
        station04_capacity
    ]
    .copy()
)


print(
    "\n"
    + "=" * 120
)

print(
    "STATION04 POWER DISTRIBUTION"
)

print(
    "=" * 120
)


station04_quantiles = (
    station04[
        "power_MW"
    ]
    .quantile(
        [
            0.50,
            0.90,
            0.95,
            0.975,
            0.99,
            0.995,
            0.999,
            1.00,
        ]
    )
    .rename(
        "power_MW"
    )
    .to_frame()
)


station04_quantiles[
    "fraction_of_metadata_capacity"
] = (
    station04_quantiles[
        "power_MW"
    ]
    /
    station04_capacity
)


display(
    station04_quantiles
)


# ============================================================
# 4. When do station04 exceedances occur?
# ============================================================

station04_exceed[
    "utc_date"
] = (
    station04_exceed[
        "timestamp_utc"
    ]
    .dt
    .date
)


station04_exceed[
    "china_date"
] = (
    station04_exceed[
        "timestamp_china"
    ]
    .dt
    .date
)


station04_exceed[
    "china_month"
] = (
    station04_exceed[
        "timestamp_china"
    ]
    .dt
    .to_period(
        "M"
    )
    .astype(str)
)


station04_exceed[
    "china_hour"
] = (
    station04_exceed[
        "timestamp_china"
    ]
    .dt
    .hour
)


print(
    "\n"
    + "=" * 120
)

print(
    "STATION04 EXCEEDANCES BY MONTH"
)

print(
    "=" * 120
)


station04_monthly_exceedance = (
    station04_exceed
    .groupby(
        "china_month",
        sort=True,
    )
    .agg(
        exceedance_rows=(
            "power_MW",
            "size",
        ),

        exceedance_days=(
            "china_date",
            "nunique",
        ),

        max_power_MW=(
            "power_MW",
            "max",
        ),

        median_power_MW=(
            "power_MW",
            "median",
        ),

        max_ratio=(
            "power_capacity_ratio",
            "max",
        ),

        median_lmd_GHI=(
            "lmd_totalirrad",
            "median",
        ),

        max_lmd_GHI=(
            "lmd_totalirrad",
            "max",
        ),
    )
    .reset_index()
)


display(
    station04_monthly_exceedance
)


# ============================================================
# 5. Daily persistence
# ============================================================

station04_daily_exceedance = (
    station04_exceed
    .groupby(
        "china_date",
        sort=True,
    )
    .agg(
        exceedance_rows=(
            "power_MW",
            "size",
        ),

        max_power_MW=(
            "power_MW",
            "max",
        ),

        max_ratio=(
            "power_capacity_ratio",
            "max",
        ),

        max_lmd_GHI=(
            "lmd_totalirrad",
            "max",
        ),
    )
    .reset_index()
)


print(
    "\n"
    + "=" * 120
)

print(
    "STATION04 EXCEEDANCE-DAY SUMMARY"
)

print(
    "=" * 120
)


print(
    "Number of affected local dates:",
    station04_daily_exceedance[
        "china_date"
    ]
    .nunique()
)


print(
    "Maximum exceedance rows on one day:",
    int(
        station04_daily_exceedance[
            "exceedance_rows"
        ]
        .max()
    )
)


print(
    "Median exceedance rows per affected day:",
    float(
        station04_daily_exceedance[
            "exceedance_rows"
        ]
        .median()
    )
)


print(
    "\nTop 15 exceedance days:"
)


display(
    station04_daily_exceedance
    .sort_values(
        [
            "max_ratio",
            "exceedance_rows",
        ],
        ascending=False,
    )
    .head(
        15
    )
)


# ============================================================
# 6. Largest individual station04 observations
# ============================================================

print(
    "\n"
    + "=" * 120
)

print(
    "STATION04 TOP 25 POWER OBSERVATIONS"
)

print(
    "=" * 120
)


display(
    station04[
        [
            "timestamp_utc",
            "timestamp_china",
            "power_MW",
            "power_capacity_ratio",
            "lmd_totalirrad",
            "lmd_diffuseirrad",
            "lmd_temperature",
            "nwp_globalirrad",
            "nwp_temperature",
        ]
    ]
    .sort_values(
        "power_MW",
        ascending=False,
    )
    .head(
        25
    )
)


# ============================================================
# 7. Station05 single/small exceedance inspection
# ============================================================

if (
    "station05"
    in
    affected_stations
):

    station05_exceed = (
        exceedance_tables[
            "station05"
        ]
        .copy()
    )


    print(
        "\n"
        + "=" * 120
    )

    print(
        "STATION05 ABOVE-CAPACITY OBSERVATIONS"
    )

    print(
        "=" * 120
    )


    display(
        station05_exceed
    )


# ============================================================
# 8. Basic irradiance association for station04
# ============================================================

station04_daylike = (
    station04[
        "lmd_totalirrad"
    ]
    >
    0
)


station04_positive_ghi = (
    station04.loc[
        station04_daylike,
        [
            "power_MW",
            "lmd_totalirrad",
        ]
    ]
    .dropna()
)


station04_power_ghi_corr = float(
    station04_positive_ghi[
        "power_MW"
    ]
    .corr(
        station04_positive_ghi[
            "lmd_totalirrad"
        ]
    )
)


station04_exceed_power_ghi_corr = (
    float(
        station04_exceed[
            "power_MW"
        ]
        .corr(
            station04_exceed[
                "lmd_totalirrad"
            ]
        )
    )
    if len(
        station04_exceed
    ) > 1
    else np.nan
)


print(
    "\n"
    + "=" * 120
)

print(
    "STATION04 POWER / IRRADIANCE ASSOCIATION"
)

print(
    "=" * 120
)


print(
    "Positive-GHI rows:",
    len(
        station04_positive_ghi
    )
)


print(
    "Power vs LMD GHI correlation, positive-GHI rows:",
    station04_power_ghi_corr
)


print(
    "Power vs LMD GHI correlation, above-capacity rows:",
    station04_exceed_power_ghi_corr
)


# ============================================================
# 9. Policy status
# ============================================================

print(
    "\n"
    + "=" * 120
)

print(
    "PVOD-01D CAPACITY DIAGNOSTIC COMPLETE"
)

print(
    "=" * 120
)


print(
    "\nNo power clipped."
)

print(
    "No above-capacity row removed."
)

print(
    "No capacity metadata changed."
)

print(
    "No split created."
)

print(
    "No model trained."
)

STATION-LEVEL CAPACITY EXCEEDANCE SUMMARY


,station,rows,capacity_MW,power_max_MW,power_p99_MW,power_p999_MW,max_capacity_ratio,above_capacity_rows,above_capacity_fraction,above_105pct_rows,above_110pct_rows,above_120pct_rows
0,station00,28896,6.6,5.523046,4.447266,4.901150,0.836825,0,0.000000,0,0,0
1,station01,33408,20.0,19.997459,18.292521,19.908661,0.999873,0,0.000000,0,0,0
2,station02,30432,17.0,16.048201,13.797269,14.967710,0.944012,0,0.000000,0,0,0
3,station03,14688,20.0,17.419300,16.483407,17.096958,0.870965,0,0.000000,0,0,0
4,station04,33408,20.0,26.768000,23.172800,24.997812,1.338400,1578,0.047234,1143,716,149
5,station05,9696,35.0,35.118130,30.041993,32.992626,1.003375,1,0.000103,0,0,0
6,station06,31104,15.0,11.741000,9.745910,10.682897,0.782733,0,0.000000,0,0,0
7,station07,32928,20.0,17.280581,14.523640,16.358934,0.864029,0,0.000000,0,0,0
8,station08,33120,20.0,17.866020,15.056806,16.622500,0.893301,0,0.000000,0,0,0
9,station09,24288,20.0,12.044960,8.273970,11.007186,0.602248,0,0.000000,0,0,0



Affected stations: ['station04', 'station05']

STATION04 POWER DISTRIBUTION


,power_MW,fraction_of_metadata_capacity
0.500,0.000000,0.000000
0.900,16.716000,0.835800
0.950,19.784800,0.989240
0.975,21.693421,1.084671
0.990,23.172800,1.158640
0.995,23.928605,1.196430
0.999,24.997812,1.249891
1.000,26.768000,1.338400



STATION04 EXCEEDANCES BY MONTH


...


STATION04 TOP 25 POWER OBSERVATIONS


,timestamp_utc,timestamp_china,power_MW,power_capacity_ratio,lmd_totalirrad,lmd_diffuseirrad,lmd_temperature,nwp_globalirrad,nwp_temperature
26258,2019-03-31 04:30:00+00:00,2019-03-31 12:30:00+08:00,26.768000,1.33840,1222,169,6.2,825.98,4.78
25296,2019-03-21 04:00:00+00:00,2019-03-21 12:00:00+08:00,25.978399,1.29892,1388,94,1.0,792.95,0.53
29137,2019-04-30 04:15:00+00:00,2019-04-30 12:15:00+08:00,25.793600,1.28968,1193,655,12.0,925.65,12.10
29136,2019-04-30 04:00:00+00:00,2019-04-30 12:00:00+08:00,25.748800,1.28744,1168,622,11.4,922.31,11.83
29138,2019-04-30 04:30:00+00:00,2019-04-30 12:30:00+08:00,25.692801,1.28464,1196,682,12.5,925.33,12.35
29135,2019-04-30 03:45:00+00:00,2019-04-30 11:45:00+08:00,25.670401,1.28352,1131,595,11.9,917.26,11.54
29617,2019-05-05 04:15:00+00:00,2019-05-05 12:15:00+08:00,25.642401,1.28212,1158,926,13.8,906.71,14.57
25295,2019-03-21 03:45:00+00:00,2019-03-21 11:45:00+08:00,25.603201,1.28016,1330,85,0.1,785.63,0.28
29139,2019-04-30 04:45:00+00:00,2019-04-30 12:45:00+08:00,25.519199,1.27596,1191,696,12.7,920.82,12.58
25299,2019-03-21 04:45:00+00:00,2019-03-21 12:45:00+08:00,25.463200,1.27316,1433,70,0.9,798.48,1.17



STATION05 ABOVE-CAPACITY OBSERVATIONS


,timestamp_utc,timestamp_china,power_MW,capacity_MW,lmd_totalirrad,lmd_diffuseirrad,lmd_temperature,nwp_globalirrad,nwp_temperature,power_capacity_ratio
7251,2019-05-19 04:45:00+00:00,2019-05-19 12:45:00+08:00,35.11813,35.0,1139,336,24.1,931.43,23.0,1.003375



STATION04 POWER / IRRADIANCE ASSOCIATION
Positive-GHI rows: 16555
Power vs LMD GHI correlation, positive-GHI rows: 0.9526926599542381
Power vs LMD GHI correlation, above-capacity rows: 0.40259779346185093

PVOD-01D CAPACITY DIAGNOSTIC COMPLETE

No power clipped.
No above-capacity row removed.
No capacity metadata changed.
No split created.
No model trained.


In [5]:
# ============================================================
# PVOD-02A
# Candidate chronological split + gap-aware sample audit
#
# NO forecasting feature values are created here.
# NO F3/F4 comparison is performed.
# NO model is trained.
#
# Candidate protocol
# ------------------
# Train      : first 65% of each station's temporal span
# Validation : next 15%
# Test       : final 20%
#
# Assignment is based on TARGET valid time.
#
# Sample eligibility requires:
#
#   recent history:
#       t-60, t-45, t-30, t-15, t
#
#   daily PV reference:
#       t-24h
#
#   target:
#       t+h
#
# Every timestamp must physically exist in the station data.
# Therefore no forecasting window may bridge a preserved gap.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. Frozen temporal constants
# ============================================================

PVOD_CADENCE = pd.Timedelta(
    minutes=15
)


PVOD_RECENT_HISTORY_OFFSETS = [
    pd.Timedelta(
        minutes=minutes
    )
    for minutes in [
        60,
        45,
        30,
        15,
        0,
    ]
]


PVOD_DAILY_LAG = pd.Timedelta(
    hours=24
)


PVOD_HORIZONS = {
    15: pd.Timedelta(
        minutes=15
    ),

    30: pd.Timedelta(
        minutes=30
    ),

    60: pd.Timedelta(
        minutes=60
    ),
}


TRAIN_FRACTION = 0.65
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.20


assert np.isclose(
    TRAIN_FRACTION
    +
    VALIDATION_FRACTION
    +
    TEST_FRACTION,
    1.0,
)


# ============================================================
# 2. Helper: snap a boundary UP to the next 15-minute point
# ============================================================

def ceil_to_15min(
    timestamp: pd.Timestamp,
) -> pd.Timestamp:

    return timestamp.ceil(
        "15min"
    )


# ============================================================
# 3. Build station-specific chronological boundaries
# ============================================================

split_rows = []


for station_id, df in pvod_clean.items():

    timestamps = pd.DatetimeIndex(
        df[
            "timestamp_utc"
        ]
    )


    station_start = (
        timestamps.min()
    )


    station_end = (
        timestamps.max()
    )


    # Include one cadence interval so that the span corresponds
    # to the number of 15-minute grid positions.
    span = (
        station_end
        -
        station_start
        +
        PVOD_CADENCE
    )


    train_end_candidate = (
        station_start
        +
        span
        *
        TRAIN_FRACTION
    )


    validation_end_candidate = (
        station_start
        +
        span
        *
        (
            TRAIN_FRACTION
            +
            VALIDATION_FRACTION
        )
    )


    validation_start = (
        ceil_to_15min(
            train_end_candidate
        )
    )


    test_start = (
        ceil_to_15min(
            validation_end_candidate
        )
    )


    # Half-open intervals:
    #
    # train: [station_start, validation_start)
    # val  : [validation_start, test_start)
    # test : [test_start, station_end + 15 min)

    station_stop_exclusive = (
        station_end
        +
        PVOD_CADENCE
    )


    assert (
        station_start
        <
        validation_start
        <
        test_start
        <
        station_stop_exclusive
    )


    split_rows.append(
        {
            "station":
                station_id,

            "station_start_utc":
                station_start,

            "validation_start_utc":
                validation_start,

            "test_start_utc":
                test_start,

            "station_stop_exclusive_utc":
                station_stop_exclusive,

            "train_span_days":
                (
                    validation_start
                    -
                    station_start
                )
                /
                pd.Timedelta(
                    days=1
                ),

            "validation_span_days":
                (
                    test_start
                    -
                    validation_start
                )
                /
                pd.Timedelta(
                    days=1
                ),

            "test_span_days":
                (
                    station_stop_exclusive
                    -
                    test_start
                )
                /
                pd.Timedelta(
                    days=1
                ),
        }
    )


pvod_split_boundaries = pd.DataFrame(
    split_rows
)


print(
    "=" * 125
)

print(
    "CANDIDATE STATION-SPECIFIC SPLIT BOUNDARIES"
)

print(
    "=" * 125
)


display(
    pvod_split_boundaries
)


# ============================================================
# 4. Gap-aware sample eligibility
# ============================================================

eligibility_rows = []


for station_id, df in pvod_clean.items():

    timestamps = pd.DatetimeIndex(
        df[
            "timestamp_utc"
        ]
    )


    timestamp_set = set(
        timestamps
    )


    boundary = (
        pvod_split_boundaries
        .set_index(
            "station"
        )
        .loc[
            station_id
        ]
    )


    validation_start = (
        boundary[
            "validation_start_utc"
        ]
    )


    test_start = (
        boundary[
            "test_start_utc"
        ]
    )


    stop_exclusive = (
        boundary[
            "station_stop_exclusive_utc"
        ]
    )


    for horizon_minutes, horizon_delta in (
        PVOD_HORIZONS.items()
    ):

        split_issue_times = {
            "train": [],
            "validation": [],
            "test": [],
        }


        # Candidate issue time t.
        #
        # We deliberately iterate over actual timestamps rather
        # than creating/filling a complete artificial grid.

        for issue_time in timestamps:

            target_time = (
                issue_time
                +
                horizon_delta
            )


            # ------------------------------------------------
            # Required recent-history timestamps
            # ------------------------------------------------

            history_times = [
                issue_time
                -
                offset

                for offset
                in PVOD_RECENT_HISTORY_OFFSETS
            ]


            recent_history_exists = all(
                required_time
                in
                timestamp_set

                for required_time
                in history_times
            )


            # ------------------------------------------------
            # Previous-day PV timestamp
            # ------------------------------------------------

            daily_reference_time = (
                issue_time
                -
                PVOD_DAILY_LAG
            )


            daily_reference_exists = (
                daily_reference_time
                in
                timestamp_set
            )


            # ------------------------------------------------
            # Target must genuinely exist
            # ------------------------------------------------

            target_exists = (
                target_time
                in
                timestamp_set
            )


            if not (
                recent_history_exists
                and
                daily_reference_exists
                and
                target_exists
            ):

                continue


            # ------------------------------------------------
            # Split assignment uses TARGET VALID TIME
            # ------------------------------------------------

            if (
                target_time
                <
                validation_start
            ):

                split_name = (
                    "train"
                )


            elif (
                target_time
                <
                test_start
            ):

                split_name = (
                    "validation"
                )


            elif (
                target_time
                <
                stop_exclusive
            ):

                split_name = (
                    "test"
                )


            else:

                continue


            split_issue_times[
                split_name
            ].append(
                issue_time
            )


        # ----------------------------------------------------
        # Summarize each split
        # ----------------------------------------------------

        for split_name, issue_times in (
            split_issue_times.items()
        ):

            if len(
                issue_times
            ) == 0:

                eligibility_rows.append(
                    {
                        "station":
                            station_id,

                        "horizon_min":
                            horizon_minutes,

                        "split":
                            split_name,

                        "samples":
                            0,

                        "issue_start_utc":
                            pd.NaT,

                        "issue_end_utc":
                            pd.NaT,

                        "target_start_utc":
                            pd.NaT,

                        "target_end_utc":
                            pd.NaT,

                        "target_utc_days":
                            0,

                        "target_china_days":
                            0,
                    }
                )

                continue


            issue_index = pd.DatetimeIndex(
                issue_times
            )


            target_index = (
                issue_index
                +
                horizon_delta
            )


            target_china = (
                target_index
                .tz_convert(
                    "Asia/Shanghai"
                )
            )


            eligibility_rows.append(
                {
                    "station":
                        station_id,

                    "horizon_min":
                        horizon_minutes,

                    "split":
                        split_name,

                    "samples":
                        int(
                            len(
                                issue_index
                            )
                        ),

                    "issue_start_utc":
                        issue_index.min(),

                    "issue_end_utc":
                        issue_index.max(),

                    "target_start_utc":
                        target_index.min(),

                    "target_end_utc":
                        target_index.max(),

                    "target_utc_days":
                        int(
                            pd.Index(
                                target_index.date
                            )
                            .nunique()
                        ),

                    "target_china_days":
                        int(
                            pd.Index(
                                target_china.date
                            )
                            .nunique()
                        ),
                }
            )


pvod_sample_eligibility = pd.DataFrame(
    eligibility_rows
)


# ============================================================
# 5. Integrity assertions
# ============================================================

expected_rows = (
    10
    *
    len(
        PVOD_HORIZONS
    )
    *
    3
)


assert len(
    pvod_sample_eligibility
) == expected_rows


assert (
    pvod_sample_eligibility[
        "samples"
    ]
    >
    0
).all(), (
    "At least one station/horizon/split has zero eligible samples."
)


# ============================================================
# 6. Sample-count pivot
# ============================================================

sample_count_pivot = (
    pvod_sample_eligibility
    .pivot_table(
        index=[
            "station",
            "horizon_min",
        ],

        columns="split",

        values="samples",

        aggfunc="first",
    )
    .reset_index()
)


sample_count_pivot = (
    sample_count_pivot[
        [
            "station",
            "horizon_min",
            "train",
            "validation",
            "test",
        ]
    ]
)


print(
    "\n"
    + "=" * 125
)

print(
    "GAP-AWARE ELIGIBLE SAMPLE COUNTS"
)

print(
    "=" * 125
)


display(
    sample_count_pivot
)


# ============================================================
# 7. Test-day counts
# ============================================================

test_day_audit = (
    pvod_sample_eligibility.loc[
        pvod_sample_eligibility[
            "split"
        ]
        ==
        "test",
        [
            "station",
            "horizon_min",
            "samples",
            "target_utc_days",
            "target_china_days",
            "target_start_utc",
            "target_end_utc",
        ]
    ]
    .sort_values(
        [
            "station",
            "horizon_min",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 125
)

print(
    "HELD-OUT TEST COVERAGE"
)

print(
    "=" * 125
)


display(
    test_day_audit
)


# ============================================================
# 8. Station-level minimum sample/day diagnostics
# ============================================================

print(
    "\n"
    + "=" * 125
)

print(
    "MINIMUM ELIGIBILITY DIAGNOSTICS"
)

print(
    "=" * 125
)


for split_name in [
    "train",
    "validation",
    "test",
]:

    split_table = (
        pvod_sample_eligibility.loc[
            pvod_sample_eligibility[
                "split"
            ]
            ==
            split_name
        ]
    )


    print(
        f"{split_name.upper():10s} "
        f"minimum samples = "
        f"{int(split_table['samples'].min())}"
    )


print(
    "\nMinimum held-out China-local test days:",
    int(
        test_day_audit[
            "target_china_days"
        ]
        .min()
    )
)


print(
    "Maximum held-out China-local test days:",
    int(
        test_day_audit[
            "target_china_days"
        ]
        .max()
    )
)


# ============================================================
# 9. Protocol status
# ============================================================

print(
    "\n"
    + "=" * 125
)

print(
    "PVOD-02A CANDIDATE SPLIT AUDIT: PASSED"
)

print(
    "=" * 125
)


print(
    "\nCandidate split proportions: 65% / 15% / 20%."
)

print(
    "Split assignment uses target valid time."
)

print(
    "Recent physical history window: 60 minutes."
)

print(
    "Daily PV lag: 24 hours."
)

print(
    "Horizon mapping: 15 / 30 / 60 minutes."
)

print(
    "All temporal gaps preserved."
)

print(
    "No eligible sample crosses a missing timestamp."
)

print(
    "\nNo feature family generated."
)

print(
    "No scaler fitted."
)

print(
    "No model trained."
)

print(
    "No F3/F4 result observed."
)

CANDIDATE STATION-SPECIFIC SPLIT BOUNDARIES


,station,station_start_utc,validation_start_utc,test_start_utc,station_stop_exclusive_utc,train_span_days,validation_span_days,test_span_days
0,station00,2018-08-15 16:00:00+00:00,2019-02-27 23:15:00+00:00,2019-04-14 06:30:00+00:00,2019-06-13 16:00:00+00:00,196.302083,45.302083,60.395833
1,station01,2018-06-30 16:00:00+00:00,2019-02-11 21:00:00+00:00,2019-04-05 01:45:00+00:00,2019-06-13 16:00:00+00:00,226.208333,52.197917,69.593750
2,station02,2018-07-22 16:00:00+00:00,2019-02-17 15:00:00+00:00,2019-04-07 01:45:00+00:00,2019-06-10 16:00:00+00:00,209.958333,48.447917,64.593750
3,station03,2019-01-11 16:00:00+00:00,2019-04-21 03:00:00+00:00,2019-05-14 01:45:00+00:00,2019-06-13 16:00:00+00:00,99.458333,22.947917,30.593750
4,station04,2018-06-30 16:00:00+00:00,2019-02-11 21:00:00+00:00,2019-04-05 01:45:00+00:00,2019-06-13 16:00:00+00:00,226.208333,52.197917,69.593750
5,station05,2019-03-04 16:00:00+00:00,2019-05-09 07:45:00+00:00,2019-05-24 11:15:00+00:00,2019-06-13 16:00:00+00:00,65.656250,15.145833,20.197917
6,station06,2018-07-13 16:00:00+00:00,2019-02-16 10:00:00+00:00,2019-04-07 16:00:00+00:00,2019-06-13 16:00:00+00:00,217.750000,50.250000,67.000000
7,station07,2018-06-30 16:00:00+00:00,2019-02-11 21:00:00+00:00,2019-04-05 01:45:00+00:00,2019-06-13 16:00:00+00:00,226.208333,52.197917,69.593750
8,station08,2018-06-30 16:00:00+00:00,2019-02-11 21:00:00+00:00,2019-04-05 01:45:00+00:00,2019-06-13 16:00:00+00:00,226.208333,52.197917,69.593750
9,station09,2018-09-25 16:00:00+00:00,2019-03-14 07:45:00+00:00,2019-04-22 11:15:00+00:00,2019-06-13 16:00:00+00:00,169.656250,39.145833,52.197917



GAP-AWARE ELIGIBLE SAMPLE COUNTS


split,station,horizon_min,train,validation,test
0,station00,15,18555,4349,5798
1,station00,30,18553,4349,5798
2,station00,60,18549,4349,5798
3,station01,15,21619,5011,6681
4,station01,30,21618,5011,6681
5,station01,60,21616,5011,6681
6,station02,15,19674,4651,5719
7,station02,30,19672,4651,5717
8,station02,60,19668,4651,5713
9,station03,15,9451,2203,2937



HELD-OUT TEST COVERAGE


,station,horizon_min,samples,target_utc_days,target_china_days,target_start_utc,target_end_utc
0,station00,15,5798,61,61,2019-04-14 06:30:00+00:00,2019-06-13 15:45:00+00:00
1,station00,30,5798,61,61,2019-04-14 06:30:00+00:00,2019-06-13 15:45:00+00:00
2,station00,60,5798,61,61,2019-04-14 06:30:00+00:00,2019-06-13 15:45:00+00:00
3,station01,15,6681,70,70,2019-04-05 01:45:00+00:00,2019-06-13 15:45:00+00:00
4,station01,30,6681,70,70,2019-04-05 01:45:00+00:00,2019-06-13 15:45:00+00:00
5,station01,60,6681,70,70,2019-04-05 01:45:00+00:00,2019-06-13 15:45:00+00:00
6,station02,15,5719,62,60,2019-04-07 01:45:00+00:00,2019-06-10 15:45:00+00:00
7,station02,30,5717,62,60,2019-04-07 01:45:00+00:00,2019-06-10 15:45:00+00:00
8,station02,60,5713,62,60,2019-04-07 01:45:00+00:00,2019-06-10 15:45:00+00:00
9,station03,15,2937,31,31,2019-05-14 01:45:00+00:00,2019-06-13 15:45:00+00:00



MINIMUM ELIGIBILITY DIAGNOSTICS
TRAIN      minimum samples = 6203
VALIDATION minimum samples = 1454
TEST       minimum samples = 1939

Minimum held-out China-local test days: 21
Maximum held-out China-local test days: 70

PVOD-02A CANDIDATE SPLIT AUDIT: PASSED

Candidate split proportions: 65% / 15% / 20%.
Split assignment uses target valid time.
Recent physical history window: 60 minutes.
Daily PV lag: 24 hours.
Horizon mapping: 15 / 30 / 60 minutes.
All temporal gaps preserved.
No eligible sample crosses a missing timestamp.

No feature family generated.
No scaler fitted.
No model trained.
No F3/F4 result observed.


In [6]:
# ============================================================
# PVOD-02B
# Freeze external-validation feature protocol
#
# PURPOSE
# -------
# Define exactly which PVOD variables belong to the common
# forecasting backbone BEFORE generating F1/F2/F3/F4.
#
# NO astronomical feature is calculated.
# NO model is trained.
# NO F3/F4 result is observed.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Raw PVOD meteorological channels
# ============================================================
#
# Use every released non-target weather channel.
#
# We deliberately do NOT try to manufacture an exact
# one-to-one SOLETE sensor mapping because PVOD has a
# different observational schema.
#
# These variables are identical for every representation
# comparison, so F3 vs F4 differs only in astronomical /
# Panchang representation.
# ============================================================

PVOD_METEOROLOGICAL_COLUMNS = [

    # Numerical weather prediction channels
    "nwp_globalirrad",
    "nwp_directirrad",
    "nwp_temperature",
    "nwp_humidity",
    "nwp_windspeed",
    "nwp_winddirection",
    "nwp_pressure",

    # Local measurement data channels
    "lmd_totalirrad",
    "lmd_diffuseirrad",
    "lmd_temperature",
    "lmd_pressure",
    "lmd_winddirection",
    "lmd_windspeed",
]


PVOD_TARGET_COLUMN = (
    "power_MW"
)


PVOD_NORMALIZED_TARGET_COLUMN = (
    "power_pu"
)


# ============================================================
# 2. Dynamic history protocol
# ============================================================
#
# All weather channels and current PV power are available
# through the preceding physical hour:
#
#     t-60, t-45, t-30, t-15, t
#
# This preserves the same one-hour physical history concept
# used in SOLETE despite the different sampling cadence.
# ============================================================

PVOD_DYNAMIC_SOURCE_COLUMNS = [
    *PVOD_METEOROLOGICAL_COLUMNS,
    PVOD_TARGET_COLUMN,
]


PVOD_HISTORY_OFFSETS_MIN = [
    60,
    45,
    30,
    15,
    0,
]


# Same-time previous-day PV reference.
PVOD_DAILY_POWER_LAG_MIN = (
    24
    *
    60
)


# ============================================================
# 3. Deterministic representation blocks
# ============================================================
#
# F0_external
#   PVOD dynamic history + previous-day PV
#
# F1_external
#   F0 + Gregorian temporal representation
#
# F2_external
#   F1 + deterministic solar geometry
#
# F3_external
#   F2 + continuous lunar / Sun-Moon geometry
#
# F4_external
#   F3 + categorical Panchang one-hot representation
#
# Important:
# F4 remains an augmentation of F3, matching the original
# project hierarchy rather than replacing F3.
# ============================================================

PVOD_FEATURE_BLOCKS = {

    "F0": {
        "description":
            "PVOD meteorology + PV history + previous-day PV",

        "dynamic_sources":
            PVOD_DYNAMIC_SOURCE_COLUMNS,

        "history_offsets_min":
            PVOD_HISTORY_OFFSETS_MIN,

        "daily_power_lag_min":
            PVOD_DAILY_POWER_LAG_MIN,
    },

    "F1": {
        "parent":
            "F0",

        "description":
            "F0 + Gregorian temporal features",
    },

    "F2": {
        "parent":
            "F1",

        "description":
            "F1 + deterministic solar geometry",
    },

    "F3": {
        "parent":
            "F2",

        "description":
            "F2 + continuous lunar/Sun-Moon geometry",
    },

    "F4": {
        "parent":
            "F3",

        "description":
            "F3 + categorical Panchang one-hot features",
    },
}


# ============================================================
# 4. Frozen astronomy conventions
# ============================================================
#
# Geometry is evaluated at TARGET VALID TIME.
#
# Because PVOD is exactly 15-minute sampled, we do not carry
# over SOLETE's +2-minute interval-midpoint correction.
# That +2-minute offset was specific to the validated
# SOLETE 5-minute timestamp convention.
#
# For PVOD:
#
#     astronomical reference time = target valid timestamp
#
# The same reference time will be used for both F3 and F4.
# ============================================================

PVOD_ASTRONOMY_REFERENCE = (
    "target_valid_time"
)


PVOD_ASTRONOMY_OFFSET_MIN = (
    0
)


PVOD_SIDEREAL_CONVENTION = (
    "Lahiri"
)


PVOD_EPHEMERIS = (
    "JPL DE440s"
)


# ============================================================
# 5. Site geometry from metadata
# ============================================================

pvod_site_registry = (
    metadata[
        [
            "Station_ID",
            "Latitude",
            "Longitude",
            "Capacity",
        ]
    ]
    .rename(
        columns={
            "Station_ID":
                "station",

            "Latitude":
                "latitude_deg",

            "Longitude":
                "longitude_deg",

            "Capacity":
                "capacity_kW",
        }
    )
    .copy()
)


pvod_site_registry[
    "capacity_MW"
] = (
    pvod_site_registry[
        "capacity_kW"
    ]
    /
    1000.0
)


# ============================================================
# 6. Schema validation against every cleaned station
# ============================================================

schema_audit_rows = []


required_columns = set(
    [
        *PVOD_METEOROLOGICAL_COLUMNS,
        PVOD_TARGET_COLUMN,
        PVOD_NORMALIZED_TARGET_COLUMN,
        "timestamp_utc",
        "timestamp_china",
        "capacity_MW",
    ]
)


for station_id, df in pvod_clean.items():

    missing_columns = sorted(
        required_columns
        -
        set(
            df.columns
        )
    )


    nonfinite_counts = {}


    for column in [
        *PVOD_METEOROLOGICAL_COLUMNS,
        PVOD_TARGET_COLUMN,
    ]:

        values = (
            pd.to_numeric(
                df[
                    column
                ],
                errors="coerce",
            )
            .to_numpy(
                dtype=float
            )
        )


        nonfinite_counts[
            column
        ] = int(
            (
                ~np.isfinite(
                    values
                )
            )
            .sum()
        )


    schema_audit_rows.append(
        {
            "station":
                station_id,

            "rows":
                int(
                    len(df)
                ),

            "required_columns_present":
                (
                    len(
                        missing_columns
                    )
                    ==
                    0
                ),

            "missing_columns":
                missing_columns,

            "nonfinite_weather_or_power":
                int(
                    sum(
                        nonfinite_counts.values()
                    )
                ),

            "latitude_deg":
                float(
                    pvod_site_registry
                    .set_index(
                        "station"
                    )
                    .loc[
                        station_id,
                        "latitude_deg",
                    ]
                ),

            "longitude_deg":
                float(
                    pvod_site_registry
                    .set_index(
                        "station"
                    )
                    .loc[
                        station_id,
                        "longitude_deg",
                    ]
                ),
        }
    )


pvod_feature_schema_audit = pd.DataFrame(
    schema_audit_rows
)


# ============================================================
# 7. Hard protocol assertions
# ============================================================

assert len(
    PVOD_METEOROLOGICAL_COLUMNS
) == 13


assert len(
    PVOD_DYNAMIC_SOURCE_COLUMNS
) == 14


assert (
    pvod_feature_schema_audit[
        "required_columns_present"
    ]
    .all()
)


assert (
    pvod_feature_schema_audit[
        "nonfinite_weather_or_power"
    ]
    .eq(0)
    .all()
)


assert len(
    pvod_site_registry
) == 10


assert (
    pvod_site_registry[
        [
            "latitude_deg",
            "longitude_deg",
        ]
    ]
    .notna()
    .all()
    .all()
)


# ============================================================
# 8. Dimension bookkeeping before F1/F2/F3/F4
# ============================================================

dynamic_history_dimension = (
    len(
        PVOD_DYNAMIC_SOURCE_COLUMNS
    )
    *
    len(
        PVOD_HISTORY_OFFSETS_MIN
    )
)


daily_lag_dimension = 1


f0_dimension_expected = (
    dynamic_history_dimension
    +
    daily_lag_dimension
)


dimension_registry = pd.DataFrame(
    [
        {
            "component":
                "dynamic history",

            "sources":
                len(
                    PVOD_DYNAMIC_SOURCE_COLUMNS
                ),

            "time_points":
                len(
                    PVOD_HISTORY_OFFSETS_MIN
                ),

            "expected_columns":
                dynamic_history_dimension,
        },

        {
            "component":
                "previous-day PV",

            "sources":
                1,

            "time_points":
                1,

            "expected_columns":
                daily_lag_dimension,
        },
    ]
)


# ============================================================
# 9. Display frozen protocol
# ============================================================

print(
    "=" * 120
)

print(
    "PVOD EXTERNAL FEATURE PROTOCOL"
)

print(
    "=" * 120
)


print(
    "\nMeteorological channels:",
    len(
        PVOD_METEOROLOGICAL_COLUMNS
    )
)


for column in (
    PVOD_METEOROLOGICAL_COLUMNS
):

    print(
        " ",
        column
    )


print(
    "\nDynamic history sources:",
    len(
        PVOD_DYNAMIC_SOURCE_COLUMNS
    )
)


print(
    "History offsets (minutes):",
    PVOD_HISTORY_OFFSETS_MIN
)


print(
    "Previous-day PV lag (minutes):",
    PVOD_DAILY_POWER_LAG_MIN
)


print(
    "Expected F0 dimension:",
    f0_dimension_expected
)


print(
    "\nAstronomy reference:",
    PVOD_ASTRONOMY_REFERENCE
)


print(
    "Astronomy offset minutes:",
    PVOD_ASTRONOMY_OFFSET_MIN
)


print(
    "Ephemeris:",
    PVOD_EPHEMERIS
)


print(
    "Sidereal convention:",
    PVOD_SIDEREAL_CONVENTION
)


print(
    "\n"
    + "=" * 120
)

print(
    "SITE REGISTRY"
)

print(
    "=" * 120
)


display(
    pvod_site_registry
)


print(
    "\n"
    + "=" * 120
)

print(
    "FEATURE SCHEMA AUDIT"
)

print(
    "=" * 120
)


display(
    pvod_feature_schema_audit
)


print(
    "\n"
    + "=" * 120
)

print(
    "F0 DIMENSION BOOKKEEPING"
)

print(
    "=" * 120
)


display(
    dimension_registry
)


print(
    "\nPVOD-02B EXTERNAL FEATURE PROTOCOL: FROZEN"
)


print(
    "\nNo astronomy calculated."
)

print(
    "No Panchang calculated."
)

print(
    "No scaler fitted."
)

print(
    "No model trained."
)

print(
    "No F3/F4 result observed."
)

PVOD EXTERNAL FEATURE PROTOCOL

Meteorological channels: 13
  nwp_globalirrad
  nwp_directirrad
  nwp_temperature
  nwp_humidity
  nwp_windspeed
  nwp_winddirection
  nwp_pressure
  lmd_totalirrad
  lmd_diffuseirrad
  lmd_temperature
  lmd_pressure
  lmd_winddirection
  lmd_windspeed

Dynamic history sources: 14
History offsets (minutes): [60, 45, 30, 15, 0]
Previous-day PV lag (minutes): 1440
Expected F0 dimension: 71

Astronomy reference: target_valid_time
Astronomy offset minutes: 0
Ephemeris: JPL DE440s
Sidereal convention: Lahiri

SITE REGISTRY


,station,latitude_deg,longitude_deg,capacity_kW,capacity_MW
0,station00,38.047780,114.951390,6600,6.6
1,station01,38.183060,117.457220,20000,20.0
2,station02,38.057280,114.198870,17000,17.0
3,station03,38.109560,114.114940,20000,20.0
4,station04,39.515500,114.867670,20000,20.0
5,station05,38.235500,114.123600,35000,35.0
6,station06,36.898910,114.548410,15000,15.0
7,station07,36.644030,113.641870,20000,20.0
8,station08,36.707610,113.899990,20000,20.0
9,station09,38.731417,115.059855,20000,20.0



FEATURE SCHEMA AUDIT


,station,rows,required_columns_present,missing_columns,nonfinite_weather_or_power,latitude_deg,longitude_deg
0,station00,28896,True,[],0,38.047780,114.951390
1,station01,33408,True,[],0,38.183060,117.457220
2,station02,30432,True,[],0,38.057280,114.198870
3,station03,14688,True,[],0,38.109560,114.114940
4,station04,33408,True,[],0,39.515500,114.867670
5,station05,9696,True,[],0,38.235500,114.123600
6,station06,31104,True,[],0,36.898910,114.548410
7,station07,32928,True,[],0,36.644030,113.641870
8,station08,33120,True,[],0,36.707610,113.899990
9,station09,24288,True,[],0,38.731417,115.059855



F0 DIMENSION BOOKKEEPING


,component,sources,time_points,expected_columns
0,dynamic history,14,5,70
1,previous-day PV,1,1,1



PVOD-02B EXTERNAL FEATURE PROTOCOL: FROZEN

No astronomy calculated.
No Panchang calculated.
No scaler fitted.
No model trained.
No F3/F4 result observed.


In [7]:
# ============================================================
# PVOD-03A
# Deterministic target-time representation generation
#
# Builds reusable lookup tables for:
#
#   F1 : Gregorian UTC features              15
#   F2 : site-specific solar geometry         5
#   F3 : continuous lunar/Sun-Moon geometry   9
#   F4 : canonical Panchang one-hot           97
#
# IMPORTANT
# ---------
# Astronomical reference time:
#
#     exact TARGET VALID TIME
#
# There is NO SOLETE +2-minute offset.
#
# No train/validation/test values are used to fit anything.
# No scaler is fitted.
# No model is trained.
# No forecasting result is computed.
# ============================================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import pvlib


# ============================================================
# 1. Locate validated project modules
# ============================================================

astronomy_matches = list(
    PROJECT_ROOT.rglob(
        "astronomy_features.py"
    )
)

vedic_matches = list(
    PROJECT_ROOT.rglob(
        "vedic_calendar.py"
    )
)

preprocessing_matches = list(
    PROJECT_ROOT.rglob(
        "preprocessing.py"
    )
)


assert len(astronomy_matches) >= 1, (
    "astronomy_features.py not found"
)

assert len(vedic_matches) >= 1, (
    "vedic_calendar.py not found"
)

assert len(preprocessing_matches) >= 1, (
    "preprocessing.py not found"
)


ASTRONOMY_MODULE_PATH = (
    astronomy_matches[0]
)

VEDIC_MODULE_PATH = (
    vedic_matches[0]
)

PREPROCESSING_MODULE_PATH = (
    preprocessing_matches[0]
)


for module_path in [
    ASTRONOMY_MODULE_PATH,
    VEDIC_MODULE_PATH,
    PREPROCESSING_MODULE_PATH,
]:

    parent = str(
        module_path.parent
    )

    if parent not in sys.path:
        sys.path.insert(
            0,
            parent,
        )


print(
    "astronomy_features.py:",
    ASTRONOMY_MODULE_PATH.relative_to(
        PROJECT_ROOT
    ),
)

print(
    "vedic_calendar.py:",
    VEDIC_MODULE_PATH.relative_to(
        PROJECT_ROOT
    ),
)

print(
    "preprocessing.py:",
    PREPROCESSING_MODULE_PATH.relative_to(
        PROJECT_ROOT
    ),
)


# ============================================================
# 2. Import ONLY reusable validated primitives
# ============================================================

from preprocessing import (
    build_gregorian_features,
)

from astronomy_features import (
    load_jpl_ephemeris,
    skyfield_times_from_utc,
    apparent_tropical_ecliptic_longitudes,
    moon_sun_phase_angle_deg,
    phase_cyclic_features,
    longitude_cyclic_features,
    moon_ecliptic_latitude_and_distance,
    moon_illumination_fraction,
)

from vedic_calendar import (
    lahiri_ayanamsha_for_utc_times,
    build_panchang_f4_raw,
    KARANA_SEQUENCE,
    NAKSHATRA_NAMES,
    YOGA_NAMES,
)


# ============================================================
# 3. Locate DE440s ephemeris
# ============================================================

ephemeris_matches = list(
    PROJECT_ROOT.rglob(
        "de440s.bsp"
    )
)


assert len(ephemeris_matches) >= 1, (
    "de440s.bsp not found in project."
)


EPHEMERIS_PATH = (
    ephemeris_matches[0]
)


print(
    "\nEphemeris:",
    EPHEMERIS_PATH.relative_to(
        PROJECT_ROOT
    ),
)


ephemeris = load_jpl_ephemeris(
    EPHEMERIS_PATH
)


# ============================================================
# 4. Global unique PVOD timestamps
#
# Gregorian, lunar geometry and Panchang are independent of
# PV station identity in this project's instantaneous
# representation.
#
# Solar elevation/azimuth will remain site-specific.
# ============================================================

all_timestamp_values = set()


for station_id, df in (
    pvod_clean.items()
):

    all_timestamp_values.update(
        pd.DatetimeIndex(
            df[
                "timestamp_utc"
            ]
        )
    )


pvod_global_times = pd.DatetimeIndex(
    sorted(
        all_timestamp_values
    )
)


assert pvod_global_times.tz is not None

assert (
    pvod_global_times
    .is_monotonic_increasing
)

assert (
    pvod_global_times
    .duplicated()
    .sum()
    ==
    0
)


print(
    "\nUnique global PVOD timestamps:",
    len(
        pvod_global_times
    )
)

print(
    "Global start:",
    pvod_global_times.min()
)

print(
    "Global end:",
    pvod_global_times.max()
)


# ============================================================
# 5. F1 — canonical Gregorian UTC features
# ============================================================

pvod_f1_global = (
    build_gregorian_features(
        pvod_global_times
    )
)


assert (
    pvod_f1_global.index
    .equals(
        pvod_global_times
    )
)


assert (
    pvod_f1_global.shape[1]
    ==
    15
), (
    f"Expected 15 F1 columns, "
    f"found {pvod_f1_global.shape[1]}"
)


# ============================================================
# 6. Exact-time Skyfield geometry
#
# DO NOT call build_lunar_f3_features(), because that
# high-level function applies SOLETE's +2-minute offset.
#
# Here Skyfield receives exact PVOD target-valid timestamps.
# ============================================================

pvod_skyfield_times = (
    skyfield_times_from_utc(
        pvod_global_times
    )
)


(
    sun_tropical_deg,
    moon_tropical_deg,
) = (
    apparent_tropical_ecliptic_longitudes(
        ephemeris,
        pvod_skyfield_times,
    )
)


sun_tropical_series = pd.Series(
    sun_tropical_deg,
    index=pvod_global_times,
    name="sun_tropical_longitude_deg",
)


moon_tropical_series = pd.Series(
    moon_tropical_deg,
    index=pvod_global_times,
    name="moon_tropical_longitude_deg",
)


# ============================================================
# 7. F3 — nine continuous lunar/Sun-Moon features
# ============================================================

phase_angle_deg = (
    moon_sun_phase_angle_deg(
        sun_tropical_deg,
        moon_tropical_deg,
    )
)


(
    phase_sin,
    phase_cos,
) = phase_cyclic_features(
    phase_angle_deg
)


(
    moon_longitude_sin,
    moon_longitude_cos,
) = longitude_cyclic_features(
    moon_tropical_deg
)


(
    sun_longitude_sin,
    sun_longitude_cos,
) = longitude_cyclic_features(
    sun_tropical_deg
)


(
    moon_latitude_deg,
    moon_distance_km,
) = (
    moon_ecliptic_latitude_and_distance(
        ephemeris,
        pvod_skyfield_times,
    )
)


moon_illumination = (
    moon_illumination_fraction(
        ephemeris,
        pvod_skyfield_times,
    )
)


pvod_f3_global = pd.DataFrame(
    {
        "moon_phase_sin":
            phase_sin,

        "moon_phase_cos":
            phase_cos,

        "moon_longitude_sin":
            moon_longitude_sin,

        "moon_longitude_cos":
            moon_longitude_cos,

        "sun_ecliptic_longitude_sin":
            sun_longitude_sin,

        "sun_ecliptic_longitude_cos":
            sun_longitude_cos,

        "moon_ecliptic_latitude_deg":
            moon_latitude_deg,

        "moon_illumination_fraction":
            moon_illumination,

        "earth_moon_distance_km":
            moon_distance_km,
    },

    index=pvod_global_times,
)


assert (
    pvod_f3_global.shape[1]
    ==
    9
)


# ============================================================
# 8. Canonical extended Lahiri ayanamsha
# ============================================================

pvod_ayanamsha_deg = pd.Series(
    lahiri_ayanamsha_for_utc_times(
        pvod_global_times
    ),
    index=pvod_global_times,
    name="lahiri_ayanamsha_deg",
)


# ============================================================
# 9. F4 raw Panchang categories
# ============================================================

pvod_f4_raw_global = (
    build_panchang_f4_raw(
        sun_tropical_series,
        moon_tropical_series,
        pvod_ayanamsha_deg,
    )
)


assert list(
    pvod_f4_raw_global.columns
) == [
    "tithi",
    "paksha",
    "karana",
    "nakshatra",
    "yoga",
]


# ============================================================
# 10. Explicit canonical category registries
#
# Full category sets are frozen BEFORE modelling so every
# station/split receives exactly the same 97 dummy columns,
# even if a particular station period does not observe every
# category.
# ============================================================

TITHI_LEVELS = list(
    range(
        1,
        31,
    )
)


PAKSHA_LEVELS = [
    "Shukla",
    "Krishna",
]


# Preserve first occurrence of each traditional Karana name.
KARANA_LEVELS = list(
    dict.fromkeys(
        KARANA_SEQUENCE
    )
)


NAKSHATRA_LEVELS = list(
    NAKSHATRA_NAMES
)


YOGA_LEVELS = list(
    YOGA_NAMES
)


assert len(
    TITHI_LEVELS
) == 30

assert len(
    PAKSHA_LEVELS
) == 2

assert len(
    KARANA_LEVELS
) == 11

assert len(
    NAKSHATRA_LEVELS
) == 27

assert len(
    YOGA_LEVELS
) == 27


PANCHANG_LEVELS = {
    "tithi":
        TITHI_LEVELS,

    "paksha":
        PAKSHA_LEVELS,

    "karana":
        KARANA_LEVELS,

    "nakshatra":
        NAKSHATRA_LEVELS,

    "yoga":
        YOGA_LEVELS,
}


# ============================================================
# 11. Canonical 97-column one-hot F4 representation
# ============================================================

f4_dummy_blocks = []


for family, levels in (
    PANCHANG_LEVELS.items()
):

    categorical = pd.Categorical(
        pvod_f4_raw_global[
            family
        ],
        categories=levels,
        ordered=True,
    )


    dummy = pd.get_dummies(
        categorical,
        prefix=family,
        dtype=np.int8,
    )


    dummy.index = (
        pvod_global_times
    )


    assert (
        dummy.shape[1]
        ==
        len(
            levels
        )
    )


    # Exactly one active level per Panchang family.
    assert (
        dummy.sum(
            axis=1
        )
        .eq(1)
        .all()
    )


    f4_dummy_blocks.append(
        dummy
    )


pvod_f4_onehot_global = pd.concat(
    f4_dummy_blocks,
    axis=1,
)


assert (
    pvod_f4_onehot_global.shape[1]
    ==
    97
), (
    f"Expected 97 Panchang dummy columns, "
    f"found {pvod_f4_onehot_global.shape[1]}"
)


assert (
    pvod_f4_onehot_global
    .sum(
        axis=1
    )
    .eq(5)
    .all()
), (
    "Each timestamp should activate exactly "
    "one level in each of five Panchang families."
)


# ============================================================
# 12. F2 — site-specific solar geometry
#
# Unlike F1/F3/F4, solar elevation and azimuth depend on
# station latitude/longitude.
# ============================================================

site_lookup = (
    pvod_site_registry
    .set_index(
        "station"
    )
)


pvod_f2_by_station = {}


for station_id, df in (
    pvod_clean.items()
):

    timestamps = pd.DatetimeIndex(
        df[
            "timestamp_utc"
        ]
    )


    latitude_deg = float(
        site_lookup.loc[
            station_id,
            "latitude_deg",
        ]
    )


    longitude_deg = float(
        site_lookup.loc[
            station_id,
            "longitude_deg",
        ]
    )


    solar_position = (
        pvlib.solarposition
        .get_solarposition(
            time=timestamps,
            latitude=latitude_deg,
            longitude=longitude_deg,
            method="nrel_numpy",
        )
    )


    solar_elevation_deg = (
        solar_position[
            "elevation"
        ]
        .to_numpy(
            dtype=float
        )
    )


    solar_zenith_deg = (
        solar_position[
            "zenith"
        ]
        .to_numpy(
            dtype=float
        )
    )


    solar_azimuth_deg = (
        solar_position[
            "azimuth"
        ]
        .to_numpy(
            dtype=float
        )
    )


    solar_azimuth_rad = (
        np.deg2rad(
            solar_azimuth_deg
        )
    )


    f2 = pd.DataFrame(
        {
            "solar_elevation_deg":
                solar_elevation_deg,

            "solar_cos_zenith":
                np.cos(
                    np.deg2rad(
                        solar_zenith_deg
                    )
                ),

            "solar_azimuth_sin":
                np.sin(
                    solar_azimuth_rad
                ),

            "solar_azimuth_cos":
                np.cos(
                    solar_azimuth_rad
                ),

            "solar_daylight":
                (
                    solar_elevation_deg
                    >
                    0.0
                )
                .astype(
                    np.int8
                ),
        },

        index=timestamps,
    )


    assert (
        f2.shape
        ==
        (
            len(
                timestamps
            ),
            5,
        )
    )


    assert np.isfinite(
        f2.to_numpy(
            dtype=float
        )
    ).all()


    pvod_f2_by_station[
        station_id
    ] = f2


# ============================================================
# 13. Global finite-value checks
# ============================================================

assert np.isfinite(
    pvod_f1_global
    .to_numpy(
        dtype=float
    )
).all()


assert np.isfinite(
    pvod_f3_global
    .to_numpy(
        dtype=float
    )
).all()


assert np.isfinite(
    pvod_ayanamsha_deg
    .to_numpy(
        dtype=float
    )
).all()


assert (
    pvod_f4_onehot_global
    .isna()
    .sum()
    .sum()
    ==
    0
)


# ============================================================
# 14. Panchang observed-level audit
# ============================================================

panchang_level_audit_rows = []


for family, levels in (
    PANCHANG_LEVELS.items()
):

    observed = (
        pvod_f4_raw_global[
            family
        ]
        .astype(
            object
        )
        .dropna()
        .unique()
        .tolist()
    )


    panchang_level_audit_rows.append(
        {
            "family":
                family,

            "canonical_levels":
                len(
                    levels
                ),

            "observed_levels":
                len(
                    observed
                ),

            "missing_canonical_levels":
                [
                    level
                    for level
                    in levels
                    if level
                    not in observed
                ],
        }
    )


pvod_panchang_level_audit = pd.DataFrame(
    panchang_level_audit_rows
)


# ============================================================
# 15. Feature-dimension registry
# ============================================================

PVOD_EXPECTED_DIMS = {
    "F0": 71,
    "F1": 71 + 15,
    "F2": 71 + 15 + 5,
    "F3": 71 + 15 + 5 + 9,
    "F4": 71 + 15 + 5 + 9 + 97,
}


assert PVOD_EXPECTED_DIMS == {
    "F0": 71,
    "F1": 86,
    "F2": 91,
    "F3": 100,
    "F4": 197,
}


pvod_dimension_registry = pd.DataFrame(
    {
        "feature_block":
            list(
                PVOD_EXPECTED_DIMS.keys()
            ),

        "expected_dimension":
            list(
                PVOD_EXPECTED_DIMS.values()
            ),
    }
)


# ============================================================
# 16. Display audit
# ============================================================

print(
    "=" * 120
)

print(
    "PVOD DETERMINISTIC REPRESENTATION AUDIT"
)

print(
    "=" * 120
)


print(
    "\nGlobal timestamps:",
    len(
        pvod_global_times
    )
)


print(
    "F1 Gregorian columns:",
    pvod_f1_global.shape[1]
)


print(
    "F3 continuous astronomy columns:",
    pvod_f3_global.shape[1]
)


print(
    "F4 raw Panchang columns:",
    pvod_f4_raw_global.shape[1]
)


print(
    "F4 one-hot columns:",
    pvod_f4_onehot_global.shape[1]
)


print(
    "Site-specific F2 tables:",
    len(
        pvod_f2_by_station
    )
)


print(
    "\n"
    + "=" * 120
)

print(
    "PANCHANG LEVEL AUDIT"
)

print(
    "=" * 120
)


display(
    pvod_panchang_level_audit
)


print(
    "\n"
    + "=" * 120
)

print(
    "PVOD FEATURE DIMENSION REGISTRY"
)

print(
    "=" * 120
)


display(
    pvod_dimension_registry
)


print(
    "\n"
    + "=" * 120
)

print(
    "SAMPLE RAW PANCHANG"
)

print(
    "=" * 120
)


display(
    pvod_f4_raw_global.head(
        10
    )
)


print(
    "\nPVOD-03A DETERMINISTIC REPRESENTATIONS: PASSED"
)


print(
    "\nAstronomy reference = exact target valid time."
)

print(
    "SOLETE +2-minute offset was NOT used."
)

print(
    "Canonical F4 one-hot dimension = 97."
)

print(
    "No train/test information used to construct deterministic features."
)

print(
    "No scaler fitted."
)

print(
    "No model trained."
)

print(
    "No F3/F4 forecasting result observed."
)

astronomy_features.py: src/astronomy_features.py
vedic_calendar.py: src/vedic_calendar.py
preprocessing.py: src/preprocessing.py

Ephemeris: ephemeris/de440s.bsp

Unique global PVOD timestamps: 33408
Global start: 2018-06-30 16:00:00+00:00
Global end: 2019-06-13 15:45:00+00:00
PVOD DETERMINISTIC REPRESENTATION AUDIT

Global timestamps: 33408
F1 Gregorian columns: 15
F3 continuous astronomy columns: 9
F4 raw Panchang columns: 5
F4 one-hot columns: 97
Site-specific F2 tables: 10

PANCHANG LEVEL AUDIT


,family,canonical_levels,observed_levels,missing_canonical_levels
0,tithi,30,30,[]
1,paksha,2,2,[]
2,karana,11,11,[]
3,nakshatra,27,27,[]
4,yoga,27,27,[]



PVOD FEATURE DIMENSION REGISTRY


,feature_block,expected_dimension
0,F0,71
1,F1,86
2,F2,91
3,F3,100
4,F4,197



SAMPLE RAW PANCHANG


,tithi,paksha,karana,nakshatra,yoga
2018-06-30 16:00:00+00:00,18,Krishna,Vanija,Shravana,Vaidhriti
2018-06-30 16:15:00+00:00,18,Krishna,Vanija,Shravana,Vaidhriti
2018-06-30 16:30:00+00:00,18,Krishna,Vanija,Shravana,Vaidhriti
2018-06-30 16:45:00+00:00,18,Krishna,Vanija,Shravana,Vaidhriti
2018-06-30 17:00:00+00:00,18,Krishna,Vanija,Shravana,Vaidhriti
2018-06-30 17:15:00+00:00,18,Krishna,Vanija,Shravana,Vaidhriti
2018-06-30 17:30:00+00:00,18,Krishna,Vanija,Shravana,Vaidhriti
2018-06-30 17:45:00+00:00,18,Krishna,Vanija,Shravana,Vaidhriti
2018-06-30 18:00:00+00:00,18,Krishna,Vanija,Shravana,Vaidhriti
2018-06-30 18:15:00+00:00,18,Krishna,Vanija,Shravana,Vaidhriti



PVOD-03A DETERMINISTIC REPRESENTATIONS: PASSED

Astronomy reference = exact target valid time.
SOLETE +2-minute offset was NOT used.
Canonical F4 one-hot dimension = 97.
No train/test information used to construct deterministic features.
No scaler fitted.
No model trained.
No F3/F4 forecasting result observed.


In [8]:
# ============================================================
# PVOD-03B
# Gap-safe forecasting sample assembly
#
# PURPOSE
# -------
# Build:
#
#   1. one reusable F0 history table per station;
#   2. one horizon-specific sample-index table per
#      station x horizon;
#   3. a lazy materializer for F0/F1/F2/F3/F4 matrices.
#
# PRIMARY EXTERNAL TARGET
# -----------------------
#     power_pu = power_MW / nominal metadata capacity_MW
#
# Raw MW target is retained as a secondary quantity.
#
# IMPORTANT
# ---------
# - no missing timestamp is filled;
# - no sample crosses a required missing history timestamp;
# - every future 15-min point up to the target must exist;
# - split assignment uses TARGET VALID TIME;
# - F1/F2/F3/F4 attach at TARGET VALID TIME;
# - no scaler is fitted;
# - no model is trained;
# - no F3/F4 performance result is observed.
# ============================================================

import gc

import numpy as np
import pandas as pd


# ============================================================
# 1. Freeze target convention
# ============================================================

PVOD_PRIMARY_TARGET = (
    "power_pu"
)

PVOD_SECONDARY_TARGET = (
    "power_MW"
)


print(
    "Primary modelling target:",
    PVOD_PRIMARY_TARGET,
)

print(
    "Secondary physical target:",
    PVOD_SECONDARY_TARGET,
)


# ============================================================
# 2. Reusable indexed station tables
# ============================================================

pvod_indexed = {}


for station_id, df in (
    pvod_clean.items()
):

    indexed = (
        df
        .copy()
        .set_index(
            "timestamp_utc",
            drop=False,
        )
    )


    assert indexed.index.is_unique
    assert indexed.index.is_monotonic_increasing


    pvod_indexed[
        station_id
    ] = indexed


# ============================================================
# 3. Build F0 history table once per station
#
# 14 dynamic sources:
#
#   13 weather variables
#   + current PV power_MW
#
# at:
#
#   t-60
#   t-45
#   t-30
#   t-15
#   t
#
# plus:
#
#   power_MW at t-24h
#
# Expected dimension:
#
#   14 * 5 + 1 = 71
# ============================================================

pvod_f0_by_station = {}

f0_audit_rows = []


for station_id, indexed in (
    pvod_indexed.items()
):

    issue_times = pd.DatetimeIndex(
        indexed.index
    )


    f0 = pd.DataFrame(
        index=issue_times
    )


    # --------------------------------------------------------
    # Dynamic history
    # --------------------------------------------------------

    for source_column in (
        PVOD_DYNAMIC_SOURCE_COLUMNS
    ):

        source_series = (
            indexed[
                source_column
            ]
        )


        for lag_min in (
            PVOD_HISTORY_OFFSETS_MIN
        ):

            required_times = (
                issue_times
                -
                pd.Timedelta(
                    minutes=lag_min
                )
            )


            feature_name = (
                f"{source_column}"
                f"__lag_{lag_min:04d}m"
            )


            f0[
                feature_name
            ] = (
                source_series
                .reindex(
                    required_times
                )
                .to_numpy(
                    dtype=float
                )
            )


    # --------------------------------------------------------
    # Previous-day PV
    # --------------------------------------------------------

    daily_reference_times = (
        issue_times
        -
        pd.Timedelta(
            minutes=
                PVOD_DAILY_POWER_LAG_MIN
        )
    )


    f0[
        "power_MW__lag_1440m"
    ] = (
        indexed[
            "power_MW"
        ]
        .reindex(
            daily_reference_times
        )
        .to_numpy(
            dtype=float
        )
    )


    assert (
        f0.shape[1]
        ==
        PVOD_EXPECTED_DIMS[
            "F0"
        ]
    )


    assert (
        f0.columns.is_unique
    )


    valid_f0_mask = (
        f0
        .notna()
        .all(
            axis=1
        )
    )


    f0_audit_rows.append(
        {
            "station":
                station_id,

            "station_rows":
                int(
                    len(
                        indexed
                    )
                ),

            "F0_columns":
                int(
                    f0.shape[1]
                ),

            "complete_F0_rows":
                int(
                    valid_f0_mask
                    .sum()
                ),

            "incomplete_F0_rows":
                int(
                    (
                        ~valid_f0_mask
                    )
                    .sum()
                ),
        }
    )


    # float32 is sufficient for model design matrices and
    # substantially reduces memory usage.
    pvod_f0_by_station[
        station_id
    ] = (
        f0
        .astype(
            np.float32
        )
    )


pvod_f0_audit = pd.DataFrame(
    f0_audit_rows
)


# ============================================================
# 4. Split-boundary lookup
# ============================================================

split_boundary_lookup = (
    pvod_split_boundaries
    .set_index(
        "station"
    )
)


# ============================================================
# 5. Build horizon-specific sample indices
#
# Each candidate issue time must satisfy:
#
# A. all 71 F0 predictors exist;
#
# B. every future 15-minute point from issue time through
#    target valid time exists;
#
# C. target PV value exists;
#
# D. target time belongs to train / validation / test
#    according to the frozen station-specific boundaries.
# ============================================================

pvod_sample_index = {}

sample_audit_rows = []


for station_id, indexed in (
    pvod_indexed.items()
):

    f0 = (
        pvod_f0_by_station[
            station_id
        ]
    )


    timestamp_set = set(
        indexed.index
    )


    boundary = (
        split_boundary_lookup
        .loc[
            station_id
        ]
    )


    validation_start = (
        boundary[
            "validation_start_utc"
        ]
    )


    test_start = (
        boundary[
            "test_start_utc"
        ]
    )


    stop_exclusive = (
        boundary[
            "station_stop_exclusive_utc"
        ]
    )


    complete_f0_times = (
        f0.index[
            f0
            .notna()
            .all(
                axis=1
            )
        ]
    )


    for horizon_min in sorted(
        PVOD_HORIZONS
    ):

        horizon_delta = (
            PVOD_HORIZONS[
                horizon_min
            ]
        )


        horizon_steps = (
            horizon_min
            //
            15
        )


        assert (
            horizon_steps
            *
            15
            ==
            horizon_min
        )


        records = []


        for issue_time in (
            complete_f0_times
        ):

            # -----------------------------------------------
            # Explicit future-path integrity
            #
            # For 60-min horizon, require:
            #
            #   t+15
            #   t+30
            #   t+45
            #   t+60
            #
            # to all genuinely exist.
            # -----------------------------------------------

            future_times = [
                issue_time
                +
                pd.Timedelta(
                    minutes=
                        15
                        *
                        step
                )

                for step in range(
                    1,
                    horizon_steps + 1,
                )
            ]


            if not all(
                future_time
                in
                timestamp_set

                for future_time
                in future_times
            ):

                continue


            target_time = (
                issue_time
                +
                horizon_delta
            )


            # -----------------------------------------------
            # Split assignment by TARGET VALID TIME
            # -----------------------------------------------

            if (
                target_time
                <
                validation_start
            ):

                split_name = (
                    "train"
                )


            elif (
                target_time
                <
                test_start
            ):

                split_name = (
                    "validation"
                )


            elif (
                target_time
                <
                stop_exclusive
            ):

                split_name = (
                    "test"
                )


            else:

                continue


            target_row = (
                indexed
                .loc[
                    target_time
                ]
            )


            target_power_mw = float(
                target_row[
                    "power_MW"
                ]
            )


            target_power_pu = float(
                target_row[
                    "power_pu"
                ]
            )


            # Target-time solar daylight flag.
            target_daylight = int(
                pvod_f2_by_station[
                    station_id
                ]
                .loc[
                    target_time,
                    "solar_daylight",
                ]
            )


            records.append(
                {
                    "issue_time_utc":
                        issue_time,

                    "target_time_utc":
                        target_time,

                    "split":
                        split_name,

                    "target_power_MW":
                        target_power_mw,

                    "target_power_pu":
                        target_power_pu,

                    "target_solar_daylight":
                        target_daylight,
                }
            )


        sample_table = pd.DataFrame(
            records
        )


        assert len(
            sample_table
        ) > 0


        assert (
            sample_table[
                "issue_time_utc"
            ]
            .is_unique
        )


        assert (
            sample_table[
                "target_time_utc"
            ]
            .is_unique
        )


        assert (
            sample_table[
                "issue_time_utc"
            ]
            .is_monotonic_increasing
        )


        # -----------------------------------------------
        # Verify target timing
        # -----------------------------------------------

        assert (
            (
                sample_table[
                    "target_time_utc"
                ]
                -
                sample_table[
                    "issue_time_utc"
                ]
            )
            .eq(
                horizon_delta
            )
            .all()
        )


        # -----------------------------------------------
        # No nonfinite target values
        # -----------------------------------------------

        assert np.isfinite(
            sample_table[
                [
                    "target_power_MW",
                    "target_power_pu",
                ]
            ]
            .to_numpy(
                dtype=float
            )
        ).all()


        # -----------------------------------------------
        # Preserve >1 per-unit observations.
        # They are allowed by the frozen PVOD policy.
        # -----------------------------------------------

        assert (
            sample_table[
                "target_power_pu"
            ]
            .min()
            >=
            0.0
        )


        # -----------------------------------------------
        # Store
        # -----------------------------------------------

        pvod_sample_index[
            (
                station_id,
                horizon_min,
            )
        ] = sample_table


        split_counts = (
            sample_table[
                "split"
            ]
            .value_counts()
        )


        sample_audit_rows.append(
            {
                "station":
                    station_id,

                "horizon_min":
                    horizon_min,

                "samples_total":
                    int(
                        len(
                            sample_table
                        )
                    ),

                "train":
                    int(
                        split_counts.get(
                            "train",
                            0,
                        )
                    ),

                "validation":
                    int(
                        split_counts.get(
                            "validation",
                            0,
                        )
                    ),

                "test":
                    int(
                        split_counts.get(
                            "test",
                            0,
                        )
                    ),

                "daylight_test":
                    int(
                        (
                            (
                                sample_table[
                                    "split"
                                ]
                                ==
                                "test"
                            )
                            &
                            (
                                sample_table[
                                    "target_solar_daylight"
                                ]
                                ==
                                1
                            )
                        )
                        .sum()
                    ),

                "target_pu_max":
                    float(
                        sample_table[
                            "target_power_pu"
                        ]
                        .max()
                    ),
            }
        )


pvod_sample_matrix_audit = pd.DataFrame(
    sample_audit_rows
)


assert len(
    pvod_sample_index
) == (
    10
    *
    3
)


# ============================================================
# 6. Reconcile against PVOD-02A
#
# PVOD-03B adds the explicit requirement that all intermediate
# future 15-min points exist.
#
# Because observed PVOD gaps are >= 1 day, counts should remain
# identical to the earlier audit.
# ============================================================

earlier_counts = (
    sample_count_pivot
    .set_index(
        [
            "station",
            "horizon_min",
        ]
    )
)


new_counts = (
    pvod_sample_matrix_audit
    .set_index(
        [
            "station",
            "horizon_min",
        ]
    )
)


for split_name in [
    "train",
    "validation",
    "test",
]:

    assert (
        new_counts[
            split_name
        ]
        .astype(int)
        .equals(
            earlier_counts[
                split_name
            ]
            .astype(int)
        )
    ), (
        f"PVOD-03B {split_name} counts "
        "do not match PVOD-02A."
    )


# ============================================================
# 7. Lazy feature-matrix materializer
#
# This avoids storing 30 full F4 matrices simultaneously.
#
# feature_block:
#
#   F0 = history only
#   F1 = F0 + Gregorian(target)
#   F2 = F1 + solar(target, site-specific)
#   F3 = F2 + continuous astronomy(target)
#   F4 = F3 + Panchang one-hot(target)
# ============================================================

def materialize_pvod_design(
    station_id: str,
    horizon_min: int,
    feature_block: str,
    split: str | None = None,
):
    """
    Materialize one PVOD forecasting design matrix.

    Returns
    -------
    dict
        {
            "X": DataFrame,
            "y_pu": Series,
            "y_MW": Series,
            "meta": DataFrame,
        }
    """

    if station_id not in pvod_clean:
        raise KeyError(
            f"Unknown station: {station_id}"
        )


    if horizon_min not in PVOD_HORIZONS:
        raise KeyError(
            f"Unknown horizon: {horizon_min}"
        )


    if feature_block not in (
        PVOD_EXPECTED_DIMS
    ):
        raise KeyError(
            f"Unknown feature block: {feature_block}"
        )


    samples = (
        pvod_sample_index[
            (
                station_id,
                horizon_min,
            )
        ]
        .copy()
    )


    if split is not None:

        if split not in {
            "train",
            "validation",
            "test",
        }:

            raise ValueError(
                f"Unknown split: {split}"
            )


        samples = (
            samples.loc[
                samples[
                    "split"
                ]
                ==
                split
            ]
            .copy()
        )


    samples = (
        samples
        .reset_index(
            drop=True
        )
    )


    issue_times = pd.DatetimeIndex(
        samples[
            "issue_time_utc"
        ]
    )


    target_times = pd.DatetimeIndex(
        samples[
            "target_time_utc"
        ]
    )


    # --------------------------------------------------------
    # F0
    # --------------------------------------------------------

    blocks = [
        (
            pvod_f0_by_station[
                station_id
            ]
            .loc[
                issue_times
            ]
            .copy()
        )
    ]


    # Every final design matrix is indexed by issue time.
    blocks[0].index = (
        issue_times
    )


    # --------------------------------------------------------
    # F1
    # --------------------------------------------------------

    if feature_block in {
        "F1",
        "F2",
        "F3",
        "F4",
    }:

        f1 = (
            pvod_f1_global
            .loc[
                target_times
            ]
            .copy()
        )

        f1.index = (
            issue_times
        )

        blocks.append(
            f1
        )


    # --------------------------------------------------------
    # F2
    # --------------------------------------------------------

    if feature_block in {
        "F2",
        "F3",
        "F4",
    }:

        f2 = (
            pvod_f2_by_station[
                station_id
            ]
            .loc[
                target_times
            ]
            .copy()
        )

        f2.index = (
            issue_times
        )

        blocks.append(
            f2
        )


    # --------------------------------------------------------
    # F3
    # --------------------------------------------------------

    if feature_block in {
        "F3",
        "F4",
    }:

        f3 = (
            pvod_f3_global
            .loc[
                target_times
            ]
            .copy()
        )

        f3.index = (
            issue_times
        )

        blocks.append(
            f3
        )


    # --------------------------------------------------------
    # F4
    # --------------------------------------------------------

    if feature_block == "F4":

        f4 = (
            pvod_f4_onehot_global
            .loc[
                target_times
            ]
            .copy()
        )

        f4.index = (
            issue_times
        )

        blocks.append(
            f4
        )


    # --------------------------------------------------------
    # Combine
    # --------------------------------------------------------

    X = pd.concat(
        blocks,
        axis=1,
    )


    assert (
        X.shape[0]
        ==
        len(
            samples
        )
    )


    assert (
        X.shape[1]
        ==
        PVOD_EXPECTED_DIMS[
            feature_block
        ]
    ), (
        f"{station_id} h={horizon_min} "
        f"{feature_block}: expected "
        f"{PVOD_EXPECTED_DIMS[feature_block]} columns, "
        f"found {X.shape[1]}"
    )


    assert (
        X.columns.is_unique
    )


    assert (
        X.index.equals(
            issue_times
        )
    )


    assert np.isfinite(
        X.to_numpy(
            dtype=float
        )
    ).all()


    # Memory-efficient modelling representation.
    X = X.astype(
        np.float32
    )


    y_pu = pd.Series(
        samples[
            "target_power_pu"
        ]
        .to_numpy(
            dtype=np.float32
        ),
        index=issue_times,
        name="target_power_pu",
    )


    y_MW = pd.Series(
        samples[
            "target_power_MW"
        ]
        .to_numpy(
            dtype=np.float32
        ),
        index=issue_times,
        name="target_power_MW",
    )


    meta = (
        samples
        .set_index(
            "issue_time_utc",
            drop=False,
        )
    )


    return {
        "X":
            X,

        "y_pu":
            y_pu,

        "y_MW":
            y_MW,

        "meta":
            meta,
    }


# ============================================================
# 8. Full design audit
#
# Materialize F4 sequentially for all station x horizon pairs,
# verify 197 columns and finite values, then release each
# temporary matrix immediately.
# ============================================================

design_audit_rows = []


for station_id in sorted(
    pvod_clean
):

    for horizon_min in sorted(
        PVOD_HORIZONS
    ):

        design = (
            materialize_pvod_design(
                station_id=
                    station_id,

                horizon_min=
                    horizon_min,

                feature_block=
                    "F4",
            )
        )


        X = design[
            "X"
        ]


        meta = design[
            "meta"
        ]


        design_audit_rows.append(
            {
                "station":
                    station_id,

                "horizon_min":
                    horizon_min,

                "rows":
                    int(
                        len(
                            X
                        )
                    ),

                "F0_dim":
                    PVOD_EXPECTED_DIMS[
                        "F0"
                    ],

                "F1_dim":
                    PVOD_EXPECTED_DIMS[
                        "F1"
                    ],

                "F2_dim":
                    PVOD_EXPECTED_DIMS[
                        "F2"
                    ],

                "F3_dim":
                    PVOD_EXPECTED_DIMS[
                        "F3"
                    ],

                "F4_dim":
                    int(
                        X.shape[1]
                    ),

                "train":
                    int(
                        (
                            meta[
                                "split"
                            ]
                            ==
                            "train"
                        )
                        .sum()
                    ),

                "validation":
                    int(
                        (
                            meta[
                                "split"
                            ]
                            ==
                            "validation"
                        )
                        .sum()
                    ),

                "test":
                    int(
                        (
                            meta[
                                "split"
                            ]
                            ==
                            "test"
                        )
                        .sum()
                    ),

                "test_daylight":
                    int(
                        (
                            (
                                meta[
                                    "split"
                                ]
                                ==
                                "test"
                            )
                            &
                            (
                                meta[
                                    "target_solar_daylight"
                                ]
                                ==
                                1
                            )
                        )
                        .sum()
                    ),

                "max_target_power_pu":
                    float(
                        design[
                            "y_pu"
                        ]
                        .max()
                    ),
            }
        )


        del design
        del X
        del meta

        gc.collect()


pvod_design_audit = pd.DataFrame(
    design_audit_rows
)


# ============================================================
# 9. One explicit nested-dimension test
# ============================================================

dimension_test_rows = []


for feature_block in [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]:

    example = (
        materialize_pvod_design(
            station_id=
                "station00",

            horizon_min=
                60,

            feature_block=
                feature_block,

            split=
                "train",
        )
    )


    dimension_test_rows.append(
        {
            "feature_block":
                feature_block,

            "rows":
                int(
                    example[
                        "X"
                    ]
                    .shape[0]
                ),

            "columns":
                int(
                    example[
                        "X"
                    ]
                    .shape[1]
                ),

            "expected_columns":
                PVOD_EXPECTED_DIMS[
                    feature_block
                ],
        }
    )


    del example

    gc.collect()


pvod_nested_dimension_test = pd.DataFrame(
    dimension_test_rows
)


assert (
    pvod_nested_dimension_test[
        "columns"
    ]
    .equals(
        pvod_nested_dimension_test[
            "expected_columns"
        ]
    )
)


# ============================================================
# 10. Display audits
# ============================================================

print(
    "=" * 125
)

print(
    "F0 HISTORY TABLE AUDIT"
)

print(
    "=" * 125
)


display(
    pvod_f0_audit
)


print(
    "\n"
    + "=" * 125
)

print(
    "GAP-SAFE SAMPLE INDEX AUDIT"
)

print(
    "=" * 125
)


display(
    pvod_sample_matrix_audit
)


print(
    "\n"
    + "=" * 125
)

print(
    "FULL DESIGN MATRIX AUDIT"
)

print(
    "=" * 125
)


display(
    pvod_design_audit
)


print(
    "\n"
    + "=" * 125
)

print(
    "NESTED FEATURE-DIMENSION TEST"
)

print(
    "=" * 125
)


display(
    pvod_nested_dimension_test
)


print(
    "\nPVOD-03B GAP-SAFE SAMPLE ASSEMBLY: PASSED"
)


print(
    "\nPrimary target = power_pu."
)

print(
    "Raw MW target retained as secondary."
)

print(
    "F0 = 71."
)

print(
    "F1 = 86."
)

print(
    "F2 = 91."
)

print(
    "F3 = 100."
)

print(
    "F4 = 197."
)

print(
    "F1/F2/F3/F4 attach at target valid time."
)

print(
    "No temporal gap filled."
)

print(
    "No required history or future-path gap crossed."
)

print(
    "No scaler fitted."
)

print(
    "No model trained."
)

print(
    "No F3/F4 forecasting performance observed."
)

Primary modelling target: power_pu
Secondary physical target: power_MW
F0 HISTORY TABLE AUDIT


,station,station_rows,F0_columns,complete_F0_rows,incomplete_F0_rows
0,station00,28896,71,28704,192
1,station01,33408,71,33312,96
2,station02,30432,71,30048,384
3,station03,14688,71,14592,96
4,station04,33408,71,33312,96
5,station05,9696,71,9600,96
6,station06,31104,71,30528,576
7,station07,32928,71,32736,192
8,station08,33120,71,32928,192
9,station09,24288,71,23808,480



GAP-SAFE SAMPLE INDEX AUDIT


,station,horizon_min,samples_total,train,validation,test,daylight_test,target_pu_max
0,station00,15,28702,18555,4349,5798,3366,0.836825
1,station00,30,28700,18553,4349,5798,3366,0.836825
2,station00,60,28696,18549,4349,5798,3366,0.836825
3,station01,15,33311,21619,5011,6681,3850,0.999873
4,station01,30,33310,21618,5011,6681,3850,0.999873
5,station01,60,33308,21616,5011,6681,3850,0.999873
6,station02,15,30044,19674,4651,5719,3296,0.944012
7,station02,30,30040,19672,4651,5717,3296,0.944012
8,station02,60,30032,19668,4651,5713,3296,0.944012
9,station03,15,14591,9451,2203,2937,1764,0.870965



FULL DESIGN MATRIX AUDIT


,station,horizon_min,rows,F0_dim,F1_dim,F2_dim,F3_dim,F4_dim,train,validation,test,test_daylight,max_target_power_pu
0,station00,15,28702,71,86,91,100,197,18555,4349,5798,3366,0.836825
1,station00,30,28700,71,86,91,100,197,18553,4349,5798,3366,0.836825
2,station00,60,28696,71,86,91,100,197,18549,4349,5798,3366,0.836825
3,station01,15,33311,71,86,91,100,197,21619,5011,6681,3850,0.999873
4,station01,30,33310,71,86,91,100,197,21618,5011,6681,3850,0.999873
5,station01,60,33308,71,86,91,100,197,21616,5011,6681,3850,0.999873
6,station02,15,30044,71,86,91,100,197,19674,4651,5719,3296,0.944012
7,station02,30,30040,71,86,91,100,197,19672,4651,5717,3296,0.944012
8,station02,60,30032,71,86,91,100,197,19668,4651,5713,3296,0.944012
9,station03,15,14591,71,86,91,100,197,9451,2203,2937,1764,0.870965



NESTED FEATURE-DIMENSION TEST


,feature_block,rows,columns,expected_columns
0,F0,18549,71,71
1,F1,18549,86,86
2,F2,18549,91,91
3,F3,18549,100,100
4,F4,18549,197,197



PVOD-03B GAP-SAFE SAMPLE ASSEMBLY: PASSED

Primary target = power_pu.
Raw MW target retained as secondary.
F0 = 71.
F1 = 86.
F2 = 91.
F3 = 100.
F4 = 197.
F1/F2/F3/F4 attach at target valid time.
No temporal gap filled.
No required history or future-path gap crossed.
No scaler fitted.
No model trained.
No F3/F4 forecasting performance observed.


In [9]:
# ============================================================
# PVOD-04C
# Freeze external-validation model/scaling/evaluation protocol
#
# IMPORTANT
# ---------
# This cell performs NO model fitting.
# It freezes all remaining choices BEFORE external results.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. External-validation identity
# ============================================================

PVOD_EXTERNAL_PROTOCOL_NAME = (
    "pvod_external_validation_v1"
)

PVOD_EXTERNAL_MODELS = [
    "Ridge",
    "LightGBM",
    "LSTM",
    "TCN",
]

PVOD_PRIMARY_VARIANTS = [
    "F3",
    "F4_real",
]

PVOD_EXTERNAL_HORIZONS = [
    15,
    30,
    60,
]

PVOD_EXTERNAL_STATIONS = sorted(
    pvod_clean.keys()
)


assert len(
    PVOD_EXTERNAL_STATIONS
) == 10


# ============================================================
# 2. Experimental unit
#
# One independent model is refitted for each:
#
#     station × horizon × architecture × representation
#
# We DO NOT pool stations into one training dataset.
#
# This mirrors the original single-system SOLETE setup while
# testing transfer across ten independent PV installations.
# ============================================================

PVOD_MODEL_UNIT = (
    "station_specific_refit"
)


# ============================================================
# 3. Primary / secondary targets
# ============================================================

PVOD_PRIMARY_TARGET = (
    "power_pu"
)

PVOD_SECONDARY_TARGET = (
    "power_MW"
)

PVOD_PREDICTION_CLIPPING = False


# ============================================================
# 4. Frozen classical model configurations
# ============================================================

PVOD_RIDGE_CONFIG = {
    15: {
        "alpha": 0.01,
    },
    30: {
        "alpha": 0.01,
    },
    60: {
        "alpha": 0.01,
    },
}


PVOD_LIGHTGBM_BASE_CONFIG = {
    "boosting_type":
        "gbdt",

    "colsample_bytree":
        0.8,

    "deterministic":
        True,

    "force_col_wise":
        True,

    "learning_rate":
        0.03,

    "max_depth":
        -1,

    "min_child_samples":
        20,

    "n_estimators":
        300,

    "n_jobs":
        -1,

    "num_leaves":
        15,

    "objective":
        "regression",

    "random_state":
        42,

    "reg_alpha":
        0.0,

    "reg_lambda":
        1.0,

    "subsample":
        0.8,

    "subsample_freq":
        1,

    "verbosity":
        -1,
}


PVOD_LIGHTGBM_CONFIG = {
    horizon:
        PVOD_LIGHTGBM_BASE_CONFIG.copy()

    for horizon
    in PVOD_EXTERNAL_HORIZONS
}


# ============================================================
# 5. Frozen LSTM configuration
# ============================================================

PVOD_LSTM_ARCHITECTURE = {
    "candidate_id":
        "RNN_C1",

    "units":
        32,

    "n_recurrent_layers":
        1,

    "dropout":
        0.0,

    "fusion_dense_units":
        32,
}


PVOD_LSTM_EPOCHS = {
    15: 7,
    30: 1,
    60: 2,
}


# ============================================================
# 6. Frozen TCN configuration
# ============================================================

PVOD_TCN_ARCHITECTURE = {
    "candidate_id":
        "TCN_C1",

    "filters":
        32,

    "kernel_size":
        3,

    "dilations":
        [
            1,
            2,
            4,
        ],

    "convolutions_per_block":
        2,

    "dropout":
        0.0,

    "fusion_dense_units":
        32,

    # Historical SOLETE architecture property.
    # The PVOD input sequence itself is only 5 time points
    # because the physical history window remains 60 minutes
    # at 15-minute sampling.
    "receptive_field_timesteps":
        29,
}


PVOD_TCN_EPOCHS = {
    15: 4,
    30: 1,
    60: 1,
}


# ============================================================
# 7. Neural execution constants
# ============================================================

PVOD_NEURAL_SEED = 42

PVOD_NEURAL_BATCH_SIZE = 256

PVOD_NEURAL_SHUFFLE = True

PVOD_NEURAL_CALLBACKS = None

PVOD_NEURAL_VALIDATION_DATA = None


# ============================================================
# 8. PVOD neural sequence geometry
#
# SOLETE:
#     13 time points × 9 channels
#
# PVOD:
#      5 time points × 14 channels
#
# Physical history duration remains 60 minutes.
#
# Architecture hyperparameters stay frozen.
# Only input geometry adapts to the external dataset cadence
# and observational schema.
# ============================================================

PVOD_SEQUENCE_OFFSETS_MIN = [
    60,
    45,
    30,
    15,
    0,
]


PVOD_SEQUENCE_CHANNELS = (
    list(
        PVOD_DYNAMIC_SOURCE_COLUMNS
    )
)


PVOD_SEQUENCE_LENGTH = len(
    PVOD_SEQUENCE_OFFSETS_MIN
)

PVOD_SEQUENCE_CHANNEL_COUNT = len(
    PVOD_SEQUENCE_CHANNELS
)


assert (
    PVOD_SEQUENCE_LENGTH
    ==
    5
)

assert (
    PVOD_SEQUENCE_CHANNEL_COUNT
    ==
    14
)


# ============================================================
# 9. Neural auxiliary geometry
#
# Recent 60-minute history appears ONLY in sequence branch.
#
# Auxiliary F3:
#
#   previous-day PV          1
#   Gregorian              +15
#   site solar             + 5
#   continuous astronomy   + 9
#                         ----
#                           30
#
# F4:
#
#   F3 auxiliary            30
#   Panchang one-hot       +97
#                         ----
#                          127
# ============================================================

PVOD_F3_AUXILIARY_DIM = 30
PVOD_F4_AUXILIARY_DIM = 127


assert (
    PVOD_F3_AUXILIARY_DIM
    ==
    PVOD_EXPECTED_DIMS["F3"]
    -
    (
        PVOD_SEQUENCE_LENGTH
        *
        PVOD_SEQUENCE_CHANNEL_COUNT
    )
)


assert (
    PVOD_F4_AUXILIARY_DIM
    ==
    PVOD_EXPECTED_DIMS["F4"]
    -
    (
        PVOD_SEQUENCE_LENGTH
        *
        PVOD_SEQUENCE_CHANNEL_COUNT
    )
)


# ============================================================
# 10. Identify binary auxiliary features
#
# Notebook 13 leaves exactly these feature TYPES unscaled:
#
#   - Gregorian weekend indicator
#   - solar daylight indicator
#   - Panchang one-hot indicators
#
# Everything else in the auxiliary branch is continuous and
# is standardized from TRAIN ONLY.
# ============================================================

def is_pvod_binary_auxiliary_column(
    column_name: str,
) -> bool:

    column_name = str(
        column_name
    )

    if (
        column_name
        in {
            "utc_is_weekend",
            "valid_utc_is_weekend",
            "solar_daylight",
        }
    ):
        return True

    panchang_prefixes = (
        "tithi_",
        "paksha_",
        "karana_",
        "nakshatra_",
        "yoga_",
    )

    return column_name.startswith(
        panchang_prefixes
    )


# ============================================================
# 11. Scaling policy
# ============================================================

PVOD_SCALING_POLICY = {

    "Ridge":
        (
            "StandardScaler inside sklearn Pipeline; "
            "fit on final train+validation population."
        ),

    "LightGBM":
        (
            "No predictor scaling; fit directly on "
            "train+validation."
        ),

    "LSTM_sequence":
        (
            "Per-channel mean/std from TRAIN ONLY, "
            "computed across sample and timestep axes."
        ),

    "TCN_sequence":
        (
            "Per-channel mean/std from TRAIN ONLY, "
            "computed across sample and timestep axes."
        ),

    "neural_auxiliary_continuous":
        (
            "Per-column mean/std from TRAIN ONLY."
        ),

    "neural_auxiliary_binary":
        (
            "Leave exactly 0/1; no scaling."
        ),

    "neural_target":
        (
            "Mean/std of power_pu from TRAIN ONLY."
        ),

    "neural_final_fit":
        (
            "Use frozen scalers, concatenate "
            "TRAIN + VALIDATION, and refit for "
            "SOLETE-frozen epoch count."
        ),
}


# ============================================================
# 12. Validation-data policy
#
# Critical external-validation rule:
#
# PVOD validation data DOES NOT:
#
# - select architecture;
# - select hyperparameters;
# - select epoch count;
# - fit neural scaling statistics.
#
# It joins train only for the final fixed-protocol model fit.
# ============================================================

PVOD_VALIDATION_POLICY = {
    "hyperparameter_search":
        False,

    "architecture_selection":
        False,

    "epoch_selection":
        False,

    "neural_scaler_fit":
        False,

    "final_refit_population":
        "train_plus_validation",
}


# ============================================================
# 13. Evaluation scopes
# ============================================================

PVOD_EVALUATION_SCOPES = [
    "all",
    "daylight",
]


PVOD_PRIMARY_METRIC = (
    "MAE_power_pu"
)

PVOD_SECONDARY_METRICS = [
    "RMSE_power_pu",
    "R2_power_pu",
    "MAE_MW",
    "RMSE_MW",
    "R2_MW",
]


# ============================================================
# 14. Paired F3-vs-F4 effect definition
#
# Primary comparison:
#
#     delta_MAE =
#         MAE(F4_real)
#         -
#         MAE(F3)
#
# Therefore:
#
#     delta < 0  -> F4 has lower error
#     delta > 0  -> F3 has lower error
#
# Same exact test rows are required for the pair.
# ============================================================

PVOD_PRIMARY_COMPARISON = {
    "reference":
        "F3",

    "candidate":
        "F4_real",

    "effect":
        "MAE_F4_minus_MAE_F3",

    "lower_is_better":
        True,
}


# ============================================================
# 15. External uncertainty protocol
#
# Hierarchical paired block bootstrap:
#
# 1. resample stations with replacement;
# 2. within each selected station, resample
#    China-local test calendar days with replacement;
# 3. preserve paired F3/F4 predictions;
# 4. compute station-level MAE delta;
# 5. equally weight stations in the aggregate.
#
# This allows uncertainty to reflect both:
#   - site heterogeneity;
#   - day-to-day temporal variability.
# ============================================================

PVOD_BOOTSTRAP_REPLICATES = 10_000

PVOD_BOOTSTRAP_SEED = 42

PVOD_BOOTSTRAP_INTERVAL = (
    "95_percentile"
)

PVOD_BOOTSTRAP_DAY_TIMEZONE = (
    "Asia/Shanghai"
)

PVOD_BOOTSTRAP_STATION_WEIGHTING = (
    "equal_station_weight"
)

PVOD_BOOTSTRAP_RETRAIN_MODELS = False


# ============================================================
# 16. Result-interpretation rule
# ============================================================

PVOD_EFFECT_CLASSIFICATION = {
    "F4_lower":
        "95% CI entirely below zero",

    "F3_lower":
        "95% CI entirely above zero",

    "crosses_zero":
        "95% CI includes zero",
}


# ============================================================
# 17. Secondary Phase-2 controls
#
# Definitions are frozen NOW, before primary PVOD results.
#
# They are NOT part of the first primary execution.
# If run later, they retain exactly the original Phase-2
# definitions rather than being chosen after seeing PVOD.
# ============================================================

PVOD_SECONDARY_FROZEN_CONTROLS = {
    "reference_coding":
        "F4_reference",

    "circular_shifts_days":
        [
            17,
            37,
            61,
        ],

    "joint_permutation_seeds":
        [
            1301,
            1302,
            1303,
        ],
}


# ============================================================
# 18. Strong protocol assertions
# ============================================================

assert set(
    PVOD_EXTERNAL_MODELS
) == {
    "Ridge",
    "LightGBM",
    "LSTM",
    "TCN",
}


assert PVOD_LSTM_EPOCHS == {
    15: 7,
    30: 1,
    60: 2,
}


assert PVOD_TCN_EPOCHS == {
    15: 4,
    30: 1,
    60: 1,
}


assert (
    PVOD_NEURAL_SEED
    ==
    42
)


assert (
    PVOD_NEURAL_BATCH_SIZE
    ==
    256
)


assert (
    PVOD_PREDICTION_CLIPPING
    is False
)


assert (
    PVOD_F3_AUXILIARY_DIM
    ==
    30
)


assert (
    PVOD_F4_AUXILIARY_DIM
    ==
    127
)


# ============================================================
# 19. Compact registry for display
# ============================================================

pvod_external_model_registry = pd.DataFrame(
    [
        {
            "model":
                "Ridge",

            "15min":
                "alpha=0.01",

            "30min":
                "alpha=0.01",

            "60min":
                "alpha=0.01",

            "scaling":
                "StandardScaler(train+validation)",
        },

        {
            "model":
                "LightGBM",

            "15min":
                "fixed LGBM config",

            "30min":
                "fixed LGBM config",

            "60min":
                "fixed LGBM config",

            "scaling":
                "none",
        },

        {
            "model":
                "LSTM",

            "15min":
                "RNN_C1 / 7 epochs",

            "30min":
                "RNN_C1 / 1 epoch",

            "60min":
                "RNN_C1 / 2 epochs",

            "scaling":
                "train-only neural scalers",
        },

        {
            "model":
                "TCN",

            "15min":
                "TCN_C1 / 4 epochs",

            "30min":
                "TCN_C1 / 1 epoch",

            "60min":
                "TCN_C1 / 1 epoch",

            "scaling":
                "train-only neural scalers",
        },
    ]
)


# ============================================================
# 20. Display frozen protocol
# ============================================================

print(
    "=" * 125
)

print(
    "PVOD EXTERNAL MODEL PROTOCOL"
)

print(
    "=" * 125
)


display(
    pvod_external_model_registry
)


print(
    "\nStations:",
    len(
        PVOD_EXTERNAL_STATIONS
    ),
)


print(
    "Station-specific refits:",
    PVOD_MODEL_UNIT,
)


print(
    "Primary variants:",
    PVOD_PRIMARY_VARIANTS,
)


print(
    "Primary target:",
    PVOD_PRIMARY_TARGET,
)


print(
    "\nNeural sequence shape:"
)

print(
    " ",
    (
        PVOD_SEQUENCE_LENGTH,
        PVOD_SEQUENCE_CHANNEL_COUNT,
    ),
)


print(
    "F3 auxiliary dimension:",
    PVOD_F3_AUXILIARY_DIM,
)


print(
    "F4 auxiliary dimension:",
    PVOD_F4_AUXILIARY_DIM,
)


print(
    "\nValidation selects hyperparameters:",
    PVOD_VALIDATION_POLICY[
        "hyperparameter_search"
    ],
)


print(
    "Validation selects epochs:",
    PVOD_VALIDATION_POLICY[
        "epoch_selection"
    ],
)


print(
    "Neural scalers use validation:",
    PVOD_VALIDATION_POLICY[
        "neural_scaler_fit"
    ],
)


print(
    "Final refit population:",
    PVOD_VALIDATION_POLICY[
        "final_refit_population"
    ],
)


print(
    "\nPrimary comparison:",
    PVOD_PRIMARY_COMPARISON[
        "effect"
    ],
)


print(
    "Bootstrap replicates:",
    PVOD_BOOTSTRAP_REPLICATES,
)


print(
    "Bootstrap day timezone:",
    PVOD_BOOTSTRAP_DAY_TIMEZONE,
)


print(
    "Bootstrap station weighting:",
    PVOD_BOOTSTRAP_STATION_WEIGHTING,
)


print(
    "\nSecondary control definitions frozen:",
    PVOD_SECONDARY_FROZEN_CONTROLS,
)


print(
    "\nPVOD-04C MODEL + STATISTICAL PROTOCOL: FROZEN"
)


print(
    "\nNo PVOD scaler fitted."
)

print(
    "No PVOD model fitted."
)

print(
    "No PVOD prediction generated."
)

print(
    "No external F3/F4 result observed."
)

PVOD EXTERNAL MODEL PROTOCOL


,model,15min,30min,60min,scaling
0,Ridge,alpha=0.01,alpha=0.01,alpha=0.01,StandardScaler(train+validation)
1,LightGBM,fixed LGBM config,fixed LGBM config,fixed LGBM config,none
2,LSTM,RNN_C1 / 7 epochs,RNN_C1 / 1 epoch,RNN_C1 / 2 epochs,train-only neural scalers
3,TCN,TCN_C1 / 4 epochs,TCN_C1 / 1 epoch,TCN_C1 / 1 epoch,train-only neural scalers



Stations: 10
Station-specific refits: station_specific_refit
Primary variants: ['F3', 'F4_real']
Primary target: power_pu

Neural sequence shape:
  (5, 14)
F3 auxiliary dimension: 30
F4 auxiliary dimension: 127

Validation selects hyperparameters: False
Validation selects epochs: False
Neural scalers use validation: False
Final refit population: train_plus_validation

Primary comparison: MAE_F4_minus_MAE_F3
Bootstrap replicates: 10000
Bootstrap day timezone: Asia/Shanghai
Bootstrap station weighting: equal_station_weight

Secondary control definitions frozen: {'reference_coding': 'F4_reference', 'circular_shifts_days': [17, 37, 61], 'joint_permutation_seeds': [1301, 1302, 1303]}

PVOD-04C MODEL + STATISTICAL PROTOCOL: FROZEN

No PVOD scaler fitted.
No PVOD model fitted.
No PVOD prediction generated.
No external F3/F4 result observed.


In [10]:
# ============================================================
# PVOD-05A
# Neural input + train-only scaling preflight
#
# PURPOSE
# -------
# Verify, for every station x horizon:
#
# 1. sequence geometry = (n, 5, 14);
# 2. F3 auxiliary width = 30;
# 3. F4 auxiliary width = 127;
# 4. F3/F4 rows and targets are perfectly paired;
# 5. sequence channels have non-zero TRAIN variance;
# 6. continuous auxiliary features have non-zero TRAIN variance;
# 7. binary auxiliary features remain exact 0/1;
# 8. target power_pu has non-zero TRAIN variance.
#
# NO scaler is applied.
# NO model is built.
# NO model is trained.
# NO prediction is generated.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. Exact recent-history columns
#
# Sequence order:
#
#     t-60
#     t-45
#     t-30
#     t-15
#     t
#
# Each timestep contains the same 14 dynamic channels.
# ============================================================

PVOD_RECENT_HISTORY_COLUMNS = [
    f"{source_column}__lag_{lag_min:04d}m"

    for source_column
    in PVOD_SEQUENCE_CHANNELS

    for lag_min
    in PVOD_SEQUENCE_OFFSETS_MIN
]


assert len(
    PVOD_RECENT_HISTORY_COLUMNS
) == (
    PVOD_SEQUENCE_CHANNEL_COUNT
    *
    PVOD_SEQUENCE_LENGTH
)


assert len(
    PVOD_RECENT_HISTORY_COLUMNS
) == 70


# ============================================================
# 2. Helper: construct raw neural sequence
# ============================================================

def build_pvod_raw_sequence(
    X: pd.DataFrame,
) -> np.ndarray:

    sequence_timesteps = []


    for lag_min in (
        PVOD_SEQUENCE_OFFSETS_MIN
    ):

        timestep_columns = [
            f"{source_column}__lag_{lag_min:04d}m"

            for source_column
            in PVOD_SEQUENCE_CHANNELS
        ]


        assert all(
            column
            in
            X.columns

            for column
            in timestep_columns
        )


        timestep_values = (
            X[
                timestep_columns
            ]
            .to_numpy(
                dtype=np.float32
            )
        )


        sequence_timesteps.append(
            timestep_values
        )


    sequence = np.stack(
        sequence_timesteps,
        axis=1,
    )


    assert sequence.shape == (
        len(X),
        PVOD_SEQUENCE_LENGTH,
        PVOD_SEQUENCE_CHANNEL_COUNT,
    )


    assert np.isfinite(
        sequence
    ).all()


    return sequence


# ============================================================
# 3. Helper: neural auxiliary frame
#
# Recent-history values are represented exclusively by the
# sequence branch.
#
# Auxiliary branch contains:
#
#     previous-day PV
#     +
#     F1
#     +
#     F2
#     +
#     F3
#     [+ F4 one-hot]
# ============================================================

def build_pvod_raw_auxiliary(
    X: pd.DataFrame,
) -> pd.DataFrame:

    auxiliary_columns = [
        column
        for column
        in X.columns

        if column
        not in PVOD_RECENT_HISTORY_COLUMNS
    ]


    auxiliary = (
        X[
            auxiliary_columns
        ]
        .copy()
    )


    return auxiliary


# ============================================================
# 4. Audit all station x horizon cases
# ============================================================

audit_rows = []
problem_rows = []

observed_binary_column_names = set()


for station_id in (
    PVOD_EXTERNAL_STATIONS
):

    for horizon_min in (
        PVOD_EXTERNAL_HORIZONS
    ):

        # ----------------------------------------------------
        # Materialize paired F3 and F4 designs
        # ----------------------------------------------------

        design_f3 = (
            materialize_pvod_design(
                station_id=
                    station_id,

                horizon_min=
                    horizon_min,

                feature_block=
                    "F3",
            )
        )


        design_f4 = (
            materialize_pvod_design(
                station_id=
                    station_id,

                horizon_min=
                    horizon_min,

                feature_block=
                    "F4",
            )
        )


        X_f3 = (
            design_f3[
                "X"
            ]
        )


        X_f4 = (
            design_f4[
                "X"
            ]
        )


        meta_f3 = (
            design_f3[
                "meta"
            ]
        )


        meta_f4 = (
            design_f4[
                "meta"
            ]
        )


        y_f3 = (
            design_f3[
                "y_pu"
            ]
        )


        y_f4 = (
            design_f4[
                "y_pu"
            ]
        )


        # ----------------------------------------------------
        # Strong F3/F4 pairing checks
        # ----------------------------------------------------

        assert X_f3.index.equals(
            X_f4.index
        )


        assert meta_f3[
            "target_time_utc"
        ].equals(
            meta_f4[
                "target_time_utc"
            ]
        )


        assert meta_f3[
            "split"
        ].equals(
            meta_f4[
                "split"
            ]
        )


        assert np.array_equal(
            y_f3.to_numpy(),
            y_f4.to_numpy(),
        )


        # ----------------------------------------------------
        # Frozen split masks
        # ----------------------------------------------------

        train_mask = (
            meta_f3[
                "split"
            ]
            .eq(
                "train"
            )
            .to_numpy()
        )


        validation_mask = (
            meta_f3[
                "split"
            ]
            .eq(
                "validation"
            )
            .to_numpy()
        )


        test_mask = (
            meta_f3[
                "split"
            ]
            .eq(
                "test"
            )
            .to_numpy()
        )


        assert (
            train_mask.sum()
            +
            validation_mask.sum()
            +
            test_mask.sum()
            ==
            len(
                X_f3
            )
        )


        # ----------------------------------------------------
        # Sequence branch
        #
        # F3 and F4 must contain exactly the same sequence.
        # ----------------------------------------------------

        sequence_f3 = (
            build_pvod_raw_sequence(
                X_f3
            )
        )


        sequence_f4 = (
            build_pvod_raw_sequence(
                X_f4
            )
        )


        assert np.array_equal(
            sequence_f3,
            sequence_f4,
        )


        train_sequence = (
            sequence_f3[
                train_mask
            ]
            .astype(
                np.float64,
                copy=False,
            )
        )


        sequence_mean = (
            train_sequence
            .mean(
                axis=(
                    0,
                    1,
                )
            )
        )


        sequence_std = (
            train_sequence
            .std(
                axis=(
                    0,
                    1,
                ),
                ddof=0,
            )
        )


        zero_sequence_channels = [
            PVOD_SEQUENCE_CHANNELS[
                index
            ]

            for index
            in np.flatnonzero(
                sequence_std
                <=
                0.0
            )
        ]


        # ----------------------------------------------------
        # Target scaling audit
        # ----------------------------------------------------

        raw_target = (
            y_f3
            .to_numpy(
                dtype=np.float64
            )
        )


        train_target = (
            raw_target[
                train_mask
            ]
        )


        target_mean = float(
            train_target.mean()
        )


        target_std = float(
            train_target.std(
                ddof=0
            )
        )


        # ----------------------------------------------------
        # Audit both auxiliary representations
        # ----------------------------------------------------

        for variant_name, X_variant in [
            (
                "F3",
                X_f3,
            ),
            (
                "F4_real",
                X_f4,
            ),
        ]:

            auxiliary = (
                build_pvod_raw_auxiliary(
                    X_variant
                )
            )


            expected_auxiliary_dim = {
                "F3":
                    PVOD_F3_AUXILIARY_DIM,

                "F4_real":
                    PVOD_F4_AUXILIARY_DIM,
            }[
                variant_name
            ]


            assert (
                auxiliary.shape[
                    1
                ]
                ==
                expected_auxiliary_dim
            )


            # -----------------------------------------------
            # Partition binary / continuous auxiliary columns
            # -----------------------------------------------

            binary_columns = [
                column
                for column
                in auxiliary.columns

                if is_pvod_binary_auxiliary_column(
                    column
                )
            ]


            continuous_columns = [
                column
                for column
                in auxiliary.columns

                if column
                not in binary_columns
            ]


            observed_binary_column_names.update(
                binary_columns
            )


            # -----------------------------------------------
            # Binary-value integrity
            # -----------------------------------------------

            bad_binary_columns = []


            for column in (
                binary_columns
            ):

                values = (
                    auxiliary[
                        column
                    ]
                    .to_numpy(
                        dtype=float
                    )
                )


                unique_values = np.unique(
                    values
                )


                if not np.all(
                    np.isin(
                        unique_values,
                        [
                            0.0,
                            1.0,
                        ],
                    )
                ):

                    bad_binary_columns.append(
                        column
                    )


            # -----------------------------------------------
            # Continuous train-only variance
            # -----------------------------------------------

            train_continuous = (
                auxiliary[
                    continuous_columns
                ]
                .to_numpy(
                    dtype=np.float64
                )[
                    train_mask
                ]
            )


            continuous_std = (
                train_continuous
                .std(
                    axis=0,
                    ddof=0,
                )
            )


            zero_continuous_columns = [
                continuous_columns[
                    index
                ]

                for index
                in np.flatnonzero(
                    continuous_std
                    <=
                    0.0
                )
            ]


            # -----------------------------------------------
            # Expected partition widths
            #
            # SOLETE Phase 2:
            #
            # F3:
            #     28 continuous + 2 binary = 30
            #
            # F4:
            #     28 continuous + 99 binary = 127
            #
            # PVOD should preserve this deterministic
            # auxiliary geometry.
            # -----------------------------------------------

            expected_partition = {
                "F3": {
                    "continuous":
                        28,

                    "binary":
                        2,
                },

                "F4_real": {
                    "continuous":
                        28,

                    "binary":
                        99,
                },
            }[
                variant_name
            ]


            partition_matches = (
                len(
                    continuous_columns
                )
                ==
                expected_partition[
                    "continuous"
                ]
                and
                len(
                    binary_columns
                )
                ==
                expected_partition[
                    "binary"
                ]
            )


            audit_rows.append(
                {
                    "station":
                        station_id,

                    "horizon_min":
                        horizon_min,

                    "variant":
                        variant_name,

                    "rows":
                        int(
                            len(
                                X_variant
                            )
                        ),

                    "train":
                        int(
                            train_mask.sum()
                        ),

                    "validation":
                        int(
                            validation_mask.sum()
                        ),

                    "test":
                        int(
                            test_mask.sum()
                        ),

                    "sequence_shape":
                        (
                            PVOD_SEQUENCE_LENGTH,
                            PVOD_SEQUENCE_CHANNEL_COUNT,
                        ),

                    "sequence_zero_std_channels":
                        len(
                            zero_sequence_channels
                        ),

                    "auxiliary_dim":
                        int(
                            auxiliary.shape[
                                1
                            ]
                        ),

                    "continuous_aux":
                        len(
                            continuous_columns
                        ),

                    "binary_aux":
                        len(
                            binary_columns
                        ),

                    "partition_matches":
                        bool(
                            partition_matches
                        ),

                    "zero_std_continuous_aux":
                        len(
                            zero_continuous_columns
                        ),

                    "bad_binary_columns":
                        len(
                            bad_binary_columns
                        ),

                    "target_train_std":
                        target_std,
                }
            )


            # -----------------------------------------------
            # Preserve details for any problem
            # -----------------------------------------------

            if (
                zero_sequence_channels
                or
                zero_continuous_columns
                or
                bad_binary_columns
                or
                target_std <= 0.0
                or
                not partition_matches
            ):

                problem_rows.append(
                    {
                        "station":
                            station_id,

                        "horizon_min":
                            horizon_min,

                        "variant":
                            variant_name,

                        "zero_sequence_channels":
                            zero_sequence_channels,

                        "zero_continuous_columns":
                            zero_continuous_columns,

                        "bad_binary_columns":
                            bad_binary_columns,

                        "target_train_std":
                            target_std,

                        "continuous_aux":
                            len(
                                continuous_columns
                            ),

                        "binary_aux":
                            len(
                                binary_columns
                            ),
                    }
                )


# ============================================================
# 5. Aggregate results
# ============================================================

pvod_neural_preflight_audit = pd.DataFrame(
    audit_rows
)


pvod_neural_preflight_problems = pd.DataFrame(
    problem_rows
)


# ============================================================
# 6. Global pairing / geometry assertions
# ============================================================

assert len(
    pvod_neural_preflight_audit
) == (
    10
    *
    3
    *
    2
)


assert (
    pvod_neural_preflight_audit[
        "partition_matches"
    ]
    .all()
), (
    "PVOD auxiliary partition differs from frozen "
    "Phase-2 geometry."
)


assert (
    pvod_neural_preflight_audit[
        "bad_binary_columns"
    ]
    .eq(0)
    .all()
), (
    "At least one supposed binary auxiliary feature "
    "contains a non-binary value."
)


# ============================================================
# 7. Compact station x horizon table
# ============================================================

compact_preflight = (
    pvod_neural_preflight_audit
    .pivot_table(
        index=[
            "station",
            "horizon_min",
        ],

        columns="variant",

        values=[
            "sequence_zero_std_channels",
            "zero_std_continuous_aux",
            "target_train_std",
        ],

        aggfunc="first",
    )
)


# ============================================================
# 8. Display
# ============================================================

print(
    "=" * 125
)

print(
    "PVOD NEURAL INPUT + SCALING PREFLIGHT"
)

print(
    "=" * 125
)


print(
    "\nSequence geometry:",
    (
        PVOD_SEQUENCE_LENGTH,
        PVOD_SEQUENCE_CHANNEL_COUNT,
    ),
)


print(
    "F3 auxiliary dimension:",
    PVOD_F3_AUXILIARY_DIM,
)


print(
    "F4 auxiliary dimension:",
    PVOD_F4_AUXILIARY_DIM,
)


print(
    "\nObserved binary auxiliary columns:"
)


for column in sorted(
    observed_binary_column_names
):
    print(
        " ",
        column
    )


print(
    "\n"
    + "=" * 125
)

print(
    "FULL PREFLIGHT AUDIT"
)

print(
    "=" * 125
)


display(
    pvod_neural_preflight_audit
)


print(
    "\n"
    + "=" * 125
)

print(
    "PROBLEM AUDIT"
)

print(
    "=" * 125
)


if (
    pvod_neural_preflight_problems.empty
):

    print(
        "No zero-variance, binary-integrity, "
        "partition, or target-scaling problems detected."
    )

else:

    display(
        pvod_neural_preflight_problems
    )


print(
    "\n"
    + "=" * 125
)

print(
    "SUMMARY"
)

print(
    "=" * 125
)


print(
    "Cases audited:",
    len(
        pvod_neural_preflight_audit
    )
)


print(
    "Cases with zero-variance sequence channel:",
    int(
        (
            pvod_neural_preflight_audit[
                "sequence_zero_std_channels"
            ]
            >
            0
        )
        .sum()
    )
)


print(
    "Cases with zero-variance continuous auxiliary:",
    int(
        (
            pvod_neural_preflight_audit[
                "zero_std_continuous_aux"
            ]
            >
            0
        )
        .sum()
    )
)


print(
    "Cases with invalid binary auxiliary:",
    int(
        (
            pvod_neural_preflight_audit[
                "bad_binary_columns"
            ]
            >
            0
        )
        .sum()
    )
)


print(
    "Cases with non-positive target std:",
    int(
        (
            pvod_neural_preflight_audit[
                "target_train_std"
            ]
            <=
            0.0
        )
        .sum()
    )
)


print(
    "\nPVOD-05A NEURAL SCALING PREFLIGHT: COMPLETE"
)


print(
    "\nNo scaler fitted."
)

print(
    "No model built."
)

print(
    "No model trained."
)

print(
    "No prediction generated."
)

print(
    "No external F3/F4 result observed."
)

PVOD NEURAL INPUT + SCALING PREFLIGHT

Sequence geometry: (5, 14)
F3 auxiliary dimension: 30
F4 auxiliary dimension: 127

Observed binary auxiliary columns:
  karana_Balava
  karana_Bava
  karana_Chatushpada
  karana_Gara
  karana_Kaulava
  karana_Kimstughna
  karana_Naga
  karana_Shakuni
  karana_Taitila
  karana_Vanija
  karana_Vishti
  nakshatra_Anuradha
  nakshatra_Ardra
  nakshatra_Ashlesha
  nakshatra_Ashwini
  nakshatra_Bharani
  nakshatra_Chitra
  nakshatra_Dhanishta
  nakshatra_Hasta
  nakshatra_Jyeshtha
  nakshatra_Krittika
  nakshatra_Magha
  nakshatra_Mrigashirsha
  nakshatra_Mula
  nakshatra_Punarvasu
  nakshatra_Purva Ashadha
  nakshatra_Purva Bhadrapada
  nakshatra_Purva Phalguni
  nakshatra_Pushya
  nakshatra_Revati
  nakshatra_Rohini
  nakshatra_Shatabhisha
  nakshatra_Shravana
  nakshatra_Swati
  nakshatra_Uttara Ashadha
  nakshatra_Uttara Bhadrapada
  nakshatra_Uttara Phalguni
  nakshatra_Vishakha
  paksha_Krishna
  paksha_Shukla
  solar_daylight
  tithi_1
  tithi_10

,station,horizon_min,variant,rows,train,validation,test,sequence_shape,sequence_zero_std_channels,auxiliary_dim,continuous_aux,binary_aux,partition_matches,zero_std_continuous_aux,bad_binary_columns,target_train_std
0,station00,15,F3,28702,18555,4349,5798,"(5, 14)",0,30,28,2,True,0,0,0.166448
1,station00,15,F4_real,28702,18555,4349,5798,"(5, 14)",0,127,28,99,True,0,0,0.166448
2,station00,30,F3,28700,18553,4349,5798,"(5, 14)",0,30,28,2,True,0,0,0.166454
3,station00,30,F4_real,28700,18553,4349,5798,"(5, 14)",0,127,28,99,True,0,0,0.166454
4,station00,60,F3,28696,18549,4349,5798,"(5, 14)",0,30,28,2,True,0,0,0.166465
5,station00,60,F4_real,28696,18549,4349,5798,"(5, 14)",0,127,28,99,True,0,0,0.166465
6,station01,15,F3,33311,21619,5011,6681,"(5, 14)",0,30,28,2,True,0,0,0.261372
7,station01,15,F4_real,33311,21619,5011,6681,"(5, 14)",0,127,28,99,True,0,0,0.261372
8,station01,30,F3,33310,21618,5011,6681,"(5, 14)",0,30,28,2,True,0,0,0.261375
9,station01,30,F4_real,33310,21618,5011,6681,"(5, 14)",0,127,28,99,True,0,0,0.261375



PROBLEM AUDIT


,station,horizon_min,variant,zero_sequence_channels,zero_continuous_columns,bad_binary_columns,target_train_std,continuous_aux,binary_aux
0,station03,15,F3,[],[utc_year],[],0.246213,28,2
1,station03,15,F4_real,[],[utc_year],[],0.246213,28,99
2,station03,30,F3,[],[utc_year],[],0.246221,28,2
3,station03,30,F4_real,[],[utc_year],[],0.246221,28,99
4,station03,60,F3,[],[utc_year],[],0.246236,28,2
5,station03,60,F4_real,[],[utc_year],[],0.246236,28,99
6,station05,15,F3,[],[utc_year],[],0.277237,28,2
7,station05,15,F4_real,[],[utc_year],[],0.277237,28,99
8,station05,30,F3,[],[utc_year],[],0.277248,28,2
9,station05,30,F4_real,[],[utc_year],[],0.277248,28,99



SUMMARY
Cases audited: 60
Cases with zero-variance sequence channel: 0
Cases with zero-variance continuous auxiliary: 12
Cases with invalid binary auxiliary: 0
Cases with non-positive target std: 0

PVOD-05A NEURAL SCALING PREFLIGHT: COMPLETE

No scaler fitted.
No model built.
No model trained.
No prediction generated.
No external F3/F4 result observed.


In [11]:
# ============================================================
# PVOD-05B
# Frozen zero-variance-safe neural scaling
#
# PURPOSE
# -------
# Implement the exact train-only Phase-2 scaling policy with
# one external-data safeguard discovered prospectively in
# PVOD-05A:
#
#   continuous auxiliary train std == 0
#       -> retain feature
#       -> use effective std = 1.0
#
# Therefore, after centering, a train-constant feature becomes
# exactly zero.
#
# IMPORTANT
# ---------
# - feature is NOT dropped;
# - feature dimension is NOT changed;
# - binary columns remain exact 0/1;
# - target zero variance is NOT tolerated;
# - no model is built or trained.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. Freeze zero-variance policy
# ============================================================

PVOD_ZERO_VARIANCE_CONTINUOUS_POLICY = (
    "retain_center_and_use_unit_scale"
)


def make_safe_std(
    raw_std: np.ndarray,
):

    raw_std = np.asarray(
        raw_std,
        dtype=np.float64,
    )


    assert np.isfinite(
        raw_std
    ).all()


    zero_mask = (
        raw_std
        ==
        0.0
    )


    safe_std = (
        raw_std.copy()
    )


    safe_std[
        zero_mask
    ] = 1.0


    assert (
        safe_std
        >
        0.0
    ).all()


    return (
        safe_std,
        zero_mask,
    )


# ============================================================
# 2. Full neural-preparation helper
# ============================================================

def prepare_pvod_neural_arrays(
    station_id: str,
    horizon_min: int,
    variant_name: str,
):

    if variant_name == "F3":

        feature_block = (
            "F3"
        )

        expected_aux_dim = (
            PVOD_F3_AUXILIARY_DIM
        )


    elif variant_name == "F4_real":

        feature_block = (
            "F4"
        )

        expected_aux_dim = (
            PVOD_F4_AUXILIARY_DIM
        )


    else:

        raise ValueError(
            f"Unsupported variant: "
            f"{variant_name}"
        )


    # --------------------------------------------------------
    # Materialize frozen forecasting design
    # --------------------------------------------------------

    design = (
        materialize_pvod_design(
            station_id=
                station_id,

            horizon_min=
                horizon_min,

            feature_block=
                feature_block,
        )
    )


    X = (
        design[
            "X"
        ]
    )


    y_pu = (
        design[
            "y_pu"
        ]
    )


    y_MW = (
        design[
            "y_MW"
        ]
    )


    meta = (
        design[
            "meta"
        ]
    )


    # --------------------------------------------------------
    # Frozen split masks
    # --------------------------------------------------------

    train_mask = (
        meta[
            "split"
        ]
        .eq(
            "train"
        )
        .to_numpy()
    )


    validation_mask = (
        meta[
            "split"
        ]
        .eq(
            "validation"
        )
        .to_numpy()
    )


    test_mask = (
        meta[
            "split"
        ]
        .eq(
            "test"
        )
        .to_numpy()
    )


    assert (
        train_mask.sum()
        >
        0
    )


    assert (
        validation_mask.sum()
        >
        0
    )


    assert (
        test_mask.sum()
        >
        0
    )


    # ========================================================
    # A. RAW SEQUENCE
    # ========================================================

    raw_sequence = (
        build_pvod_raw_sequence(
            X
        )
    )


    assert raw_sequence.shape == (
        len(X),
        PVOD_SEQUENCE_LENGTH,
        PVOD_SEQUENCE_CHANNEL_COUNT,
    )


    # ========================================================
    # B. TRAIN-ONLY SEQUENCE SCALING
    # ========================================================

    train_sequence_64 = (
        raw_sequence[
            train_mask
        ]
        .astype(
            np.float64,
            copy=False,
        )
    )


    sequence_mean = (
        train_sequence_64
        .mean(
            axis=(
                0,
                1,
            )
        )
    )


    sequence_std_raw = (
        train_sequence_64
        .std(
            axis=(
                0,
                1,
            ),
            ddof=0,
        )
    )


    (
        sequence_std,
        sequence_zero_mask,
    ) = make_safe_std(
        sequence_std_raw
    )


    # PVOD-05A established that no sequence channel is
    # train-constant. Keep this as a hard invariant.
    assert not (
        sequence_zero_mask.any()
    ), (
        f"{station_id} h={horizon_min}: "
        "unexpected zero-variance sequence channel."
    )


    scaled_sequence = (
        (
            raw_sequence.astype(
                np.float64
            )
            -
            sequence_mean
        )
        /
        sequence_std
    ).astype(
        np.float32
    )


    # ========================================================
    # C. TRAIN-ONLY TARGET SCALING
    # ========================================================

    raw_target = (
        y_pu
        .to_numpy(
            dtype=np.float64
        )
    )


    train_target = (
        raw_target[
            train_mask
        ]
    )


    target_mean = float(
        train_target.mean()
    )


    target_std = float(
        train_target.std(
            ddof=0
        )
    )


    # A constant training target would be a fundamentally
    # different problem, so do NOT silently guard it.
    assert (
        np.isfinite(
            target_std
        )
        and
        target_std
        >
        0.0
    )


    scaled_target = (
        (
            raw_target
            -
            target_mean
        )
        /
        target_std
    ).astype(
        np.float32
    )


    # ========================================================
    # D. RAW AUXILIARY REPRESENTATION
    # ========================================================

    raw_auxiliary_df = (
        build_pvod_raw_auxiliary(
            X
        )
    )


    assert (
        raw_auxiliary_df.shape[
            1
        ]
        ==
        expected_aux_dim
    )


    auxiliary_columns = list(
        raw_auxiliary_df.columns
    )


    binary_columns = [
        column
        for column
        in auxiliary_columns

        if is_pvod_binary_auxiliary_column(
            column
        )
    ]


    continuous_columns = [
        column
        for column
        in auxiliary_columns

        if column
        not in binary_columns
    ]


    # Frozen Phase-2 auxiliary geometry.
    if variant_name == "F3":

        assert len(
            continuous_columns
        ) == 28

        assert len(
            binary_columns
        ) == 2


    else:

        assert len(
            continuous_columns
        ) == 28

        assert len(
            binary_columns
        ) == 99


    raw_auxiliary = (
        raw_auxiliary_df
        .to_numpy(
            dtype=np.float32
        )
    )


    scaled_auxiliary = (
        raw_auxiliary.copy()
    )


    # ========================================================
    # E. TRAIN-ONLY CONTINUOUS AUXILIARY SCALING
    # ========================================================

    continuous_positions = np.array(
        [
            auxiliary_columns.index(
                column
            )

            for column
            in continuous_columns
        ],
        dtype=np.int64,
    )


    binary_positions = np.array(
        [
            auxiliary_columns.index(
                column
            )

            for column
            in binary_columns
        ],
        dtype=np.int64,
    )


    train_continuous = (
        raw_auxiliary[
            train_mask
        ][
            :,
            continuous_positions,
        ]
        .astype(
            np.float64
        )
    )


    auxiliary_mean = (
        train_continuous
        .mean(
            axis=0
        )
    )


    auxiliary_std_raw = (
        train_continuous
        .std(
            axis=0,
            ddof=0,
        )
    )


    (
        auxiliary_std,
        auxiliary_zero_mask,
    ) = make_safe_std(
        auxiliary_std_raw
    )


    zero_variance_continuous_columns = [
        continuous_columns[
            index
        ]

        for index
        in np.flatnonzero(
            auxiliary_zero_mask
        )
    ]


    scaled_continuous = (
        (
            raw_auxiliary[
                :,
                continuous_positions,
            ]
            .astype(
                np.float64
            )
            -
            auxiliary_mean
        )
        /
        auxiliary_std
    ).astype(
        np.float32
    )


    scaled_auxiliary[
        :,
        continuous_positions,
    ] = (
        scaled_continuous
    )


    # ========================================================
    # F. Binary columns remain EXACTLY 0/1
    # ========================================================

    if len(
        binary_positions
    ) > 0:

        assert np.array_equal(
            scaled_auxiliary[
                :,
                binary_positions,
            ],
            raw_auxiliary[
                :,
                binary_positions,
            ],
        )


        observed_binary_values = np.unique(
            scaled_auxiliary[
                :,
                binary_positions,
            ]
        )


        assert np.all(
            np.isin(
                observed_binary_values,
                [
                    0.0,
                    1.0,
                ],
            )
        )


    # ========================================================
    # G. Strong train-scaling audits
    # ========================================================

    scaled_train_sequence = (
        scaled_sequence[
            train_mask
        ]
        .astype(
            np.float64
        )
    )


    assert np.allclose(
        scaled_train_sequence.mean(
            axis=(
                0,
                1,
            )
        ),
        0.0,
        atol=1e-5,
        rtol=0.0,
    )


    assert np.allclose(
        scaled_train_sequence.std(
            axis=(
                0,
                1,
            ),
            ddof=0,
        ),
        1.0,
        atol=1e-5,
        rtol=0.0,
    )


    scaled_train_target = (
        scaled_target[
            train_mask
        ]
        .astype(
            np.float64
        )
    )


    assert np.isclose(
        scaled_train_target.mean(),
        0.0,
        atol=1e-5,
        rtol=0.0,
    )


    assert np.isclose(
        scaled_train_target.std(
            ddof=0
        ),
        1.0,
        atol=1e-5,
        rtol=0.0,
    )


    scaled_train_continuous = (
        scaled_auxiliary[
            train_mask
        ][
            :,
            continuous_positions,
        ]
        .astype(
            np.float64
        )
    )


    # --------------------------------------------------------
    # Non-constant continuous columns -> mean 0 / std 1
    # --------------------------------------------------------

    nonzero_positions = np.flatnonzero(
        ~auxiliary_zero_mask
    )


    if len(
        nonzero_positions
    ) > 0:

        assert np.allclose(
            scaled_train_continuous[
                :,
                nonzero_positions,
            ]
            .mean(
                axis=0
            ),
            0.0,
            atol=1e-5,
            rtol=0.0,
        )


        assert np.allclose(
            scaled_train_continuous[
                :,
                nonzero_positions,
            ]
            .std(
                axis=0,
                ddof=0,
            ),
            1.0,
            atol=1e-5,
            rtol=0.0,
        )


    # --------------------------------------------------------
    # Train-constant continuous columns -> exactly zero
    # --------------------------------------------------------

    zero_positions = np.flatnonzero(
        auxiliary_zero_mask
    )


    if len(
        zero_positions
    ) > 0:

        assert np.allclose(
            scaled_train_continuous[
                :,
                zero_positions,
            ],
            0.0,
            atol=1e-7,
            rtol=0.0,
        )


    # ========================================================
    # H. Final finite checks
    # ========================================================

    assert np.isfinite(
        scaled_sequence
    ).all()


    assert np.isfinite(
        scaled_auxiliary
    ).all()


    assert np.isfinite(
        scaled_target
    ).all()


    return {
        "sequence":
            np.ascontiguousarray(
                scaled_sequence,
                dtype=np.float32,
            ),

        "auxiliary":
            np.ascontiguousarray(
                scaled_auxiliary,
                dtype=np.float32,
            ),

        "target_scaled":
            np.ascontiguousarray(
                scaled_target,
                dtype=np.float32,
            ),

        "target_pu":
            np.ascontiguousarray(
                raw_target,
                dtype=np.float64,
            ),

        "target_MW":
            np.ascontiguousarray(
                y_MW.to_numpy(
                    dtype=np.float64
                ),
            ),

        "train_mask":
            train_mask.copy(),

        "validation_mask":
            validation_mask.copy(),

        "test_mask":
            test_mask.copy(),

        "meta":
            meta.copy(),

        "auxiliary_columns":
            tuple(
                auxiliary_columns
            ),

        "continuous_columns":
            tuple(
                continuous_columns
            ),

        "binary_columns":
            tuple(
                binary_columns
            ),

        "zero_variance_continuous_columns":
            tuple(
                zero_variance_continuous_columns
            ),

        "sequence_mean":
            sequence_mean.copy(),

        "sequence_std":
            sequence_std.copy(),

        "auxiliary_mean":
            auxiliary_mean.copy(),

        "auxiliary_std_raw":
            auxiliary_std_raw.copy(),

        "auxiliary_std_effective":
            auxiliary_std.copy(),

        "target_mean":
            target_mean,

        "target_std":
            target_std,
    }


# ============================================================
# 3. Audit helper across all 60 station/horizon/variant cases
# ============================================================

safe_scaling_audit_rows = []


for station_id in (
    PVOD_EXTERNAL_STATIONS
):

    for horizon_min in (
        PVOD_EXTERNAL_HORIZONS
    ):

        for variant_name in (
            PVOD_PRIMARY_VARIANTS
        ):

            prepared = (
                prepare_pvod_neural_arrays(
                    station_id=
                        station_id,

                    horizon_min=
                        horizon_min,

                    variant_name=
                        variant_name,
                )
            )


            zero_columns = (
                prepared[
                    "zero_variance_continuous_columns"
                ]
            )


            safe_scaling_audit_rows.append(
                {
                    "station":
                        station_id,

                    "horizon_min":
                        horizon_min,

                    "variant":
                        variant_name,

                    "sequence_shape":
                        prepared[
                            "sequence"
                        ].shape,

                    "auxiliary_dim":
                        prepared[
                            "auxiliary"
                        ].shape[1],

                    "target_std":
                        prepared[
                            "target_std"
                        ],

                    "zero_variance_continuous_count":
                        len(
                            zero_columns
                        ),

                    "zero_variance_continuous_columns":
                        zero_columns,

                    "all_finite":
                        bool(
                            np.isfinite(
                                prepared[
                                    "sequence"
                                ]
                            ).all()
                            and
                            np.isfinite(
                                prepared[
                                    "auxiliary"
                                ]
                            ).all()
                            and
                            np.isfinite(
                                prepared[
                                    "target_scaled"
                                ]
                            ).all()
                        ),
                }
            )


            del prepared


pvod_safe_scaling_audit = pd.DataFrame(
    safe_scaling_audit_rows
)


# ============================================================
# 4. Strong expected-result assertions
# ============================================================

assert len(
    pvod_safe_scaling_audit
) == 60


assert (
    pvod_safe_scaling_audit[
        "all_finite"
    ]
    .all()
)


zero_cases = (
    pvod_safe_scaling_audit.loc[
        pvod_safe_scaling_audit[
            "zero_variance_continuous_count"
        ]
        >
        0
    ]
    .copy()
)


# Only the prospectively diagnosed utc_year cases are allowed.
assert set(
    zero_cases[
        "station"
    ]
) == {
    "station03",
    "station05",
}


assert (
    zero_cases[
        "zero_variance_continuous_columns"
    ]
    .apply(
        lambda value:
            value
            ==
            (
                "utc_year",
            )
    )
    .all()
)


assert len(
    zero_cases
) == 12


# ============================================================
# 5. Display
# ============================================================

print(
    "=" * 125
)

print(
    "PVOD ZERO-VARIANCE-SAFE SCALING AUDIT"
)

print(
    "=" * 125
)


print(
    "\nPolicy:",
    PVOD_ZERO_VARIANCE_CONTINUOUS_POLICY,
)


print(
    "\nZero-variance cases:"
)


display(
    zero_cases[
        [
            "station",
            "horizon_min",
            "variant",
            "zero_variance_continuous_columns",
            "target_std",
            "all_finite",
        ]
    ]
)


print(
    "\n"
    + "=" * 125
)

print(
    "SUMMARY"
)

print(
    "=" * 125
)


print(
    "Cases audited:",
    len(
        pvod_safe_scaling_audit
    )
)


print(
    "Finite scaled cases:",
    int(
        pvod_safe_scaling_audit[
            "all_finite"
        ]
        .sum()
    ),
)


print(
    "Cases using zero-variance safeguard:",
    len(
        zero_cases
    ),
)


print(
    "Affected stations:",
    sorted(
        zero_cases[
            "station"
        ]
        .unique()
        .tolist()
    ),
)


print(
    "Affected feature(s):",
    sorted(
        {
            feature

            for columns
            in zero_cases[
                "zero_variance_continuous_columns"
            ]

            for feature
            in columns
        }
    ),
)


print(
    "\nPVOD-05B ZERO-VARIANCE-SAFE SCALING: PASSED"
)


print(
    "\nFeature dimensions preserved."
)

print(
    "No feature dropped."
)

print(
    "No target scaling safeguard required."
)

print(
    "No model built."
)

print(
    "No model trained."
)

print(
    "No prediction generated."
)

PVOD ZERO-VARIANCE-SAFE SCALING AUDIT

Policy: retain_center_and_use_unit_scale

Zero-variance cases:


,station,horizon_min,variant,zero_variance_continuous_columns,target_std,all_finite
18,station03,15,F3,"(utc_year,)",0.246213,True
19,station03,15,F4_real,"(utc_year,)",0.246213,True
20,station03,30,F3,"(utc_year,)",0.246221,True
21,station03,30,F4_real,"(utc_year,)",0.246221,True
22,station03,60,F3,"(utc_year,)",0.246236,True
23,station03,60,F4_real,"(utc_year,)",0.246236,True
30,station05,15,F3,"(utc_year,)",0.277237,True
31,station05,15,F4_real,"(utc_year,)",0.277237,True
32,station05,30,F3,"(utc_year,)",0.277248,True
33,station05,30,F4_real,"(utc_year,)",0.277248,True



SUMMARY
Cases audited: 60
Finite scaled cases: 60
Cases using zero-variance safeguard: 12
Affected stations: ['station03', 'station05']
Affected feature(s): ['utc_year']

PVOD-05B ZERO-VARIANCE-SAFE SCALING: PASSED

Feature dimensions preserved.
No feature dropped.
No target scaling safeguard required.
No model built.
No model trained.
No prediction generated.


In [12]:
# ============================================================
# PVOD-05C
# Frozen model execution functions + UNTRAINED smoke tests
#
# PURPOSE
# -------
# Define the final PVOD execution machinery for:
#
#   Ridge
#   LightGBM
#   LSTM
#   TCN
#
# using the protocol frozen in PVOD-04C / 05B.
#
# THIS CELL DOES NOT FIT ANY MODEL.
#
# Neural smoke tests only:
#   - build fresh models;
#   - verify architecture;
#   - perform untrained forward passes;
#   - DO NOT compute forecasting metrics.
# ============================================================

import gc
import random
import time

import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# ============================================================
# 1. LightGBM import
# ============================================================

try:
    from lightgbm import LGBMRegressor

except ImportError as exc:
    raise ImportError(
        "lightgbm is required for the frozen PVOD protocol."
    ) from exc


# ============================================================
# 2. TensorFlow import
# ============================================================

import tensorflow as tf


# ============================================================
# 3. Frozen neural constants
# ============================================================

PVOD_NEURAL_LEARNING_RATE = 1e-3
PVOD_NEURAL_CLIPNORM = 1.0
PVOD_NEURAL_LOSS = "mae"


assert PVOD_NEURAL_SEED == 42
assert PVOD_NEURAL_BATCH_SIZE == 256

assert np.isclose(
    PVOD_NEURAL_LEARNING_RATE,
    1e-3,
)

assert np.isclose(
    PVOD_NEURAL_CLIPNORM,
    1.0,
)

assert PVOD_NEURAL_LOSS == "mae"


# ============================================================
# 4. Deterministic neural reset
# ============================================================

def reset_pvod_neural_state(
    seed: int = PVOD_NEURAL_SEED,
):

    tf.keras.backend.clear_session()

    gc.collect()

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    tf.keras.utils.set_random_seed(
        seed
    )

    try:
        tf.config.experimental.enable_op_determinism()

    except Exception:
        pass


# ============================================================
# 5. Frozen classical model factory
# ============================================================

def build_pvod_classical_model(
    model_name: str,
    horizon_min: int,
):

    if horizon_min not in (
        PVOD_EXTERNAL_HORIZONS
    ):
        raise ValueError(
            f"Unknown horizon: {horizon_min}"
        )


    if model_name == "Ridge":

        config = (
            PVOD_RIDGE_CONFIG[
                horizon_min
            ]
        )

        model = Pipeline(
            steps=[
                (
                    "scaler",
                    StandardScaler(),
                ),
                (
                    "ridge",
                    Ridge(
                        alpha=float(
                            config[
                                "alpha"
                            ]
                        )
                    ),
                ),
            ]
        )


    elif model_name == "LightGBM":

        model = LGBMRegressor(
            **PVOD_LIGHTGBM_CONFIG[
                horizon_min
            ]
        )


    else:

        raise ValueError(
            f"Unsupported classical model: "
            f"{model_name}"
        )


    return model


# ============================================================
# 6. Frozen LSTM sequence branch
# ============================================================

def build_pvod_lstm_sequence_branch(
    sequence_input,
    architecture: dict,
):

    units = int(
        architecture[
            "units"
        ]
    )

    n_layers = int(
        architecture[
            "n_recurrent_layers"
        ]
    )

    dropout = float(
        architecture[
            "dropout"
        ]
    )


    assert n_layers in {
        1,
        2,
    }


    x = sequence_input


    for layer_index in range(
        n_layers
    ):

        is_last = (
            layer_index
            ==
            n_layers - 1
        )


        x = tf.keras.layers.LSTM(
            units=units,

            return_sequences=(
                not is_last
            ),

            dropout=dropout,

            recurrent_dropout=0.0,

            name=(
                f"lstm_"
                f"{layer_index + 1}"
            ),
        )(
            x
        )


    return x


# ============================================================
# 7. Corrected causal residual TCN block
#
# Frozen Phase-2 requirements:
#
#   causal convolutions
#   He-normal initialization
#   residual connection
#   causal 1x1 projection when channel count differs
# ============================================================

def pvod_tcn_residual_block(
    x,
    filters: int,
    kernel_size: int,
    dilation_rate: int,
    dropout: float,
    convolutions_per_block: int,
    block_index: int,
):

    assert (
        convolutions_per_block
        >=
        1
    )


    residual = x


    for convolution_index in range(
        convolutions_per_block
    ):

        x = tf.keras.layers.Conv1D(
            filters=filters,

            kernel_size=kernel_size,

            dilation_rate=dilation_rate,

            padding="causal",

            activation="relu",

            kernel_initializer="he_normal",

            name=(
                f"tcn_block_{block_index}_"
                f"conv_{convolution_index + 1}_"
                f"d{dilation_rate}"
            ),
        )(
            x
        )


        if dropout > 0.0:

            x = tf.keras.layers.Dropout(
                rate=dropout,

                name=(
                    f"tcn_block_{block_index}_"
                    f"dropout_"
                    f"{convolution_index + 1}"
                ),
            )(
                x
            )


    residual_channels = (
        residual.shape[
            -1
        ]
    )


    if (
        residual_channels is None
        or
        int(
            residual_channels
        )
        !=
        filters
    ):

        residual = tf.keras.layers.Conv1D(
            filters=filters,

            kernel_size=1,

            padding="causal",

            kernel_initializer="he_normal",

            name=(
                f"tcn_block_{block_index}_"
                "residual_projection"
            ),
        )(
            residual
        )


    x = tf.keras.layers.Add(
        name=(
            f"tcn_block_{block_index}_"
            "residual_add"
        )
    )(
        [
            x,
            residual,
        ]
    )


    x = tf.keras.layers.Activation(
        "relu",

        name=(
            f"tcn_block_{block_index}_"
            "output_relu"
        ),
    )(
        x
    )


    return x


# ============================================================
# 8. Corrected TCN sequence branch
#
# IMPORTANT:
# Final representation is the LAST temporal position.
#
# No GlobalAveragePooling1D.
# ============================================================

def build_pvod_tcn_sequence_branch(
    sequence_input,
    architecture: dict,
):

    filters = int(
        architecture[
            "filters"
        ]
    )

    kernel_size = int(
        architecture[
            "kernel_size"
        ]
    )

    dropout = float(
        architecture[
            "dropout"
        ]
    )

    dilations = tuple(
        int(
            value
        )
        for value
        in architecture[
            "dilations"
        ]
    )

    convolutions_per_block = int(
        architecture.get(
            "convolutions_per_block",
            2,
        )
    )


    x = sequence_input


    for block_index, dilation_rate in enumerate(
        dilations,
        start=1,
    ):

        x = pvod_tcn_residual_block(
            x=x,

            filters=filters,

            kernel_size=kernel_size,

            dilation_rate=dilation_rate,

            dropout=dropout,

            convolutions_per_block=(
                convolutions_per_block
            ),

            block_index=block_index,
        )


    # --------------------------------------------------------
    # Correct Notebook-11 / Phase-2 behavior:
    # LAST temporal position only.
    # --------------------------------------------------------

    x = tf.keras.layers.Lambda(
        lambda tensor:
            tensor[
                :,
                -1,
                :,
            ],

        name="tcn_last_position",
    )(
        x
    )


    return x


# ============================================================
# 9. Frozen PVOD neural model factory
# ============================================================

def build_pvod_neural_model(
    model_name: str,
    auxiliary_dim: int,
    architecture: dict,
    seed: int = PVOD_NEURAL_SEED,
):

    assert model_name in {
        "LSTM",
        "TCN",
    }

    assert auxiliary_dim > 0


    reset_pvod_neural_state(
        seed=seed
    )


    # --------------------------------------------------------
    # INPUT 1 — 60-minute historical sequence
    #
    # PVOD geometry:
    #
    #     5 time points
    #     14 dynamic channels
    # --------------------------------------------------------

    sequence_input = tf.keras.layers.Input(
        shape=(
            PVOD_SEQUENCE_LENGTH,
            PVOD_SEQUENCE_CHANNEL_COUNT,
        ),

        dtype=tf.float32,

        name="sequence_input",
    )


    if model_name == "LSTM":

        sequence_embedding = (
            build_pvod_lstm_sequence_branch(
                sequence_input=
                    sequence_input,

                architecture=
                    architecture,
            )
        )


    else:

        sequence_embedding = (
            build_pvod_tcn_sequence_branch(
                sequence_input=
                    sequence_input,

                architecture=
                    architecture,
            )
        )


    # --------------------------------------------------------
    # INPUT 2 — deterministic auxiliary representation
    # --------------------------------------------------------

    auxiliary_input = tf.keras.layers.Input(
        shape=(
            auxiliary_dim,
        ),

        dtype=tf.float32,

        name="auxiliary_input",
    )


    # --------------------------------------------------------
    # Fusion
    # --------------------------------------------------------

    fused = tf.keras.layers.Concatenate(
        name="sequence_auxiliary_fusion"
    )(
        [
            sequence_embedding,
            auxiliary_input,
        ]
    )


    fusion_dense_units = int(
        architecture[
            "fusion_dense_units"
        ]
    )


    fused = tf.keras.layers.Dense(
        units=fusion_dense_units,

        activation="relu",

        name="fusion_dense",
    )(
        fused
    )


    output = tf.keras.layers.Dense(
        units=1,

        activation="linear",

        name="scaled_power_output",
    )(
        fused
    )


    model = tf.keras.Model(
        inputs=[
            sequence_input,
            auxiliary_input,
        ],

        outputs=output,

        name=(
            f"pvod_"
            f"{model_name.lower()}"
        ),
    )


    optimizer = (
        tf.keras.optimizers.Adam(
            learning_rate=(
                PVOD_NEURAL_LEARNING_RATE
            ),

            clipnorm=(
                PVOD_NEURAL_CLIPNORM
            ),
        )
    )


    model.compile(
        optimizer=optimizer,
        loss=PVOD_NEURAL_LOSS,
    )


    return model


# ============================================================
# 10. Frozen neural architecture registry helper
# ============================================================

def get_pvod_neural_architecture(
    model_name: str,
):

    if model_name == "LSTM":

        return (
            PVOD_LSTM_ARCHITECTURE
            .copy()
        )


    if model_name == "TCN":

        architecture = (
            PVOD_TCN_ARCHITECTURE
            .copy()
        )

        architecture[
            "dilations"
        ] = list(
            architecture[
                "dilations"
            ]
        )

        return architecture


    raise ValueError(
        f"Unsupported neural model: "
        f"{model_name}"
    )


# ============================================================
# 11. Frozen epoch helper
# ============================================================

def get_pvod_frozen_epochs(
    model_name: str,
    horizon_min: int,
) -> int:

    if model_name == "LSTM":

        return int(
            PVOD_LSTM_EPOCHS[
                horizon_min
            ]
        )


    if model_name == "TCN":

        return int(
            PVOD_TCN_EPOCHS[
                horizon_min
            ]
        )


    raise ValueError(
        f"Unsupported neural model: "
        f"{model_name}"
    )


# ============================================================
# 12. FINAL classical execution helper
#
# FUNCTION DEFINITION ONLY.
#
# Fit population:
#     train + validation
#
# Test remains held out.
# ============================================================

def fit_predict_pvod_classical(
    station_id: str,
    horizon_min: int,
    model_name: str,
    variant_name: str,
):

    if variant_name == "F3":
        feature_block = "F3"

    elif variant_name == "F4_real":
        feature_block = "F4"

    else:
        raise ValueError(
            f"Unsupported variant: "
            f"{variant_name}"
        )


    design = materialize_pvod_design(
        station_id=
            station_id,

        horizon_min=
            horizon_min,

        feature_block=
            feature_block,
    )


    X = design[
        "X"
    ]

    y = design[
        "y_pu"
    ]

    meta = design[
        "meta"
    ]


    trainval_mask = (
        meta[
            "split"
        ]
        .isin(
            [
                "train",
                "validation",
            ]
        )
        .to_numpy()
    )


    test_mask = (
        meta[
            "split"
        ]
        .eq(
            "test"
        )
        .to_numpy()
    )


    X_trainval = (
        X.iloc[
            np.flatnonzero(
                trainval_mask
            )
        ]
    )


    y_trainval = (
        y.iloc[
            np.flatnonzero(
                trainval_mask
            )
        ]
    )


    X_test = (
        X.iloc[
            np.flatnonzero(
                test_mask
            )
        ]
    )


    y_test = (
        y.iloc[
            np.flatnonzero(
                test_mask
            )
        ]
    )


    model = (
        build_pvod_classical_model(
            model_name=
                model_name,

            horizon_min=
                horizon_min,
        )
    )


    fit_start = (
        time.perf_counter()
    )


    model.fit(
        X_trainval,
        y_trainval,
    )


    fit_seconds = float(
        time.perf_counter()
        -
        fit_start
    )


    predict_start = (
        time.perf_counter()
    )


    prediction_pu = np.asarray(
        model.predict(
            X_test
        ),
        dtype=np.float64,
    ).reshape(
        -1
    )


    predict_seconds = float(
        time.perf_counter()
        -
        predict_start
    )


    assert prediction_pu.shape == (
        len(
            y_test
        ),
    )


    assert np.isfinite(
        prediction_pu
    ).all()


    test_meta = (
        meta.iloc[
            np.flatnonzero(
                test_mask
            )
        ]
        .copy()
    )


    return {
        "model":
            model,

        "prediction_pu":
            prediction_pu,

        "target_pu":
            y_test.to_numpy(
                dtype=np.float64
            ),

        "target_MW":
            design[
                "y_MW"
            ]
            .iloc[
                np.flatnonzero(
                    test_mask
                )
            ]
            .to_numpy(
                dtype=np.float64
            ),

        "meta":
            test_meta,

        "fit_seconds":
            fit_seconds,

        "predict_seconds":
            predict_seconds,
    }


# ============================================================
# 13. FINAL neural execution helper
#
# FUNCTION DEFINITION ONLY.
#
# Scaling:
#     TRAIN ONLY
#
# Model refit:
#     TRAIN + VALIDATION
#
# Epoch count:
#     frozen from SOLETE
#
# Test:
#     completely held out
# ============================================================

def fit_predict_pvod_neural(
    station_id: str,
    horizon_min: int,
    model_name: str,
    variant_name: str,
    verbose: int = 0,
):

    assert model_name in {
        "LSTM",
        "TCN",
    }


    prepared = (
        prepare_pvod_neural_arrays(
            station_id=
                station_id,

            horizon_min=
                horizon_min,

            variant_name=
                variant_name,
        )
    )


    sequence_all = np.asarray(
        prepared[
            "sequence"
        ],
        dtype=np.float32,
    )


    auxiliary_all = np.asarray(
        prepared[
            "auxiliary"
        ],
        dtype=np.float32,
    )


    target_scaled_all = np.asarray(
        prepared[
            "target_scaled"
        ],
        dtype=np.float32,
    )


    target_pu_all = np.asarray(
        prepared[
            "target_pu"
        ],
        dtype=np.float64,
    )


    target_MW_all = np.asarray(
        prepared[
            "target_MW"
        ],
        dtype=np.float64,
    )


    train_mask = (
        prepared[
            "train_mask"
        ]
    )


    validation_mask = (
        prepared[
            "validation_mask"
        ]
    )


    test_mask = (
        prepared[
            "test_mask"
        ]
    )


    trainval_mask = (
        train_mask
        |
        validation_mask
    )


    # --------------------------------------------------------
    # Train + validation refit population
    # --------------------------------------------------------

    trainval_sequence = (
        sequence_all[
            trainval_mask
        ]
    )


    trainval_auxiliary = (
        auxiliary_all[
            trainval_mask
        ]
    )


    trainval_target_scaled = (
        target_scaled_all[
            trainval_mask
        ]
    )


    test_sequence = (
        sequence_all[
            test_mask
        ]
    )


    test_auxiliary = (
        auxiliary_all[
            test_mask
        ]
    )


    target_test_pu = (
        target_pu_all[
            test_mask
        ]
    )


    target_test_MW = (
        target_MW_all[
            test_mask
        ]
    )


    auxiliary_dim = int(
        auxiliary_all.shape[
            1
        ]
    )


    architecture = (
        get_pvod_neural_architecture(
            model_name
        )
    )


    frozen_epochs = (
        get_pvod_frozen_epochs(
            model_name=
                model_name,

            horizon_min=
                horizon_min,
        )
    )


    model = (
        build_pvod_neural_model(
            model_name=
                model_name,

            auxiliary_dim=
                auxiliary_dim,

            architecture=
                architecture,

            seed=
                PVOD_NEURAL_SEED,
        )
    )


    # --------------------------------------------------------
    # Preserve historical model-specific target shape
    # --------------------------------------------------------

    if model_name == "LSTM":

        final_fit_target = (
            trainval_target_scaled
            .reshape(
                -1
            )
        )


    else:

        final_fit_target = (
            trainval_target_scaled
            .reshape(
                -1,
                1,
            )
        )


    fit_start = (
        time.perf_counter()
    )


    history_object = model.fit(
        x=[
            trainval_sequence,
            trainval_auxiliary,
        ],

        y=final_fit_target,

        epochs=frozen_epochs,

        batch_size=
            PVOD_NEURAL_BATCH_SIZE,

        shuffle=True,

        callbacks=None,

        validation_data=None,

        verbose=verbose,
    )


    fit_seconds = float(
        time.perf_counter()
        -
        fit_start
    )


    history_df = pd.DataFrame(
        history_object.history
    )


    assert len(
        history_df
    ) == frozen_epochs


    assert (
        "loss"
        in
        history_df.columns
    )


    assert (
        "val_loss"
        not in
        history_df.columns
    )


    # --------------------------------------------------------
    # Held-out TEST prediction
    # --------------------------------------------------------

    predict_start = (
        time.perf_counter()
    )


    prediction_scaled = np.asarray(
        model.predict(
            [
                test_sequence,
                test_auxiliary,
            ],

            batch_size=
                PVOD_NEURAL_BATCH_SIZE,

            verbose=0,
        ),

        dtype=np.float64,
    ).reshape(
        -1
    )


    predict_seconds = float(
        time.perf_counter()
        -
        predict_start
    )


    prediction_pu = (
        prediction_scaled
        *
        float(
            prepared[
                "target_std"
            ]
        )
        +
        float(
            prepared[
                "target_mean"
            ]
        )
    )


    assert prediction_pu.shape == (
        len(
            target_test_pu
        ),
    )


    assert np.isfinite(
        prediction_pu
    ).all()


    test_meta = (
        prepared[
            "meta"
        ]
        .loc[
            test_mask
        ]
        .copy()
    )


    return {
        "model":
            model,

        "history":
            history_df,

        "prediction_pu":
            prediction_pu,

        "target_pu":
            target_test_pu,

        "target_MW":
            target_test_MW,

        "meta":
            test_meta,

        "fit_seconds":
            fit_seconds,

        "predict_seconds":
            predict_seconds,

        "frozen_epochs":
            frozen_epochs,

        "zero_variance_continuous_columns":
            prepared[
                "zero_variance_continuous_columns"
            ],
    }


# ============================================================
# 14. CLASSICAL structural smoke test
#
# Build only. DO NOT fit.
# ============================================================

ridge_smoke = (
    build_pvod_classical_model(
        model_name=
            "Ridge",

        horizon_min=
            60,
    )
)


lgbm_smoke = (
    build_pvod_classical_model(
        model_name=
            "LightGBM",

        horizon_min=
            60,
    )
)


assert isinstance(
    ridge_smoke,
    Pipeline,
)


assert np.isclose(
    ridge_smoke[
        "ridge"
    ].alpha,
    0.01,
)


assert isinstance(
    lgbm_smoke,
    LGBMRegressor,
)


# ============================================================
# 15. NEURAL untrained smoke test
#
# Use station00 / 60 min.
# No fit occurs.
# No metric is calculated.
# ============================================================

smoke_rows = []


for model_name in [
    "LSTM",
    "TCN",
]:

    architecture = (
        get_pvod_neural_architecture(
            model_name
        )
    )


    for variant_name in (
        PVOD_PRIMARY_VARIANTS
    ):

        prepared = (
            prepare_pvod_neural_arrays(
                station_id=
                    "station00",

                horizon_min=
                    60,

                variant_name=
                    variant_name,
            )
        )


        auxiliary_dim = int(
            prepared[
                "auxiliary"
            ]
            .shape[
                1
            ]
        )


        model = (
            build_pvod_neural_model(
                model_name=
                    model_name,

                auxiliary_dim=
                    auxiliary_dim,

                architecture=
                    architecture,

                seed=
                    PVOD_NEURAL_SEED,
            )
        )


        smoke_sequence = (
            prepared[
                "sequence"
            ][
                :8
            ]
        )


        smoke_auxiliary = (
            prepared[
                "auxiliary"
            ][
                :8
            ]
        )


        # UNTRAINED forward pass only.
        smoke_prediction = np.asarray(
            model(
                [
                    smoke_sequence,
                    smoke_auxiliary,
                ],

                training=False,
            )
            .numpy(),
            dtype=float,
        ).reshape(
            -1
        )


        assert smoke_prediction.shape == (
            8,
        )


        assert np.isfinite(
            smoke_prediction
        ).all()


        # ----------------------------------------------------
        # Architecture-specific assertions
        # ----------------------------------------------------

        if model_name == "LSTM":

            lstm_layers = [
                layer
                for layer
                in model.layers

                if isinstance(
                    layer,
                    tf.keras.layers.LSTM,
                )
            ]


            assert len(
                lstm_layers
            ) == 1


            assert (
                lstm_layers[
                    0
                ].units
                ==
                32
            )


            assert np.isclose(
                lstm_layers[
                    0
                ].dropout,
                0.0,
            )


            assert np.isclose(
                lstm_layers[
                    0
                ].recurrent_dropout,
                0.0,
            )


        else:

            conv_layers = [
                layer
                for layer
                in model.layers

                if isinstance(
                    layer,
                    tf.keras.layers.Conv1D,
                )
            ]


            # 6 causal block convolutions
            # + 1 residual projection in first block.
            assert len(
                conv_layers
            ) == 7


            assert all(
                layer.padding
                ==
                "causal"

                for layer
                in conv_layers
            )


            assert any(
                layer.name
                ==
                "tcn_last_position"

                for layer
                in model.layers
            )


            assert not any(
                isinstance(
                    layer,
                    tf.keras.layers.GlobalAveragePooling1D,
                )

                for layer
                in model.layers
            )


        smoke_rows.append(
            {
                "model":
                    model_name,

                "variant":
                    variant_name,

                "sequence_shape":
                    tuple(
                        smoke_sequence.shape
                    ),

                "auxiliary_dim":
                    auxiliary_dim,

                "parameters":
                    int(
                        model.count_params()
                    ),

                "output_shape":
                    tuple(
                        model.output.shape
                    ),

                "frozen_epochs":
                    get_pvod_frozen_epochs(
                        model_name=
                            model_name,

                        horizon_min=
                            60,
                    ),

                "untrained_forward_finite":
                    bool(
                        np.isfinite(
                            smoke_prediction
                        )
                        .all()
                    ),
            }
        )


        del model
        del prepared

        gc.collect()


pvod_execution_smoke_audit = (
    pd.DataFrame(
        smoke_rows
    )
)


# ============================================================
# 16. Display
# ============================================================

print(
    "=" * 125
)

print(
    "PVOD FROZEN EXECUTION SMOKE AUDIT"
)

print(
    "=" * 125
)


display(
    pvod_execution_smoke_audit
)


print(
    "\nClassical factories:"
)

print(
    "  Ridge: PASS"
)

print(
    "  LightGBM: PASS"
)


print(
    "\nNeural constants:"
)

print(
    "  learning_rate =",
    PVOD_NEURAL_LEARNING_RATE,
)

print(
    "  clipnorm      =",
    PVOD_NEURAL_CLIPNORM,
)

print(
    "  loss          =",
    PVOD_NEURAL_LOSS,
)


print(
    "\nPVOD-05C EXECUTION FUNCTIONS + UNTRAINED SMOKE TESTS: PASSED"
)


print(
    "\nTraining functions defined but NOT executed."
)

print(
    "No classical model fitted."
)

print(
    "No neural model fitted."
)

print(
    "No forecasting metric calculated."
)

print(
    "No external F3/F4 performance observed."
)

2026-10-01 20:44:35.086338: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-10-01 20:44:35.086400: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-10-01 20:44:35.086410: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
I0000 00:00:1790867675.086619 52940962 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1790867675.086688 52940962 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


PVOD FROZEN EXECUTION SMOKE AUDIT


,model,variant,sequence_shape,auxiliary_dim,parameters,output_shape,frozen_epochs,untrained_forward_finite
0,LSTM,F3,"(8, 5, 14)",30,8065,"(None, 1)",2,True
1,LSTM,F4_real,"(8, 5, 14)",127,11169,"(None, 1)",2,True
2,TCN,F3,"(8, 5, 14)",30,19425,"(None, 1)",1,True
3,TCN,F4_real,"(8, 5, 14)",127,22529,"(None, 1)",1,True



Classical factories:
  Ridge: PASS
  LightGBM: PASS

Neural constants:
  learning_rate = 0.001
  clipnorm      = 1.0
  loss          = mae

PVOD-05C EXECUTION FUNCTIONS + UNTRAINED SMOKE TESTS: PASSED

Training functions defined but NOT executed.
No classical model fitted.
No neural model fitted.
No forecasting metric calculated.
No external F3/F4 performance observed.


In [13]:
# ============================================================
# PVOD-06A-PREFLIGHT
# Verify notebook state after kernel restart
# ============================================================

required_objects = [
    "PROJECT_ROOT",
    "pvod_clean",
    "pvod_site_registry",
    "pvod_f1_global",
    "pvod_f2_by_station",
    "pvod_f3_global",
    "pvod_f4_onehot_global",
    "pvod_sample_index",
    "PVOD_EXTERNAL_STATIONS",
    "PVOD_EXTERNAL_HORIZONS",
    "PVOD_PRIMARY_VARIANTS",
    "PVOD_EVALUATION_SCOPES",
    "PVOD_RIDGE_CONFIG",
    "PVOD_LIGHTGBM_CONFIG",
    "materialize_pvod_design",
    "fit_predict_pvod_classical",
    "prepare_pvod_neural_arrays",
    "build_pvod_neural_model",
    "fit_predict_pvod_neural",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

print(
    "Required objects:",
    len(required_objects)
)

print(
    "Missing objects:",
    len(missing_objects)
)

if missing_objects:
    print("\nMissing:")
    for name in missing_objects:
        print(" ", name)

    raise RuntimeError(
        "Notebook state is incomplete. "
        "Run all cells above PVOD-06A."
    )

print(
    "\nPVOD-06A notebook-state preflight: PASSED"
)

print(
    "Stations:",
    PVOD_EXTERNAL_STATIONS,
)

print(
    "Horizons:",
    PVOD_EXTERNAL_HORIZONS,
)

print(
    "Variants:",
    PVOD_PRIMARY_VARIANTS,
)

print(
    "\nSafe to run PVOD-06A."
)

Required objects: 19
Missing objects: 0

PVOD-06A notebook-state preflight: PASSED
Stations: ['station00', 'station01', 'station02', 'station03', 'station04', 'station05', 'station06', 'station07', 'station08', 'station09']
Horizons: [15, 30, 60]
Variants: ['F3', 'F4_real']

Safe to run PVOD-06A.


In [14]:
# ============================================================
# PVOD-06A
# PRIMARY classical external-validation execution — canonical precision
#
# Models:
#     Ridge
#     LightGBM
#
# Representations:
#     F3
#     F4_real
#
# Stations:
#     10
#
# Horizons:
#     15 / 30 / 60 min
#
# Total fitted models:
#
#     2 models
#   x 10 stations
#   x 3 horizons
#   x 2 representations
#   = 120 fits
#
# IMPORTANT
# ---------
# Protocol is already frozen.
#
# No hyperparameter tuning.
# No validation-based selection.
# No prediction clipping.
# No bootstrap yet.
# No neural training yet.
#
# Precision policy:
# - Ridge is fit directly from genuine source-float64 features.
# - LightGBM retains the frozen float32 path.
# - No provisional Ridge rerun is required.
# ============================================================

import gc
import time

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)


# ============================================================
# 1. Frozen execution registry
# ============================================================

PVOD_CLASSICAL_MODELS = [
    "Ridge",
    "LightGBM",
]


assert set(
    PVOD_CLASSICAL_MODELS
) == {
    "Ridge",
    "LightGBM",
}


expected_fit_count = (
    len(
        PVOD_CLASSICAL_MODELS
    )
    *
    len(
        PVOD_EXTERNAL_STATIONS
    )
    *
    len(
        PVOD_EXTERNAL_HORIZONS
    )
    *
    len(
        PVOD_PRIMARY_VARIANTS
    )
)


assert expected_fit_count == 120


# ============================================================
# 2. Metric helper
# ============================================================

def calculate_pvod_regression_metrics(
    y_true_pu,
    y_pred_pu,
    y_true_MW,
    y_pred_MW,
):

    y_true_pu = np.asarray(
        y_true_pu,
        dtype=np.float64,
    ).reshape(-1)


    y_pred_pu = np.asarray(
        y_pred_pu,
        dtype=np.float64,
    ).reshape(-1)


    y_true_MW = np.asarray(
        y_true_MW,
        dtype=np.float64,
    ).reshape(-1)


    y_pred_MW = np.asarray(
        y_pred_MW,
        dtype=np.float64,
    ).reshape(-1)


    assert (
        len(y_true_pu)
        ==
        len(y_pred_pu)
        ==
        len(y_true_MW)
        ==
        len(y_pred_MW)
    )


    assert len(
        y_true_pu
    ) > 1


    assert np.isfinite(
        y_true_pu
    ).all()


    assert np.isfinite(
        y_pred_pu
    ).all()


    assert np.isfinite(
        y_true_MW
    ).all()


    assert np.isfinite(
        y_pred_MW
    ).all()


    return {
        "n":
            int(
                len(
                    y_true_pu
                )
            ),

        "MAE_power_pu":
            float(
                mean_absolute_error(
                    y_true_pu,
                    y_pred_pu,
                )
            ),

        "RMSE_power_pu":
            float(
                np.sqrt(
                    mean_squared_error(
                        y_true_pu,
                        y_pred_pu,
                    )
                )
            ),

        "R2_power_pu":
            float(
                r2_score(
                    y_true_pu,
                    y_pred_pu,
                )
            ),

        "MAE_MW":
            float(
                mean_absolute_error(
                    y_true_MW,
                    y_pred_MW,
                )
            ),

        "RMSE_MW":
            float(
                np.sqrt(
                    mean_squared_error(
                        y_true_MW,
                        y_pred_MW,
                    )
                )
            ),

        "R2_MW":
            float(
                r2_score(
                    y_true_MW,
                    y_pred_MW,
                )
            ),
    }


# ============================================================
# 2. Reconstruct F0 directly from original cleaned source
#
# IMPORTANT
# ---------
# Do NOT use pvod_f0_by_station values here except to verify
# column ordering.
#
# The source data in pvod_indexed retain float64/int precision.
# ============================================================

def build_pvod_f0_float64(
    station_id: str,
    issue_times: pd.DatetimeIndex,
) -> pd.DataFrame:

    source = (
        pvod_indexed[
            station_id
        ]
    )


    assert isinstance(
        source.index,
        pd.DatetimeIndex,
    )


    assert source.index.is_unique


    issue_times = pd.DatetimeIndex(
        issue_times
    )


    columns = {}


    # --------------------------------------------------------
    # 60-minute recent history
    # --------------------------------------------------------

    for source_column in (
        PVOD_DYNAMIC_SOURCE_COLUMNS
    ):

        for lag_min in (
            PVOD_HISTORY_OFFSETS_MIN
        ):

            feature_name = (
                f"{source_column}"
                f"__lag_{lag_min:04d}m"
            )


            source_times = (
                issue_times
                -
                pd.Timedelta(
                    minutes=int(
                        lag_min
                    )
                )
            )


            values = (
                source[
                    source_column
                ]
                .reindex(
                    source_times
                )
                .to_numpy(
                    dtype=np.float64
                )
            )


            assert np.isfinite(
                values
            ).all(), (
                f"{station_id}: missing/nonfinite "
                f"{feature_name}"
            )


            columns[
                feature_name
            ] = values


    # --------------------------------------------------------
    # Previous-day PV
    # --------------------------------------------------------

    daily_feature_name = (
        f"power_MW"
        f"__lag_{PVOD_DAILY_POWER_LAG_MIN:04d}m"
    )


    daily_times = (
        issue_times
        -
        pd.Timedelta(
            minutes=int(
                PVOD_DAILY_POWER_LAG_MIN
            )
        )
    )


    daily_values = (
        source[
            "power_MW"
        ]
        .reindex(
            daily_times
        )
        .to_numpy(
            dtype=np.float64
        )
    )


    assert np.isfinite(
        daily_values
    ).all()


    columns[
        daily_feature_name
    ] = daily_values


    f0 = pd.DataFrame(
        columns,
        index=issue_times,
        dtype=np.float64,
    )


    # --------------------------------------------------------
    # Exact canonical F0 column ordering
    # --------------------------------------------------------

    canonical_columns = list(
        pvod_f0_by_station[
            station_id
        ]
        .columns
    )


    assert set(
        f0.columns
    ) == set(
        canonical_columns
    )


    f0 = f0[
        canonical_columns
    ]


    assert f0.shape == (
        len(
            issue_times
        ),
        71,
    )


    assert np.isfinite(
        f0.to_numpy(
            dtype=np.float64
        )
    ).all()


    assert all(
        dtype
        ==
        np.dtype(
            "float64"
        )

        for dtype
        in f0.dtypes
    )


    return f0


# ============================================================
# 3. Genuine float64 Ridge materializer
# ============================================================

def materialize_pvod_ridge_float64_design(
    station_id: str,
    horizon_min: int,
    variant_name: str,
):

    if variant_name == "F3":

        feature_block = "F3"

        expected_dim = (
            PVOD_EXPECTED_DIMS[
                "F3"
            ]
        )


    elif variant_name == "F4_real":

        feature_block = "F4"

        expected_dim = (
            PVOD_EXPECTED_DIMS[
                "F4"
            ]
        )


    else:

        raise ValueError(
            f"Unsupported variant: "
            f"{variant_name}"
        )


    samples = (
        pvod_sample_index[
            (
                station_id,
                horizon_min,
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    issue_times = pd.DatetimeIndex(
        samples[
            "issue_time_utc"
        ]
    )


    target_times = pd.DatetimeIndex(
        samples[
            "target_time_utc"
        ]
    )


    # ========================================================
    # F0 — rebuilt from original source precision
    # ========================================================

    blocks = [
        build_pvod_f0_float64(
            station_id=
                station_id,

            issue_times=
                issue_times,
        )
    ]


    # ========================================================
    # F1 — target-valid-time Gregorian
    # ========================================================

    f1 = (
        pvod_f1_global
        .loc[
            target_times
        ]
        .copy()
    )


    f1.index = (
        issue_times
    )


    blocks.append(
        f1
    )


    # ========================================================
    # F2 — target-valid-time site solar
    # ========================================================

    f2 = (
        pvod_f2_by_station[
            station_id
        ]
        .loc[
            target_times
        ]
        .copy()
    )


    f2.index = (
        issue_times
    )


    blocks.append(
        f2
    )


    # ========================================================
    # F3 — continuous astronomy
    # ========================================================

    f3 = (
        pvod_f3_global
        .loc[
            target_times
        ]
        .copy()
    )


    f3.index = (
        issue_times
    )


    blocks.append(
        f3
    )


    # ========================================================
    # F4 — Panchang one-hot
    # ========================================================

    if feature_block == "F4":

        f4 = (
            pvod_f4_onehot_global
            .loc[
                target_times
            ]
            .copy()
        )


        f4.index = (
            issue_times
        )


        blocks.append(
            f4
        )


    # ========================================================
    # Combine in genuine float64
    # ========================================================

    X = pd.concat(
        blocks,
        axis=1,
    )


    assert X.shape == (
        len(
            samples
        ),
        expected_dim,
    )


    assert X.columns.is_unique


    assert X.index.equals(
        issue_times
    )


    # Explicit conversion here is only a numeric container
    # conversion. No source value was previously rounded to
    # float32 in this construction.
    X = X.astype(
        np.float64
    )


    assert np.isfinite(
        X.to_numpy()
    ).all()


    y_pu = pd.Series(
        samples[
            "target_power_pu"
        ]
        .to_numpy(
            dtype=np.float64
        ),
        index=issue_times,
        name="target_power_pu",
    )


    y_MW = pd.Series(
        samples[
            "target_power_MW"
        ]
        .to_numpy(
            dtype=np.float64
        ),
        index=issue_times,
        name="target_power_MW",
    )


    meta = (
        samples
        .set_index(
            "issue_time_utc",
            drop=False,
        )
    )


    return {
        "X":
            X,

        "y_pu":
            y_pu,

        "y_MW":
            y_MW,

        "meta":
            meta,
    }

# ============================================================
# Canonical classical fit helper
#
# Ridge:
#   genuine source-float64 materialization.
#
# LightGBM:
#   original float32 materialization.
# ============================================================

def fit_predict_pvod_classical_canonical(
    station_id: str,
    horizon_min: int,
    model_name: str,
    variant_name: str,
):

    if variant_name == "F3":
        feature_block = "F3"
    elif variant_name == "F4_real":
        feature_block = "F4"
    else:
        raise ValueError(
            f"Unsupported primary variant: {variant_name}"
        )

    if model_name == "Ridge":
        design = materialize_pvod_ridge_float64_design(
            station_id=station_id,
            horizon_min=horizon_min,
            variant_name=variant_name,
        )
        X = design["X"]
        assert all(
            dtype == np.dtype("float64")
            for dtype in X.dtypes
        )

    elif model_name == "LightGBM":
        design = materialize_pvod_design(
            station_id=station_id,
            horizon_min=horizon_min,
            feature_block=feature_block,
        )
        X = design["X"]
        assert all(
            dtype == np.dtype("float32")
            for dtype in X.dtypes
        )

    else:
        raise ValueError(
            f"Unsupported classical model: {model_name}"
        )

    y_pu = design["y_pu"]
    y_MW = design["y_MW"]
    meta = design["meta"]

    trainval_mask = (
        meta["split"]
        .isin(["train", "validation"])
        .to_numpy()
    )
    test_mask = (
        meta["split"]
        .eq("test")
        .to_numpy()
    )

    trainval_positions = np.flatnonzero(trainval_mask)
    test_positions = np.flatnonzero(test_mask)

    X_trainval = X.iloc[trainval_positions]
    X_test = X.iloc[test_positions]
    y_trainval = y_pu.iloc[trainval_positions]

    target_test_pu = (
        y_pu.iloc[test_positions]
        .to_numpy(dtype=np.float64)
    )
    target_test_MW = (
        y_MW.iloc[test_positions]
        .to_numpy(dtype=np.float64)
    )
    test_meta = (
        meta.iloc[test_positions]
        .copy()
        .reset_index(drop=True)
    )

    model = build_pvod_classical_model(
        model_name=model_name,
        horizon_min=horizon_min,
    )

    fit_start = time.perf_counter()
    model.fit(X_trainval, y_trainval)
    fit_seconds = float(time.perf_counter() - fit_start)

    predict_start = time.perf_counter()
    prediction_pu = np.asarray(
        model.predict(X_test),
        dtype=np.float64,
    ).reshape(-1)
    predict_seconds = float(
        time.perf_counter() - predict_start
    )

    assert prediction_pu.shape == target_test_pu.shape
    assert np.isfinite(prediction_pu).all()

    result = {
        "prediction_pu": prediction_pu,
        "target_pu": target_test_pu,
        "target_MW": target_test_MW,
        "meta": test_meta,
        "fit_seconds": fit_seconds,
        "predict_seconds": predict_seconds,
    }

    del model
    del design
    gc.collect()

    return result


# ============================================================
# 3. Run complete primary classical grid
# ============================================================

prediction_frames = []
metric_rows = []
runtime_rows = []


grid_start = time.perf_counter()

fit_counter = 0


for model_name in (
    PVOD_CLASSICAL_MODELS
):

    print(
        "\n"
        + "=" * 125
    )

    print(
        f"MODEL: {model_name}"
    )

    print(
        "=" * 125
    )


    for horizon_min in (
        PVOD_EXTERNAL_HORIZONS
    ):

        print(
            f"\n--- Horizon: "
            f"{horizon_min} min ---"
        )


        for station_id in (
            PVOD_EXTERNAL_STATIONS
        ):

            station_start = (
                time.perf_counter()
            )


            station_variant_results = {}


            for variant_name in (
                PVOD_PRIMARY_VARIANTS
            ):

                fit_counter += 1


                result = (
                    fit_predict_pvod_classical_canonical(
                        station_id=
                            station_id,

                        horizon_min=
                            horizon_min,

                        model_name=
                            model_name,

                        variant_name=
                            variant_name,
                    )
                )


                prediction_pu = (
                    np.asarray(
                        result[
                            "prediction_pu"
                        ],
                        dtype=np.float64,
                    )
                    .reshape(
                        -1
                    )
                )


                target_pu = (
                    np.asarray(
                        result[
                            "target_pu"
                        ],
                        dtype=np.float64,
                    )
                    .reshape(
                        -1
                    )
                )


                target_MW = (
                    np.asarray(
                        result[
                            "target_MW"
                        ],
                        dtype=np.float64,
                    )
                    .reshape(
                        -1
                    )
                )


                test_meta = (
                    result[
                        "meta"
                    ]
                    .copy()
                    .reset_index(
                        drop=True
                    )
                )


                assert (
                    len(
                        prediction_pu
                    )
                    ==
                    len(
                        target_pu
                    )
                    ==
                    len(
                        target_MW
                    )
                    ==
                    len(
                        test_meta
                    )
                )


                # --------------------------------------------
                # Nominal station capacity
                #
                # power_pu was defined using this same
                # denominator.
                # --------------------------------------------

                capacity_MW = float(
                    pvod_site_registry
                    .set_index(
                        "station"
                    )
                    .loc[
                        station_id,
                        "capacity_MW",
                    ]
                )


                prediction_MW = (
                    prediction_pu
                    *
                    capacity_MW
                )


                # --------------------------------------------
                # Strong target consistency
                # --------------------------------------------

                assert np.allclose(
                    target_MW,
                    (
                        target_pu
                        *
                        capacity_MW
                    ),
                    atol=1e-6,
                    rtol=1e-6,
                )


                # --------------------------------------------
                # Construct row-level prediction table
                # --------------------------------------------

                prediction_frame = pd.DataFrame(
                    {
                        "model":
                            model_name,

                        "station":
                            station_id,

                        "horizon_min":
                            horizon_min,

                        "variant":
                            variant_name,

                        "issue_time_utc":
                            pd.DatetimeIndex(
                                test_meta[
                                    "issue_time_utc"
                                ]
                            ),

                        "target_time_utc":
                            pd.DatetimeIndex(
                                test_meta[
                                    "target_time_utc"
                                ]
                            ),

                        "target_solar_daylight":
                            test_meta[
                                "target_solar_daylight"
                            ]
                            .to_numpy(
                                dtype=np.int8
                            ),

                        "target_power_pu":
                            target_pu,

                        "prediction_power_pu":
                            prediction_pu,

                        "target_power_MW":
                            target_MW,

                        "prediction_power_MW":
                            prediction_MW,

                        "capacity_MW":
                            capacity_MW,
                    }
                )


                prediction_frame[
                    "target_time_china"
                ] = (
                    prediction_frame[
                        "target_time_utc"
                    ]
                    .dt
                    .tz_convert(
                        "Asia/Shanghai"
                    )
                )


                prediction_frame[
                    "target_china_date"
                ] = (
                    prediction_frame[
                        "target_time_china"
                    ]
                    .dt
                    .date
                )


                prediction_frame[
                    "absolute_error_pu"
                ] = np.abs(
                    prediction_frame[
                        "prediction_power_pu"
                    ]
                    -
                    prediction_frame[
                        "target_power_pu"
                    ]
                )


                prediction_frame[
                    "absolute_error_MW"
                ] = np.abs(
                    prediction_frame[
                        "prediction_power_MW"
                    ]
                    -
                    prediction_frame[
                        "target_power_MW"
                    ]
                )


                prediction_frames.append(
                    prediction_frame
                )


                # --------------------------------------------
                # Evaluation scopes
                # --------------------------------------------

                scope_masks = {
                    "all":
                        np.ones(
                            len(
                                prediction_frame
                            ),
                            dtype=bool,
                        ),

                    "daylight":
                        prediction_frame[
                            "target_solar_daylight"
                        ]
                        .eq(1)
                        .to_numpy(),
                }


                for scope_name, scope_mask in (
                    scope_masks.items()
                ):

                    assert (
                        scope_mask.sum()
                        >
                        1
                    )


                    metrics = (
                        calculate_pvod_regression_metrics(
                            y_true_pu=
                                prediction_frame.loc[
                                    scope_mask,
                                    "target_power_pu",
                                ],

                            y_pred_pu=
                                prediction_frame.loc[
                                    scope_mask,
                                    "prediction_power_pu",
                                ],

                            y_true_MW=
                                prediction_frame.loc[
                                    scope_mask,
                                    "target_power_MW",
                                ],

                            y_pred_MW=
                                prediction_frame.loc[
                                    scope_mask,
                                    "prediction_power_MW",
                                ],
                        )
                    )


                    metric_rows.append(
                        {
                            "model":
                                model_name,

                            "station":
                                station_id,

                            "horizon_min":
                                horizon_min,

                            "variant":
                                variant_name,

                            "scope":
                                scope_name,

                            **metrics,
                        }
                    )


                runtime_rows.append(
                    {
                        "model":
                            model_name,

                        "station":
                            station_id,

                        "horizon_min":
                            horizon_min,

                        "variant":
                            variant_name,

                        "fit_seconds":
                            float(
                                result[
                                    "fit_seconds"
                                ]
                            ),

                        "predict_seconds":
                            float(
                                result[
                                    "predict_seconds"
                                ]
                            ),
                    }
                )


                station_variant_results[
                    variant_name
                ] = {
                    "target_time_utc":
                        prediction_frame[
                            "target_time_utc"
                        ]
                        .copy(),

                    "target_power_pu":
                        target_pu.copy(),

                    "rows":
                        len(
                            prediction_frame
                        ),
                }


                # Do NOT retain fitted model objects.
                del result

                gc.collect()


            # =================================================
            # Immediate F3/F4 pairing assertion
            # =================================================

            f3_result = (
                station_variant_results[
                    "F3"
                ]
            )


            f4_result = (
                station_variant_results[
                    "F4_real"
                ]
            )


            assert (
                f3_result[
                    "rows"
                ]
                ==
                f4_result[
                    "rows"
                ]
            )


            assert (
                f3_result[
                    "target_time_utc"
                ]
                .equals(
                    f4_result[
                        "target_time_utc"
                    ]
                )
            )


            assert np.array_equal(
                f3_result[
                    "target_power_pu"
                ],
                f4_result[
                    "target_power_pu"
                ],
            )


            station_seconds = float(
                time.perf_counter()
                -
                station_start
            )


            print(
                f"{station_id}: "
                f"paired rows="
                f"{f3_result['rows']:,} "
                f"| completed in "
                f"{station_seconds:.2f}s"
            )


# ============================================================
# 4. Aggregate outputs
# ============================================================

pvod_classical_predictions_corrected = (
    pd.concat(
        prediction_frames,
        ignore_index=True,
    )
)


pvod_classical_metrics_corrected = pd.DataFrame(
    metric_rows
)


pvod_classical_runtime = pd.DataFrame(
    runtime_rows
)


grid_seconds = float(
    time.perf_counter()
    -
    grid_start
)


# ============================================================
# 5. Strong global execution assertions
# ============================================================

assert fit_counter == 120


assert len(
    pvod_classical_runtime
) == 120


assert len(
    pvod_classical_metrics_corrected
) == (
    120
    *
    len(
        PVOD_EVALUATION_SCOPES
    )
)


assert (
    pvod_classical_predictions_corrected[
        [
            "target_power_pu",
            "prediction_power_pu",
            "target_power_MW",
            "prediction_power_MW",
        ]
    ]
    .apply(
        np.isfinite
    )
    .all()
    .all()
)


# ============================================================
# 6. Global F3/F4 paired-row audit
# ============================================================

pair_key_columns = [
    "model",
    "station",
    "horizon_min",
    "issue_time_utc",
    "target_time_utc",
]


pair_count = (
    pvod_classical_predictions_corrected
    .groupby(
        pair_key_columns,
        sort=False,
    )[
        "variant"
    ]
    .nunique()
)


assert (
    pair_count
    .eq(2)
    .all()
)


assert (
    set(
        pvod_classical_predictions_corrected[
            "variant"
        ]
        .unique()
    )
    ==
    {
        "F3",
        "F4_real",
    }
)


# ============================================================
# 7. Primary F3/F4 MAE effect table
#
# effect =
#
#     MAE(F4_real)
#     -
#     MAE(F3)
#
# negative -> F4 lower
# positive -> F3 lower
# ============================================================

mae_wide = (
    pvod_classical_metrics_corrected[
        [
            "model",
            "station",
            "horizon_min",
            "scope",
            "variant",
            "MAE_power_pu",
        ]
    ]
    .pivot(
        index=[
            "model",
            "station",
            "horizon_min",
            "scope",
        ],

        columns=
            "variant",

        values=
            "MAE_power_pu",
    )
    .reset_index()
)


assert (
    {
        "F3",
        "F4_real",
    }
    .issubset(
        mae_wide.columns
    )
)


mae_wide[
    "delta_MAE_F4_minus_F3"
] = (
    mae_wide[
        "F4_real"
    ]
    -
    mae_wide[
        "F3"
    ]
)


mae_wide[
    "lower_point_estimate"
] = np.select(
    [
        mae_wide[
            "delta_MAE_F4_minus_F3"
        ]
        <
        0.0,

        mae_wide[
            "delta_MAE_F4_minus_F3"
        ]
        >
        0.0,
    ],
    [
        "F4",
        "F3",
    ],
    default="tie",
)


pvod_classical_primary_effects_corrected = (
    mae_wide
    .sort_values(
        [
            "scope",
            "model",
            "horizon_min",
            "station",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 8. Aggregate descriptive effects across stations
#
# IMPORTANT:
# These are DESCRIPTIVE point estimates only.
#
# Formal uncertainty comes in the next bootstrap cell.
# ============================================================

pvod_classical_station_aggregate_corrected = (
    pvod_classical_primary_effects_corrected
    .groupby(
        [
            "scope",
            "model",
            "horizon_min",
        ],
        as_index=False,
    )
    .agg(
        stations=(
            "station",
            "nunique",
        ),

        mean_station_delta_MAE=(
            "delta_MAE_F4_minus_F3",
            "mean",
        ),

        median_station_delta_MAE=(
            "delta_MAE_F4_minus_F3",
            "median",
        ),

        stations_F4_lower_point=(
            "lower_point_estimate",
            lambda values:
                int(
                    (
                        values
                        ==
                        "F4"
                    )
                    .sum()
                ),
        ),

        stations_F3_lower_point=(
            "lower_point_estimate",
            lambda values:
                int(
                    (
                        values
                        ==
                        "F3"
                    )
                    .sum()
                ),
        ),

        stations_tie_point=(
            "lower_point_estimate",
            lambda values:
                int(
                    (
                        values
                        ==
                        "tie"
                    )
                    .sum()
                ),
        ),
    )
)


# ============================================================
# 9. Compact metric pivot
# ============================================================

pvod_classical_mae_summary_corrected = (
    pvod_classical_metrics_corrected
    .pivot_table(
        index=[
            "model",
            "station",
            "horizon_min",
            "scope",
        ],

        columns="variant",

        values="MAE_power_pu",

        aggfunc="first",
    )
    .reset_index()
)


pvod_classical_mae_summary_corrected[
    "F4_minus_F3"
] = (
    pvod_classical_mae_summary_corrected[
        "F4_real"
    ]
    -
    pvod_classical_mae_summary_corrected[
        "F3"
    ]
)


# ============================================================
# 10. Runtime summary
# ============================================================

runtime_summary = (
    pvod_classical_runtime
    .groupby(
        "model",
        as_index=False,
    )
    .agg(
        fits=(
            "variant",
            "size",
        ),

        total_fit_seconds=(
            "fit_seconds",
            "sum",
        ),

        total_predict_seconds=(
            "predict_seconds",
            "sum",
        ),

        median_fit_seconds=(
            "fit_seconds",
            "median",
        ),
    )
)


# ============================================================
# 11. Display
# ============================================================

print(
    "\n"
    + "=" * 125
)

print(
    "PVOD PRIMARY CLASSICAL GRID COMPLETE"
)

print(
    "=" * 125
)


print(
    "\nModels fitted:",
    fit_counter,
)


print(
    "Total execution seconds:",
    grid_seconds,
)


print(
    "Prediction rows stored:",
    len(
        pvod_classical_predictions_corrected
    ),
)


print(
    "\n"
    + "=" * 125
)

print(
    "RUNTIME SUMMARY"
)

print(
    "=" * 125
)


display(
    runtime_summary
)


print(
    "\n"
    + "=" * 125
)

print(
    "ALL-TEST MAE: F3 vs F4"
)

print(
    "=" * 125
)


display(
    pvod_classical_mae_summary_corrected.loc[
        pvod_classical_mae_summary_corrected[
            "scope"
        ]
        ==
        "all"
    ]
    .sort_values(
        [
            "model",
            "horizon_min",
            "station",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 125
)

print(
    "DAYLIGHT MAE: F3 vs F4"
)

print(
    "=" * 125
)


display(
    pvod_classical_mae_summary_corrected.loc[
        pvod_classical_mae_summary_corrected[
            "scope"
        ]
        ==
        "daylight"
    ]
    .sort_values(
        [
            "model",
            "horizon_min",
            "station",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 125
)

print(
    "DESCRIPTIVE EQUAL-STATION F3/F4 EFFECT SUMMARY"
)

print(
    "=" * 125
)


display(
    pvod_classical_station_aggregate_corrected
    .sort_values(
        [
            "scope",
            "model",
            "horizon_min",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nPVOD-06A PRIMARY CLASSICAL EXECUTION: PASSED"
)


print(
    "\nPrimary effect = MAE(F4_real) - MAE(F3)."
)

print(
    "Negative point estimate -> F4 lower MAE."
)

print(
    "Positive point estimate -> F3 lower MAE."
)

print(
    "\nThese are POINT ESTIMATES only."
)

print(
    "No confidence interval computed yet."
)

print(
    "No bootstrap performed yet."
)

print(
    "No neural model trained yet."
)

# Canonical compatibility aliases.
# These reference the same objects; no duplicate DataFrame is created.
pvod_classical_predictions = pvod_classical_predictions_corrected
pvod_classical_metrics = pvod_classical_metrics_corrected
pvod_classical_primary_effects = pvod_classical_primary_effects_corrected
pvod_classical_station_aggregate = pvod_classical_station_aggregate_corrected
pvod_classical_mae_summary = pvod_classical_mae_summary_corrected

print("Canonical Ridge precision path: source-float64 from first fit.")



MODEL: Ridge

--- Horizon: 15 min ---
station00: paired rows=5,798 | completed in 1.36s
station01: paired rows=6,681 | completed in 1.31s
station02: paired rows=5,719 | completed in 1.28s
station03: paired rows=2,937 | completed in 1.15s
station04: paired rows=6,681 | completed in 1.26s
station05: paired rows=1,939 | completed in 1.12s
station06: paired rows=6,239 | completed in 1.32s
station07: paired rows=6,681 | completed in 1.33s
station08: paired rows=6,681 | completed in 1.33s
station09: paired rows=4,529 | completed in 1.32s

--- Horizon: 30 min ---
station00: paired rows=5,798 | completed in 1.29s
station01: paired rows=6,681 | completed in 1.33s
station02: paired rows=5,717 | completed in 1.29s
station03: paired rows=2,937 | completed in 1.20s
station04: paired rows=6,681 | completed in 1.32s
station05: paired rows=1,939 | completed in 1.22s
station06: paired rows=6,238 | completed in 1.35s
station07: paired rows=6,681 | completed in 1.37s
station08: paired rows=6,681 | compl

,model,fits,total_fit_seconds,total_predict_seconds,median_fit_seconds
0,LightGBM,60,100.674038,1.147678,1.676918
1,Ridge,60,1.884456,0.226770,0.024975



ALL-TEST MAE: F3 vs F4


variant,model,station,horizon_min,scope,F3,F4_real,F4_minus_F3
0,LightGBM,station00,15,all,0.019901,0.020294,0.000393
1,LightGBM,station01,15,all,0.022608,0.022285,-0.000323
2,LightGBM,station02,15,all,0.022999,0.023091,0.000092
3,LightGBM,station03,15,all,0.022309,0.022122,-0.000188
4,LightGBM,station04,15,all,0.037816,0.037992,0.000177
5,LightGBM,station05,15,all,0.034594,0.035346,0.000752
6,LightGBM,station06,15,all,0.019762,0.019830,0.000068
7,LightGBM,station07,15,all,0.021418,0.021379,-0.000039
8,LightGBM,station08,15,all,0.021589,0.021549,-0.000040
9,LightGBM,station09,15,all,0.014990,0.015340,0.000349



DAYLIGHT MAE: F3 vs F4


variant,model,station,horizon_min,scope,F3,F4_real,F4_minus_F3
0,LightGBM,station00,15,daylight,0.034165,0.034851,0.000686
1,LightGBM,station01,15,daylight,0.038762,0.038266,-0.000496
2,LightGBM,station02,15,daylight,0.039777,0.039889,0.000112
3,LightGBM,station03,15,daylight,0.036558,0.036527,-0.000031
4,LightGBM,station04,15,daylight,0.065033,0.065334,0.000301
5,LightGBM,station05,15,daylight,0.044738,0.046997,0.002259
6,LightGBM,station06,15,daylight,0.034216,0.034371,0.000154
7,LightGBM,station07,15,daylight,0.037217,0.037139,-0.000078
8,LightGBM,station08,15,daylight,0.037492,0.037449,-0.000042
9,LightGBM,station09,15,daylight,0.025504,0.026068,0.000564



DESCRIPTIVE EQUAL-STATION F3/F4 EFFECT SUMMARY


,scope,model,horizon_min,stations,mean_station_delta_MAE,median_station_delta_MAE,stations_F4_lower_point,stations_F3_lower_point,stations_tie_point
0,all,LightGBM,15,10,0.000124,0.000080,4,6,0
1,all,LightGBM,30,10,0.000456,0.000016,5,5,0
2,all,LightGBM,60,10,-0.000290,0.000036,5,5,0
3,all,Ridge,15,10,0.002909,0.002069,1,9,0
4,all,Ridge,30,10,0.005677,0.003465,1,9,0
5,all,Ridge,60,10,0.011207,0.006487,0,10,0
6,daylight,LightGBM,15,10,0.000343,0.000133,4,6,0
7,daylight,LightGBM,30,10,0.000311,-0.000065,6,4,0
8,daylight,LightGBM,60,10,-0.000660,-0.000247,7,3,0
9,daylight,Ridge,15,10,0.002087,0.001384,2,8,0



PVOD-06A PRIMARY CLASSICAL EXECUTION: PASSED

Primary effect = MAE(F4_real) - MAE(F3).
Negative point estimate -> F4 lower MAE.
Positive point estimate -> F3 lower MAE.

These are POINT ESTIMATES only.
No confidence interval computed yet.
No bootstrap performed yet.
No neural model trained yet.
Canonical Ridge precision path: source-float64 from first fit.


In [15]:
# ============================================================
# PVOD-06B
# PRIMARY CLASSICAL HIERARCHICAL PAIRED BOOTSTRAP
#
# Frozen protocol:
#
#   replicates       = 10,000
#   seed             = 42
#   day timezone     = Asia/Shanghai
#   station weighting = equal
#
# For each replicate:
#
#   1. sample 10 stations WITH replacement;
#   2. for each selected station occurrence:
#        sample that station's China-local TEST days
#        WITH replacement;
#   3. preserve paired F3/F4 predictions within sampled days;
#   4. compute station-level:
#
#        Delta MAE = MAE(F4_real) - MAE(F3)
#
#   5. average the 10 selected station-level deltas equally.
#
# IMPORTANT
# ---------
# NO retraining.
# NO model selection.
# NO prediction clipping.
#
# Bootstrap captures station/day prediction variability only.
# It does NOT capture training stochasticity.
# ============================================================

import time

import numpy as np
import pandas as pd


# ============================================================
# 1. Frozen bootstrap constants
# ============================================================

PVOD_CLASSICAL_BOOTSTRAP_REPLICATES = 10_000
PVOD_CLASSICAL_BOOTSTRAP_SEED = 42
PVOD_CLASSICAL_BOOTSTRAP_TIMEZONE = (
    "Asia/Shanghai"
)


assert (
    PVOD_CLASSICAL_BOOTSTRAP_REPLICATES
    ==
    10_000
)

assert (
    PVOD_CLASSICAL_BOOTSTRAP_SEED
    ==
    42
)


# ============================================================
# 2. Canonical corrected predictions
# ============================================================

assert (
    "pvod_classical_predictions_corrected"
    in globals()
)


bootstrap_source = (
    pvod_classical_predictions_corrected
    .copy()
)


assert set(
    bootstrap_source[
        "model"
    ]
    .unique()
) == {
    "Ridge",
    "LightGBM",
}


assert set(
    bootstrap_source[
        "variant"
    ]
    .unique()
) == {
    "F3",
    "F4_real",
}


assert set(
    bootstrap_source[
        "horizon_min"
    ]
    .unique()
) == {
    15,
    30,
    60,
}


# ============================================================
# 3. Reconstruct China-local target calendar day
#
# Do not rely on previously cached date objects.
# ============================================================

bootstrap_source[
    "bootstrap_target_time_china"
] = (
    pd.to_datetime(
        bootstrap_source[
            "target_time_utc"
        ],
        utc=True,
    )
    .dt
    .tz_convert(
        PVOD_CLASSICAL_BOOTSTRAP_TIMEZONE
    )
)


bootstrap_source[
    "bootstrap_china_date"
] = (
    bootstrap_source[
        "bootstrap_target_time_china"
    ]
    .dt
    .date
)


# ============================================================
# 4. Strong paired-row audit
# ============================================================

pair_keys = [
    "model",
    "station",
    "horizon_min",
    "target_time_utc",
]


pair_audit = (
    bootstrap_source
    .groupby(
        pair_keys,
        sort=False,
    )
    .agg(
        variants=(
            "variant",
            "nunique",
        ),

        target_count=(
            "target_power_pu",
            "count",
        ),
    )
)


assert (
    pair_audit[
        "variants"
    ]
    .eq(2)
    .all()
)


# ============================================================
# 5. Daily paired sufficient statistics
#
# For each:
#
#   model
#   station
#   horizon
#   scope
#   China-local day
#
# retain:
#
#   sum absolute errors F3
#   sum absolute errors F4
#   paired row count
#
# This is sufficient to reconstruct MAE after day resampling.
# ============================================================

daily_frames = []


for scope_name in [
    "all",
    "daylight",
]:

    if scope_name == "all":

        scoped = (
            bootstrap_source
        )


    else:

        scoped = (
            bootstrap_source.loc[
                bootstrap_source[
                    "target_solar_daylight"
                ]
                .eq(1)
            ]
        )


    daily_long = (
        scoped
        .groupby(
            [
                "model",
                "station",
                "horizon_min",
                "bootstrap_china_date",
                "variant",
            ],
            as_index=False,
            sort=True,
        )
        .agg(
            error_sum=(
                "absolute_error_pu",
                "sum",
            ),

            row_count=(
                "absolute_error_pu",
                "size",
            ),
        )
    )


    daily_wide = (
        daily_long
        .pivot(
            index=[
                "model",
                "station",
                "horizon_min",
                "bootstrap_china_date",
            ],

            columns="variant",

            values=[
                "error_sum",
                "row_count",
            ],
        )
        .reset_index()
    )


    daily_wide.columns = [
        (
            "_".join(
                [
                    str(part)
                    for part in column
                    if str(part) != ""
                ]
            )
            if isinstance(
                column,
                tuple,
            )
            else column
        )

        for column
        in daily_wide.columns
    ]


    daily_wide[
        "scope"
    ] = scope_name


    # --------------------------------------------------------
    # Every retained row must remain paired F3/F4.
    # --------------------------------------------------------

    required_columns = {
        "error_sum_F3",
        "error_sum_F4_real",
        "row_count_F3",
        "row_count_F4_real",
    }


    assert required_columns.issubset(
        daily_wide.columns
    )


    assert (
        daily_wide[
            [
                "error_sum_F3",
                "error_sum_F4_real",
                "row_count_F3",
                "row_count_F4_real",
            ]
        ]
        .notna()
        .all()
        .all()
    )


    assert np.array_equal(
        daily_wide[
            "row_count_F3"
        ]
        .to_numpy(),
        daily_wide[
            "row_count_F4_real"
        ]
        .to_numpy(),
    )


    daily_wide[
        "paired_row_count"
    ] = (
        daily_wide[
            "row_count_F3"
        ]
        .astype(
            np.int64
        )
    )


    daily_frames.append(
        daily_wide
    )


pvod_classical_daily_bootstrap_table = (
    pd.concat(
        daily_frames,
        ignore_index=True,
    )
)


# ============================================================
# 6. Station/day audit
# ============================================================

station_day_audit = (
    pvod_classical_daily_bootstrap_table
    .groupby(
        [
            "scope",
            "model",
            "horizon_min",
            "station",
        ],
        as_index=False,
    )
    .agg(
        local_days=(
            "bootstrap_china_date",
            "nunique",
        ),

        paired_rows=(
            "paired_row_count",
            "sum",
        ),
    )
)


assert (
    station_day_audit[
        "local_days"
    ]
    .gt(0)
    .all()
)


assert (
    station_day_audit[
        "paired_rows"
    ]
    .gt(0)
    .all()
)


# ============================================================
# 7. Convert one condition into compact station arrays
# ============================================================

def make_station_day_payload(
    model_name: str,
    horizon_min: int,
    scope_name: str,
):

    condition = (
        pvod_classical_daily_bootstrap_table.loc[
            (
                pvod_classical_daily_bootstrap_table[
                    "model"
                ]
                ==
                model_name
            )
            &
            (
                pvod_classical_daily_bootstrap_table[
                    "horizon_min"
                ]
                ==
                horizon_min
            )
            &
            (
                pvod_classical_daily_bootstrap_table[
                    "scope"
                ]
                ==
                scope_name
            )
        ]
        .copy()
    )


    stations = sorted(
        condition[
            "station"
        ]
        .unique()
        .tolist()
    )


    assert stations == sorted(
        PVOD_EXTERNAL_STATIONS
    )


    payload = []


    for station_id in stations:

        station_days = (
            condition.loc[
                condition[
                    "station"
                ]
                ==
                station_id
            ]
            .sort_values(
                "bootstrap_china_date"
            )
            .reset_index(
                drop=True
            )
        )


        error_f3 = (
            station_days[
                "error_sum_F3"
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        error_f4 = (
            station_days[
                "error_sum_F4_real"
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        counts = (
            station_days[
                "paired_row_count"
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        assert (
            len(
                error_f3
            )
            ==
            len(
                error_f4
            )
            ==
            len(
                counts
            )
        )


        assert (
            len(
                counts
            )
            >
            0
        )


        assert (
            counts
            >
            0
        ).all()


        # ----------------------------------------------------
        # Original station-level point estimate
        # ----------------------------------------------------

        mae_f3 = float(
            error_f3.sum()
            /
            counts.sum()
        )


        mae_f4 = float(
            error_f4.sum()
            /
            counts.sum()
        )


        delta = float(
            mae_f4
            -
            mae_f3
        )


        payload.append(
            {
                "station":
                    station_id,

                "error_f3":
                    error_f3,

                "error_f4":
                    error_f4,

                "counts":
                    counts,

                "n_days":
                    len(
                        counts
                    ),

                "point_delta":
                    delta,
            }
        )


    assert len(
        payload
    ) == 10


    return payload


# ============================================================
# 8. Hierarchical paired bootstrap
# ============================================================

def hierarchical_paired_bootstrap(
    model_name: str,
    horizon_min: int,
    scope_name: str,
    rng: np.random.Generator,
    n_replicates: int,
):

    payload = (
        make_station_day_payload(
            model_name=
                model_name,

            horizon_min=
                horizon_min,

            scope_name=
                scope_name,
        )
    )


    n_stations = len(
        payload
    )


    assert n_stations == 10


    # --------------------------------------------------------
    # Equal-station point estimate
    # --------------------------------------------------------

    point_station_deltas = np.array(
        [
            station_data[
                "point_delta"
            ]

            for station_data
            in payload
        ],
        dtype=np.float64,
    )


    point_estimate = float(
        point_station_deltas.mean()
    )


    bootstrap_values = np.empty(
        n_replicates,
        dtype=np.float64,
    )


    # --------------------------------------------------------
    # Each duplicated station occurrence receives an
    # INDEPENDENT day resample.
    # --------------------------------------------------------

    for replicate_index in range(
        n_replicates
    ):

        station_draw = (
            rng.integers(
                low=0,
                high=n_stations,
                size=n_stations,
            )
        )


        station_delta_draw = np.empty(
            n_stations,
            dtype=np.float64,
        )


        for draw_position, station_index in enumerate(
            station_draw
        ):

            station_data = (
                payload[
                    int(
                        station_index
                    )
                ]
            )


            n_days = int(
                station_data[
                    "n_days"
                ]
            )


            day_draw = (
                rng.integers(
                    low=0,
                    high=n_days,
                    size=n_days,
                )
            )


            sampled_counts = (
                station_data[
                    "counts"
                ][
                    day_draw
                ]
            )


            sampled_error_f3 = (
                station_data[
                    "error_f3"
                ][
                    day_draw
                ]
            )


            sampled_error_f4 = (
                station_data[
                    "error_f4"
                ][
                    day_draw
                ]
            )


            denominator = float(
                sampled_counts.sum()
            )


            assert denominator > 0.0


            sampled_mae_f3 = float(
                sampled_error_f3.sum()
                /
                denominator
            )


            sampled_mae_f4 = float(
                sampled_error_f4.sum()
                /
                denominator
            )


            station_delta_draw[
                draw_position
            ] = (
                sampled_mae_f4
                -
                sampled_mae_f3
            )


        # Equal weight for each sampled station occurrence.
        bootstrap_values[
            replicate_index
        ] = float(
            station_delta_draw.mean()
        )


    return {
        "point_estimate":
            point_estimate,

        "point_station_deltas":
            point_station_deltas,

        "bootstrap_values":
            bootstrap_values,
    }


# ============================================================
# 9. Execute all 12 classical model/horizon/scope conditions
# ============================================================

rng = np.random.default_rng(
    PVOD_CLASSICAL_BOOTSTRAP_SEED
)


bootstrap_summary_rows = []
bootstrap_draw_frames = []


bootstrap_start = (
    time.perf_counter()
)


for scope_name in [
    "all",
    "daylight",
]:

    for model_name in [
        "LightGBM",
        "Ridge",
    ]:

        for horizon_min in (
            PVOD_EXTERNAL_HORIZONS
        ):

            print(
                f"Bootstrapping "
                f"{scope_name:8s} | "
                f"{model_name:8s} | "
                f"{horizon_min:2d} min"
            )


            result = (
                hierarchical_paired_bootstrap(
                    model_name=
                        model_name,

                    horizon_min=
                        horizon_min,

                    scope_name=
                        scope_name,

                    rng=
                        rng,

                    n_replicates=
                        PVOD_CLASSICAL_BOOTSTRAP_REPLICATES,
                )
            )


            bootstrap_values = (
                result[
                    "bootstrap_values"
                ]
            )


            ci_low = float(
                np.quantile(
                    bootstrap_values,
                    0.025,
                )
            )


            ci_high = float(
                np.quantile(
                    bootstrap_values,
                    0.975,
                )
            )


            if ci_high < 0.0:

                classification = (
                    "F4_lower"
                )


            elif ci_low > 0.0:

                classification = (
                    "F3_lower"
                )


            else:

                classification = (
                    "crosses_zero"
                )


            bootstrap_summary_rows.append(
                {
                    "scope":
                        scope_name,

                    "model":
                        model_name,

                    "horizon_min":
                        horizon_min,

                    "stations":
                        10,

                    "replicates":
                        PVOD_CLASSICAL_BOOTSTRAP_REPLICATES,

                    "point_delta_MAE":
                        result[
                            "point_estimate"
                        ],

                    "bootstrap_mean":
                        float(
                            bootstrap_values.mean()
                        ),

                    "bootstrap_median":
                        float(
                            np.median(
                                bootstrap_values
                            )
                        ),

                    "ci95_low":
                        ci_low,

                    "ci95_high":
                        ci_high,

                    "classification":
                        classification,
                }
            )


            bootstrap_draw_frames.append(
                pd.DataFrame(
                    {
                        "scope":
                            scope_name,

                        "model":
                            model_name,

                        "horizon_min":
                            horizon_min,

                        "replicate":
                            np.arange(
                                1,
                                PVOD_CLASSICAL_BOOTSTRAP_REPLICATES
                                +
                                1,
                                dtype=np.int32,
                            ),

                        "delta_MAE_F4_minus_F3":
                            bootstrap_values,
                    }
                )
            )


bootstrap_seconds = float(
    time.perf_counter()
    -
    bootstrap_start
)


# ============================================================
# 10. Canonical bootstrap objects
# ============================================================

pvod_classical_hierarchical_bootstrap_summary = (
    pd.DataFrame(
        bootstrap_summary_rows
    )
    .sort_values(
        [
            "scope",
            "model",
            "horizon_min",
        ]
    )
    .reset_index(
        drop=True
    )
)


pvod_classical_hierarchical_bootstrap_draws = (
    pd.concat(
        bootstrap_draw_frames,
        ignore_index=True,
    )
)


assert len(
    pvod_classical_hierarchical_bootstrap_summary
) == 12


assert len(
    pvod_classical_hierarchical_bootstrap_draws
) == (
    12
    *
    PVOD_CLASSICAL_BOOTSTRAP_REPLICATES
)


# ============================================================
# 11. Point-estimate reconciliation
#
# Bootstrap point estimate must equal the corrected
# descriptive equal-station mean delta already calculated.
# ============================================================

expected_points = (
    pvod_classical_station_aggregate_corrected[
        [
            "scope",
            "model",
            "horizon_min",
            "mean_station_delta_MAE",
        ]
    ]
    .rename(
        columns={
            "mean_station_delta_MAE":
                "expected_point_delta_MAE",
        }
    )
)


point_check = (
    pvod_classical_hierarchical_bootstrap_summary
    .merge(
        expected_points,
        on=[
            "scope",
            "model",
            "horizon_min",
        ],
        how="left",
        validate="one_to_one",
    )
)


point_check[
    "absolute_difference"
] = np.abs(
    point_check[
        "point_delta_MAE"
    ]
    -
    point_check[
        "expected_point_delta_MAE"
    ]
)


assert (
    point_check[
        "absolute_difference"
    ]
    .max()
    <
    1e-12
)


# ============================================================
# 12. Classification counts
# ============================================================

classification_counts = (
    pvod_classical_hierarchical_bootstrap_summary
    .groupby(
        [
            "scope",
            "model",
            "classification",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size":
                "comparisons",
        }
    )
)


# ============================================================
# 13. Display
# ============================================================

print(
    "\n"
    + "=" * 125
)

print(
    "PVOD PRIMARY CLASSICAL HIERARCHICAL BOOTSTRAP"
)

print(
    "=" * 125
)


display(
    pvod_classical_hierarchical_bootstrap_summary
)


print(
    "\n"
    + "=" * 125
)

print(
    "CLASSIFICATION COUNTS"
)

print(
    "=" * 125
)


display(
    classification_counts
)


print(
    "\n"
    + "=" * 125
)

print(
    "BOOTSTRAP AUDIT"
)

print(
    "=" * 125
)


print(
    "Replicates per comparison:",
    PVOD_CLASSICAL_BOOTSTRAP_REPLICATES,
)


print(
    "Seed:",
    PVOD_CLASSICAL_BOOTSTRAP_SEED,
)


print(
    "China-local day timezone:",
    PVOD_CLASSICAL_BOOTSTRAP_TIMEZONE,
)


print(
    "Station weighting: equal"
)


print(
    "Total bootstrap draws stored:",
    len(
        pvod_classical_hierarchical_bootstrap_draws
    ),
)


print(
    "Maximum point-estimate reconciliation error:",
    float(
        point_check[
            "absolute_difference"
        ]
        .max()
    ),
)


print(
    "Bootstrap execution seconds:",
    bootstrap_seconds,
)


print(
    "\nPVOD-06B PRIMARY CLASSICAL HIERARCHICAL BOOTSTRAP: PASSED"
)


print(
    "\nClassification rule:"
)

print(
    "  CI entirely below zero -> F4_lower"
)

print(
    "  CI entirely above zero -> F3_lower"
)

print(
    "  otherwise              -> crosses_zero"
)


print(
    "\nNo model retrained."
)

print(
    "No hyperparameter changed."
)

print(
    "No prediction clipped."
)

print(
    "Uncertainty reflects station/day resampling only."
)

print(
    "Neural training stochasticity is not represented."
)

Bootstrapping all      | LightGBM | 15 min
Bootstrapping all      | LightGBM | 30 min
Bootstrapping all      | LightGBM | 60 min
Bootstrapping all      | Ridge    | 15 min
Bootstrapping all      | Ridge    | 30 min
Bootstrapping all      | Ridge    | 60 min
Bootstrapping daylight | LightGBM | 15 min
Bootstrapping daylight | LightGBM | 30 min
Bootstrapping daylight | LightGBM | 60 min
Bootstrapping daylight | Ridge    | 15 min
Bootstrapping daylight | Ridge    | 30 min
Bootstrapping daylight | Ridge    | 60 min

PVOD PRIMARY CLASSICAL HIERARCHICAL BOOTSTRAP


,scope,model,horizon_min,stations,replicates,point_delta_MAE,bootstrap_mean,bootstrap_median,ci95_low,ci95_high,classification
0,all,LightGBM,15,10,10000,0.000124,0.000122,0.000119,-0.000071,0.000332,crosses_zero
1,all,LightGBM,30,10,10000,0.000456,0.000459,0.000432,-0.000229,0.001611,crosses_zero
2,all,LightGBM,60,10,10000,-0.000290,-0.000287,-0.000244,-0.001040,0.000211,crosses_zero
3,all,Ridge,15,10,10000,0.002909,0.002897,0.002763,0.001121,0.005528,F3_lower
4,all,Ridge,30,10,10000,0.005677,0.005688,0.005431,0.002067,0.011352,F3_lower
5,all,Ridge,60,10,10000,0.011207,0.011155,0.010631,0.003701,0.023092,F3_lower
6,daylight,LightGBM,15,10,10000,0.000343,0.000348,0.000327,-0.000050,0.000870,crosses_zero
7,daylight,LightGBM,30,10,10000,0.000311,0.000310,0.000271,-0.000364,0.001265,crosses_zero
8,daylight,LightGBM,60,10,10000,-0.000660,-0.000662,-0.000607,-0.001811,0.000119,crosses_zero
9,daylight,Ridge,15,10,10000,0.002087,0.002090,0.001965,0.000597,0.004397,F3_lower



CLASSIFICATION COUNTS


,scope,model,classification,comparisons
0,all,LightGBM,crosses_zero,3
1,all,Ridge,F3_lower,3
2,daylight,LightGBM,crosses_zero,3
3,daylight,Ridge,F3_lower,3



BOOTSTRAP AUDIT
Replicates per comparison: 10000
Seed: 42
China-local day timezone: Asia/Shanghai
Station weighting: equal
Total bootstrap draws stored: 120000
Maximum point-estimate reconciliation error: 1.0408340855860843e-17
Bootstrap execution seconds: 14.135332665871829

PVOD-06B PRIMARY CLASSICAL HIERARCHICAL BOOTSTRAP: PASSED

Classification rule:
  CI entirely below zero -> F4_lower
  CI entirely above zero -> F3_lower
  otherwise              -> crosses_zero

No model retrained.
No hyperparameter changed.
No prediction clipped.
Uncertainty reflects station/day resampling only.
Neural training stochasticity is not represented.


In [16]:
# ============================================================
# PVOD-07A
# CHECKPOINTED PRIMARY NEURAL EXTERNAL-VALIDATION EXECUTION
#
# Models:
#     LSTM
#     TCN
#
# Representations:
#     F3
#     F4_real
#
# 2 models × 10 stations × 3 horizons × 2 variants = 120 fits
#
# EXECUTION-SAFETY POLICY
# -----------------------
# - Every completed fit is atomically checkpointed to disk.
# - Row-level primary predictions are checkpointed because they
#   are the canonical F4_real pairing source for later controls.
# - Re-running this notebook skips valid completed primary
#   neural checkpoints, preserving the same primary predictions
#   across a kernel restart.
# - Frozen architecture / epochs / seed / scaling are unchanged.
# ============================================================

import gc
import json
import os
import time

import numpy as np
import pandas as pd
import tensorflow as tf


# ============================================================
# 1. Preconditions / frozen registry
# ============================================================

required_objects = [
    "fit_predict_pvod_neural",
    "calculate_pvod_regression_metrics",
    "pvod_site_registry",
    "pvod_sample_index",
    "PVOD_EXTERNAL_STATIONS",
    "PVOD_EXTERNAL_HORIZONS",
    "PVOD_PRIMARY_VARIANTS",
    "PROJECT_ROOT",
]
missing = [
    name for name in required_objects
    if name not in globals()
]
assert not missing, f"Missing required objects: {missing}"

PVOD_NEURAL_MODELS = ["LSTM", "TCN"]
expected_neural_fit_count = (
    len(PVOD_NEURAL_MODELS)
    * len(PVOD_EXTERNAL_STATIONS)
    * len(PVOD_EXTERNAL_HORIZONS)
    * len(PVOD_PRIMARY_VARIANTS)
)
assert expected_neural_fit_count == 120

PVOD_PRIMARY_NEURAL_CHECKPOINT_VERSION = 2
PVOD_PRIMARY_NEURAL_CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "results"
    / "pvod_primary_checkpoints_v2"
    / "neural"
)
PVOD_PRIMARY_NEURAL_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

pvod_capacity_lookup = (
    pvod_site_registry
    .set_index("station")["capacity_MW"]
    .astype(float)
    .to_dict()
)


def primary_neural_checkpoint_paths(
    model_name,
    station_id,
    horizon_min,
    variant_name,
):
    key = (
        f"{model_name}"
        f"__h{int(horizon_min):03d}"
        f"__{station_id}"
        f"__{variant_name}"
    )
    return (
        PVOD_PRIMARY_NEURAL_CHECKPOINT_DIR
        / f"{key}__predictions.csv",
        PVOD_PRIMARY_NEURAL_CHECKPOINT_DIR
        / f"{key}__metadata.json",
    )


def primary_neural_expected_identity(
    model_name,
    station_id,
    horizon_min,
    variant_name,
):
    samples = (
        pvod_sample_index[(station_id, horizon_min)]
        .loc[lambda frame: frame["split"].eq("test")]
        .reset_index(drop=True)
    )
    assert len(samples) > 0
    return {
        "checkpoint_version": PVOD_PRIMARY_NEURAL_CHECKPOINT_VERSION,
        "family": "primary_neural",
        "model": model_name,
        "station": station_id,
        "horizon_min": int(horizon_min),
        "variant": variant_name,
        "test_rows": int(len(samples)),
        "first_target_time_utc": str(
            samples["target_time_utc"].iloc[0]
        ),
        "last_target_time_utc": str(
            samples["target_time_utc"].iloc[-1]
        ),
    }


def primary_neural_checkpoint_is_valid(
    prediction_path,
    metadata_path,
    expected_identity,
):
    if not prediction_path.exists() or not metadata_path.exists():
        return False
    try:
        with metadata_path.open("r", encoding="utf-8") as handle:
            metadata = json.load(handle)
        frame = pd.read_csv(prediction_path, nrows=5)
    except Exception:
        return False

    for key, value in expected_identity.items():
        if metadata.get(key) != value:
            return False

    required_columns = {
        "model",
        "station",
        "horizon_min",
        "variant",
        "issue_time_utc",
        "target_time_utc",
        "target_solar_daylight",
        "target_power_pu",
        "prediction_power_pu",
        "target_power_MW",
        "prediction_power_MW",
        "capacity_MW",
        "absolute_error_pu",
        "absolute_error_MW",
    }
    return required_columns.issubset(frame.columns)


def write_primary_neural_checkpoint(
    prediction_path,
    metadata_path,
    prediction_frame,
    metadata,
):
    prediction_tmp = prediction_path.with_name(
        prediction_path.name + ".tmp"
    )
    metadata_tmp = metadata_path.with_name(
        metadata_path.name + ".tmp"
    )
    prediction_frame.to_csv(prediction_tmp, index=False)
    with metadata_tmp.open("w", encoding="utf-8") as handle:
        json.dump(metadata, handle, indent=2)
    os.replace(prediction_tmp, prediction_path)
    os.replace(metadata_tmp, metadata_path)


def load_primary_neural_prediction(path):
    frame = pd.read_csv(path)
    frame["issue_time_utc"] = pd.to_datetime(
        frame["issue_time_utc"],
        utc=True,
    )
    frame["target_time_utc"] = pd.to_datetime(
        frame["target_time_utc"],
        utc=True,
    )
    frame["target_time_china"] = (
        frame["target_time_utc"]
        .dt
        .tz_convert("Asia/Shanghai")
    )
    frame["target_china_date"] = (
        frame["target_time_china"]
        .dt
        .date
    )
    return frame


def clear_pvod_tensorflow_session():
    try:
        tf.keras.backend.clear_session(free_memory=True)
    except TypeError:
        tf.keras.backend.clear_session()
    gc.collect()


# ============================================================
# 2. Execute / resume the 120 primary neural fits
# ============================================================

new_fit_count = 0
skipped_fit_count = 0
grid_start = time.perf_counter()

for model_name in PVOD_NEURAL_MODELS:
    print("\n" + "=" * 125)
    print(f"MODEL: {model_name}")
    print("=" * 125)

    for horizon_min in PVOD_EXTERNAL_HORIZONS:
        print(f"\n--- Horizon: {horizon_min} min ---")

        for station_id in PVOD_EXTERNAL_STATIONS:
            station_start = time.perf_counter()
            station_variant_pairing = {}
            station_new = 0
            station_skipped = 0

            for variant_name in PVOD_PRIMARY_VARIANTS:
                prediction_path, metadata_path = (
                    primary_neural_checkpoint_paths(
                        model_name,
                        station_id,
                        horizon_min,
                        variant_name,
                    )
                )
                expected_identity = primary_neural_expected_identity(
                    model_name,
                    station_id,
                    horizon_min,
                    variant_name,
                )

                if primary_neural_checkpoint_is_valid(
                    prediction_path,
                    metadata_path,
                    expected_identity,
                ):
                    prediction_frame = (
                        load_primary_neural_prediction(
                            prediction_path
                        )
                    )
                    skipped_fit_count += 1
                    station_skipped += 1

                else:
                    clear_pvod_tensorflow_session()
                    result = fit_predict_pvod_neural(
                        station_id=station_id,
                        horizon_min=horizon_min,
                        model_name=model_name,
                        variant_name=variant_name,
                        verbose=0,
                    )

                    prediction_pu = np.asarray(
                        result["prediction_pu"],
                        dtype=np.float64,
                    ).reshape(-1)
                    target_pu = np.asarray(
                        result["target_pu"],
                        dtype=np.float64,
                    ).reshape(-1)
                    target_MW = np.asarray(
                        result["target_MW"],
                        dtype=np.float64,
                    ).reshape(-1)
                    test_meta = (
                        result["meta"]
                        .copy()
                        .reset_index(drop=True)
                    )

                    assert (
                        len(prediction_pu)
                        == len(target_pu)
                        == len(target_MW)
                        == len(test_meta)
                        == expected_identity["test_rows"]
                    )
                    assert str(
                        test_meta["target_time_utc"].iloc[0]
                    ) == expected_identity[
                        "first_target_time_utc"
                    ]
                    assert str(
                        test_meta["target_time_utc"].iloc[-1]
                    ) == expected_identity[
                        "last_target_time_utc"
                    ]

                    capacity_MW = float(
                        pvod_capacity_lookup[station_id]
                    )
                    prediction_MW = prediction_pu * capacity_MW
                    assert np.allclose(
                        target_MW,
                        target_pu * capacity_MW,
                        atol=1e-6,
                        rtol=1e-6,
                    )

                    prediction_frame = pd.DataFrame({
                        "model": model_name,
                        "station": station_id,
                        "horizon_min": int(horizon_min),
                        "variant": variant_name,
                        "issue_time_utc": pd.DatetimeIndex(
                            test_meta["issue_time_utc"]
                        ),
                        "target_time_utc": pd.DatetimeIndex(
                            test_meta["target_time_utc"]
                        ),
                        "target_solar_daylight": test_meta[
                            "target_solar_daylight"
                        ].to_numpy(dtype=np.int8),
                        "target_power_pu": target_pu,
                        "prediction_power_pu": prediction_pu,
                        "target_power_MW": target_MW,
                        "prediction_power_MW": prediction_MW,
                        "capacity_MW": capacity_MW,
                    })
                    prediction_frame["target_time_china"] = (
                        prediction_frame["target_time_utc"]
                        .dt
                        .tz_convert("Asia/Shanghai")
                    )
                    prediction_frame["target_china_date"] = (
                        prediction_frame["target_time_china"]
                        .dt
                        .date
                    )
                    prediction_frame["absolute_error_pu"] = np.abs(
                        prediction_frame["prediction_power_pu"]
                        - prediction_frame["target_power_pu"]
                    )
                    prediction_frame["absolute_error_MW"] = np.abs(
                        prediction_frame["prediction_power_MW"]
                        - prediction_frame["target_power_MW"]
                    )

                    history_df = result["history"]
                    frozen_epochs = int(result["frozen_epochs"])
                    assert len(history_df) == frozen_epochs
                    assert "loss" in history_df.columns
                    assert "val_loss" not in history_df.columns

                    metadata = {
                        **expected_identity,
                        "frozen_epochs": frozen_epochs,
                        "fit_seconds": float(result["fit_seconds"]),
                        "predict_seconds": float(
                            result["predict_seconds"]
                        ),
                        "final_train_loss": float(
                            history_df["loss"].iloc[-1]
                        ),
                        "zero_variance_continuous_columns": list(
                            result[
                                "zero_variance_continuous_columns"
                            ]
                        ),
                    }
                    write_primary_neural_checkpoint(
                        prediction_path,
                        metadata_path,
                        prediction_frame,
                        metadata,
                    )
                    new_fit_count += 1
                    station_new += 1

                    del result
                    del history_df
                    clear_pvod_tensorflow_session()

                station_variant_pairing[variant_name] = {
                    "target_time_utc": prediction_frame[
                        "target_time_utc"
                    ].copy(),
                    "target_power_pu": prediction_frame[
                        "target_power_pu"
                    ].to_numpy(dtype=np.float64),
                    "rows": len(prediction_frame),
                }
                del prediction_frame

            pair_f3 = station_variant_pairing["F3"]
            pair_f4 = station_variant_pairing["F4_real"]
            assert pair_f3["rows"] == pair_f4["rows"]
            assert pair_f3["target_time_utc"].equals(
                pair_f4["target_time_utc"]
            )
            assert np.array_equal(
                pair_f3["target_power_pu"],
                pair_f4["target_power_pu"],
            )

            station_seconds = float(
                time.perf_counter() - station_start
            )
            print(
                f"{station_id}: paired rows={pair_f3['rows']:,} | "
                f"new={station_new}, skipped={station_skipped} | "
                f"{station_seconds:.2f}s"
            )


# ============================================================
# 3. Reconstruct canonical primary neural objects from disk
# ============================================================

neural_prediction_frames = []
neural_runtime_rows = []

for model_name in PVOD_NEURAL_MODELS:
    for horizon_min in PVOD_EXTERNAL_HORIZONS:
        for station_id in PVOD_EXTERNAL_STATIONS:
            for variant_name in PVOD_PRIMARY_VARIANTS:
                prediction_path, metadata_path = (
                    primary_neural_checkpoint_paths(
                        model_name,
                        station_id,
                        horizon_min,
                        variant_name,
                    )
                )
                expected_identity = primary_neural_expected_identity(
                    model_name,
                    station_id,
                    horizon_min,
                    variant_name,
                )
                assert primary_neural_checkpoint_is_valid(
                    prediction_path,
                    metadata_path,
                    expected_identity,
                )
                neural_prediction_frames.append(
                    load_primary_neural_prediction(prediction_path)
                )
                with metadata_path.open(
                    "r",
                    encoding="utf-8",
                ) as handle:
                    neural_runtime_rows.append(json.load(handle))

pvod_neural_predictions = pd.concat(
    neural_prediction_frames,
    ignore_index=True,
)
pvod_neural_runtime = pd.DataFrame(neural_runtime_rows)
del neural_prediction_frames

grid_seconds = float(time.perf_counter() - grid_start)

assert len(pvod_neural_runtime) == 120
assert set(pvod_neural_predictions["variant"]) == {
    "F3", "F4_real"
}
assert set(pvod_neural_predictions["model"]) == {
    "LSTM", "TCN"
}
assert (
    pvod_neural_predictions[
        [
            "target_power_pu",
            "prediction_power_pu",
            "target_power_MW",
            "prediction_power_MW",
        ]
    ]
    .apply(np.isfinite)
    .all()
    .all()
)

# Exact global F3/F4 pairing.
neural_pair_count = (
    pvod_neural_predictions
    .groupby(
        [
            "model",
            "station",
            "horizon_min",
            "issue_time_utc",
            "target_time_utc",
        ],
        sort=False,
    )["variant"]
    .nunique()
)
assert neural_pair_count.eq(2).all()


# ============================================================
# 4. Metrics / descriptive primary effect tables
# ============================================================

neural_metric_rows = []
for (
    model_name,
    station_id,
    horizon_min,
    variant_name,
), group in pvod_neural_predictions.groupby(
    ["model", "station", "horizon_min", "variant"],
    sort=False,
):
    for scope_name, mask in {
        "all": np.ones(len(group), dtype=bool),
        "daylight": group[
            "target_solar_daylight"
        ].eq(1).to_numpy(),
    }.items():
        metrics = calculate_pvod_regression_metrics(
            y_true_pu=group.loc[
                mask, "target_power_pu"
            ],
            y_pred_pu=group.loc[
                mask, "prediction_power_pu"
            ],
            y_true_MW=group.loc[
                mask, "target_power_MW"
            ],
            y_pred_MW=group.loc[
                mask, "prediction_power_MW"
            ],
        )
        neural_metric_rows.append({
            "model": model_name,
            "station": station_id,
            "horizon_min": int(horizon_min),
            "variant": variant_name,
            "scope": scope_name,
            **metrics,
        })

pvod_neural_metrics = pd.DataFrame(neural_metric_rows)
assert len(pvod_neural_metrics) == 240

neural_mae_wide = (
    pvod_neural_metrics[
        [
            "model",
            "station",
            "horizon_min",
            "scope",
            "variant",
            "MAE_power_pu",
        ]
    ]
    .pivot(
        index=["model", "station", "horizon_min", "scope"],
        columns="variant",
        values="MAE_power_pu",
    )
    .reset_index()
)
neural_mae_wide["delta_MAE_F4_minus_F3"] = (
    neural_mae_wide["F4_real"] - neural_mae_wide["F3"]
)
neural_mae_wide["lower_point_estimate"] = np.select(
    [
        neural_mae_wide["delta_MAE_F4_minus_F3"] < 0.0,
        neural_mae_wide["delta_MAE_F4_minus_F3"] > 0.0,
    ],
    ["F4", "F3"],
    default="tie",
)

pvod_neural_primary_effects = (
    neural_mae_wide
    .sort_values(
        ["scope", "model", "horizon_min", "station"]
    )
    .reset_index(drop=True)
)

pvod_neural_station_aggregate = (
    pvod_neural_primary_effects
    .groupby(
        ["scope", "model", "horizon_min"],
        as_index=False,
    )
    .agg(
        stations=("station", "nunique"),
        mean_station_delta_MAE=(
            "delta_MAE_F4_minus_F3", "mean"
        ),
        median_station_delta_MAE=(
            "delta_MAE_F4_minus_F3", "median"
        ),
        stations_F4_lower_point=(
            "lower_point_estimate",
            lambda values: int((values == "F4").sum()),
        ),
        stations_F3_lower_point=(
            "lower_point_estimate",
            lambda values: int((values == "F3").sum()),
        ),
        stations_tie_point=(
            "lower_point_estimate",
            lambda values: int((values == "tie").sum()),
        ),
    )
)

pvod_neural_mae_summary = (
    pvod_neural_metrics
    .pivot_table(
        index=["model", "station", "horizon_min", "scope"],
        columns="variant",
        values="MAE_power_pu",
        aggfunc="first",
    )
    .reset_index()
)
pvod_neural_mae_summary["F4_minus_F3"] = (
    pvod_neural_mae_summary["F4_real"]
    - pvod_neural_mae_summary["F3"]
)

pvod_neural_runtime["zero_variance_continuous_columns"] = (
    pvod_neural_runtime[
        "zero_variance_continuous_columns"
    ].apply(tuple)
)
pvod_neural_runtime_summary = (
    pvod_neural_runtime
    .groupby(["model", "horizon_min"], as_index=False)
    .agg(
        fits=("variant", "size"),
        frozen_epochs=("frozen_epochs", "first"),
        total_fit_seconds=("fit_seconds", "sum"),
        median_fit_seconds=("fit_seconds", "median"),
        total_predict_seconds=("predict_seconds", "sum"),
    )
)

zero_variance_execution_audit = (
    pvod_neural_runtime.loc[
        pvod_neural_runtime[
            "zero_variance_continuous_columns"
        ].apply(len) > 0,
        [
            "model",
            "station",
            "horizon_min",
            "variant",
            "zero_variance_continuous_columns",
        ],
    ]
    .reset_index(drop=True)
)
assert len(zero_variance_execution_audit) == 24
assert set(zero_variance_execution_audit["station"]) == {
    "station03", "station05"
}
assert (
    zero_variance_execution_audit[
        "zero_variance_continuous_columns"
    ].map(lambda value: tuple(value) == ("utc_year",))
).all()

print("\n" + "=" * 125)
print("PVOD PRIMARY NEURAL GRID COMPLETE")
print("=" * 125)
print("Logical fits:", 120)
print("New fits this run:", new_fit_count)
print("Valid checkpoints skipped:", skipped_fit_count)
print("Prediction rows stored:", len(pvod_neural_predictions))
print("Total execution seconds this run:", grid_seconds)
print("\nNEURAL RUNTIME SUMMARY")
display(pvod_neural_runtime_summary)
print("\nALL-TEST NEURAL MAE: F3 vs F4")
display(
    pvod_neural_mae_summary.loc[
        pvod_neural_mae_summary["scope"].eq("all")
    ].sort_values(["model", "horizon_min", "station"])
)
print("\nDAYLIGHT NEURAL MAE: F3 vs F4")
display(
    pvod_neural_mae_summary.loc[
        pvod_neural_mae_summary["scope"].eq("daylight")
    ].sort_values(["model", "horizon_min", "station"])
)
print("\nDESCRIPTIVE EQUAL-STATION NEURAL EFFECT SUMMARY")
display(pvod_neural_station_aggregate)
print("\nZERO-VARIANCE SAFEGUARD EXECUTION AUDIT")
display(zero_variance_execution_audit)
print("\nPVOD-07A PRIMARY NEURAL EXECUTION: PASSED")
print("Primary effect = MAE(F4_real) - MAE(F3).")
print("Negative point estimate -> F4 lower MAE.")
print("Positive point estimate -> F3 lower MAE.")
print("These are point estimates only; bootstrap follows in PVOD-07B.")



MODEL: LSTM

--- Horizon: 15 min ---
station00: paired rows=5,798 | new=0, skipped=2 | 0.07s
station01: paired rows=6,681 | new=0, skipped=2 | 0.08s
station02: paired rows=5,719 | new=0, skipped=2 | 0.07s
station03: paired rows=2,937 | new=0, skipped=2 | 0.04s
station04: paired rows=6,681 | new=0, skipped=2 | 0.08s
station05: paired rows=1,939 | new=0, skipped=2 | 0.03s
station06: paired rows=6,239 | new=0, skipped=2 | 0.08s
station07: paired rows=6,681 | new=0, skipped=2 | 0.08s
station08: paired rows=6,681 | new=0, skipped=2 | 0.08s
station09: paired rows=4,529 | new=0, skipped=2 | 0.05s

--- Horizon: 30 min ---
station00: paired rows=5,798 | new=0, skipped=2 | 0.07s
station01: paired rows=6,681 | new=0, skipped=2 | 0.08s
station02: paired rows=5,717 | new=0, skipped=2 | 0.06s
station03: paired rows=2,937 | new=0, skipped=2 | 0.04s
station04: paired rows=6,681 | new=0, skipped=2 | 0.08s
station05: paired rows=1,939 | new=0, skipped=2 | 0.03s
station06: paired rows=6,238 | new=0, ski

,model,horizon_min,fits,frozen_epochs,total_fit_seconds,median_fit_seconds,total_predict_seconds
0,LSTM,15,20,7,263.369417,12.765694,8.510341
1,LSTM,30,20,1,63.370131,3.127504,7.147766
2,LSTM,60,20,2,92.999173,4.807829,7.898026
3,TCN,15,20,4,407.324177,21.759690,20.500136
4,TCN,30,20,1,204.075196,10.368077,22.719758
5,TCN,60,20,1,209.340074,10.665565,24.110596



ALL-TEST NEURAL MAE: F3 vs F4


variant,model,station,horizon_min,scope,F3,F4_real,F4_minus_F3
0,LSTM,station00,15,all,0.026083,0.023118,-0.002965
6,LSTM,station01,15,all,0.028854,0.026961,-0.001893
12,LSTM,station02,15,all,0.032154,0.029055,-0.003098
18,LSTM,station03,15,all,0.032376,0.030176,-0.002200
24,LSTM,station04,15,all,0.044431,0.043645,-0.000785
30,LSTM,station05,15,all,0.038554,0.043934,0.005380
36,LSTM,station06,15,all,0.029493,0.026135,-0.003359
42,LSTM,station07,15,all,0.026305,0.025935,-0.000370
48,LSTM,station08,15,all,0.026159,0.025583,-0.000576
54,LSTM,station09,15,all,0.020040,0.019891,-0.000149



DAYLIGHT NEURAL MAE: F3 vs F4


...


DESCRIPTIVE EQUAL-STATION NEURAL EFFECT SUMMARY


,scope,model,horizon_min,stations,mean_station_delta_MAE,median_station_delta_MAE,stations_F4_lower_point,stations_F3_lower_point,stations_tie_point
0,all,LSTM,15,10,-0.001002,-0.001339,9,1,0
1,all,LSTM,30,10,-0.004835,-0.002760,9,1,0
2,all,LSTM,60,10,-0.002388,-0.001679,9,1,0
3,all,TCN,15,10,-0.005827,-0.006172,6,4,0
4,all,TCN,30,10,-0.011507,-0.007447,9,1,0
5,all,TCN,60,10,-0.012775,-0.008001,9,1,0
6,daylight,LSTM,15,10,-0.000717,-0.001169,8,2,0
7,daylight,LSTM,30,10,-0.005222,-0.003669,10,0,0
8,daylight,LSTM,60,10,-0.003331,-0.003294,9,1,0
9,daylight,TCN,15,10,-0.011766,-0.007335,6,4,0



ZERO-VARIANCE SAFEGUARD EXECUTION AUDIT


,model,station,horizon_min,variant,zero_variance_continuous_columns
0,LSTM,station03,15,F3,"(utc_year,)"
1,LSTM,station03,15,F4_real,"(utc_year,)"
2,LSTM,station05,15,F3,"(utc_year,)"
3,LSTM,station05,15,F4_real,"(utc_year,)"
4,LSTM,station03,30,F3,"(utc_year,)"
5,LSTM,station03,30,F4_real,"(utc_year,)"
6,LSTM,station05,30,F3,"(utc_year,)"
7,LSTM,station05,30,F4_real,"(utc_year,)"
8,LSTM,station03,60,F3,"(utc_year,)"
9,LSTM,station03,60,F4_real,"(utc_year,)"



PVOD-07A PRIMARY NEURAL EXECUTION: PASSED
Primary effect = MAE(F4_real) - MAE(F3).
Negative point estimate -> F4 lower MAE.
Positive point estimate -> F3 lower MAE.
These are point estimates only; bootstrap follows in PVOD-07B.


In [17]:
# ============================================================
# PVOD-07B
# PRIMARY NEURAL HIERARCHICAL PAIRED BOOTSTRAP
#
# Identical inferential protocol to PVOD-06B:
#
#   replicates        = 10,000
#   seed              = 42
#   day timezone      = Asia/Shanghai
#   station weighting = equal
#
# Per bootstrap replicate:
#
#   1. sample 10 stations WITH replacement;
#   2. independently resample China-local TEST days within
#      each selected station occurrence;
#   3. preserve paired F3/F4 rows;
#   4. compute station-level
#
#        Delta MAE = MAE(F4_real) - MAE(F3)
#
#   5. average station deltas equally.
#
# NO retraining.
# Training stochasticity is NOT included.
# ============================================================

import time

import numpy as np
import pandas as pd


# ============================================================
# 1. Frozen constants
# ============================================================

PVOD_NEURAL_BOOTSTRAP_REPLICATES = 10_000
PVOD_NEURAL_BOOTSTRAP_SEED = 42
PVOD_NEURAL_BOOTSTRAP_TIMEZONE = "Asia/Shanghai"


assert PVOD_NEURAL_BOOTSTRAP_REPLICATES == 10_000
assert PVOD_NEURAL_BOOTSTRAP_SEED == 42


# ============================================================
# 2. Canonical neural prediction source
# ============================================================

assert "pvod_neural_predictions" in globals()


neural_bootstrap_source = (
    pvod_neural_predictions
    .copy()
)


assert set(
    neural_bootstrap_source["model"].unique()
) == {
    "LSTM",
    "TCN",
}


assert set(
    neural_bootstrap_source["variant"].unique()
) == {
    "F3",
    "F4_real",
}


assert set(
    neural_bootstrap_source["horizon_min"].unique()
) == {
    15,
    30,
    60,
}


# ============================================================
# 3. Reconstruct China-local target date
# ============================================================

neural_bootstrap_source[
    "bootstrap_target_time_china"
] = (
    pd.to_datetime(
        neural_bootstrap_source[
            "target_time_utc"
        ],
        utc=True,
    )
    .dt
    .tz_convert(
        PVOD_NEURAL_BOOTSTRAP_TIMEZONE
    )
)


neural_bootstrap_source[
    "bootstrap_china_date"
] = (
    neural_bootstrap_source[
        "bootstrap_target_time_china"
    ]
    .dt
    .date
)


# ============================================================
# 4. Exact F3/F4 pairing audit
# ============================================================

neural_pair_audit = (
    neural_bootstrap_source
    .groupby(
        [
            "model",
            "station",
            "horizon_min",
            "target_time_utc",
        ],
        sort=False,
    )[
        "variant"
    ]
    .nunique()
)


assert (
    neural_pair_audit
    .eq(2)
    .all()
)


# ============================================================
# 5. Build paired daily sufficient statistics
# ============================================================

neural_daily_frames = []


for scope_name in [
    "all",
    "daylight",
]:

    if scope_name == "all":

        scoped = (
            neural_bootstrap_source
        )

    else:

        scoped = (
            neural_bootstrap_source.loc[
                neural_bootstrap_source[
                    "target_solar_daylight"
                ]
                .eq(1)
            ]
        )


    daily_long = (
        scoped
        .groupby(
            [
                "model",
                "station",
                "horizon_min",
                "bootstrap_china_date",
                "variant",
            ],
            as_index=False,
            sort=True,
        )
        .agg(
            error_sum=(
                "absolute_error_pu",
                "sum",
            ),

            row_count=(
                "absolute_error_pu",
                "size",
            ),
        )
    )


    daily_wide = (
        daily_long
        .pivot(
            index=[
                "model",
                "station",
                "horizon_min",
                "bootstrap_china_date",
            ],

            columns="variant",

            values=[
                "error_sum",
                "row_count",
            ],
        )
        .reset_index()
    )


    daily_wide.columns = [
        (
            "_".join(
                [
                    str(part)
                    for part in column
                    if str(part) != ""
                ]
            )
            if isinstance(
                column,
                tuple,
            )
            else column
        )

        for column in daily_wide.columns
    ]


    required_columns = {
        "error_sum_F3",
        "error_sum_F4_real",
        "row_count_F3",
        "row_count_F4_real",
    }


    assert required_columns.issubset(
        daily_wide.columns
    )


    assert (
        daily_wide[
            [
                "error_sum_F3",
                "error_sum_F4_real",
                "row_count_F3",
                "row_count_F4_real",
            ]
        ]
        .notna()
        .all()
        .all()
    )


    assert np.array_equal(
        daily_wide[
            "row_count_F3"
        ].to_numpy(),
        daily_wide[
            "row_count_F4_real"
        ].to_numpy(),
    )


    daily_wide[
        "paired_row_count"
    ] = (
        daily_wide[
            "row_count_F3"
        ]
        .astype(
            np.int64
        )
    )


    daily_wide[
        "scope"
    ] = scope_name


    neural_daily_frames.append(
        daily_wide
    )


pvod_neural_daily_bootstrap_table = (
    pd.concat(
        neural_daily_frames,
        ignore_index=True,
    )
)


# ============================================================
# 6. Day-count audit
# ============================================================

pvod_neural_station_day_audit = (
    pvod_neural_daily_bootstrap_table
    .groupby(
        [
            "scope",
            "model",
            "horizon_min",
            "station",
        ],
        as_index=False,
    )
    .agg(
        local_days=(
            "bootstrap_china_date",
            "nunique",
        ),

        paired_rows=(
            "paired_row_count",
            "sum",
        ),
    )
)


assert (
    pvod_neural_station_day_audit[
        "local_days"
    ]
    .gt(0)
    .all()
)


assert (
    pvod_neural_station_day_audit[
        "paired_rows"
    ]
    .gt(0)
    .all()
)


# ============================================================
# 7. Build one model/horizon/scope station payload
# ============================================================

def make_neural_station_day_payload(
    model_name: str,
    horizon_min: int,
    scope_name: str,
):

    condition = (
        pvod_neural_daily_bootstrap_table.loc[
            (
                pvod_neural_daily_bootstrap_table[
                    "model"
                ]
                ==
                model_name
            )
            &
            (
                pvod_neural_daily_bootstrap_table[
                    "horizon_min"
                ]
                ==
                horizon_min
            )
            &
            (
                pvod_neural_daily_bootstrap_table[
                    "scope"
                ]
                ==
                scope_name
            )
        ]
        .copy()
    )


    stations = sorted(
        condition[
            "station"
        ]
        .unique()
        .tolist()
    )


    assert stations == sorted(
        PVOD_EXTERNAL_STATIONS
    )


    payload = []


    for station_id in stations:

        station_days = (
            condition.loc[
                condition[
                    "station"
                ]
                ==
                station_id
            ]
            .sort_values(
                "bootstrap_china_date"
            )
            .reset_index(
                drop=True
            )
        )


        error_f3 = (
            station_days[
                "error_sum_F3"
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        error_f4 = (
            station_days[
                "error_sum_F4_real"
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        counts = (
            station_days[
                "paired_row_count"
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        assert (
            len(error_f3)
            ==
            len(error_f4)
            ==
            len(counts)
        )


        assert len(counts) > 0
        assert (counts > 0).all()


        point_mae_f3 = float(
            error_f3.sum()
            /
            counts.sum()
        )


        point_mae_f4 = float(
            error_f4.sum()
            /
            counts.sum()
        )


        payload.append(
            {
                "station":
                    station_id,

                "error_f3":
                    error_f3,

                "error_f4":
                    error_f4,

                "counts":
                    counts,

                "n_days":
                    len(
                        counts
                    ),

                "point_delta":
                    float(
                        point_mae_f4
                        -
                        point_mae_f3
                    ),
            }
        )


    assert len(payload) == 10


    return payload


# ============================================================
# 8. Hierarchical paired bootstrap
# ============================================================

def neural_hierarchical_paired_bootstrap(
    model_name: str,
    horizon_min: int,
    scope_name: str,
    rng: np.random.Generator,
    n_replicates: int,
):

    payload = (
        make_neural_station_day_payload(
            model_name=
                model_name,

            horizon_min=
                horizon_min,

            scope_name=
                scope_name,
        )
    )


    n_stations = len(
        payload
    )


    assert n_stations == 10


    point_station_deltas = np.array(
        [
            station_data[
                "point_delta"
            ]

            for station_data
            in payload
        ],
        dtype=np.float64,
    )


    point_estimate = float(
        point_station_deltas.mean()
    )


    bootstrap_values = np.empty(
        n_replicates,
        dtype=np.float64,
    )


    for replicate_index in range(
        n_replicates
    ):

        # ----------------------------------------------------
        # Level 1: station resampling
        # ----------------------------------------------------

        station_draw = (
            rng.integers(
                low=0,
                high=n_stations,
                size=n_stations,
            )
        )


        station_delta_draw = np.empty(
            n_stations,
            dtype=np.float64,
        )


        # ----------------------------------------------------
        # Level 2: independent within-station day resampling
        #
        # If a station is drawn twice, each occurrence gets an
        # independent day bootstrap.
        # ----------------------------------------------------

        for draw_position, station_index in enumerate(
            station_draw
        ):

            station_data = (
                payload[
                    int(
                        station_index
                    )
                ]
            )


            n_days = int(
                station_data[
                    "n_days"
                ]
            )


            day_draw = (
                rng.integers(
                    low=0,
                    high=n_days,
                    size=n_days,
                )
            )


            sampled_counts = (
                station_data[
                    "counts"
                ][
                    day_draw
                ]
            )


            sampled_error_f3 = (
                station_data[
                    "error_f3"
                ][
                    day_draw
                ]
            )


            sampled_error_f4 = (
                station_data[
                    "error_f4"
                ][
                    day_draw
                ]
            )


            denominator = float(
                sampled_counts.sum()
            )


            assert denominator > 0.0


            sampled_mae_f3 = float(
                sampled_error_f3.sum()
                /
                denominator
            )


            sampled_mae_f4 = float(
                sampled_error_f4.sum()
                /
                denominator
            )


            station_delta_draw[
                draw_position
            ] = (
                sampled_mae_f4
                -
                sampled_mae_f3
            )


        bootstrap_values[
            replicate_index
        ] = float(
            station_delta_draw.mean()
        )


    return {
        "point_estimate":
            point_estimate,

        "point_station_deltas":
            point_station_deltas,

        "bootstrap_values":
            bootstrap_values,
    }


# ============================================================
# 9. Execute 12 neural inferential comparisons
# ============================================================

rng = np.random.default_rng(
    PVOD_NEURAL_BOOTSTRAP_SEED
)


neural_bootstrap_summary_rows = []
neural_bootstrap_draw_frames = []


bootstrap_start = (
    time.perf_counter()
)


for scope_name in [
    "all",
    "daylight",
]:

    for model_name in [
        "LSTM",
        "TCN",
    ]:

        for horizon_min in (
            PVOD_EXTERNAL_HORIZONS
        ):

            print(
                f"Bootstrapping "
                f"{scope_name:8s} | "
                f"{model_name:4s} | "
                f"{horizon_min:2d} min"
            )


            result = (
                neural_hierarchical_paired_bootstrap(
                    model_name=
                        model_name,

                    horizon_min=
                        horizon_min,

                    scope_name=
                        scope_name,

                    rng=
                        rng,

                    n_replicates=
                        PVOD_NEURAL_BOOTSTRAP_REPLICATES,
                )
            )


            bootstrap_values = (
                result[
                    "bootstrap_values"
                ]
            )


            ci_low = float(
                np.quantile(
                    bootstrap_values,
                    0.025,
                )
            )


            ci_high = float(
                np.quantile(
                    bootstrap_values,
                    0.975,
                )
            )


            if ci_high < 0.0:

                classification = (
                    "F4_lower"
                )

            elif ci_low > 0.0:

                classification = (
                    "F3_lower"
                )

            else:

                classification = (
                    "crosses_zero"
                )


            neural_bootstrap_summary_rows.append(
                {
                    "scope":
                        scope_name,

                    "model":
                        model_name,

                    "horizon_min":
                        horizon_min,

                    "stations":
                        10,

                    "replicates":
                        PVOD_NEURAL_BOOTSTRAP_REPLICATES,

                    "point_delta_MAE":
                        result[
                            "point_estimate"
                        ],

                    "bootstrap_mean":
                        float(
                            bootstrap_values.mean()
                        ),

                    "bootstrap_median":
                        float(
                            np.median(
                                bootstrap_values
                            )
                        ),

                    "ci95_low":
                        ci_low,

                    "ci95_high":
                        ci_high,

                    "classification":
                        classification,
                }
            )


            neural_bootstrap_draw_frames.append(
                pd.DataFrame(
                    {
                        "scope":
                            scope_name,

                        "model":
                            model_name,

                        "horizon_min":
                            horizon_min,

                        "replicate":
                            np.arange(
                                1,
                                PVOD_NEURAL_BOOTSTRAP_REPLICATES
                                +
                                1,
                                dtype=np.int32,
                            ),

                        "delta_MAE_F4_minus_F3":
                            bootstrap_values,
                    }
                )
            )


bootstrap_seconds = float(
    time.perf_counter()
    -
    bootstrap_start
)


# ============================================================
# 10. Canonical neural bootstrap objects
# ============================================================

pvod_neural_hierarchical_bootstrap_summary = (
    pd.DataFrame(
        neural_bootstrap_summary_rows
    )
    .sort_values(
        [
            "scope",
            "model",
            "horizon_min",
        ]
    )
    .reset_index(
        drop=True
    )
)


pvod_neural_hierarchical_bootstrap_draws = (
    pd.concat(
        neural_bootstrap_draw_frames,
        ignore_index=True,
    )
)


assert len(
    pvod_neural_hierarchical_bootstrap_summary
) == 12


assert len(
    pvod_neural_hierarchical_bootstrap_draws
) == 120_000


# ============================================================
# 11. Reconcile bootstrap point estimate with PVOD-07A
# ============================================================

expected_neural_points = (
    pvod_neural_station_aggregate[
        [
            "scope",
            "model",
            "horizon_min",
            "mean_station_delta_MAE",
        ]
    ]
    .rename(
        columns={
            "mean_station_delta_MAE":
                "expected_point_delta_MAE",
        }
    )
)


neural_point_check = (
    pvod_neural_hierarchical_bootstrap_summary
    .merge(
        expected_neural_points,
        on=[
            "scope",
            "model",
            "horizon_min",
        ],
        how="left",
        validate="one_to_one",
    )
)


neural_point_check[
    "absolute_difference"
] = np.abs(
    neural_point_check[
        "point_delta_MAE"
    ]
    -
    neural_point_check[
        "expected_point_delta_MAE"
    ]
)


assert (
    neural_point_check[
        "absolute_difference"
    ]
    .max()
    <
    1e-12
)


# ============================================================
# 12. Classification counts
# ============================================================

pvod_neural_bootstrap_classification_counts = (
    pvod_neural_hierarchical_bootstrap_summary
    .groupby(
        [
            "scope",
            "model",
            "classification",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size":
                "comparisons",
        }
    )
)


# ============================================================
# 13. Display
# ============================================================

print(
    "\n"
    + "=" * 125
)

print(
    "PVOD PRIMARY NEURAL HIERARCHICAL BOOTSTRAP"
)

print(
    "=" * 125
)


display(
    pvod_neural_hierarchical_bootstrap_summary
)


print(
    "\n"
    + "=" * 125
)

print(
    "NEURAL CLASSIFICATION COUNTS"
)

print(
    "=" * 125
)


display(
    pvod_neural_bootstrap_classification_counts
)


print(
    "\n"
    + "=" * 125
)

print(
    "NEURAL BOOTSTRAP AUDIT"
)

print(
    "=" * 125
)


print(
    "Replicates per comparison:",
    PVOD_NEURAL_BOOTSTRAP_REPLICATES,
)


print(
    "Seed:",
    PVOD_NEURAL_BOOTSTRAP_SEED,
)


print(
    "China-local day timezone:",
    PVOD_NEURAL_BOOTSTRAP_TIMEZONE,
)


print(
    "Station weighting: equal"
)


print(
    "Total bootstrap draws stored:",
    len(
        pvod_neural_hierarchical_bootstrap_draws
    ),
)


print(
    "Maximum point-estimate reconciliation error:",
    float(
        neural_point_check[
            "absolute_difference"
        ]
        .max()
    ),
)


print(
    "Bootstrap execution seconds:",
    bootstrap_seconds,
)


print(
    "\nPVOD-07B PRIMARY NEURAL HIERARCHICAL BOOTSTRAP: PASSED"
)


print(
    "\nClassification rule:"
)

print(
    "  CI entirely below zero -> F4_lower"
)

print(
    "  CI entirely above zero -> F3_lower"
)

print(
    "  otherwise              -> crosses_zero"
)


print(
    "\nNo model retrained."
)

print(
    "No hyperparameter changed."
)

print(
    "No epoch count changed."
)

print(
    "No prediction clipped."
)

print(
    "Uncertainty reflects station/day resampling only."
)

print(
    "Neural training stochasticity is NOT represented."
)

Bootstrapping all      | LSTM | 15 min
Bootstrapping all      | LSTM | 30 min
Bootstrapping all      | LSTM | 60 min
Bootstrapping all      | TCN  | 15 min
Bootstrapping all      | TCN  | 30 min
Bootstrapping all      | TCN  | 60 min
Bootstrapping daylight | LSTM | 15 min
Bootstrapping daylight | LSTM | 30 min
Bootstrapping daylight | LSTM | 60 min
Bootstrapping daylight | TCN  | 15 min
Bootstrapping daylight | TCN  | 30 min
Bootstrapping daylight | TCN  | 60 min

PVOD PRIMARY NEURAL HIERARCHICAL BOOTSTRAP


,scope,model,horizon_min,stations,replicates,point_delta_MAE,bootstrap_mean,bootstrap_median,ci95_low,ci95_high,classification
0,all,LSTM,15,10,10000,-0.001002,-0.001015,-0.001117,-0.002300,0.000784,crosses_zero
1,all,LSTM,30,10,10000,-0.004835,-0.004826,-0.004606,-0.009518,-0.001278,F4_lower
2,all,LSTM,60,10,10000,-0.002388,-0.002399,-0.002367,-0.005218,0.000305,crosses_zero
3,all,TCN,15,10,10000,-0.005827,-0.005809,-0.005957,-0.014556,0.003307,crosses_zero
4,all,TCN,30,10,10000,-0.011507,-0.011489,-0.011034,-0.020179,-0.005358,F4_lower
5,all,TCN,60,10,10000,-0.012775,-0.012726,-0.012150,-0.023781,-0.005613,F4_lower
6,daylight,LSTM,15,10,10000,-0.000717,-0.000705,-0.000773,-0.001623,0.000564,crosses_zero
7,daylight,LSTM,30,10,10000,-0.005222,-0.005225,-0.004873,-0.009825,-0.002339,F4_lower
8,daylight,LSTM,60,10,10000,-0.003331,-0.003310,-0.003306,-0.006009,-0.000644,F4_lower
9,daylight,TCN,15,10,10000,-0.011766,-0.011752,-0.011158,-0.026385,-0.000018,F4_lower



NEURAL CLASSIFICATION COUNTS


,scope,model,classification,comparisons
0,all,LSTM,F4_lower,1
1,all,LSTM,crosses_zero,2
2,all,TCN,F4_lower,2
3,all,TCN,crosses_zero,1
4,daylight,LSTM,F4_lower,2
5,daylight,LSTM,crosses_zero,1
6,daylight,TCN,F4_lower,3



NEURAL BOOTSTRAP AUDIT
Replicates per comparison: 10000
Seed: 42
China-local day timezone: Asia/Shanghai
Station weighting: equal
Total bootstrap draws stored: 120000
Maximum point-estimate reconciliation error: 1.3877787807814457e-17
Bootstrap execution seconds: 14.342907374724746

PVOD-07B PRIMARY NEURAL HIERARCHICAL BOOTSTRAP: PASSED

Classification rule:
  CI entirely below zero -> F4_lower
  CI entirely above zero -> F3_lower
  otherwise              -> crosses_zero

No model retrained.
No hyperparameter changed.
No epoch count changed.
No prediction clipped.
Uncertainty reflects station/day resampling only.
Neural training stochasticity is NOT represented.


In [18]:
# ============================================================
# PVOD-08A
# Consolidate PRIMARY PVOD external-validation evidence
#
# NO fitting.
# NO bootstrap.
# NO new inference.
#
# Simply combines the already-completed classical and neural
# hierarchical bootstrap results into one canonical table.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. Preconditions
# ============================================================

assert (
    "pvod_classical_hierarchical_bootstrap_summary"
    in globals()
)

assert (
    "pvod_neural_hierarchical_bootstrap_summary"
    in globals()
)


# ============================================================
# 2. Combine all four architectures
# ============================================================

classical_primary = (
    pvod_classical_hierarchical_bootstrap_summary
    .copy()
)

classical_primary[
    "architecture_family"
] = "classical"


neural_primary = (
    pvod_neural_hierarchical_bootstrap_summary
    .copy()
)

neural_primary[
    "architecture_family"
] = "neural"


pvod_primary_external_validation_summary = (
    pd.concat(
        [
            classical_primary,
            neural_primary,
        ],
        ignore_index=True,
    )
)


# ============================================================
# 3. Strong shape/content assertions
# ============================================================

assert len(
    pvod_primary_external_validation_summary
) == 24


assert set(
    pvod_primary_external_validation_summary[
        "model"
    ]
) == {
    "Ridge",
    "LightGBM",
    "LSTM",
    "TCN",
}


assert set(
    pvod_primary_external_validation_summary[
        "scope"
    ]
) == {
    "all",
    "daylight",
}


assert set(
    pvod_primary_external_validation_summary[
        "horizon_min"
    ]
) == {
    15,
    30,
    60,
}


assert (
    pvod_primary_external_validation_summary[
        "replicates"
    ]
    .eq(
        10_000
    )
    .all()
)


# ============================================================
# 4. Resolution flag
# ============================================================

pvod_primary_external_validation_summary[
    "resolved"
] = (
    pvod_primary_external_validation_summary[
        "classification"
    ]
    !=
    "crosses_zero"
)


# ============================================================
# 5. Stable architecture ordering
# ============================================================

model_order = [
    "Ridge",
    "LightGBM",
    "LSTM",
    "TCN",
]


scope_order = [
    "all",
    "daylight",
]


pvod_primary_external_validation_summary[
    "model"
] = pd.Categorical(
    pvod_primary_external_validation_summary[
        "model"
    ],
    categories=model_order,
    ordered=True,
)


pvod_primary_external_validation_summary[
    "scope"
] = pd.Categorical(
    pvod_primary_external_validation_summary[
        "scope"
    ],
    categories=scope_order,
    ordered=True,
)


pvod_primary_external_validation_summary = (
    pvod_primary_external_validation_summary
    .sort_values(
        [
            "scope",
            "model",
            "horizon_min",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 6. Classification matrices
# ============================================================

all_test_matrix = (
    pvod_primary_external_validation_summary.loc[
        pvod_primary_external_validation_summary[
            "scope"
        ]
        ==
        "all"
    ]
    .pivot(
        index="model",
        columns="horizon_min",
        values="classification",
    )
    .reindex(
        model_order
    )
)


daylight_matrix = (
    pvod_primary_external_validation_summary.loc[
        pvod_primary_external_validation_summary[
            "scope"
        ]
        ==
        "daylight"
    ]
    .pivot(
        index="model",
        columns="horizon_min",
        values="classification",
    )
    .reindex(
        model_order
    )
)


# ============================================================
# 7. Architecture-level classification counts
# ============================================================

pvod_primary_architecture_counts = (
    pvod_primary_external_validation_summary
    .groupby(
        [
            "scope",
            "model",
            "classification",
        ],
        observed=True,
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size":
                "comparisons",
        }
    )
)


# ============================================================
# 8. Compact effect table
# ============================================================

pvod_primary_effect_table = (
    pvod_primary_external_validation_summary[
        [
            "scope",
            "architecture_family",
            "model",
            "horizon_min",
            "point_delta_MAE",
            "ci95_low",
            "ci95_high",
            "classification",
            "resolved",
        ]
    ]
    .copy()
)


# ============================================================
# 9. Primary all-test counts
# ============================================================

all_primary = (
    pvod_primary_external_validation_summary.loc[
        pvod_primary_external_validation_summary[
            "scope"
        ]
        ==
        "all"
    ]
)


primary_all_counts = (
    all_primary[
        "classification"
    ]
    .value_counts()
    .reindex(
        [
            "F4_lower",
            "F3_lower",
            "crosses_zero",
        ],
        fill_value=0,
    )
)


# ============================================================
# 10. Display
# ============================================================

print(
    "=" * 125
)

print(
    "PVOD PRIMARY EXTERNAL-VALIDATION SYNTHESIS"
)

print(
    "=" * 125
)


print(
    "\nALL-TEST CLASSIFICATION MATRIX"
)

display(
    all_test_matrix
)


print(
    "\nDAYLIGHT CLASSIFICATION MATRIX"
)

display(
    daylight_matrix
)


print(
    "\n"
    + "=" * 125
)

print(
    "ARCHITECTURE-LEVEL CLASSIFICATION COUNTS"
)

print(
    "=" * 125
)


display(
    pvod_primary_architecture_counts
)


print(
    "\n"
    + "=" * 125
)

print(
    "ALL-TEST PRIMARY COUNTS"
)

print(
    "=" * 125
)


print(
    "F4_lower:",
    int(
        primary_all_counts[
            "F4_lower"
        ]
    ),
)

print(
    "F3_lower:",
    int(
        primary_all_counts[
            "F3_lower"
        ]
    ),
)

print(
    "crosses_zero:",
    int(
        primary_all_counts[
            "crosses_zero"
        ]
    ),
)


print(
    "\nCanonical combined object:"
)

print(
    "  pvod_primary_external_validation_summary"
)


print(
    "\nPVOD-08A PRIMARY EXTERNAL-VALIDATION SYNTHESIS: PASSED"
)


print(
    "\nNo model fitted."
)

print(
    "No bootstrap rerun."
)

print(
    "No result changed."
)

print(
    "Primary F3-vs-F4 external validation is now complete."
)

PVOD PRIMARY EXTERNAL-VALIDATION SYNTHESIS

ALL-TEST CLASSIFICATION MATRIX


horizon_min,15,30,60
model,,,
Ridge,F3_lower,F3_lower,F3_lower
LightGBM,crosses_zero,crosses_zero,crosses_zero
LSTM,crosses_zero,F4_lower,crosses_zero
TCN,crosses_zero,F4_lower,F4_lower



DAYLIGHT CLASSIFICATION MATRIX


horizon_min,15,30,60
model,,,
Ridge,F3_lower,F3_lower,F3_lower
LightGBM,crosses_zero,crosses_zero,crosses_zero
LSTM,crosses_zero,F4_lower,F4_lower
TCN,F4_lower,F4_lower,F4_lower



ARCHITECTURE-LEVEL CLASSIFICATION COUNTS


,scope,model,classification,comparisons
0,all,Ridge,F3_lower,3
1,all,LightGBM,crosses_zero,3
2,all,LSTM,F4_lower,1
3,all,LSTM,crosses_zero,2
4,all,TCN,F4_lower,2
5,all,TCN,crosses_zero,1
6,daylight,Ridge,F3_lower,3
7,daylight,LightGBM,crosses_zero,3
8,daylight,LSTM,F4_lower,2
9,daylight,LSTM,crosses_zero,1



ALL-TEST PRIMARY COUNTS
F4_lower: 3
F3_lower: 3
crosses_zero: 6

Canonical combined object:
  pvod_primary_external_validation_summary

PVOD-08A PRIMARY EXTERNAL-VALIDATION SYNTHESIS: PASSED

No model fitted.
No bootstrap rerun.
No result changed.
Primary F3-vs-F4 external validation is now complete.


In [19]:
# ============================================================
# PVOD-08B
# SECONDARY CONTROL REPRESENTATION CONSTRUCTION + AUDIT
#
# NO model fitting.
# NO prediction.
# NO metric.
#
# Controls:
#
#   F4_reference
#   F4_shift_17d
#   F4_shift_37d
#   F4_shift_61d
#   F4_perm_1301
#   F4_perm_1302
#   F4_perm_1303
#
# Exact Phase-2 semantics:
#
#   reference:
#       drop one canonical indicator from each Panchang family
#
#   shifts:
#       circular row shift WITHIN train / validation / test
#
#   permutations:
#       jointly permute complete Panchang rows WITHIN each split
#       seed = base_seed + split_offset
#
# PVOD adaptation:
#       15-minute resolution -> 96 rows/day
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. Frozen control registry
# ============================================================

PVOD_SECONDARY_CONTROL_VARIANTS = [
    "F4_reference",
    "F4_shift_17d",
    "F4_shift_37d",
    "F4_shift_61d",
    "F4_perm_1301",
    "F4_perm_1302",
    "F4_perm_1303",
]


PVOD_CONTROL_SHIFT_DAYS = {
    "F4_shift_17d": 17,
    "F4_shift_37d": 37,
    "F4_shift_61d": 61,
}


PVOD_CONTROL_PERMUTATION_SEEDS = {
    "F4_perm_1301": 1301,
    "F4_perm_1302": 1302,
    "F4_perm_1303": 1303,
}


PVOD_CONTROL_RESOLUTION_MINUTES = 15
PVOD_CONTROL_STEPS_PER_DAY = 96


assert (
    (24 * 60)
    //
    PVOD_CONTROL_RESOLUTION_MINUTES
    ==
    PVOD_CONTROL_STEPS_PER_DAY
)


# ============================================================
# 2. Recover canonical Panchang family ordering
# ============================================================

pvod_panchang_columns = list(
    pvod_f4_onehot_global.columns
)


assert len(
    pvod_panchang_columns
) == 97


PVOD_PANCHANG_FAMILY_PREFIXES = {
    "tithi":
        "tithi_",

    "paksha":
        "paksha_",

    "karana":
        "karana_",

    "nakshatra":
        "nakshatra_",

    "yoga":
        "yoga_",
}


PVOD_EXPECTED_PANCHANG_FAMILY_WIDTHS = {
    "tithi": 30,
    "paksha": 2,
    "karana": 11,
    "nakshatra": 27,
    "yoga": 27,
}


pvod_panchang_family_columns = {}


for family_name, prefix in (
    PVOD_PANCHANG_FAMILY_PREFIXES.items()
):

    family_columns = [
        column
        for column
        in pvod_panchang_columns
        if column.startswith(
            prefix
        )
    ]


    assert len(
        family_columns
    ) == (
        PVOD_EXPECTED_PANCHANG_FAMILY_WIDTHS[
            family_name
        ]
    )


    pvod_panchang_family_columns[
        family_name
    ] = family_columns


assert (
    sum(
        len(columns)
        for columns
        in pvod_panchang_family_columns.values()
    )
    ==
    97
)


# ============================================================
# 3. Freeze reference coding
#
# "Canonical indicator" is operationalized prospectively for
# PVOD as:
#
#   first indicator in the already-canonical one-hot column
#   order of each family.
#
# This is frozen BEFORE any secondary-control forecasting
# result is observed.
# ============================================================

PVOD_F4_REFERENCE_DROP_COLUMNS = tuple(
    pvod_panchang_family_columns[
        family_name
    ][0]

    for family_name in [
        "tithi",
        "paksha",
        "karana",
        "nakshatra",
        "yoga",
    ]
)


PVOD_F4_REFERENCE_COLUMNS = [
    column
    for column
    in pvod_panchang_columns
    if column
    not in
    PVOD_F4_REFERENCE_DROP_COLUMNS
]


assert len(
    PVOD_F4_REFERENCE_DROP_COLUMNS
) == 5


assert len(
    PVOD_F4_REFERENCE_COLUMNS
) == 92


# ============================================================
# 4. Retrieve correctly aligned Panchang block for one
#    station/horizon sample population
# ============================================================

def get_pvod_real_panchang_sample_block(
    station_id: str,
    horizon_min: int,
):

    samples = (
        pvod_sample_index[
            (
                station_id,
                horizon_min,
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    issue_times = pd.DatetimeIndex(
        samples[
            "issue_time_utc"
        ]
    )


    target_times = pd.DatetimeIndex(
        samples[
            "target_time_utc"
        ]
    )


    block = (
        pvod_f4_onehot_global
        .loc[
            target_times
        ]
        .copy()
    )


    # Modelling rows are indexed by issue time.
    block.index = issue_times


    metadata = (
        samples.copy()
    )

    metadata.index = issue_times


    assert block.shape == (
        len(
            samples
        ),
        97,
    )


    assert block.index.equals(
        metadata.index
    )


    block = block.astype(
        np.int8
    )


    values = block.to_numpy(
        dtype=np.int8
    )


    assert (
        (
            values
            ==
            0
        )
        |
        (
            values
            ==
            1
        )
    ).all()


    # Exactly one active indicator in each of 5 families.
    assert np.array_equal(
        values.sum(
            axis=1
        ),
        np.full(
            len(
                block
            ),
            5,
            dtype=np.int64,
        ),
    )


    return block, metadata


# ============================================================
# 5. Exact splitwise joint permutation
# ============================================================

def pvod_splitwise_joint_permutation(
    block: pd.DataFrame,
    metadata: pd.DataFrame,
    seed: int,
):

    out = pd.DataFrame(
        index=block.index,
        columns=block.columns,
        dtype=np.int8,
    )


    split_names = [
        "train",
        "validation",
        "test",
    ]


    for split_offset, split_name in enumerate(
        split_names
    ):

        split_mask = (
            metadata[
                "split"
            ]
            .eq(
                split_name
            )
        )


        split_index = (
            metadata.index[
                split_mask
            ]
        )


        values = (
            block.loc[
                split_index
            ]
            .to_numpy(
                copy=True
            )
        )


        assert len(
            values
        ) > 0


        rng = np.random.default_rng(
            int(
                seed
            )
            +
            int(
                split_offset
            )
        )


        order = rng.permutation(
            len(
                values
            )
        )


        out.loc[
            split_index,
            :
        ] = values[
            order
        ]


    return out.astype(
        np.int8
    )


# ============================================================
# 6. Exact splitwise circular shift
# ============================================================

def pvod_splitwise_circular_shift(
    block: pd.DataFrame,
    metadata: pd.DataFrame,
    shift_days: int,
):

    shift_steps = (
        int(
            shift_days
        )
        *
        PVOD_CONTROL_STEPS_PER_DAY
    )


    out = pd.DataFrame(
        index=block.index,
        columns=block.columns,
        dtype=np.int8,
    )


    for split_name in [
        "train",
        "validation",
        "test",
    ]:

        split_mask = (
            metadata[
                "split"
            ]
            .eq(
                split_name
            )
        )


        split_index = (
            metadata.index[
                split_mask
            ]
        )


        values = (
            block.loc[
                split_index
            ]
            .to_numpy(
                copy=True
            )
        )


        assert len(
            values
        ) > 0


        effective_shift = (
            shift_steps
            %
            len(
                values
            )
        )


        shifted = np.roll(
            values,
            shift=
                effective_shift,
            axis=0,
        )


        out.loc[
            split_index,
            :
        ] = shifted


    return out.astype(
        np.int8
    )


# ============================================================
# 7. Unified secondary-control block builder
# ============================================================

def build_pvod_secondary_panchang_block(
    station_id: str,
    horizon_min: int,
    variant_name: str,
):

    if variant_name not in (
        PVOD_SECONDARY_CONTROL_VARIANTS
    ):

        raise KeyError(
            f"Unknown PVOD control variant: "
            f"{variant_name}"
        )


    real_block, metadata = (
        get_pvod_real_panchang_sample_block(
            station_id=
                station_id,

            horizon_min=
                horizon_min,
        )
    )


    # --------------------------------------------------------
    # Reference coding
    # --------------------------------------------------------

    if variant_name == "F4_reference":

        control = (
            real_block[
                PVOD_F4_REFERENCE_COLUMNS
            ]
            .copy()
            .astype(
                np.int8
            )
        )


    # --------------------------------------------------------
    # Circular shifts
    # --------------------------------------------------------

    elif variant_name in (
        PVOD_CONTROL_SHIFT_DAYS
    ):

        control = (
            pvod_splitwise_circular_shift(
                block=
                    real_block,

                metadata=
                    metadata,

                shift_days=
                    PVOD_CONTROL_SHIFT_DAYS[
                        variant_name
                    ],
            )
        )


    # --------------------------------------------------------
    # Joint permutations
    # --------------------------------------------------------

    else:

        control = (
            pvod_splitwise_joint_permutation(
                block=
                    real_block,

                metadata=
                    metadata,

                seed=
                    PVOD_CONTROL_PERMUTATION_SEEDS[
                        variant_name
                    ],
            )
        )


    return {
        "block":
            control,

        "metadata":
            metadata,

        "real_block":
            real_block,
    }


# ============================================================
# 8. Full control-integrity audit
# ============================================================

control_audit_rows = []
effective_shift_rows = []


for station_id in (
    PVOD_EXTERNAL_STATIONS
):

    for horizon_min in (
        PVOD_EXTERNAL_HORIZONS
    ):

        real_block, metadata = (
            get_pvod_real_panchang_sample_block(
                station_id=
                    station_id,

                horizon_min=
                    horizon_min,
            )
        )


        # ----------------------------------------------------
        # Shift wrap/effective-distance audit
        # ----------------------------------------------------

        for variant_name, shift_days in (
            PVOD_CONTROL_SHIFT_DAYS.items()
        ):

            raw_shift_rows = (
                int(
                    shift_days
                )
                *
                PVOD_CONTROL_STEPS_PER_DAY
            )


            for split_name in [
                "train",
                "validation",
                "test",
            ]:

                n_rows = int(
                    metadata[
                        "split"
                    ]
                    .eq(
                        split_name
                    )
                    .sum()
                )


                effective_rows = int(
                    raw_shift_rows
                    %
                    n_rows
                )


                effective_shift_rows.append(
                    {
                        "station":
                            station_id,

                        "horizon_min":
                            int(
                                horizon_min
                            ),

                        "split":
                            split_name,

                        "variant":
                            variant_name,

                        "requested_shift_days":
                            int(
                                shift_days
                            ),

                        "split_rows":
                            n_rows,

                        "effective_shift_rows":
                            effective_rows,

                        "effective_shift_days":
                            float(
                                effective_rows
                                *
                                PVOD_CONTROL_RESOLUTION_MINUTES
                                /
                                (
                                    24
                                    *
                                    60
                                )
                            ),
                    }
                )


        # ----------------------------------------------------
        # Variant integrity
        # ----------------------------------------------------

        for variant_name in (
            PVOD_SECONDARY_CONTROL_VARIANTS
        ):

            built = (
                build_pvod_secondary_panchang_block(
                    station_id=
                        station_id,

                    horizon_min=
                        horizon_min,

                    variant_name=
                        variant_name,
                )
            )


            control = (
                built[
                    "block"
                ]
            )


            expected_width = (
                92
                if variant_name
                ==
                "F4_reference"
                else
                97
            )


            assert control.shape == (
                len(
                    metadata
                ),
                expected_width,
            )


            assert control.index.equals(
                real_block.index
            )


            control_values = (
                control.to_numpy(
                    dtype=np.int8
                )
            )


            assert (
                (
                    control_values
                    ==
                    0
                )
                |
                (
                    control_values
                    ==
                    1
                )
            ).all()


            # ------------------------------------------------
            # Full-width nulls retain exactly five active
            # Panchang indicators per row.
            # ------------------------------------------------

            if variant_name != "F4_reference":

                assert np.array_equal(
                    control_values.sum(
                        axis=1
                    ),
                    np.full(
                        len(
                            control
                        ),
                        5,
                        dtype=np.int64,
                    ),
                )


                # --------------------------------------------
                # Each split retains exact marginal category
                # frequencies because the operation is only a
                # row permutation / circular row shift.
                # --------------------------------------------

                for split_name in [
                    "train",
                    "validation",
                    "test",
                ]:

                    split_index = (
                        metadata.index[
                            metadata[
                                "split"
                            ]
                            .eq(
                                split_name
                            )
                        ]
                    )


                    assert np.array_equal(
                        control.loc[
                            split_index
                        ]
                        .sum(
                            axis=0
                        )
                        .to_numpy(),

                        real_block.loc[
                            split_index
                        ]
                        .sum(
                            axis=0
                        )
                        .to_numpy(),
                    )


            else:

                # Reference block is exactly the real block
                # minus five frozen canonical indicators.
                assert control.columns.tolist() == (
                    PVOD_F4_REFERENCE_COLUMNS
                )


                assert np.array_equal(
                    control.to_numpy(),
                    real_block[
                        PVOD_F4_REFERENCE_COLUMNS
                    ]
                    .to_numpy(),
                )


            control_audit_rows.append(
                {
                    "station":
                        station_id,

                    "horizon_min":
                        int(
                            horizon_min
                        ),

                    "variant":
                        variant_name,

                    "rows":
                        len(
                            control
                        ),

                    "columns":
                        control.shape[
                            1
                        ],

                    "binary":
                        True,

                    "index_aligned":
                        True,
                }
            )


pvod_secondary_control_audit = (
    pd.DataFrame(
        control_audit_rows
    )
)


pvod_secondary_effective_shift_audit = (
    pd.DataFrame(
        effective_shift_rows
    )
)


# ============================================================
# 9. Strong expected-case assertions
# ============================================================

assert len(
    pvod_secondary_control_audit
) == (
    10
    *
    3
    *
    7
)


assert (
    pvod_secondary_control_audit[
        "binary"
    ]
    .all()
)


assert (
    pvod_secondary_control_audit[
        "index_aligned"
    ]
    .all()
)


# Reference:
# F3 total = 100
# + 92 categorical = 192 total
#
# Neural auxiliary:
# F3 auxiliary = 30
# + 92 categorical = 122

PVOD_F4_REFERENCE_TOTAL_DIM = 192
PVOD_F4_REFERENCE_NEURAL_AUX_DIM = 122


assert (
    PVOD_EXPECTED_DIMS[
        "F3"
    ]
    +
    92
    ==
    PVOD_F4_REFERENCE_TOTAL_DIM
)


assert (
    30
    +
    92
    ==
    PVOD_F4_REFERENCE_NEURAL_AUX_DIM
)


# ============================================================
# 10. Effective-shift summary
# ============================================================

pvod_effective_shift_summary = (
    pvod_secondary_effective_shift_audit
    .groupby(
        [
            "variant",
            "split",
        ],
        as_index=False,
    )
    .agg(
        cases=(
            "station",
            "size",
        ),

        min_effective_days=(
            "effective_shift_days",
            "min",
        ),

        median_effective_days=(
            "effective_shift_days",
            "median",
        ),

        max_effective_days=(
            "effective_shift_days",
            "max",
        ),

        zero_shift_cases=(
            "effective_shift_rows",
            lambda values:
                int(
                    (
                        values
                        ==
                        0
                    )
                    .sum()
                ),
        ),

        under_one_day_cases=(
            "effective_shift_days",
            lambda values:
                int(
                    (
                        values
                        <
                        1.0
                    )
                    .sum()
                ),
        ),
    )
)


# ============================================================
# 11. Display
# ============================================================

print(
    "=" * 125
)

print(
    "PVOD SECONDARY CONTROL REPRESENTATION FREEZE"
)

print(
    "=" * 125
)


print(
    "\nReference-coded columns dropped:"
)

for column in (
    PVOD_F4_REFERENCE_DROP_COLUMNS
):

    print(
        " ",
        column
    )


print(
    "\nReference Panchang width:",
    len(
        PVOD_F4_REFERENCE_COLUMNS
    ),
)


print(
    "Reference total F4 width:",
    PVOD_F4_REFERENCE_TOTAL_DIM,
)


print(
    "Reference neural auxiliary width:",
    PVOD_F4_REFERENCE_NEURAL_AUX_DIM,
)


print(
    "\n"
    + "=" * 125
)

print(
    "CONTROL DIMENSION AUDIT"
)

print(
    "=" * 125
)


display(
    pvod_secondary_control_audit
    .groupby(
        "variant",
        as_index=False,
    )
    .agg(
        cases=(
            "station",
            "size",
        ),

        unique_widths=(
            "columns",
            lambda values:
                tuple(
                    sorted(
                        set(
                            values
                        )
                    )
                ),
        ),

        min_rows=(
            "rows",
            "min",
        ),

        max_rows=(
            "rows",
            "max",
        ),
    )
)


print(
    "\n"
    + "=" * 125
)

print(
    "CIRCULAR-SHIFT EFFECTIVE DISTANCE AUDIT"
)

print(
    "=" * 125
)


display(
    pvod_effective_shift_summary
)


print(
    "\nPVOD-08B SECONDARY CONTROL REPRESENTATIONS: PASSED"
)


print(
    "\nNo model fitted."
)

print(
    "No prediction generated."
)

print(
    "No secondary-control forecasting result observed."
)

print(
    "\nSecondary-control definitions are now frozen."
)

PVOD SECONDARY CONTROL REPRESENTATION FREEZE

Reference-coded columns dropped:
  tithi_1
  paksha_Shukla
  karana_Kimstughna
  nakshatra_Ashwini
  yoga_Vishkambha

Reference Panchang width: 92
Reference total F4 width: 192
Reference neural auxiliary width: 122

CONTROL DIMENSION AUDIT


,variant,cases,unique_widths,min_rows,max_rows
0,F4_perm_1301,30,"(97,)",9596,33311
1,F4_perm_1302,30,"(97,)",9596,33311
2,F4_perm_1303,30,"(97,)",9596,33311
3,F4_reference,30,"(92,)",9596,33311
4,F4_shift_17d,30,"(97,)",9596,33311
5,F4_shift_37d,30,"(97,)",9596,33311
6,F4_shift_61d,30,"(97,)",9596,33311



CIRCULAR-SHIFT EFFECTIVE DISTANCE AUDIT


,variant,split,cases,min_effective_days,median_effective_days,max_effective_days,zero_shift_cases,under_one_day_cases
0,F4_shift_17d,test,30,17.000000,17.000000,17.000000,0,0
1,F4_shift_17d,train,30,17.000000,17.000000,17.000000,0,0
2,F4_shift_17d,validation,30,1.854167,17.000000,17.000000,0,0
3,F4_shift_37d,test,30,6.406250,37.000000,37.000000,0,0
4,F4_shift_37d,train,30,37.000000,37.000000,37.000000,0,0
5,F4_shift_37d,validation,30,6.708333,37.000000,37.000000,0,0
6,F4_shift_61d,test,30,0.406250,45.703125,61.000000,0,6
7,F4_shift_61d,train,30,61.000000,61.000000,61.000000,0,0
8,F4_shift_61d,validation,30,0.416667,12.656250,21.854167,0,3



PVOD-08B SECONDARY CONTROL REPRESENTATIONS: PASSED

No model fitted.
No prediction generated.
No secondary-control forecasting result observed.

Secondary-control definitions are now frozen.


In [20]:
# ============================================================
# PVOD-09A
# CHECKPOINTED CLASSICAL SECONDARY-CONTROL EXECUTION
#
# Models:
#     Ridge
#     LightGBM
#
# Controls:
#     F4_reference
#     F4_shift_17d / 37d / 61d
#     F4_perm_1301 / 1302 / 1303
#
# 2 models × 10 stations × 3 horizons × 7 controls = 420 fits
#
# EXECUTION-SAFETY POLICY
# -----------------------
# - Every completed fit is atomically checkpointed to disk.
# - Only daily sufficient statistics are retained for the
#   secondary-control bootstrap; millions of row predictions
#   are not accumulated in RAM.
# - Re-running this cell skips valid completed checkpoints.
# - No F4_real-vs-control comparison is displayed here.
# ============================================================

import gc
import json
import os
import time
import warnings

import numpy as np
import pandas as pd

from scipy.linalg import LinAlgWarning


# ============================================================
# 1. Preconditions / frozen execution registry
# ============================================================

required_objects = [
    "pvod_classical_predictions_corrected",
    "materialize_pvod_ridge_float64_design",
    "materialize_pvod_design",
    "build_pvod_secondary_panchang_block",
    "build_pvod_classical_model",
    "PVOD_SECONDARY_CONTROL_VARIANTS",
    "PVOD_EXTERNAL_STATIONS",
    "PVOD_EXTERNAL_HORIZONS",
    "PROJECT_ROOT",
]

missing = [
    name for name in required_objects
    if name not in globals()
]
assert not missing, f"Missing required objects: {missing}"

PVOD_CLASSICAL_CONTROL_MODELS = ["Ridge", "LightGBM"]
assert (
    len(PVOD_CLASSICAL_CONTROL_MODELS)
    * len(PVOD_EXTERNAL_STATIONS)
    * len(PVOD_EXTERNAL_HORIZONS)
    * len(PVOD_SECONDARY_CONTROL_VARIANTS)
) == 420

PVOD_SECONDARY_CHECKPOINT_VERSION = 2
PVOD_SECONDARY_CHECKPOINT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "pvod_secondary_checkpoints_v2"
)
PVOD_SECONDARY_CHECKPOINT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# ============================================================
# 2. Station capacity lookup
# ============================================================

pvod_control_capacity_lookup = (
    pvod_site_registry
    .set_index("station")[
        "capacity_MW"
    ]
    .astype(float)
    .to_dict()
)


# ============================================================
# 3. Classical secondary-design materializer
# ============================================================

def materialize_pvod_classical_control_design(
    station_id: str,
    horizon_min: int,
    variant_name: str,
    model_name: str,
):

    if variant_name not in PVOD_SECONDARY_CONTROL_VARIANTS:
        raise KeyError(
            f"Unknown control variant: {variant_name}"
        )


    # --------------------------------------------------------
    # A. Model-specific F3 backbone
    # --------------------------------------------------------

    if model_name == "Ridge":

        # Genuine source-float64 reconstruction.
        base = (
            materialize_pvod_ridge_float64_design(
                station_id=station_id,
                horizon_min=horizon_min,
                variant_name="F3",
            )
        )

        X_base = (
            base["X"]
            .copy()
            .astype(np.float64)
        )


    elif model_name == "LightGBM":

        # Preserve original PVOD / Notebook-13 LightGBM
        # float32 execution semantics.
        base = (
            materialize_pvod_design(
                station_id=station_id,
                horizon_min=horizon_min,
                feature_block="F3",
            )
        )

        X_base = (
            base["X"]
            .copy()
            .astype(np.float32)
        )


    else:

        raise ValueError(
            f"Unsupported model: {model_name}"
        )


    # --------------------------------------------------------
    # B. Frozen secondary categorical block
    # --------------------------------------------------------

    control_result = (
        build_pvod_secondary_panchang_block(
            station_id=station_id,
            horizon_min=horizon_min,
            variant_name=variant_name,
        )
    )


    control_block = (
        control_result["block"]
        .copy()
    )


    assert X_base.index.equals(
        control_block.index
    )


    # --------------------------------------------------------
    # C. Preserve model-specific numeric path
    # --------------------------------------------------------

    if model_name == "Ridge":

        control_block = (
            control_block
            .astype(np.float64)
        )

    else:

        control_block = (
            control_block
            .astype(np.float32)
        )


    X = pd.concat(
        [
            X_base,
            control_block,
        ],
        axis=1,
    )


    assert X.columns.is_unique


    expected_dim = (
        192
        if variant_name == "F4_reference"
        else
        197
    )


    assert X.shape == (
        len(X_base),
        expected_dim,
    )


    if model_name == "Ridge":

        assert all(
            dtype == np.dtype("float64")
            for dtype in X.dtypes
        )

    else:

        assert all(
            dtype == np.dtype("float32")
            for dtype in X.dtypes
        )


    assert np.isfinite(
        X.to_numpy()
    ).all()


    return {
        "X": X,
        "y_pu": base["y_pu"],
        "y_MW": base["y_MW"],
        "meta": base["meta"],
    }


# ============================================================
# 4. Shared checkpoint / daily-stat utilities
# ============================================================

def pvod_secondary_checkpoint_paths(
    family: str,
    model_name: str,
    station_id: str,
    horizon_min: int,
    variant_name: str,
):
    directory = PVOD_SECONDARY_CHECKPOINT_ROOT / family
    directory.mkdir(parents=True, exist_ok=True)

    key = (
        f"{model_name}"
        f"__h{int(horizon_min):03d}"
        f"__{station_id}"
        f"__{variant_name}"
    )

    return (
        directory / f"{key}__daily.csv",
        directory / f"{key}__metadata.json",
    )


def make_pvod_daily_error_summary(
    model_name: str,
    station_id: str,
    horizon_min: int,
    variant_name: str,
    target_time_utc,
    target_solar_daylight,
    target_pu,
    prediction_pu,
    target_MW=None,
    prediction_MW=None,
):
    target_time_utc = pd.DatetimeIndex(target_time_utc)
    assert target_time_utc.tz is not None

    target_pu = np.asarray(target_pu, dtype=np.float64).reshape(-1)
    prediction_pu = np.asarray(
        prediction_pu,
        dtype=np.float64,
    ).reshape(-1)
    daylight = np.asarray(
        target_solar_daylight,
        dtype=np.int8,
    ).reshape(-1)

    assert (
        len(target_time_utc)
        == len(target_pu)
        == len(prediction_pu)
        == len(daylight)
    )
    assert np.isfinite(target_pu).all()
    assert np.isfinite(prediction_pu).all()
    assert set(np.unique(daylight)).issubset({0, 1})

    rows = pd.DataFrame({
        "target_china_date": (
            target_time_utc
            .tz_convert("Asia/Shanghai")
            .strftime("%Y-%m-%d")
        ),
        "target_solar_daylight": daylight,
        "absolute_error_pu": np.abs(
            prediction_pu - target_pu
        ),
    })

    include_mw = (
        target_MW is not None
        and prediction_MW is not None
    )

    if include_mw:
        target_MW = np.asarray(
            target_MW,
            dtype=np.float64,
        ).reshape(-1)
        prediction_MW = np.asarray(
            prediction_MW,
            dtype=np.float64,
        ).reshape(-1)
        assert len(target_MW) == len(rows)
        assert len(prediction_MW) == len(rows)
        rows["absolute_error_MW"] = np.abs(
            prediction_MW - target_MW
        )

    daily_frames = []
    for scope_name, mask in {
        "all": np.ones(len(rows), dtype=bool),
        "daylight": rows[
            "target_solar_daylight"
        ].eq(1).to_numpy(),
    }.items():
        scoped = rows.loc[mask].copy()
        assert len(scoped) > 0

        aggregation = {
            "n_rows": ("absolute_error_pu", "size"),
            "sum_abs_error_pu": (
                "absolute_error_pu",
                "sum",
            ),
        }
        if include_mw:
            aggregation["sum_abs_error_MW"] = (
                "absolute_error_MW",
                "sum",
            )

        daily = (
            scoped
            .groupby(
                "target_china_date",
                sort=True,
                as_index=False,
            )
            .agg(**aggregation)
        )
        daily.insert(0, "scope", scope_name)
        daily.insert(0, "variant", variant_name)
        daily.insert(0, "horizon_min", int(horizon_min))
        daily.insert(0, "station", station_id)
        daily.insert(0, "model", model_name)
        daily_frames.append(daily)

    result = pd.concat(daily_frames, ignore_index=True)
    assert np.isfinite(
        result["sum_abs_error_pu"].to_numpy()
    ).all()
    return result


def pvod_secondary_checkpoint_is_valid(
    daily_path,
    metadata_path,
    expected_identity: dict,
):
    if not daily_path.exists() or not metadata_path.exists():
        return False

    try:
        with metadata_path.open("r", encoding="utf-8") as handle:
            metadata = json.load(handle)
        daily = pd.read_csv(daily_path)
    except Exception:
        return False

    if metadata.get("checkpoint_version") != (
        PVOD_SECONDARY_CHECKPOINT_VERSION
    ):
        return False

    for key, value in expected_identity.items():
        if metadata.get(key) != value:
            return False

    required_daily_columns = {
        "model",
        "station",
        "horizon_min",
        "variant",
        "scope",
        "target_china_date",
        "n_rows",
        "sum_abs_error_pu",
    }
    if not required_daily_columns.issubset(daily.columns):
        return False

    if set(daily["scope"].unique()) != {"all", "daylight"}:
        return False

    return True


def write_pvod_secondary_checkpoint(
    daily_path,
    metadata_path,
    daily_table: pd.DataFrame,
    metadata: dict,
):
    daily_tmp = daily_path.with_name(daily_path.name + ".tmp")
    metadata_tmp = metadata_path.with_name(
        metadata_path.name + ".tmp"
    )

    daily_table.to_csv(daily_tmp, index=False)
    with metadata_tmp.open("w", encoding="utf-8") as handle:
        json.dump(metadata, handle, indent=2)

    os.replace(daily_tmp, daily_path)
    os.replace(metadata_tmp, metadata_path)


# ============================================================
# 5. Execute / resume complete classical secondary grid
# ============================================================

pvod_control_capacity_lookup = (
    pvod_site_registry
    .set_index("station")["capacity_MW"]
    .astype(float)
    .to_dict()
)

new_fit_count = 0
skipped_fit_count = 0
grid_start = time.perf_counter()

for model_name in PVOD_CLASSICAL_CONTROL_MODELS:
    print("\n" + "=" * 125)
    print(f"MODEL: {model_name}")
    print("=" * 125)

    for horizon_min in PVOD_EXTERNAL_HORIZONS:
        print(f"\n--- Horizon: {horizon_min} min ---")

        for station_id in PVOD_EXTERNAL_STATIONS:
            station_start = time.perf_counter()

            canonical_real = (
                pvod_classical_predictions_corrected.loc[
                    pvod_classical_predictions_corrected[
                        "model"
                    ].eq(model_name)
                    & pvod_classical_predictions_corrected[
                        "station"
                    ].eq(station_id)
                    & pvod_classical_predictions_corrected[
                        "horizon_min"
                    ].eq(horizon_min)
                    & pvod_classical_predictions_corrected[
                        "variant"
                    ].eq("F4_real")
                ]
                .copy()
                .reset_index(drop=True)
            )
            assert not canonical_real.empty

            station_new = 0
            station_skipped = 0

            for variant_name in PVOD_SECONDARY_CONTROL_VARIANTS:
                daily_path, metadata_path = (
                    pvod_secondary_checkpoint_paths(
                        family="classical",
                        model_name=model_name,
                        station_id=station_id,
                        horizon_min=horizon_min,
                        variant_name=variant_name,
                    )
                )

                expected_identity = {
                    "family": "classical",
                    "model": model_name,
                    "station": station_id,
                    "horizon_min": int(horizon_min),
                    "variant": variant_name,
                    "test_rows": int(len(canonical_real)),
                    "first_target_time_utc": str(
                        canonical_real[
                            "target_time_utc"
                        ].iloc[0]
                    ),
                    "last_target_time_utc": str(
                        canonical_real[
                            "target_time_utc"
                        ].iloc[-1]
                    ),
                }

                if pvod_secondary_checkpoint_is_valid(
                    daily_path,
                    metadata_path,
                    expected_identity,
                ):
                    skipped_fit_count += 1
                    station_skipped += 1
                    continue

                design = materialize_pvod_classical_control_design(
                    station_id=station_id,
                    horizon_min=horizon_min,
                    variant_name=variant_name,
                    model_name=model_name,
                )

                X = design["X"]
                y_pu = design["y_pu"]
                y_MW = design["y_MW"]
                meta = design["meta"]

                trainval_mask = (
                    meta["split"]
                    .isin(["train", "validation"])
                    .to_numpy()
                )
                test_mask = (
                    meta["split"]
                    .eq("test")
                    .to_numpy()
                )
                trainval_positions = np.flatnonzero(trainval_mask)
                test_positions = np.flatnonzero(test_mask)

                X_trainval = X.iloc[trainval_positions]
                X_test = X.iloc[test_positions]
                y_trainval = y_pu.iloc[trainval_positions]
                target_pu = (
                    y_pu.iloc[test_positions]
                    .to_numpy(dtype=np.float64)
                )
                target_MW = (
                    y_MW.iloc[test_positions]
                    .to_numpy(dtype=np.float64)
                )
                test_meta = (
                    meta.iloc[test_positions]
                    .copy()
                    .reset_index(drop=True)
                )

                assert len(test_meta) == len(canonical_real)
                assert pd.DatetimeIndex(
                    test_meta["issue_time_utc"]
                ).equals(
                    pd.DatetimeIndex(
                        canonical_real["issue_time_utc"]
                    )
                )
                assert pd.DatetimeIndex(
                    test_meta["target_time_utc"]
                ).equals(
                    pd.DatetimeIndex(
                        canonical_real["target_time_utc"]
                    )
                )

                if model_name == "Ridge":
                    assert np.array_equal(
                        target_pu,
                        canonical_real[
                            "target_power_pu"
                        ].to_numpy(dtype=np.float64),
                    )
                else:
                    assert np.array_equal(
                        target_pu.astype(np.float32),
                        canonical_real[
                            "target_power_pu"
                        ].to_numpy(dtype=np.float32),
                    )

                model = build_pvod_classical_model(
                    model_name=model_name,
                    horizon_min=horizon_min,
                )

                fit_start = time.perf_counter()
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter("always")
                    model.fit(X_trainval, y_trainval)
                fit_seconds = float(
                    time.perf_counter() - fit_start
                )

                ridge_linalg_warnings = int(sum(
                    issubclass(w.category, LinAlgWarning)
                    for w in caught
                ))
                other_warnings = int(
                    len(caught) - ridge_linalg_warnings
                )

                predict_start = time.perf_counter()
                prediction_pu = np.asarray(
                    model.predict(X_test),
                    dtype=np.float64,
                ).reshape(-1)
                predict_seconds = float(
                    time.perf_counter() - predict_start
                )

                assert prediction_pu.shape == target_pu.shape
                assert np.isfinite(prediction_pu).all()

                capacity_MW = float(
                    pvod_control_capacity_lookup[station_id]
                )
                prediction_MW = prediction_pu * capacity_MW

                daily_table = make_pvod_daily_error_summary(
                    model_name=model_name,
                    station_id=station_id,
                    horizon_min=horizon_min,
                    variant_name=variant_name,
                    target_time_utc=test_meta[
                        "target_time_utc"
                    ],
                    target_solar_daylight=test_meta[
                        "target_solar_daylight"
                    ],
                    target_pu=target_pu,
                    prediction_pu=prediction_pu,
                    target_MW=target_MW,
                    prediction_MW=prediction_MW,
                )

                metadata = {
                    "checkpoint_version": (
                        PVOD_SECONDARY_CHECKPOINT_VERSION
                    ),
                    **expected_identity,
                    "feature_dim": int(X.shape[1]),
                    "fit_seconds": fit_seconds,
                    "predict_seconds": predict_seconds,
                    "LinAlgWarnings": ridge_linalg_warnings,
                    "other_warnings": other_warnings,
                }

                write_pvod_secondary_checkpoint(
                    daily_path,
                    metadata_path,
                    daily_table,
                    metadata,
                )

                new_fit_count += 1
                station_new += 1

                del model
                del design
                del daily_table
                gc.collect()

            station_seconds = float(
                time.perf_counter() - station_start
            )
            print(
                f"{station_id}: "
                f"new={station_new}, "
                f"skipped={station_skipped}, "
                f"test rows={len(canonical_real):,} | "
                f"{station_seconds:.2f}s"
            )


# ============================================================
# 6. Reconstruct compact canonical checkpoint objects
# ============================================================

classical_daily_frames = []
classical_runtime_rows = []

for model_name in PVOD_CLASSICAL_CONTROL_MODELS:
    for horizon_min in PVOD_EXTERNAL_HORIZONS:
        for station_id in PVOD_EXTERNAL_STATIONS:
            canonical_real = (
                pvod_classical_predictions_corrected.loc[
                    pvod_classical_predictions_corrected[
                        "model"
                    ].eq(model_name)
                    & pvod_classical_predictions_corrected[
                        "station"
                    ].eq(station_id)
                    & pvod_classical_predictions_corrected[
                        "horizon_min"
                    ].eq(horizon_min)
                    & pvod_classical_predictions_corrected[
                        "variant"
                    ].eq("F4_real")
                ]
                .reset_index(drop=True)
            )

            for variant_name in PVOD_SECONDARY_CONTROL_VARIANTS:
                daily_path, metadata_path = (
                    pvod_secondary_checkpoint_paths(
                        family="classical",
                        model_name=model_name,
                        station_id=station_id,
                        horizon_min=horizon_min,
                        variant_name=variant_name,
                    )
                )
                expected_identity = {
                    "family": "classical",
                    "model": model_name,
                    "station": station_id,
                    "horizon_min": int(horizon_min),
                    "variant": variant_name,
                    "test_rows": int(len(canonical_real)),
                    "first_target_time_utc": str(
                        canonical_real[
                            "target_time_utc"
                        ].iloc[0]
                    ),
                    "last_target_time_utc": str(
                        canonical_real[
                            "target_time_utc"
                        ].iloc[-1]
                    ),
                }
                assert pvod_secondary_checkpoint_is_valid(
                    daily_path,
                    metadata_path,
                    expected_identity,
                )

                classical_daily_frames.append(
                    pd.read_csv(daily_path)
                )
                with metadata_path.open(
                    "r",
                    encoding="utf-8",
                ) as handle:
                    classical_runtime_rows.append(
                        json.load(handle)
                    )

pvod_classical_secondary_daily = pd.concat(
    classical_daily_frames,
    ignore_index=True,
)
pvod_classical_secondary_runtime = pd.DataFrame(
    classical_runtime_rows
)

del classical_daily_frames

grid_seconds = float(time.perf_counter() - grid_start)

assert len(pvod_classical_secondary_runtime) == 420
assert (
    pvod_classical_secondary_runtime
    .groupby(["model", "horizon_min", "station"])[
        "variant"
    ]
    .nunique()
    .eq(7)
    .all()
)
assert (
    pvod_classical_secondary_runtime.loc[
        pvod_classical_secondary_runtime[
            "model"
        ].eq("Ridge"),
        "LinAlgWarnings",
    ].sum()
    == 0
)

expected_prediction_rows = int(
    len(
        pvod_classical_predictions_corrected.loc[
            pvod_classical_predictions_corrected[
                "variant"
            ].eq("F4_real")
        ]
    )
    * 7
)
assert int(
    pvod_classical_secondary_runtime[
        "test_rows"
    ].sum()
) == expected_prediction_rows

warning_summary = (
    pvod_classical_secondary_runtime
    .groupby(["model", "horizon_min"], as_index=False)
    .agg(
        fits=("variant", "size"),
        LinAlgWarnings=("LinAlgWarnings", "sum"),
        other_warnings=("other_warnings", "sum"),
        min_feature_dim=("feature_dim", "min"),
        max_feature_dim=("feature_dim", "max"),
        total_fit_seconds=("fit_seconds", "sum"),
    )
)

print("\n" + "=" * 125)
print("PVOD CHECKPOINTED CLASSICAL SECONDARY-CONTROL AUDIT")
print("=" * 125)
print("Logical fits:", 420)
print("New fits this run:", new_fit_count)
print("Valid checkpoints skipped:", skipped_fit_count)
print("Prediction rows represented:", expected_prediction_rows)
print("Execution seconds this run:", grid_seconds)
display(warning_summary)
print("\nPVOD-09A CLASSICAL SECONDARY-CONTROL EXECUTION: PASSED")
print("No F4_real-vs-control comparison displayed.")
print("No secondary bootstrap performed yet.")



MODEL: Ridge

--- Horizon: 15 min ---
station00: new=0, skipped=7, test rows=5,798 | 0.11s
station01: new=0, skipped=7, test rows=6,681 | 0.11s
station02: new=0, skipped=7, test rows=5,719 | 0.11s
station03: new=0, skipped=7, test rows=2,937 | 0.11s
station04: new=0, skipped=7, test rows=6,681 | 0.11s
station05: new=0, skipped=7, test rows=1,939 | 0.11s
station06: new=0, skipped=7, test rows=6,239 | 0.11s
station07: new=0, skipped=7, test rows=6,681 | 0.11s
station08: new=0, skipped=7, test rows=6,681 | 0.11s
station09: new=0, skipped=7, test rows=4,529 | 0.11s

--- Horizon: 30 min ---
station00: new=0, skipped=7, test rows=5,798 | 0.11s
station01: new=0, skipped=7, test rows=6,681 | 0.11s
station02: new=0, skipped=7, test rows=5,717 | 0.11s
station03: new=0, skipped=7, test rows=2,937 | 0.11s
station04: new=0, skipped=7, test rows=6,681 | 0.11s
station05: new=0, skipped=7, test rows=1,939 | 0.11s
station06: new=0, skipped=7, test rows=6,238 | 0.11s
station07: new=0, skipped=7, test r

,model,horizon_min,fits,LinAlgWarnings,other_warnings,min_feature_dim,max_feature_dim,total_fit_seconds
0,LightGBM,15,70,0,0,192,197,118.205545
1,LightGBM,30,70,0,0,192,197,123.201189
2,LightGBM,60,70,0,0,192,197,115.249585
3,Ridge,15,70,0,0,192,197,2.881651
4,Ridge,30,70,0,0,192,197,2.944930
5,Ridge,60,70,0,0,192,197,2.765473



PVOD-09A CLASSICAL SECONDARY-CONTROL EXECUTION: PASSED
No F4_real-vs-control comparison displayed.
No secondary bootstrap performed yet.


In [21]:
# ============================================================
# PVOD-09B
# NEURAL SECONDARY-CONTROL PREPARATION + SCALING AUDIT
#
# NO MODEL FITTING.
# NO PREDICTION.
# NO METRIC.
#
# This adapts only the representation-materialization part.
# Sequence / target / auxiliary scaling semantics are copied
# exactly from the already-validated primary neural helper.
# ============================================================

import gc

import numpy as np
import pandas as pd


# ============================================================
# 1. Preconditions
# ============================================================

required_objects = [
    "materialize_pvod_design",
    "build_pvod_secondary_panchang_block",
    "build_pvod_raw_sequence",
    "build_pvod_raw_auxiliary",
    "is_pvod_binary_auxiliary_column",
    "make_safe_std",
    "PVOD_SECONDARY_CONTROL_VARIANTS",
    "PVOD_EXTERNAL_STATIONS",
    "PVOD_EXTERNAL_HORIZONS",
    "PVOD_SEQUENCE_LENGTH",
    "PVOD_SEQUENCE_CHANNEL_COUNT",
]

missing = [
    name
    for name in required_objects
    if name not in globals()
]

assert not missing, (
    f"Missing required objects: {missing}"
)


# ============================================================
# 2. Control-aware neural preparation helper
# ============================================================

def prepare_pvod_neural_control_arrays(
    station_id: str,
    horizon_min: int,
    variant_name: str,
):

    if variant_name not in (
        PVOD_SECONDARY_CONTROL_VARIANTS
    ):

        raise ValueError(
            f"Unsupported secondary-control variant: "
            f"{variant_name}"
        )


    expected_aux_dim = (
        122
        if variant_name == "F4_reference"
        else
        127
    )


    expected_binary_dim = (
        94
        if variant_name == "F4_reference"
        else
        99
    )


    # ========================================================
    # A. F3 forecasting backbone
    #
    # Same float32 modelling representation as primary neural
    # execution.
    # ========================================================

    design = (
        materialize_pvod_design(
            station_id=station_id,
            horizon_min=horizon_min,
            feature_block="F3",
        )
    )


    X_f3 = design["X"]
    y_pu = design["y_pu"]
    y_MW = design["y_MW"]
    meta = design["meta"]


    assert X_f3.shape[1] == 100


    # ========================================================
    # B. Frozen split masks
    # ========================================================

    train_mask = (
        meta["split"]
        .eq("train")
        .to_numpy()
    )


    validation_mask = (
        meta["split"]
        .eq("validation")
        .to_numpy()
    )


    test_mask = (
        meta["split"]
        .eq("test")
        .to_numpy()
    )


    assert train_mask.sum() > 0
    assert validation_mask.sum() > 0
    assert test_mask.sum() > 0


    # ========================================================
    # C. RAW SEQUENCE
    #
    # Identical F3/F4/control historical branch.
    # ========================================================

    raw_sequence = (
        build_pvod_raw_sequence(
            X_f3
        )
    )


    assert raw_sequence.shape == (
        len(X_f3),
        PVOD_SEQUENCE_LENGTH,
        PVOD_SEQUENCE_CHANNEL_COUNT,
    )


    # ========================================================
    # D. TRAIN-ONLY SEQUENCE SCALING
    # ========================================================

    train_sequence_64 = (
        raw_sequence[
            train_mask
        ]
        .astype(
            np.float64,
            copy=False,
        )
    )


    sequence_mean = (
        train_sequence_64
        .mean(
            axis=(0, 1)
        )
    )


    sequence_std_raw = (
        train_sequence_64
        .std(
            axis=(0, 1),
            ddof=0,
        )
    )


    (
        sequence_std,
        sequence_zero_mask,
    ) = make_safe_std(
        sequence_std_raw
    )


    assert not (
        sequence_zero_mask.any()
    ), (
        f"{station_id} h={horizon_min}: "
        "unexpected zero-variance sequence channel."
    )


    scaled_sequence = (
        (
            raw_sequence.astype(
                np.float64
            )
            -
            sequence_mean
        )
        /
        sequence_std
    ).astype(
        np.float32
    )


    # ========================================================
    # E. TRAIN-ONLY TARGET SCALING
    # ========================================================

    raw_target = (
        y_pu
        .to_numpy(
            dtype=np.float64
        )
    )


    train_target = (
        raw_target[
            train_mask
        ]
    )


    target_mean = float(
        train_target.mean()
    )


    target_std = float(
        train_target.std(
            ddof=0
        )
    )


    assert (
        np.isfinite(target_std)
        and
        target_std > 0.0
    )


    scaled_target = (
        (
            raw_target
            -
            target_mean
        )
        /
        target_std
    ).astype(
        np.float32
    )


    # ========================================================
    # F. Original F3 auxiliary block
    #
    # 30 columns:
    #   28 continuous
    #    2 binary
    # ========================================================

    f3_auxiliary_df = (
        build_pvod_raw_auxiliary(
            X_f3
        )
        .copy()
    )


    assert f3_auxiliary_df.shape == (
        len(X_f3),
        30,
    )


    # ========================================================
    # G. Frozen categorical control block
    # ========================================================

    control_result = (
        build_pvod_secondary_panchang_block(
            station_id=station_id,
            horizon_min=horizon_min,
            variant_name=variant_name,
        )
    )


    control_block = (
        control_result[
            "block"
        ]
        .copy()
    )


    assert f3_auxiliary_df.index.equals(
        control_block.index
    )


    assert control_block.shape[1] == (
        92
        if variant_name == "F4_reference"
        else
        97
    )


    # ========================================================
    # H. Combined raw auxiliary representation
    # ========================================================

    raw_auxiliary_df = (
        pd.concat(
            [
                f3_auxiliary_df,
                control_block,
            ],
            axis=1,
        )
    )


    assert (
        raw_auxiliary_df.shape[
            1
        ]
        ==
        expected_aux_dim
    )


    assert raw_auxiliary_df.columns.is_unique


    auxiliary_columns = list(
        raw_auxiliary_df.columns
    )


    binary_columns = [
        column
        for column
        in auxiliary_columns
        if is_pvod_binary_auxiliary_column(
            column
        )
    ]


    continuous_columns = [
        column
        for column
        in auxiliary_columns
        if column not in binary_columns
    ]


    # Same continuous backbone for every representation.
    assert len(
        continuous_columns
    ) == 28


    assert len(
        binary_columns
    ) == expected_binary_dim


    raw_auxiliary = (
        raw_auxiliary_df
        .to_numpy(
            dtype=np.float32
        )
    )


    scaled_auxiliary = (
        raw_auxiliary.copy()
    )


    # ========================================================
    # I. TRAIN-ONLY continuous auxiliary scaling
    # ========================================================

    continuous_positions = np.array(
        [
            auxiliary_columns.index(
                column
            )
            for column
            in continuous_columns
        ],
        dtype=np.int64,
    )


    binary_positions = np.array(
        [
            auxiliary_columns.index(
                column
            )
            for column
            in binary_columns
        ],
        dtype=np.int64,
    )


    train_continuous = (
        raw_auxiliary[
            train_mask
        ][
            :,
            continuous_positions,
        ]
        .astype(
            np.float64
        )
    )


    auxiliary_mean = (
        train_continuous
        .mean(
            axis=0
        )
    )


    auxiliary_std_raw = (
        train_continuous
        .std(
            axis=0,
            ddof=0,
        )
    )


    (
        auxiliary_std,
        auxiliary_zero_mask,
    ) = make_safe_std(
        auxiliary_std_raw
    )


    zero_variance_continuous_columns = [
        continuous_columns[index]
        for index
        in np.flatnonzero(
            auxiliary_zero_mask
        )
    ]


    scaled_continuous = (
        (
            raw_auxiliary[
                :,
                continuous_positions,
            ]
            .astype(
                np.float64
            )
            -
            auxiliary_mean
        )
        /
        auxiliary_std
    ).astype(
        np.float32
    )


    scaled_auxiliary[
        :,
        continuous_positions,
    ] = scaled_continuous


    # ========================================================
    # J. Binary columns unchanged EXACTLY
    # ========================================================

    assert np.array_equal(
        scaled_auxiliary[
            :,
            binary_positions,
        ],
        raw_auxiliary[
            :,
            binary_positions,
        ],
    )


    observed_binary_values = np.unique(
        scaled_auxiliary[
            :,
            binary_positions,
        ]
    )


    assert np.all(
        np.isin(
            observed_binary_values,
            [
                0.0,
                1.0,
            ],
        )
    )


    # ========================================================
    # K. Train-only scaling audits
    # ========================================================

    scaled_train_sequence = (
        scaled_sequence[
            train_mask
        ]
        .astype(
            np.float64
        )
    )


    assert np.allclose(
        scaled_train_sequence.mean(
            axis=(0, 1)
        ),
        0.0,
        atol=1e-5,
        rtol=0.0,
    )


    assert np.allclose(
        scaled_train_sequence.std(
            axis=(0, 1),
            ddof=0,
        ),
        1.0,
        atol=1e-5,
        rtol=0.0,
    )


    scaled_train_target = (
        scaled_target[
            train_mask
        ]
        .astype(
            np.float64
        )
    )


    assert np.isclose(
        scaled_train_target.mean(),
        0.0,
        atol=1e-5,
        rtol=0.0,
    )


    assert np.isclose(
        scaled_train_target.std(
            ddof=0
        ),
        1.0,
        atol=1e-5,
        rtol=0.0,
    )


    scaled_train_continuous = (
        scaled_auxiliary[
            train_mask
        ][
            :,
            continuous_positions,
        ]
        .astype(
            np.float64
        )
    )


    nonzero_positions = np.flatnonzero(
        ~auxiliary_zero_mask
    )


    if len(
        nonzero_positions
    ) > 0:

        assert np.allclose(
            scaled_train_continuous[
                :,
                nonzero_positions,
            ]
            .mean(
                axis=0
            ),
            0.0,
            atol=1e-5,
            rtol=0.0,
        )


        assert np.allclose(
            scaled_train_continuous[
                :,
                nonzero_positions,
            ]
            .std(
                axis=0,
                ddof=0,
            ),
            1.0,
            atol=1e-5,
            rtol=0.0,
        )


    zero_positions = np.flatnonzero(
        auxiliary_zero_mask
    )


    if len(
        zero_positions
    ) > 0:

        assert np.allclose(
            scaled_train_continuous[
                :,
                zero_positions,
            ],
            0.0,
            atol=1e-7,
            rtol=0.0,
        )


    # ========================================================
    # L. Finite checks
    # ========================================================

    assert np.isfinite(
        scaled_sequence
    ).all()


    assert np.isfinite(
        scaled_auxiliary
    ).all()


    assert np.isfinite(
        scaled_target
    ).all()


    return {
        "sequence":
            np.ascontiguousarray(
                scaled_sequence,
                dtype=np.float32,
            ),

        "auxiliary":
            np.ascontiguousarray(
                scaled_auxiliary,
                dtype=np.float32,
            ),

        "target_scaled":
            np.ascontiguousarray(
                scaled_target,
                dtype=np.float32,
            ),

        "target_pu":
            np.ascontiguousarray(
                raw_target,
                dtype=np.float64,
            ),

        "target_MW":
            np.ascontiguousarray(
                y_MW.to_numpy(
                    dtype=np.float64
                ),
            ),

        "train_mask":
            train_mask.copy(),

        "validation_mask":
            validation_mask.copy(),

        "test_mask":
            test_mask.copy(),

        "meta":
            meta.copy(),

        "auxiliary_columns":
            tuple(
                auxiliary_columns
            ),

        "continuous_columns":
            tuple(
                continuous_columns
            ),

        "binary_columns":
            tuple(
                binary_columns
            ),

        "zero_variance_continuous_columns":
            tuple(
                zero_variance_continuous_columns
            ),

        "sequence_mean":
            sequence_mean.copy(),

        "sequence_std":
            sequence_std.copy(),

        "auxiliary_mean":
            auxiliary_mean.copy(),

        "auxiliary_std_raw":
            auxiliary_std_raw.copy(),

        "auxiliary_std_effective":
            auxiliary_std.copy(),

        "target_mean":
            target_mean,

        "target_std":
            target_std,
    }


# ============================================================
# 3. Audit every station × horizon × control
#
# 10 × 3 × 7 = 210 preparation cases.
# ============================================================

preparation_audit_rows = []


for horizon_min in (
    PVOD_EXTERNAL_HORIZONS
):

    print(
        f"\n--- Neural control preparation: "
        f"{horizon_min} min ---"
    )


    for station_id in (
        PVOD_EXTERNAL_STATIONS
    ):

        for variant_name in (
            PVOD_SECONDARY_CONTROL_VARIANTS
        ):

            prepared = (
                prepare_pvod_neural_control_arrays(
                    station_id=station_id,
                    horizon_min=horizon_min,
                    variant_name=variant_name,
                )
            )


            expected_aux_dim = (
                122
                if variant_name
                ==
                "F4_reference"
                else
                127
            )


            expected_binary_dim = (
                94
                if variant_name
                ==
                "F4_reference"
                else
                99
            )


            assert prepared[
                "auxiliary"
            ].shape[1] == (
                expected_aux_dim
            )


            assert len(
                prepared[
                    "continuous_columns"
                ]
            ) == 28


            assert len(
                prepared[
                    "binary_columns"
                ]
            ) == (
                expected_binary_dim
            )


            preparation_audit_rows.append(
                {
                    "station":
                        station_id,

                    "horizon_min":
                        int(
                            horizon_min
                        ),

                    "variant":
                        variant_name,

                    "rows":
                        int(
                            len(
                                prepared[
                                    "target_pu"
                                ]
                            )
                        ),

                    "sequence_shape_tail":
                        tuple(
                            prepared[
                                "sequence"
                            ].shape[1:]
                        ),

                    "auxiliary_dim":
                        int(
                            prepared[
                                "auxiliary"
                            ].shape[1]
                        ),

                    "continuous_dim":
                        len(
                            prepared[
                                "continuous_columns"
                            ]
                        ),

                    "binary_dim":
                        len(
                            prepared[
                                "binary_columns"
                            ]
                        ),

                    "zero_variance_continuous_columns":
                        prepared[
                            "zero_variance_continuous_columns"
                        ],

                    "finite":
                        bool(
                            np.isfinite(
                                prepared[
                                    "sequence"
                                ]
                            ).all()
                            and
                            np.isfinite(
                                prepared[
                                    "auxiliary"
                                ]
                            ).all()
                            and
                            np.isfinite(
                                prepared[
                                    "target_scaled"
                                ]
                            ).all()
                        ),
                }
            )


            del prepared

            gc.collect()


        print(
            f"{station_id}: "
            f"7 preparations passed"
        )


# ============================================================
# 4. Canonical audit object
# ============================================================

pvod_neural_secondary_preparation_audit = (
    pd.DataFrame(
        preparation_audit_rows
    )
)


assert len(
    pvod_neural_secondary_preparation_audit
) == 210


assert (
    pvod_neural_secondary_preparation_audit[
        "finite"
    ]
    .all()
)


expected_sequence_tail = (
    PVOD_SEQUENCE_LENGTH,
    PVOD_SEQUENCE_CHANNEL_COUNT,
)

assert (
    pvod_neural_secondary_preparation_audit[
        "sequence_shape_tail"
    ]
    .map(
        lambda value:
            tuple(value)
            ==
            expected_sequence_tail
    )
    .all()
)


# ============================================================
# 5. Geometry summary
# ============================================================

geometry_summary = (
    pvod_neural_secondary_preparation_audit
    .groupby(
        "variant",
        as_index=False,
    )
    .agg(
        cases=(
            "station",
            "size",
        ),

        auxiliary_dims=(
            "auxiliary_dim",
            lambda values:
                tuple(
                    sorted(
                        set(values)
                    )
                ),
        ),

        continuous_dims=(
            "continuous_dim",
            lambda values:
                tuple(
                    sorted(
                        set(values)
                    )
                ),
        ),

        binary_dims=(
            "binary_dim",
            lambda values:
                tuple(
                    sorted(
                        set(values)
                    )
                ),
        ),

        finite_cases=(
            "finite",
            "sum",
        ),
    )
)


# ============================================================
# 6. Zero-variance audit
# ============================================================

zero_variance_audit = (
    pvod_neural_secondary_preparation_audit.loc[
        pvod_neural_secondary_preparation_audit[
            "zero_variance_continuous_columns"
        ]
        .apply(
            len
        )
        >
        0,
        [
            "station",
            "horizon_min",
            "variant",
            "zero_variance_continuous_columns",
        ],
    ]
    .reset_index(
        drop=True
    )
)


# Same frozen PVOD behavior:
# only utc_year at station03 / station05.

assert set(
    zero_variance_audit[
        "station"
    ]
) == {
    "station03",
    "station05",
}


assert (
    zero_variance_audit[
        "zero_variance_continuous_columns"
    ]
    .apply(
        lambda value:
            value
            ==
            (
                "utc_year",
            )
    )
    .all()
)


# 2 stations × 3 horizons × 7 controls
assert len(
    zero_variance_audit
) == 42


# ============================================================
# 7. Display
# ============================================================

print(
    "\n"
    + "=" * 125
)

print(
    "PVOD NEURAL SECONDARY-CONTROL PREPARATION AUDIT"
)

print(
    "=" * 125
)


display(
    geometry_summary
)


print(
    "\n"
    + "=" * 125
)

print(
    "ZERO-VARIANCE SAFEGUARD AUDIT"
)

print(
    "=" * 125
)


display(
    zero_variance_audit
)


print(
    "\nPreparation cases:",
    len(
        pvod_neural_secondary_preparation_audit
    ),
)


print(
    "Finite cases:",
    int(
        pvod_neural_secondary_preparation_audit[
            "finite"
        ]
        .sum()
    ),
)


print(
    "\nPVOD-09B NEURAL SECONDARY-CONTROL PREPARATION: PASSED"
)


print(
    "\nNo neural model fitted."
)

print(
    "No prediction generated."
)

print(
    "No control forecasting result observed."
)


--- Neural control preparation: 15 min ---
station00: 7 preparations passed
station01: 7 preparations passed
station02: 7 preparations passed
station03: 7 preparations passed
station04: 7 preparations passed
station05: 7 preparations passed
station06: 7 preparations passed
station07: 7 preparations passed
station08: 7 preparations passed
station09: 7 preparations passed

--- Neural control preparation: 30 min ---
station00: 7 preparations passed
station01: 7 preparations passed
station02: 7 preparations passed
station03: 7 preparations passed
station04: 7 preparations passed
station05: 7 preparations passed
station06: 7 preparations passed
station07: 7 preparations passed
station08: 7 preparations passed
station09: 7 preparations passed

--- Neural control preparation: 60 min ---
station00: 7 preparations passed
station01: 7 preparations passed
station02: 7 preparations passed
station03: 7 preparations passed
station04: 7 preparations passed
station05: 7 preparations passed
station06:

,variant,cases,auxiliary_dims,continuous_dims,binary_dims,finite_cases
0,F4_perm_1301,30,"(127,)","(28,)","(99,)",30
1,F4_perm_1302,30,"(127,)","(28,)","(99,)",30
2,F4_perm_1303,30,"(127,)","(28,)","(99,)",30
3,F4_reference,30,"(122,)","(28,)","(94,)",30
4,F4_shift_17d,30,"(127,)","(28,)","(99,)",30
5,F4_shift_37d,30,"(127,)","(28,)","(99,)",30
6,F4_shift_61d,30,"(127,)","(28,)","(99,)",30



ZERO-VARIANCE SAFEGUARD AUDIT


,station,horizon_min,variant,zero_variance_continuous_columns
0,station03,15,F4_reference,"(utc_year,)"
1,station03,15,F4_shift_17d,"(utc_year,)"
2,station03,15,F4_shift_37d,"(utc_year,)"
3,station03,15,F4_shift_61d,"(utc_year,)"
4,station03,15,F4_perm_1301,"(utc_year,)"
5,station03,15,F4_perm_1302,"(utc_year,)"
6,station03,15,F4_perm_1303,"(utc_year,)"
7,station05,15,F4_reference,"(utc_year,)"
8,station05,15,F4_shift_17d,"(utc_year,)"
9,station05,15,F4_shift_37d,"(utc_year,)"



Preparation cases: 210
Finite cases: 210

PVOD-09B NEURAL SECONDARY-CONTROL PREPARATION: PASSED

No neural model fitted.
No prediction generated.
No control forecasting result observed.


In [22]:
# ============================================================
# PVOD-09C
# CHECKPOINTED NEURAL SECONDARY-CONTROL EXECUTION
#
# Models:
#     LSTM
#     TCN
#
# Controls:
#     F4_reference
#     F4_shift_17d / 37d / 61d
#     F4_perm_1301 / 1302 / 1303
#
# 2 models × 10 stations × 3 horizons × 7 controls = 420 fits
#
# EXECUTION-SAFETY POLICY
# -----------------------
# - Every completed fit is atomically checkpointed to disk.
# - Only daily sufficient statistics are retained in RAM.
# - A rerun skips valid completed checkpoints.
# - TensorFlow session state is explicitly cleared before and
#   after each fit. This does not change the frozen model,
#   seed, epochs, batch size, or preprocessing protocol.
# - No F4_real-vs-control comparison is displayed here.
# ============================================================

import gc
import json
import time

import numpy as np
import pandas as pd
import tensorflow as tf


# ============================================================
# 1. Preconditions / frozen geometry
# ============================================================

required_objects = [
    "prepare_pvod_neural_control_arrays",
    "build_pvod_neural_model",
    "get_pvod_neural_architecture",
    "get_pvod_frozen_epochs",
    "pvod_neural_predictions",
    "pvod_site_registry",
    "PVOD_SECONDARY_CONTROL_VARIANTS",
    "PVOD_EXTERNAL_STATIONS",
    "PVOD_EXTERNAL_HORIZONS",
    "PVOD_NEURAL_SEED",
    "PVOD_NEURAL_BATCH_SIZE",
    "pvod_secondary_checkpoint_paths",
    "pvod_secondary_checkpoint_is_valid",
    "write_pvod_secondary_checkpoint",
    "make_pvod_daily_error_summary",
    "PVOD_SECONDARY_CHECKPOINT_VERSION",
]
missing = [
    name for name in required_objects
    if name not in globals()
]
assert not missing, f"Missing required objects: {missing}"

PVOD_NEURAL_CONTROL_MODELS = ["LSTM", "TCN"]
assert (
    len(PVOD_NEURAL_CONTROL_MODELS)
    * len(PVOD_EXTERNAL_STATIONS)
    * len(PVOD_EXTERNAL_HORIZONS)
    * len(PVOD_SECONDARY_CONTROL_VARIANTS)
) == 420

PVOD_EXPECTED_CONTROL_NEURAL_PARAMS = {
    "LSTM": {122: 11009, 127: 11169},
    "TCN": {122: 22369, 127: 22529},
}

pvod_neural_control_capacity_lookup = (
    pvod_site_registry
    .set_index("station")["capacity_MW"]
    .astype(float)
    .to_dict()
)


# ============================================================
# 2. Frozen neural-control fit helper
# ============================================================

def fit_predict_pvod_neural_control(
    station_id: str,
    horizon_min: int,
    model_name: str,
    variant_name: str,
    verbose: int = 0,
):

    assert model_name in {
        "LSTM",
        "TCN",
    }


    prepared = (
        prepare_pvod_neural_control_arrays(
            station_id=station_id,
            horizon_min=horizon_min,
            variant_name=variant_name,
        )
    )


    sequence_all = np.asarray(
        prepared[
            "sequence"
        ],
        dtype=np.float32,
    )


    auxiliary_all = np.asarray(
        prepared[
            "auxiliary"
        ],
        dtype=np.float32,
    )


    target_scaled_all = np.asarray(
        prepared[
            "target_scaled"
        ],
        dtype=np.float32,
    )


    target_pu_all = np.asarray(
        prepared[
            "target_pu"
        ],
        dtype=np.float64,
    )


    target_MW_all = np.asarray(
        prepared[
            "target_MW"
        ],
        dtype=np.float64,
    )


    train_mask = (
        prepared[
            "train_mask"
        ]
    )


    validation_mask = (
        prepared[
            "validation_mask"
        ]
    )


    test_mask = (
        prepared[
            "test_mask"
        ]
    )


    trainval_mask = (
        train_mask
        |
        validation_mask
    )


    # --------------------------------------------------------
    # Frozen final-refit population
    # --------------------------------------------------------

    trainval_sequence = (
        sequence_all[
            trainval_mask
        ]
    )


    trainval_auxiliary = (
        auxiliary_all[
            trainval_mask
        ]
    )


    trainval_target_scaled = (
        target_scaled_all[
            trainval_mask
        ]
    )


    test_sequence = (
        sequence_all[
            test_mask
        ]
    )


    test_auxiliary = (
        auxiliary_all[
            test_mask
        ]
    )


    target_test_pu = (
        target_pu_all[
            test_mask
        ]
    )


    target_test_MW = (
        target_MW_all[
            test_mask
        ]
    )


    auxiliary_dim = int(
        auxiliary_all.shape[
            1
        ]
    )


    expected_auxiliary_dim = (
        122
        if variant_name
        ==
        "F4_reference"
        else
        127
    )


    assert (
        auxiliary_dim
        ==
        expected_auxiliary_dim
    )


    # --------------------------------------------------------
    # Frozen architecture / epoch lookup
    # --------------------------------------------------------

    architecture = (
        get_pvod_neural_architecture(
            model_name
        )
    )


    frozen_epochs = (
        get_pvod_frozen_epochs(
            model_name=model_name,
            horizon_min=horizon_min,
        )
    )


    # --------------------------------------------------------
    # Model construction resets frozen seed internally.
    # --------------------------------------------------------

    try:
        tf.keras.backend.clear_session(free_memory=True)
    except TypeError:
        tf.keras.backend.clear_session()
    gc.collect()


    model = (
        build_pvod_neural_model(
            model_name=model_name,
            auxiliary_dim=auxiliary_dim,
            architecture=architecture,
            seed=PVOD_NEURAL_SEED,
        )
    )


    trainable_parameter_count = int(
        model.count_params()
    )


    assert (
        trainable_parameter_count
        ==
        PVOD_EXPECTED_CONTROL_NEURAL_PARAMS[
            model_name
        ][
            auxiliary_dim
        ]
    )


    # --------------------------------------------------------
    # Preserve exact primary model-specific target shape
    # --------------------------------------------------------

    if model_name == "LSTM":

        final_fit_target = (
            trainval_target_scaled
            .reshape(-1)
        )

    else:

        final_fit_target = (
            trainval_target_scaled
            .reshape(
                -1,
                1,
            )
        )


    # --------------------------------------------------------
    # Frozen fit
    # --------------------------------------------------------

    fit_start = (
        time.perf_counter()
    )


    history_object = model.fit(
        x=[
            trainval_sequence,
            trainval_auxiliary,
        ],

        y=final_fit_target,

        epochs=frozen_epochs,

        batch_size=
            PVOD_NEURAL_BATCH_SIZE,

        shuffle=True,

        callbacks=None,

        validation_data=None,

        verbose=verbose,
    )


    fit_seconds = float(
        time.perf_counter()
        -
        fit_start
    )


    history_df = pd.DataFrame(
        history_object.history
    )


    assert len(
        history_df
    ) == frozen_epochs


    assert (
        "loss"
        in
        history_df.columns
    )


    assert (
        "val_loss"
        not in
        history_df.columns
    )


    # --------------------------------------------------------
    # Held-out TEST prediction
    # --------------------------------------------------------

    predict_start = (
        time.perf_counter()
    )


    prediction_scaled = np.asarray(
        model.predict(
            [
                test_sequence,
                test_auxiliary,
            ],

            batch_size=
                PVOD_NEURAL_BATCH_SIZE,

            verbose=0,
        ),
        dtype=np.float64,
    ).reshape(-1)


    predict_seconds = float(
        time.perf_counter()
        -
        predict_start
    )


    prediction_pu = (
        prediction_scaled
        *
        float(
            prepared[
                "target_std"
            ]
        )
        +
        float(
            prepared[
                "target_mean"
            ]
        )
    )


    assert prediction_pu.shape == (
        len(
            target_test_pu
        ),
    )


    assert np.isfinite(
        prediction_pu
    ).all()


    test_meta = (
        prepared[
            "meta"
        ]
        .loc[
            test_mask
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    result = {
        "prediction_pu":
            prediction_pu.copy(),

        "target_pu":
            target_test_pu.copy(),

        "target_MW":
            target_test_MW.copy(),

        "meta":
            test_meta,

        "fit_seconds":
            fit_seconds,

        "predict_seconds":
            predict_seconds,

        "frozen_epochs":
            int(
                frozen_epochs
            ),

        "auxiliary_dim":
            auxiliary_dim,

        "trainable_parameter_count":
            trainable_parameter_count,

        "zero_variance_continuous_columns":
            prepared[
                "zero_variance_continuous_columns"
            ],

        "final_train_loss":
            float(
                history_df[
                    "loss"
                ]
                .iloc[-1]
            ),
    }


    # --------------------------------------------------------
    # Release TensorFlow model state immediately.
    # --------------------------------------------------------

    del model
    del history_object
    del history_df
    del prepared

    try:
        tf.keras.backend.clear_session(free_memory=True)
    except TypeError:
        tf.keras.backend.clear_session()

    gc.collect()


    return result


# ============================================================
# 3. Execute / resume all neural control fits
# ============================================================

new_fit_count = 0
skipped_fit_count = 0
grid_start = time.perf_counter()

for model_name in PVOD_NEURAL_CONTROL_MODELS:
    print("\n" + "=" * 125)
    print(f"MODEL: {model_name}")
    print("=" * 125)

    for horizon_min in PVOD_EXTERNAL_HORIZONS:
        print(f"\n--- Horizon: {horizon_min} min ---")

        for station_id in PVOD_EXTERNAL_STATIONS:
            station_start = time.perf_counter()

            canonical_real = (
                pvod_neural_predictions.loc[
                    pvod_neural_predictions["model"].eq(model_name)
                    & pvod_neural_predictions["station"].eq(station_id)
                    & pvod_neural_predictions[
                        "horizon_min"
                    ].eq(horizon_min)
                    & pvod_neural_predictions[
                        "variant"
                    ].eq("F4_real")
                ]
                .copy()
                .reset_index(drop=True)
            )
            assert not canonical_real.empty

            station_new = 0
            station_skipped = 0

            for variant_name in PVOD_SECONDARY_CONTROL_VARIANTS:
                daily_path, metadata_path = (
                    pvod_secondary_checkpoint_paths(
                        family="neural",
                        model_name=model_name,
                        station_id=station_id,
                        horizon_min=horizon_min,
                        variant_name=variant_name,
                    )
                )

                expected_identity = {
                    "family": "neural",
                    "model": model_name,
                    "station": station_id,
                    "horizon_min": int(horizon_min),
                    "variant": variant_name,
                    "test_rows": int(len(canonical_real)),
                    "first_target_time_utc": str(
                        canonical_real[
                            "target_time_utc"
                        ].iloc[0]
                    ),
                    "last_target_time_utc": str(
                        canonical_real[
                            "target_time_utc"
                        ].iloc[-1]
                    ),
                }

                if pvod_secondary_checkpoint_is_valid(
                    daily_path,
                    metadata_path,
                    expected_identity,
                ):
                    skipped_fit_count += 1
                    station_skipped += 1
                    continue

                result = fit_predict_pvod_neural_control(
                    station_id=station_id,
                    horizon_min=horizon_min,
                    model_name=model_name,
                    variant_name=variant_name,
                    verbose=0,
                )

                prediction_pu = np.asarray(
                    result["prediction_pu"],
                    dtype=np.float64,
                )
                target_pu = np.asarray(
                    result["target_pu"],
                    dtype=np.float64,
                )
                target_MW = np.asarray(
                    result["target_MW"],
                    dtype=np.float64,
                )
                test_meta = (
                    result["meta"]
                    .copy()
                    .reset_index(drop=True)
                )

                assert (
                    len(prediction_pu)
                    == len(target_pu)
                    == len(target_MW)
                    == len(test_meta)
                    == len(canonical_real)
                )
                assert pd.DatetimeIndex(
                    test_meta["issue_time_utc"]
                ).equals(
                    pd.DatetimeIndex(
                        canonical_real["issue_time_utc"]
                    )
                )
                assert pd.DatetimeIndex(
                    test_meta["target_time_utc"]
                ).equals(
                    pd.DatetimeIndex(
                        canonical_real["target_time_utc"]
                    )
                )
                # Targets originate from the frozen materializer as
                # float32. Primary-neural checkpoints are CSV files, so
                # reloading them as float64 can differ by a final binary
                # rounding bit even when the original float32 value is
                # identical. Audit exact identity at the protocol's
                # frozen target precision instead of float64 bit identity.
                assert np.array_equal(
                    target_pu.astype(np.float32),
                    canonical_real[
                        "target_power_pu"
                    ].to_numpy(dtype=np.float32),
                )
                assert np.array_equal(
                    target_MW.astype(np.float32),
                    canonical_real[
                        "target_power_MW"
                    ].to_numpy(dtype=np.float32),
                )
                assert np.array_equal(
                    test_meta[
                        "target_solar_daylight"
                    ].to_numpy(dtype=np.int8),
                    canonical_real[
                        "target_solar_daylight"
                    ].to_numpy(dtype=np.int8),
                )

                capacity_MW = float(
                    pvod_neural_control_capacity_lookup[station_id]
                )
                prediction_MW = prediction_pu * capacity_MW

                daily_table = make_pvod_daily_error_summary(
                    model_name=model_name,
                    station_id=station_id,
                    horizon_min=horizon_min,
                    variant_name=variant_name,
                    target_time_utc=test_meta[
                        "target_time_utc"
                    ],
                    target_solar_daylight=test_meta[
                        "target_solar_daylight"
                    ],
                    target_pu=target_pu,
                    prediction_pu=prediction_pu,
                    target_MW=target_MW,
                    prediction_MW=prediction_MW,
                )

                metadata = {
                    "checkpoint_version": (
                        PVOD_SECONDARY_CHECKPOINT_VERSION
                    ),
                    **expected_identity,
                    "auxiliary_dim": int(
                        result["auxiliary_dim"]
                    ),
                    "trainable_parameters": int(
                        result["trainable_parameter_count"]
                    ),
                    "frozen_epochs": int(
                        result["frozen_epochs"]
                    ),
                    "fit_seconds": float(
                        result["fit_seconds"]
                    ),
                    "predict_seconds": float(
                        result["predict_seconds"]
                    ),
                    "final_train_loss": float(
                        result["final_train_loss"]
                    ),
                    "zero_variance_continuous_columns": list(
                        result[
                            "zero_variance_continuous_columns"
                        ]
                    ),
                }

                write_pvod_secondary_checkpoint(
                    daily_path,
                    metadata_path,
                    daily_table,
                    metadata,
                )

                new_fit_count += 1
                station_new += 1

                del result
                del daily_table
                del prediction_pu
                del prediction_MW
                gc.collect()

            station_seconds = float(
                time.perf_counter() - station_start
            )
            print(
                f"{station_id}: "
                f"new={station_new}, "
                f"skipped={station_skipped}, "
                f"test rows={len(canonical_real):,} | "
                f"{station_seconds:.2f}s"
            )


# ============================================================
# 4. Reconstruct compact canonical checkpoint objects
# ============================================================

neural_daily_frames = []
neural_runtime_rows = []

for model_name in PVOD_NEURAL_CONTROL_MODELS:
    for horizon_min in PVOD_EXTERNAL_HORIZONS:
        for station_id in PVOD_EXTERNAL_STATIONS:
            canonical_real = (
                pvod_neural_predictions.loc[
                    pvod_neural_predictions["model"].eq(model_name)
                    & pvod_neural_predictions["station"].eq(station_id)
                    & pvod_neural_predictions[
                        "horizon_min"
                    ].eq(horizon_min)
                    & pvod_neural_predictions[
                        "variant"
                    ].eq("F4_real")
                ]
                .reset_index(drop=True)
            )

            for variant_name in PVOD_SECONDARY_CONTROL_VARIANTS:
                daily_path, metadata_path = (
                    pvod_secondary_checkpoint_paths(
                        family="neural",
                        model_name=model_name,
                        station_id=station_id,
                        horizon_min=horizon_min,
                        variant_name=variant_name,
                    )
                )
                expected_identity = {
                    "family": "neural",
                    "model": model_name,
                    "station": station_id,
                    "horizon_min": int(horizon_min),
                    "variant": variant_name,
                    "test_rows": int(len(canonical_real)),
                    "first_target_time_utc": str(
                        canonical_real[
                            "target_time_utc"
                        ].iloc[0]
                    ),
                    "last_target_time_utc": str(
                        canonical_real[
                            "target_time_utc"
                        ].iloc[-1]
                    ),
                }
                assert pvod_secondary_checkpoint_is_valid(
                    daily_path,
                    metadata_path,
                    expected_identity,
                )

                neural_daily_frames.append(
                    pd.read_csv(daily_path)
                )
                with metadata_path.open(
                    "r",
                    encoding="utf-8",
                ) as handle:
                    neural_runtime_rows.append(
                        json.load(handle)
                    )

pvod_neural_secondary_daily = pd.concat(
    neural_daily_frames,
    ignore_index=True,
)
pvod_neural_secondary_runtime = pd.DataFrame(
    neural_runtime_rows
)

del neural_daily_frames

grid_seconds = float(time.perf_counter() - grid_start)

assert len(pvod_neural_secondary_runtime) == 420
assert (
    pvod_neural_secondary_runtime
    .groupby(["model", "horizon_min", "station"])[
        "variant"
    ]
    .nunique()
    .eq(7)
    .all()
)

# Geometry / epoch / parameter audit.
for model_name in PVOD_NEURAL_CONTROL_MODELS:
    for horizon_min in PVOD_EXTERNAL_HORIZONS:
        subset = pvod_neural_secondary_runtime.loc[
            pvod_neural_secondary_runtime[
                "model"
            ].eq(model_name)
            & pvod_neural_secondary_runtime[
                "horizon_min"
            ].eq(horizon_min)
        ]
        assert len(subset) == 70
        assert subset["frozen_epochs"].nunique() == 1
        assert int(subset["frozen_epochs"].iloc[0]) == int(
            get_pvod_frozen_epochs(
                model_name=model_name,
                horizon_min=horizon_min,
            )
        )
        assert set(subset["auxiliary_dim"]) == {122, 127}
        for _, row in subset.iterrows():
            assert int(row["trainable_parameters"]) == (
                PVOD_EXPECTED_CONTROL_NEURAL_PARAMS[
                    model_name
                ][int(row["auxiliary_dim"])]
            )

pvod_neural_secondary_runtime[
    "zero_variance_continuous_columns"
] = pvod_neural_secondary_runtime[
    "zero_variance_continuous_columns"
].apply(tuple)

pvod_neural_secondary_zero_variance_audit = (
    pvod_neural_secondary_runtime.loc[
        pvod_neural_secondary_runtime[
            "zero_variance_continuous_columns"
        ].apply(len) > 0,
        [
            "model",
            "station",
            "horizon_min",
            "variant",
            "zero_variance_continuous_columns",
        ],
    ]
    .reset_index(drop=True)
)
assert len(pvod_neural_secondary_zero_variance_audit) == 84
assert set(
    pvod_neural_secondary_zero_variance_audit[
        "station"
    ]
) == {"station03", "station05"}
assert (
    pvod_neural_secondary_zero_variance_audit[
        "zero_variance_continuous_columns"
    ].map(
        lambda value: tuple(value) == ("utc_year",)
    )
).all()

expected_prediction_rows = int(
    len(
        pvod_neural_predictions.loc[
            pvod_neural_predictions[
                "variant"
            ].eq("F4_real")
        ]
    )
    * 7
)
assert int(
    pvod_neural_secondary_runtime[
        "test_rows"
    ].sum()
) == expected_prediction_rows

pvod_neural_secondary_execution_summary = (
    pvod_neural_secondary_runtime
    .groupby(["model", "horizon_min"], as_index=False)
    .agg(
        fits=("variant", "size"),
        frozen_epochs=("frozen_epochs", "first"),
        min_auxiliary_dim=("auxiliary_dim", "min"),
        max_auxiliary_dim=("auxiliary_dim", "max"),
        min_parameters=("trainable_parameters", "min"),
        max_parameters=("trainable_parameters", "max"),
        total_fit_seconds=("fit_seconds", "sum"),
        median_fit_seconds=("fit_seconds", "median"),
        total_predict_seconds=("predict_seconds", "sum"),
    )
)

print("\n" + "=" * 125)
print("PVOD CHECKPOINTED NEURAL SECONDARY-CONTROL AUDIT")
print("=" * 125)
print("Logical fits:", 420)
print("New fits this run:", new_fit_count)
print("Valid checkpoints skipped:", skipped_fit_count)
print("Prediction rows represented:", expected_prediction_rows)
print("Execution seconds this run:", grid_seconds)
display(pvod_neural_secondary_execution_summary)
print(
    "Zero-variance execution cases:",
    len(pvod_neural_secondary_zero_variance_audit),
)
print("\nPVOD-09C NEURAL SECONDARY-CONTROL EXECUTION: PASSED")
print("No F4_real-vs-control comparison displayed.")
print("No secondary bootstrap performed yet.")



MODEL: LSTM

--- Horizon: 15 min ---


2026-10-01 20:50:10.088028: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-10-01 20:50:10.088967: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

station00: new=3, skipped=4, test rows=5,798 | 39.04s


2026-10-01 20:51:00.907245: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-10-01 20:51:00.907599: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

station01: new=7, skipped=0, test rows=6,681 | 102.15s


...

station08: new=7, skipped=0, test rows=6,681 | 239.25s


2026-10-01 23:16:00.894211: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-10-01 23:16:00.894517: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

station09: new=7, skipped=0, test rows=4,523 | 226.05s

PVOD CHECKPOINTED NEURAL SECONDARY-CONTROL AUDIT
Logical fits: 420
New fits this run: 416
Valid checkpoints skipped: 4
Prediction rows represented: 2262890
Execution seconds this run: 8211.666072749998


,model,horizon_min,fits,frozen_epochs,min_auxiliary_dim,max_auxiliary_dim,min_parameters,max_parameters,total_fit_seconds,median_fit_seconds,total_predict_seconds
0,LSTM,15,70,7,122,127,11009,11169,777.350688,11.605965,28.204830
1,LSTM,30,70,1,122,127,11009,11169,288.792225,4.039104,33.174214
2,LSTM,60,70,2,122,127,11009,11169,337.327287,4.778366,28.720836
3,TCN,15,70,4,122,127,22369,22529,1207.524235,18.059850,68.227146
4,TCN,30,70,1,122,127,22369,22529,758.982646,10.640999,79.961835
5,TCN,60,70,1,122,127,22369,22529,998.794355,14.314147,107.399550


Zero-variance execution cases: 84

PVOD-09C NEURAL SECONDARY-CONTROL EXECUTION: PASSED
No F4_real-vs-control comparison displayed.
No secondary bootstrap performed yet.


In [23]:
# ============================================================
# PVOD-10A
# SECONDARY-CONTROL DESCRIPTIVE COMPARISONS
#
# Effect convention:
#
#   delta = MAE(F4_real) - MAE(control)
#
# negative -> aligned F4_real has lower MAE
# positive -> control has lower MAE
#
# Point estimates only. Formal uncertainty follows in 10B.
# ============================================================

import numpy as np
import pandas as pd

required_objects = [
    "pvod_classical_predictions_corrected",
    "pvod_neural_predictions",
    "pvod_classical_secondary_daily",
    "pvod_neural_secondary_daily",
    "PVOD_SECONDARY_CONTROL_VARIANTS",
]
missing = [
    name for name in required_objects
    if name not in globals()
]
assert not missing, f"Missing required objects: {missing}"


def aggregate_primary_f4_real_daily(predictions: pd.DataFrame):
    source = predictions.loc[
        predictions["variant"].eq("F4_real")
    ].copy()
    assert not source.empty

    frames = []
    for (model_name, station_id, horizon_min), group in (
        source.groupby(
            ["model", "station", "horizon_min"],
            sort=False,
        )
    ):
        group = group.copy()
        group["target_china_date"] = (
            pd.DatetimeIndex(group["target_time_utc"])
            .tz_convert("Asia/Shanghai")
            .strftime("%Y-%m-%d")
        )
        group["absolute_error_pu"] = np.abs(
            group["prediction_power_pu"].to_numpy(
                dtype=np.float64
            )
            - group["target_power_pu"].to_numpy(
                dtype=np.float64
            )
        )

        for scope_name, mask in {
            "all": np.ones(len(group), dtype=bool),
            "daylight": group[
                "target_solar_daylight"
            ].eq(1).to_numpy(),
        }.items():
            scoped = group.loc[mask]
            daily = (
                scoped
                .groupby(
                    "target_china_date",
                    sort=True,
                    as_index=False,
                )
                .agg(
                    n_rows=("absolute_error_pu", "size"),
                    sum_abs_error_pu=(
                        "absolute_error_pu",
                        "sum",
                    ),
                )
            )
            daily.insert(0, "scope", scope_name)
            daily.insert(0, "variant", "F4_real")
            daily.insert(0, "horizon_min", int(horizon_min))
            daily.insert(0, "station", station_id)
            daily.insert(0, "model", model_name)
            frames.append(daily)

    return pd.concat(frames, ignore_index=True)


pvod_secondary_real_daily = pd.concat(
    [
        aggregate_primary_f4_real_daily(
            pvod_classical_predictions_corrected
        ),
        aggregate_primary_f4_real_daily(
            pvod_neural_predictions
        ),
    ],
    ignore_index=True,
)

pvod_secondary_control_daily = pd.concat(
    [
        pvod_classical_secondary_daily,
        pvod_neural_secondary_daily,
    ],
    ignore_index=True,
)

assert set(pvod_secondary_real_daily["model"]) == {
    "Ridge", "LightGBM", "LSTM", "TCN"
}
assert set(pvod_secondary_control_daily["model"]) == {
    "Ridge", "LightGBM", "LSTM", "TCN"
}

paired_daily_frames = []

for (
    model_name,
    station_id,
    horizon_min,
    control_variant,
    scope_name,
), control_group in pvod_secondary_control_daily.groupby(
    ["model", "station", "horizon_min", "variant", "scope"],
    sort=False,
):
    real_group = pvod_secondary_real_daily.loc[
        pvod_secondary_real_daily["model"].eq(model_name)
        & pvod_secondary_real_daily["station"].eq(station_id)
        & pvod_secondary_real_daily[
            "horizon_min"
        ].eq(horizon_min)
        & pvod_secondary_real_daily["scope"].eq(scope_name)
    ].copy()

    assert not real_group.empty

    paired = real_group[
        [
            "target_china_date",
            "n_rows",
            "sum_abs_error_pu",
        ]
    ].merge(
        control_group[
            [
                "target_china_date",
                "n_rows",
                "sum_abs_error_pu",
            ]
        ],
        on="target_china_date",
        how="outer",
        suffixes=("_real", "_control"),
        validate="one_to_one",
        indicator=True,
    )

    assert paired["_merge"].eq("both").all()
    assert np.array_equal(
        paired["n_rows_real"].to_numpy(dtype=np.int64),
        paired["n_rows_control"].to_numpy(dtype=np.int64),
    )

    paired = paired.drop(columns=["_merge", "n_rows_control"])
    paired = paired.rename(
        columns={"n_rows_real": "n_rows"}
    )
    paired.insert(0, "scope", scope_name)
    paired.insert(0, "control_variant", control_variant)
    paired.insert(0, "horizon_min", int(horizon_min))
    paired.insert(0, "station", station_id)
    paired.insert(0, "model", model_name)
    paired_daily_frames.append(paired)

pvod_secondary_paired_daily = pd.concat(
    paired_daily_frames,
    ignore_index=True,
)

del paired_daily_frames

pvod_secondary_control_station_effects = (
    pvod_secondary_paired_daily
    .groupby(
        [
            "scope",
            "model",
            "station",
            "horizon_min",
            "control_variant",
        ],
        as_index=False,
    )
    .agg(
        n_rows=("n_rows", "sum"),
        real_sum_abs_error_pu=(
            "sum_abs_error_pu_real",
            "sum",
        ),
        control_sum_abs_error_pu=(
            "sum_abs_error_pu_control",
            "sum",
        ),
        n_days=("target_china_date", "nunique"),
    )
)

pvod_secondary_control_station_effects[
    "MAE_F4_real"
] = (
    pvod_secondary_control_station_effects[
        "real_sum_abs_error_pu"
    ]
    / pvod_secondary_control_station_effects["n_rows"]
)
pvod_secondary_control_station_effects[
    "MAE_control"
] = (
    pvod_secondary_control_station_effects[
        "control_sum_abs_error_pu"
    ]
    / pvod_secondary_control_station_effects["n_rows"]
)
pvod_secondary_control_station_effects[
    "delta_MAE_real_minus_control"
] = (
    pvod_secondary_control_station_effects["MAE_F4_real"]
    - pvod_secondary_control_station_effects["MAE_control"]
)

assert len(pvod_secondary_control_station_effects) == (
    2 * 4 * 3 * 7 * 10
)

pvod_secondary_control_point_summary = (
    pvod_secondary_control_station_effects
    .groupby(
        [
            "scope",
            "model",
            "horizon_min",
            "control_variant",
        ],
        as_index=False,
    )
    .agg(
        stations=("station", "nunique"),
        point_delta_MAE=(
            "delta_MAE_real_minus_control",
            "mean",
        ),
        median_station_delta_MAE=(
            "delta_MAE_real_minus_control",
            "median",
        ),
        stations_F4_real_lower_point=(
            "delta_MAE_real_minus_control",
            lambda x: int((x < 0.0).sum()),
        ),
        stations_control_lower_point=(
            "delta_MAE_real_minus_control",
            lambda x: int((x > 0.0).sum()),
        ),
    )
)
pvod_secondary_control_point_summary[
    "comparison_type"
] = np.where(
    pvod_secondary_control_point_summary[
        "control_variant"
    ].eq("F4_reference"),
    "reference_coding",
    "alignment_null",
)

assert len(pvod_secondary_control_point_summary) == 168
assert pvod_secondary_control_point_summary[
    "stations"
].eq(10).all()

print("\n" + "=" * 125)
print("PVOD SECONDARY-CONTROL POINT ESTIMATES")
print("=" * 125)
print(
    "Effect = MAE(F4_real) - MAE(control); "
    "negative means F4_real lower."
)
display(
    pvod_secondary_control_point_summary
    .sort_values(
        ["scope", "model", "horizon_min", "control_variant"]
    )
    .reset_index(drop=True)
)
print("\nPVOD-10A SECONDARY DESCRIPTIVE COMPARISON: PASSED")
print("No confidence interval computed in this cell.")



PVOD SECONDARY-CONTROL POINT ESTIMATES
Effect = MAE(F4_real) - MAE(control); negative means F4_real lower.


,scope,model,horizon_min,control_variant,stations,point_delta_MAE,median_station_delta_MAE,stations_F4_real_lower_point,stations_control_lower_point,comparison_type
0,all,LSTM,15,F4_perm_1301,10,0.000755,0.000293,4,6,alignment_null
1,all,LSTM,15,F4_perm_1302,10,0.000604,-0.000031,5,5,alignment_null
2,all,LSTM,15,F4_perm_1303,10,0.000690,0.000224,4,6,alignment_null
3,all,LSTM,15,F4_reference,10,-0.000191,0.000018,5,5,reference_coding
4,all,LSTM,15,F4_shift_17d,10,-0.001846,-0.000780,8,2,alignment_null
...,...,...,...,...,...,...,...,...,...,...
163,daylight,TCN,60,F4_perm_1303,10,-0.001652,-0.001274,6,4,alignment_null
164,daylight,TCN,60,F4_reference,10,-0.005424,-0.005029,6,4,reference_coding
165,daylight,TCN,60,F4_shift_17d,10,-0.004525,-0.003854,8,2,alignment_null
166,daylight,TCN,60,F4_shift_37d,10,-0.004834,-0.006176,7,3,alignment_null



PVOD-10A SECONDARY DESCRIPTIVE COMPARISON: PASSED
No confidence interval computed in this cell.


In [24]:
# ============================================================
# PVOD-10B
# HIERARCHICAL PAIRED BOOTSTRAP — SECONDARY CONTROLS
#
# Frozen hierarchy:
#   1. resample 10 stations with replacement;
#   2. for each sampled station occurrence, independently
#      resample that station's China-local test days;
#   3. preserve F4_real/control pairing within each day draw;
#   4. compute station-occurrence MAE difference;
#   5. equal-weight the 10 sampled station occurrences.
#
# delta = MAE(F4_real) - MAE(control)
# ============================================================

import time
import numpy as np
import pandas as pd

PVOD_SECONDARY_BOOTSTRAP_REPLICATES = 10_000
PVOD_SECONDARY_BOOTSTRAP_SEED = 42

assert "pvod_secondary_paired_daily" in globals()
assert "pvod_secondary_control_point_summary" in globals()


def secondary_hierarchical_paired_bootstrap(
    condition_daily: pd.DataFrame,
    rng: np.random.Generator,
    n_replicates: int,
):
    station_names = sorted(condition_daily["station"].unique())
    assert len(station_names) == 10

    payload = []
    point_station_deltas = []

    for station_id in station_names:
        station = (
            condition_daily.loc[
                condition_daily["station"].eq(station_id)
            ]
            .sort_values("target_china_date")
            .reset_index(drop=True)
        )
        counts = station["n_rows"].to_numpy(dtype=np.float64)
        error_real = station[
            "sum_abs_error_pu_real"
        ].to_numpy(dtype=np.float64)
        error_control = station[
            "sum_abs_error_pu_control"
        ].to_numpy(dtype=np.float64)

        assert len(counts) > 0
        assert np.all(counts > 0)
        denominator = float(counts.sum())
        point_delta = float(
            error_real.sum() / denominator
            - error_control.sum() / denominator
        )

        payload.append({
            "counts": counts,
            "error_real": error_real,
            "error_control": error_control,
            "n_days": len(counts),
        })
        point_station_deltas.append(point_delta)

    point_station_deltas = np.asarray(
        point_station_deltas,
        dtype=np.float64,
    )
    point_estimate = float(point_station_deltas.mean())

    # Station resampling for every replicate / occurrence.
    station_draw = rng.integers(
        low=0,
        high=10,
        size=(n_replicates, 10),
    )
    bootstrap_values = np.zeros(
        n_replicates,
        dtype=np.float64,
    )

    # Vectorized over replicates while preserving independent
    # day resampling for duplicated station occurrences.
    for occurrence_position in range(10):
        selected_station = station_draw[:, occurrence_position]
        occurrence_delta = np.empty(
            n_replicates,
            dtype=np.float64,
        )

        for station_index, station_data in enumerate(payload):
            replicate_positions = np.flatnonzero(
                selected_station == station_index
            )
            if len(replicate_positions) == 0:
                continue

            n_days = int(station_data["n_days"])
            day_draw = rng.integers(
                low=0,
                high=n_days,
                size=(len(replicate_positions), n_days),
            )

            sampled_counts = station_data["counts"][day_draw]
            denominator = sampled_counts.sum(axis=1)
            assert np.all(denominator > 0.0)

            sampled_real = (
                station_data["error_real"][day_draw]
                .sum(axis=1)
                / denominator
            )
            sampled_control = (
                station_data["error_control"][day_draw]
                .sum(axis=1)
                / denominator
            )
            occurrence_delta[replicate_positions] = (
                sampled_real - sampled_control
            )

        bootstrap_values += occurrence_delta / 10.0

    return {
        "point_estimate": point_estimate,
        "point_station_deltas": point_station_deltas,
        "bootstrap_values": bootstrap_values,
    }


model_order = ["Ridge", "LightGBM", "LSTM", "TCN"]
scope_order = ["all", "daylight"]

rng = np.random.default_rng(PVOD_SECONDARY_BOOTSTRAP_SEED)
summary_rows = []
pvod_secondary_hierarchical_bootstrap_draws = {}
bootstrap_start = time.perf_counter()

for scope_name in scope_order:
    for model_name in model_order:
        for horizon_min in PVOD_EXTERNAL_HORIZONS:
            for control_variant in PVOD_SECONDARY_CONTROL_VARIANTS:
                condition = pvod_secondary_paired_daily.loc[
                    pvod_secondary_paired_daily[
                        "scope"
                    ].eq(scope_name)
                    & pvod_secondary_paired_daily[
                        "model"
                    ].eq(model_name)
                    & pvod_secondary_paired_daily[
                        "horizon_min"
                    ].eq(horizon_min)
                    & pvod_secondary_paired_daily[
                        "control_variant"
                    ].eq(control_variant)
                ].copy()
                assert condition["station"].nunique() == 10

                result = secondary_hierarchical_paired_bootstrap(
                    condition,
                    rng=rng,
                    n_replicates=PVOD_SECONDARY_BOOTSTRAP_REPLICATES,
                )
                draws = result["bootstrap_values"]
                ci_low = float(np.quantile(draws, 0.025))
                ci_high = float(np.quantile(draws, 0.975))

                if ci_high < 0.0:
                    classification = "F4_real_lower"
                elif ci_low > 0.0:
                    classification = "control_lower"
                else:
                    classification = "crosses_zero"

                summary_rows.append({
                    "scope": scope_name,
                    "model": model_name,
                    "horizon_min": int(horizon_min),
                    "control_variant": control_variant,
                    "comparison_type": (
                        "reference_coding"
                        if control_variant == "F4_reference"
                        else "alignment_null"
                    ),
                    "stations": 10,
                    "replicates": (
                        PVOD_SECONDARY_BOOTSTRAP_REPLICATES
                    ),
                    "point_delta_MAE": result["point_estimate"],
                    "bootstrap_mean": float(draws.mean()),
                    "bootstrap_median": float(np.median(draws)),
                    "ci95_low": ci_low,
                    "ci95_high": ci_high,
                    "classification": classification,
                })

                pvod_secondary_hierarchical_bootstrap_draws[
                    (
                        scope_name,
                        model_name,
                        int(horizon_min),
                        control_variant,
                    )
                ] = draws

pvod_secondary_hierarchical_bootstrap_summary = pd.DataFrame(
    summary_rows
)
bootstrap_seconds = float(time.perf_counter() - bootstrap_start)

assert len(pvod_secondary_hierarchical_bootstrap_summary) == 168
assert pvod_secondary_hierarchical_bootstrap_summary[
    "replicates"
].eq(10_000).all()

# Reconcile equal-station point estimates exactly with 10A.
point_check = (
    pvod_secondary_hierarchical_bootstrap_summary[
        [
            "scope",
            "model",
            "horizon_min",
            "control_variant",
            "point_delta_MAE",
        ]
    ]
    .merge(
        pvod_secondary_control_point_summary[
            [
                "scope",
                "model",
                "horizon_min",
                "control_variant",
                "point_delta_MAE",
            ]
        ],
        on=[
            "scope",
            "model",
            "horizon_min",
            "control_variant",
        ],
        suffixes=("_bootstrap", "_descriptive"),
        validate="one_to_one",
    )
)
max_reconciliation_error = float(
    np.max(
        np.abs(
            point_check["point_delta_MAE_bootstrap"]
            - point_check["point_delta_MAE_descriptive"]
        )
    )
)
assert max_reconciliation_error < 1e-12

pvod_secondary_bootstrap_classification_counts = (
    pvod_secondary_hierarchical_bootstrap_summary
    .groupby(
        ["scope", "model", "comparison_type", "classification"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "comparisons"})
)

print("\n" + "=" * 125)
print("PVOD SECONDARY-CONTROL HIERARCHICAL BOOTSTRAP")
print("=" * 125)
print("Comparisons:", 168)
print("Replicates per comparison:", 10_000)
print("Bootstrap seconds:", bootstrap_seconds)
print("Max point-estimate reconciliation error:", max_reconciliation_error)
display(
    pvod_secondary_hierarchical_bootstrap_summary
    .sort_values(
        ["scope", "model", "horizon_min", "control_variant"]
    )
    .reset_index(drop=True)
)
print("\nClassification counts:")
display(pvod_secondary_bootstrap_classification_counts)
print("\nPVOD-10B SECONDARY HIERARCHICAL BOOTSTRAP: PASSED")
print(
    "Uncertainty reflects station/day prediction variability only; "
    "models are not retrained inside the bootstrap."
)



PVOD SECONDARY-CONTROL HIERARCHICAL BOOTSTRAP
Comparisons: 168
Replicates per comparison: 10000
Bootstrap seconds: 7.594194292090833
Max point-estimate reconciliation error: 1.5395670849294163e-17


,scope,model,horizon_min,control_variant,comparison_type,stations,replicates,point_delta_MAE,bootstrap_mean,bootstrap_median,ci95_low,ci95_high,classification
0,all,LSTM,15,F4_perm_1301,alignment_null,10,10000,0.000755,0.000763,0.000657,-0.000581,0.002753,crosses_zero
1,all,LSTM,15,F4_perm_1302,alignment_null,10,10000,0.000604,0.000603,0.000492,-0.000840,0.002830,crosses_zero
2,all,LSTM,15,F4_perm_1303,alignment_null,10,10000,0.000690,0.000684,0.000597,-0.000666,0.002465,crosses_zero
3,all,LSTM,15,F4_reference,reference_coding,10,10000,-0.000191,-0.000190,-0.000217,-0.001232,0.000969,crosses_zero
4,all,LSTM,15,F4_shift_17d,alignment_null,10,10000,-0.001846,-0.001831,-0.001736,-0.004175,-0.000124,F4_real_lower
...,...,...,...,...,...,...,...,...,...,...,...,...,...
163,daylight,TCN,60,F4_perm_1303,alignment_null,10,10000,-0.001652,-0.001643,-0.001573,-0.005468,0.001653,crosses_zero
164,daylight,TCN,60,F4_reference,reference_coding,10,10000,-0.005424,-0.005419,-0.005184,-0.013549,0.001339,crosses_zero
165,daylight,TCN,60,F4_shift_17d,alignment_null,10,10000,-0.004525,-0.004553,-0.004485,-0.009175,-0.000366,F4_real_lower
166,daylight,TCN,60,F4_shift_37d,alignment_null,10,10000,-0.004834,-0.004909,-0.004974,-0.009857,0.000446,crosses_zero



Classification counts:


,scope,model,comparison_type,classification,comparisons
0,all,LSTM,alignment_null,F4_real_lower,4
1,all,LSTM,alignment_null,control_lower,4
2,all,LSTM,alignment_null,crosses_zero,10
3,all,LSTM,reference_coding,crosses_zero,3
4,all,LightGBM,alignment_null,control_lower,8
5,all,LightGBM,alignment_null,crosses_zero,10
6,all,LightGBM,reference_coding,crosses_zero,3
7,all,Ridge,alignment_null,control_lower,10
8,all,Ridge,alignment_null,crosses_zero,8
9,all,Ridge,reference_coding,crosses_zero,3



PVOD-10B SECONDARY HIERARCHICAL BOOTSTRAP: PASSED
Uncertainty reflects station/day prediction variability only; models are not retrained inside the bootstrap.


In [25]:
# ============================================================
# PVOD-10C
# FINAL EXTERNAL-VALIDATION MECHANISM SYNTHESIS + EXPORT
#
# This cell summarizes evidence only.
# It does not fit or retrain any model.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

required_objects = [
    "pvod_primary_external_validation_summary",
    "pvod_secondary_hierarchical_bootstrap_summary",
    "pvod_secondary_control_point_summary",
    "pvod_classical_secondary_runtime",
    "pvod_neural_secondary_runtime",
]
missing = [
    name for name in required_objects
    if name not in globals()
]
assert not missing, f"Missing required objects: {missing}"

primary = (
    pvod_primary_external_validation_summary[
        [
            "scope",
            "model",
            "horizon_min",
            "point_delta_MAE",
            "ci95_low",
            "ci95_high",
            "classification",
        ]
    ]
    .copy()
    .rename(columns={
        "point_delta_MAE": "primary_F4_minus_F3",
        "ci95_low": "primary_ci95_low",
        "ci95_high": "primary_ci95_high",
        "classification": "primary_classification",
    })
)
# Remove categorical dtype before merges/CSV export.
primary["model"] = primary["model"].astype(str)
primary["scope"] = primary["scope"].astype(str)

reference = (
    pvod_secondary_hierarchical_bootstrap_summary.loc[
        pvod_secondary_hierarchical_bootstrap_summary[
            "control_variant"
        ].eq("F4_reference"),
        [
            "scope",
            "model",
            "horizon_min",
            "point_delta_MAE",
            "ci95_low",
            "ci95_high",
            "classification",
        ],
    ]
    .copy()
    .rename(columns={
        "point_delta_MAE": "real_minus_reference",
        "ci95_low": "reference_ci95_low",
        "ci95_high": "reference_ci95_high",
        "classification": "reference_classification",
    })
)

null_controls = pvod_secondary_hierarchical_bootstrap_summary.loc[
    ~pvod_secondary_hierarchical_bootstrap_summary[
        "control_variant"
    ].eq("F4_reference")
].copy()
assert null_controls["control_variant"].nunique() == 6

null_counts = (
    null_controls
    .groupby(
        ["scope", "model", "horizon_min", "classification"],
        as_index=False,
    )
    .size()
    .pivot_table(
        index=["scope", "model", "horizon_min"],
        columns="classification",
        values="size",
        fill_value=0,
        aggfunc="sum",
    )
    .reset_index()
)
for column in ["F4_real_lower", "control_lower", "crosses_zero"]:
    if column not in null_counts.columns:
        null_counts[column] = 0
null_counts = null_counts.rename(columns={
    "F4_real_lower": "nulls_F4_real_lower",
    "control_lower": "nulls_control_lower",
    "crosses_zero": "nulls_crosses_zero",
})

pvod_external_mechanism_synthesis = (
    primary
    .merge(
        reference,
        on=["scope", "model", "horizon_min"],
        validate="one_to_one",
    )
    .merge(
        null_counts,
        on=["scope", "model", "horizon_min"],
        validate="one_to_one",
    )
    .sort_values(["scope", "model", "horizon_min"])
    .reset_index(drop=True)
)

assert len(pvod_external_mechanism_synthesis) == 24
assert (
    pvod_external_mechanism_synthesis[
        [
            "nulls_F4_real_lower",
            "nulls_control_lower",
            "nulls_crosses_zero",
        ]
    ].sum(axis=1).eq(6).all()
)

RESULT_DIR = PROJECT_ROOT / "results" / "pvod_external_validation"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

export_tables = {
    "primary_external_validation_summary.csv": (
        pvod_primary_external_validation_summary.copy()
    ),
    "secondary_control_point_estimates.csv": (
        pvod_secondary_control_point_summary.copy()
    ),
    "secondary_control_bootstrap_summary.csv": (
        pvod_secondary_hierarchical_bootstrap_summary.copy()
    ),
    "mechanism_synthesis.csv": (
        pvod_external_mechanism_synthesis.copy()
    ),
    "classical_secondary_runtime.csv": (
        pvod_classical_secondary_runtime.copy()
    ),
    "neural_secondary_runtime.csv": (
        pvod_neural_secondary_runtime.copy()
    ),
}

for filename, table in export_tables.items():
    table.to_csv(RESULT_DIR / filename, index=False)

print("\n" + "=" * 125)
print("PVOD FINAL EXTERNAL-VALIDATION MECHANISM SYNTHESIS")
print("=" * 125)
display(pvod_external_mechanism_synthesis)
print("\nExport directory:")
print(RESULT_DIR)
for filename in export_tables:
    print(" ", filename)

print("\nPVOD-10C FINAL SYNTHESIS + EXPORT: PASSED")
print(
    "Interpretation boundary: these experiments test forecasting "
    "representation effects and coding/alignment sensitivity."
)
print(
    "They do not establish a causal or physical effect of Panchang "
    "categories on photovoltaic generation."
)



PVOD FINAL EXTERNAL-VALIDATION MECHANISM SYNTHESIS


,scope,model,horizon_min,primary_F4_minus_F3,primary_ci95_low,primary_ci95_high,primary_classification,real_minus_reference,reference_ci95_low,reference_ci95_high,reference_classification,nulls_F4_real_lower,nulls_control_lower,nulls_crosses_zero
0,all,LSTM,15,-0.001002,-0.002300,0.000784,crosses_zero,-1.906351e-04,-1.231933e-03,0.000969,crosses_zero,2,0,4
1,all,LSTM,30,-0.004835,-0.009518,-0.001278,F4_lower,2.587284e-03,-1.360894e-03,0.009220,crosses_zero,1,1,4
2,all,LSTM,60,-0.002388,-0.005218,0.000305,crosses_zero,2.292957e-03,-3.545181e-04,0.007199,crosses_zero,1,3,2
3,all,LightGBM,15,0.000124,-0.000071,0.000332,crosses_zero,1.095086e-04,-6.556157e-05,0.000302,crosses_zero,0,4,2
4,all,LightGBM,30,0.000456,-0.000229,0.001611,crosses_zero,4.605476e-04,-8.663326e-05,0.001378,crosses_zero,0,2,4
5,all,LightGBM,60,-0.000290,-0.001040,0.000211,crosses_zero,7.075427e-06,-2.150353e-04,0.000245,crosses_zero,0,2,4
6,all,Ridge,15,0.002909,0.001121,0.005528,F3_lower,9.326690e-07,-7.383956e-09,0.000003,crosses_zero,0,3,3
7,all,Ridge,30,0.005677,0.002067,0.011352,F3_lower,1.570500e-06,-9.219860e-09,0.000005,crosses_zero,0,3,3
8,all,Ridge,60,0.011207,0.003701,0.023092,F3_lower,2.090226e-06,-3.805961e-08,0.000007,crosses_zero,0,4,2
9,all,TCN,15,-0.005827,-0.014556,0.003307,crosses_zero,-4.687995e-04,-8.272432e-03,0.007043,crosses_zero,0,1,5



Export directory:
/Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/pvod_external_validation
  primary_external_validation_summary.csv
  secondary_control_point_estimates.csv
  secondary_control_bootstrap_summary.csv
  mechanism_synthesis.csv
  classical_secondary_runtime.csv
  neural_secondary_runtime.csv

PVOD-10C FINAL SYNTHESIS + EXPORT: PASSED
Interpretation boundary: these experiments test forecasting representation effects and coding/alignment sensitivity.
They do not establish a causal or physical effect of Panchang categories on photovoltaic generation.
